In [ ]:

# ============================================================
# 1. Установка библиотек
# ============================================================

!pip -q install -U datasets transformers accelerate sentence-transformers faiss-cpu langgraph openai scikit-learn pandas pyarrow


In [ ]:
# ============================================================
# ЯЧЕЙКА 2. НАСТРОЙКА ОКРУЖЕНИЯ И ПУТЕЙ
# Устойчивая версия: работает и с Google Drive, и без него
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Seed
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 2. Определяем устройство
# ------------------------------------------------------------

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU недоступен — проект будет работать на CPU")


# ------------------------------------------------------------
# 3. Пытаемся подключить Google Drive
# ------------------------------------------------------------

DRIVE_AVAILABLE = False

try:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    DRIVE_AVAILABLE = True

    print("Google Drive подключён")

except Exception as e:

    print()
    print("Google Drive сейчас подключить не удалось.")
    print("Ошибка:", str(e))
    print()
    print("Продолжаем работу локально в /content.")
    print("Это не мешает запускать проект.")


# ------------------------------------------------------------
# 4. Выбираем корневую папку проекта
# ------------------------------------------------------------

if DRIVE_AVAILABLE:

    PROJECT_DIR = Path(
        "/content/drive/MyDrive/"
        "ml_customer_classification_corrected"
    )

else:

    PROJECT_DIR = Path(
        "/content/"
        "ml_customer_classification_corrected"
    )


# ------------------------------------------------------------
# 5. Создаём структуру проекта
# ------------------------------------------------------------

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

SEMANTIC_DIR = (
    PROJECT_DIR
    / "semantic_search"
)

SEMANTIC_RESULTS_DIR = (
    SEMANTIC_DIR
    / "results"
)

RAG_DIR = (
    PROJECT_DIR
    / "rag"
)

RAG_DATA_DIR = (
    RAG_DIR
    / "data"
)

RAG_RESULTS_DIR = (
    RAG_DIR
    / "results"
)

DOC_RAG_DIR = (
    PROJECT_DIR
    / "document_rag"
)

DOC_RAG_DATA_DIR = (
    DOC_RAG_DIR
    / "data"
)

DOC_RAG_RESULTS_DIR = (
    DOC_RAG_DIR
    / "results"
)

LANGGRAPH_DIR = (
    PROJECT_DIR
    / "langgraph"
)

LANGGRAPH_RESULTS_DIR = (
    LANGGRAPH_DIR
    / "results"
)

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
)


all_dirs = [
    PROJECT_DIR,
    DATA_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    VALIDATION_DIR,
    SEMANTIC_DIR,
    SEMANTIC_RESULTS_DIR,
    RAG_DIR,
    RAG_DATA_DIR,
    RAG_RESULTS_DIR,
    DOC_RAG_DIR,
    DOC_RAG_DATA_DIR,
    DOC_RAG_RESULTS_DIR,
    LANGGRAPH_DIR,
    LANGGRAPH_RESULTS_DIR,
    CHECKPOINT_DIR
]


for folder in all_dirs:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 6. Итог
# ------------------------------------------------------------

print()
print("=" * 70)
print("ENVIRONMENT READY")
print("=" * 70)

print("PROJECT_DIR:")
print(PROJECT_DIR)

print()
print(
    "Storage:",
    "Google Drive"
    if DRIVE_AVAILABLE
    else "Local Colab storage"
)

print(
    "Device:",
    DEVICE
)

print("=" * 70)

Исходный `BANKING77 train` делится на `train_part` и `val_part`.
Официальный `test` не используется ни для выбора гиперпараметров, ни для threshold, ни для prompt tuning.


In [ ]:
# ============================================================
# ЯЧЕЙКА 3. ЗАГРУЗКА BANKING77 И КОРРЕКТНЫЙ TRAIN/VAL/TEST
# Совместимо с новой версией Hugging Face datasets
# ============================================================

from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd
import json

# ------------------------------------------------------------
# 1. Загружаем parquet-версию Banking77
# ------------------------------------------------------------

dataset = load_dataset(
    "mteb/banking77"
)

print("=" * 70)
print("BANKING77 LOADED")
print("=" * 70)

print(dataset)

print("\nTrain columns:")
print(dataset["train"].column_names)

print("\nTest columns:")
print(dataset["test"].column_names)


# ------------------------------------------------------------
# 2. Переводим в pandas
# ------------------------------------------------------------

train_df = (
    dataset["train"]
    .to_pandas()[
        [
            "text",
            "label",
            "label_text"
        ]
    ]
    .copy()
)

test_df = (
    dataset["test"]
    .to_pandas()[
        [
            "text",
            "label",
            "label_text"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Формируем mapping
# ------------------------------------------------------------

label_mapping_df = (
    train_df[
        [
            "label",
            "label_text"
        ]
    ]
    .drop_duplicates()
    .sort_values("label")
    .reset_index(drop=True)
)


id_to_label = dict(
    zip(
        label_mapping_df["label"].astype(int),
        label_mapping_df["label_text"].astype(str)
    )
)

label_to_id = {
    name: label_id
    for label_id, name
    in id_to_label.items()
}


# ------------------------------------------------------------
# 4. Проверки исходного датасета
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET INFO")
print("=" * 70)

print(
    "Train:",
    train_df.shape
)

print(
    "Test:",
    test_df.shape
)

print(
    "Number of classes:",
    len(id_to_label)
)

print(
    "Missing train labels:",
    train_df["label_text"].isna().sum()
)

print(
    "Missing test labels:",
    test_df["label_text"].isna().sum()
)


assert len(id_to_label) == 77
assert train_df["label_text"].isna().sum() == 0
assert test_df["label_text"].isna().sum() == 0


# ------------------------------------------------------------
# 5. Создаём TRAIN / VALIDATION
#
# ВАЖНО:
# test_df после этого не используем
# до самого финального теста.
# ------------------------------------------------------------

train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"]
)

train_part = (
    train_part
    .reset_index(drop=True)
)

val_part = (
    val_part
    .reset_index(drop=True)
)

test_df = (
    test_df
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Проверяем размеры
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED SPLIT")
print("=" * 70)

print(
    "train_part:",
    train_part.shape
)

print(
    "val_part:",
    val_part.shape
)

print(
    "test_df:",
    test_df.shape,
    "<-- LOCKED"
)

print()

print(
    "Train classes:",
    train_part["label"].nunique()
)

print(
    "Validation classes:",
    val_part["label"].nunique()
)

print(
    "Test classes:",
    test_df["label"].nunique()
)


# ------------------------------------------------------------
# 7. Проверяем пересечения текстов
# ------------------------------------------------------------

train_texts = set(
    train_part["text"]
    .astype(str)
)

val_texts = set(
    val_part["text"]
    .astype(str)
)

test_texts = set(
    test_df["text"]
    .astype(str)
)


train_val_overlap = len(
    train_texts
    & val_texts
)

train_test_overlap = len(
    train_texts
    & test_texts
)

val_test_overlap = len(
    val_texts
    & test_texts
)


print("\nTEXT OVERLAP")

print(
    "train / val:",
    train_val_overlap
)

print(
    "train / test:",
    train_test_overlap
)

print(
    "val / test:",
    val_test_overlap
)


assert train_val_overlap == 0
assert train_test_overlap == 0
assert val_test_overlap == 0


# ------------------------------------------------------------
# 8. Сохраняем split
# ------------------------------------------------------------

train_part.to_parquet(
    DATA_DIR
    / "train_part.parquet",
    index=False
)

val_part.to_parquet(
    DATA_DIR
    / "val_part.parquet",
    index=False
)

test_df.to_parquet(
    DATA_DIR
    / "test.parquet",
    index=False
)


with open(
    DATA_DIR
    / "label_mapping.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "id_to_label": {
                str(k): v
                for k, v
                in id_to_label.items()
            },

            "label_to_id":
                label_to_id
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 9. Примеры
# ------------------------------------------------------------

print("\nПримеры train_part:")

display(
    train_part[
        [
            "text",
            "label",
            "label_text"
        ]
    ]
    .head()
)


# ------------------------------------------------------------
# 10. Финальный контроль
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA PREPARATION COMPLETE")
print("=" * 70)

print(
    "ВАЖНО: test_df теперь не используем "
    "до самого последнего final evaluation."
)

In [ ]:

# ============================================================
# 4. Fixed train / validation split
# ============================================================

train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"]
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

assert len(set(train_part["text"]) & set(val_part["text"])) == 0
assert len(set(train_part["text"]) & set(test_df["text"])) == 0
assert len(set(val_part["text"]) & set(test_df["text"])) == 0

train_part.to_parquet(DATA_DIR / "train_part.parquet", index=False)
val_part.to_parquet(DATA_DIR / "val_part.parquet", index=False)
test_df.to_parquet(DATA_DIR / "test_locked.parquet", index=False)

protocol = {
    "seed": SEED,
    "train_part_size": len(train_part),
    "val_part_size": len(val_part),
    "test_size": len(test_df),
    "policy": {
        "train_part": "fit models and build retrieval corpus",
        "val_part": "threshold selection, diagnostic set, prompt tuning, architecture selection",
        "test": "single final evaluation after all decisions are frozen"
    }
}

with open(VALIDATION_DIR / "evaluation_protocol.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2)

print(protocol)


In [ ]:

# ============================================================
# 5. Минимальный EDA только на train_part
# ============================================================

print("Missing values:")
print(train_part.isna().sum())

print("\nDuplicates:", train_part["text"].duplicated().sum())

class_counts = train_part["label_text"].value_counts()
print("\nClass count summary:")
display(class_counts.describe().to_frame("count"))

lengths = train_part["text"].str.split().str.len()
print("\nText length summary:")
display(lengths.describe(percentiles=[0.5, 0.95, 0.99]).to_frame("tokens_approx"))


## 2. Baseline и DistilBERT

На этапе разработки все сравнения выполняются на `val_part`.
Финальный `test` будет запущен только в последнем разделе.


In [ ]:

# ============================================================
# 6. TF-IDF + Logistic Regression — validation baseline
# ============================================================

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=100000
)

X_train = tfidf.fit_transform(train_part["text"])
X_val = tfidf.transform(val_part["text"])

lr_model = LogisticRegression(
    max_iter=3000,
    n_jobs=-1,
    random_state=SEED
)

lr_model.fit(X_train, train_part["label"])
val_lr_pred = lr_model.predict(X_val)

val_lr_metrics = {
    "accuracy": accuracy_score(val_part["label"], val_lr_pred),
    "macro_f1": f1_score(val_part["label"], val_lr_pred, average="macro")
}

print("TF-IDF validation metrics:")
print(val_lr_metrics)


In [ ]:
# ============================================================
# ЯЧЕЙКА 7. DEBUG-ОБУЧЕНИЕ DISTILBERT НА CPU
# 1 эпоха только для проверки всего пайплайна
# train_part -> обучение
# val_part   -> validation
# test_df    -> НЕ ИСПОЛЬЗУЕМ
#
# ============================================================

import json
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ------------------------------------------------------------
# 1. Настройки
# ------------------------------------------------------------

BASE_MODEL = "distilbert-base-uncased"

FINAL_MODEL_DIR = (
    MODELS_DIR
    / "distilbert_debug_1epoch"
)

TRAINING_OUTPUT_DIR = (
    MODELS_DIR
    / "distilbert_debug_training"
)

MAX_LENGTH = 64

NUM_LABELS = len(id_to_label)

DEBUG_MODE = True

NUM_EPOCHS = 1


print("=" * 70)
print("DISTILBERT DEBUG MODE")
print("=" * 70)

print(
    "Number of labels:",
    NUM_LABELS
)

print(
    "Epochs:",
    NUM_EPOCHS
)

print(
    "Device:",
    DEVICE
)

print(
    "Mode:",
    "DEBUG / PIPELINE CHECK"
)

assert NUM_LABELS == 77


# ------------------------------------------------------------
# 2. Tokenizer
# ------------------------------------------------------------

distilbert_tokenizer = (
    AutoTokenizer.from_pretrained(
        BASE_MODEL
    )
)


# ------------------------------------------------------------
# 3. Hugging Face datasets
# ------------------------------------------------------------

train_hf = Dataset.from_pandas(
    train_part[
        [
            "text",
            "label"
        ]
    ],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_part[
        [
            "text",
            "label"
        ]
    ],
    preserve_index=False
)


# ------------------------------------------------------------
# 4. Tokenization
# ------------------------------------------------------------

def tokenize_batch(batch):

    return distilbert_tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


train_tokenized = train_hf.map(
    tokenize_batch,
    batched=True
)

val_tokenized = val_hf.map(
    tokenize_batch,
    batched=True
)


# ------------------------------------------------------------
# 5. Rename label -> labels
# ------------------------------------------------------------

if "label" in train_tokenized.column_names:

    train_tokenized = (
        train_tokenized
        .rename_column(
            "label",
            "labels"
        )
    )


if "label" in val_tokenized.column_names:

    val_tokenized = (
        val_tokenized
        .rename_column(
            "label",
            "labels"
        )
    )


# ------------------------------------------------------------
# 6. PyTorch format
# ------------------------------------------------------------

columns_to_keep = [
    "input_ids",
    "attention_mask",
    "labels"
]


train_tokenized.set_format(
    type="torch",
    columns=columns_to_keep
)

val_tokenized.set_format(
    type="torch",
    columns=columns_to_keep
)


# ------------------------------------------------------------
# 7. Metrics
# ------------------------------------------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy":
            float(accuracy),

        "macro_f1":
            float(macro_f1),

        "weighted_f1":
            float(weighted_f1)
    }


# ------------------------------------------------------------
# 8. Проверяем, есть ли уже debug-модель
# ------------------------------------------------------------

MODEL_ALREADY_EXISTS = (
    FINAL_MODEL_DIR.exists()
    and
    (FINAL_MODEL_DIR / "config.json").exists()
)


if MODEL_ALREADY_EXISTS:

    print()
    print(
        "Saved debug model found."
    )

    print(
        FINAL_MODEL_DIR
    )

    distilbert_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            FINAL_MODEL_DIR
        )
    )

    try:

        distilbert_tokenizer = (
            AutoTokenizer.from_pretrained(
                FINAL_MODEL_DIR
            )
        )

    except Exception:

        pass


else:

    print()
    print(
        "Saved debug model not found."
    )

    print(
        "Training 1 epoch from train_part only."
    )

    distilbert_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            BASE_MODEL,
            num_labels=NUM_LABELS,
            id2label=id_to_label,
            label2id=label_to_id
        )
    )


# ------------------------------------------------------------
# 9. Device
# ------------------------------------------------------------

distilbert_model.to(
    DEVICE
)

print()

print(
    "Model device:",
    next(
        distilbert_model.parameters()
    ).device
)


# ------------------------------------------------------------
# 10. Training arguments
#
# Для CPU уменьшаем batch size
# ------------------------------------------------------------

training_args = TrainingArguments(

    output_dir=str(
        TRAINING_OUTPUT_DIR
    ),

    num_train_epochs=NUM_EPOCHS,

    learning_rate=5e-5,

    per_device_train_batch_size=(
        8
        if DEVICE == "cpu"
        else 32
    ),

    per_device_eval_batch_size=(
        16
        if DEVICE == "cpu"
        else 64
    ),

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    save_total_limit=1,

    fp16=(
        DEVICE == "cuda"
    ),

    report_to="none",

    seed=SEED
)


# ------------------------------------------------------------
# 11. Trainer
# ------------------------------------------------------------

trainer = Trainer(

    model=distilbert_model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=distilbert_tokenizer,

    compute_metrics=compute_metrics
)


# ------------------------------------------------------------
# 12. Обучение
# ------------------------------------------------------------

if not MODEL_ALREADY_EXISTS:

    print()
    print("=" * 70)
    print("DISTILBERT DEBUG TRAINING — 1 EPOCH")
    print("=" * 70)

    trainer.train()


    # --------------------------------------------------------
    # 13. Сохраняем debug-модель
    # --------------------------------------------------------

    FINAL_MODEL_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    trainer.save_model(
        FINAL_MODEL_DIR
    )

    distilbert_tokenizer.save_pretrained(
        FINAL_MODEL_DIR
    )

    print()
    print(
        "Debug model saved:"
    )

    print(
        FINAL_MODEL_DIR
    )


else:

    print()
    print(
        "Training skipped: debug model already exists."
    )


# ------------------------------------------------------------
# 14. Validation
#
# Только val_part
# ------------------------------------------------------------

val_metrics = trainer.evaluate(
    eval_dataset=val_tokenized
)


print()
print("=" * 70)
print("DISTILBERT DEBUG VALIDATION METRICS")
print("=" * 70)


for key, value in val_metrics.items():

    if isinstance(
        value,
        (
            float,
            np.floating
        )
    ):

        print(
            f"{key}: {float(value):.4f}"
        )

    else:

        print(
            key,
            value
        )


# ------------------------------------------------------------
# 15. Сохраняем debug validation metrics
# ------------------------------------------------------------

DEBUG_VAL_METRICS_PATH = (
    VALIDATION_DIR
    / "distilbert_debug_1epoch_validation_metrics.json"
)


clean_val_metrics = {}

for key, value in val_metrics.items():

    if isinstance(
        value,
        (
            float,
            int,
            np.floating,
            np.integer
        )
    ):

        clean_val_metrics[key] = float(
            value
        )

    else:

        clean_val_metrics[key] = str(
            value
        )


with open(
    DEBUG_VAL_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        clean_val_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 16. Сохраняем статус debug-режима
# ------------------------------------------------------------

DEBUG_STATUS_PATH = (
    VALIDATION_DIR
    / "debug_training_status.json"
)


debug_status = {

    "debug_mode":
        True,

    "epochs":
        NUM_EPOCHS,

    "device":
        DEVICE,

    "train_size":
        int(len(train_part)),

    "validation_size":
        int(len(val_part)),

    "test_used":
        False,

    "warning":
        (
            "This model is trained for 1 epoch only "
            "to validate the pipeline. "
            "Final evaluation must use a model "
            "retrained for 6 epochs on GPU."
        )
}


with open(
    DEBUG_STATUS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        debug_status,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 17. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 7 COMPLETE")
print("=" * 70)

print(
    "MODE: DEBUG"
)

print(
    "Epochs:",
    NUM_EPOCHS
)

print(
    "train_part использован для обучения."
)

print(
    "val_part использован для validation."
)

print(
    "test_df НЕ использовался."
)

print()
print(
    "ВАЖНО: перед финальной сдачей "
    "переобучить модель на 6 эпохах на GPU."
)

print("=" * 70)

In [ ]:

# ============================================================
# 8. Batch inference helper
# ============================================================

def distilbert_predict(df, batch_size=128):
    texts = df["text"].astype(str).tolist()
    probs_all = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=64,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            logits = distilbert_model(**encoded).logits
            probs = torch.softmax(logits, dim=-1)

        probs_all.append(probs.cpu().numpy())

    probs = np.vstack(probs_all)
    pred = probs.argmax(axis=1)
    confidence = probs.max(axis=1)

    return {
        "probs": probs,
        "pred": pred,
        "confidence": confidence
    }

val_db = distilbert_predict(val_part)

print("DistilBERT validation accuracy:",
      accuracy_score(val_part["label"], val_db["pred"]))
print("DistilBERT validation Macro F1:",
      f1_score(val_part["label"], val_db["pred"], average="macro"))


## 3. Threshold выбирается только на validation


In [ ]:

# ============================================================
# 9. Threshold sweep on validation only
# ============================================================

TARGET_AUTO_ACCURACY = 0.95

rows = []

for threshold in np.arange(0.50, 0.991, 0.01):
    accepted = val_db["confidence"] >= threshold
    n_auto = int(accepted.sum())

    if n_auto == 0:
        continue

    auto_acc = float(
        (val_db["pred"][accepted] == val_part["label"].to_numpy()[accepted]).mean()
    )

    rows.append({
        "threshold": round(float(threshold), 2),
        "auto_closed": n_auto,
        "coverage": n_auto / len(val_part),
        "automatic_accuracy": auto_acc,
        "review_rate": 1 - n_auto / len(val_part)
    })

threshold_df = pd.DataFrame(rows)

eligible = threshold_df[
    threshold_df["automatic_accuracy"] >= TARGET_AUTO_ACCURACY
].copy()

if len(eligible):
    best_row = eligible.sort_values(
        ["coverage", "threshold"],
        ascending=[False, True]
    ).iloc[0]
else:
    best_row = threshold_df.sort_values(
        ["automatic_accuracy", "coverage"],
        ascending=False
    ).iloc[0]

SELECTED_THRESHOLD = float(best_row["threshold"])

display(threshold_df)
print("\nSelected threshold:", SELECTED_THRESHOLD)
print(best_row.to_dict())

with open(VALIDATION_DIR / "selected_threshold.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "target_auto_accuracy": TARGET_AUTO_ACCURACY,
            "selected_threshold": SELECTED_THRESHOLD,
            "validation_metrics": best_row.to_dict()
        },
        f,
        ensure_ascii=False,
        indent=2
    )


## 4. Semantic retrieval

FAISS строится **только по `train_part`**.
Все retrieval-метрики и диагностическая сотня до финального этапа считаются только на `val_part`.


In [ ]:

# ============================================================
# 10. Embeddings + FAISS on train_part only
# ============================================================

INTENT_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
intent_embedding_model = SentenceTransformer(INTENT_EMBED_MODEL, device="cpu")

train_embeddings = intent_embedding_model.encode(
    train_part["text"].astype(str).tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

intent_index = faiss.IndexFlatIP(train_embeddings.shape[1])
intent_index.add(train_embeddings)

print("FAISS vectors:", intent_index.ntotal)


In [ ]:

# ============================================================
# 11. Retrieval helper + validation Hit@K
# ============================================================

def semantic_retrieve_queries(texts, top_k=5):
    query_emb = intent_embedding_model.encode(
        list(texts),
        batch_size=128,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idx = intent_index.search(query_emb, top_k)
    return scores, idx

val_scores, val_idx = semantic_retrieve_queries(val_part["text"], top_k=5)

val_retrieved_labels = train_part["label"].to_numpy()[val_idx]
y_val = val_part["label"].to_numpy()

def hit_at_k(retrieved_labels, y_true, k):
    return float(
        np.mean([
            y_true[i] in retrieved_labels[i, :k]
            for i in range(len(y_true))
        ])
    )

retrieval_val_metrics = {
    "hit@1": hit_at_k(val_retrieved_labels, y_val, 1),
    "hit@3": hit_at_k(val_retrieved_labels, y_val, 3),
    "hit@5": hit_at_k(val_retrieved_labels, y_val, 5)
}

print(retrieval_val_metrics)


In [ ]:

# ============================================================
# 12. Diagnostic 100 — validation only
# 50 hard + 50 easy
# ============================================================

val_diag = val_part.copy()
val_diag["distilbert_pred"] = val_db["pred"]
val_diag["distilbert_confidence"] = val_db["confidence"]
val_diag["retrieval_top1"] = val_retrieved_labels[:, 0]
val_diag["correct_in_top5"] = [
    y_val[i] in val_retrieved_labels[i, :5]
    for i in range(len(y_val))
]

hard_pool = val_diag[
    (val_diag["distilbert_pred"] != val_diag["label"])
    & (val_diag["correct_in_top5"])
].copy()

easy_pool = val_diag[
    val_diag["distilbert_pred"] == val_diag["label"]
].copy()

n_hard = min(50, len(hard_pool))
n_easy = min(50, len(easy_pool))

diagnostic_df = pd.concat([
    hard_pool.sample(n=n_hard, random_state=SEED).assign(case_type="hard"),
    easy_pool.sample(n=n_easy, random_state=SEED).assign(case_type="easy")
], ignore_index=True)

diagnostic_df.to_csv(
    VALIDATION_DIR / "diagnostic_100_validation.csv",
    index=False
)

print(diagnostic_df["case_type"].value_counts())
print("Diagnostic size:", len(diagnostic_df))


## 5. API для prompt tuning и генерации QA



In [ ]:
# ============================================================
# ЯЧЕЙКА 13. ВВОД GROQ API KEY
# ============================================================

import os
from getpass import getpass

GROQ_API_KEY = getpass(
    "Введите ваш Groq API key: "
)

if not GROQ_API_KEY:
    raise ValueError(
        "Groq API key не введён."
    )

os.environ[
    "GROQ_API_KEY"
] = GROQ_API_KEY

print()
print("=" * 70)
print("GROQ API KEY READY")
print("=" * 70)
print(
    "Ключ успешно установлен."
)
print(
    "Сам ключ в notebook не отображается."
)

In [ ]:

# ============================================================
# 14. Intent RAG prompt — tune/evaluate only on diagnostic validation set
# ============================================================

def build_intent_candidates(query, top_k=5):
    scores, idx = semantic_retrieve_queries([query], top_k=top_k)

    candidates = []
    seen = set()

    for rank, train_i in enumerate(idx[0], start=1):
        label_id = int(train_part.iloc[int(train_i)]["label"])
        intent = id_to_label[label_id]

        if intent in seen:
            continue

        seen.add(intent)
        candidates.append({
            "intent": intent,
            "example": train_part.iloc[int(train_i)]["text"],
            "score": float(scores[0][rank - 1])
        })

    return candidates

def intent_rag_predict(query, top_k=5):
    candidates = build_intent_candidates(query, top_k=top_k)

    allowed = [x["intent"] for x in candidates]
    context = "\n".join(
        f"- {x['intent']}: {x['example']}"
        for x in candidates
    )

    prompt = f"""
Classify the banking customer request into exactly one intent.

You may choose ONLY from this list:
{allowed}

Retrieved examples:
{context}

Customer request:
{query}

Return only the intent name. No explanation.
""".strip()

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt
    )

    pred = response.output_text.strip()

    if pred not in allowed:
        return None

    return pred

# ВАЖНО:
# этот diagnostic set можно использовать для изменения prompt,
# потому что он построен только из validation.


In [ ]:

# ============================================================
# 15. Optional: diagnostic evaluation of intent RAG on validation
# ============================================================

RUN_DIAGNOSTIC_API = False

if RUN_DIAGNOSTIC_API:
    diagnostic_results = []

    for _, row in diagnostic_df.iterrows():
        pred_intent = intent_rag_predict(row["text"], top_k=5)

        diagnostic_results.append({
            "text": row["text"],
            "case_type": row["case_type"],
            "true_intent": row["label_text"],
            "rag_intent": pred_intent,
            "correct": pred_intent == row["label_text"]
        })

    diagnostic_results_df = pd.DataFrame(diagnostic_results)

    display(
        diagnostic_results_df.groupby("case_type")["correct"].mean()
    )

    diagnostic_results_df.to_csv(
        VALIDATION_DIR / "intent_rag_diagnostic_validation.csv",
        index=False
    )
else:
    print(
        "RUN_DIAGNOSTIC_API=False. "
        "Включай только если хочешь проверять/править prompt на validation."
    )


## 6. LangGraph vs simple reject baseline

Обе системы используют **один и тот же threshold, выбранный на validation**.

### Baseline
`DistilBERT → confidence >= threshold ? answer : human review`

### LangGraph
- high confidence → direct answer;
- low confidence → semantic retrieval;
- если DistilBERT и retrieval согласны → auto-close;
- иначе → human review.

Это позволяет честно измерить, даёт ли граф дополнительный coverage при сопоставимом качестве.


In [ ]:

# ============================================================
# 16. Baseline metrics helper
# ============================================================

def reject_baseline_metrics(df, db_output, threshold):
    y_true = df["label"].to_numpy()
    pred = db_output["pred"]
    conf = db_output["confidence"]

    auto = conf >= threshold
    n_auto = int(auto.sum())

    return {
        "system": "DistilBERT + reject threshold",
        "threshold": threshold,
        "n": len(df),
        "auto_closed": n_auto,
        "coverage": n_auto / len(df),
        "human_review": int((~auto).sum()),
        "review_rate": float((~auto).mean()),
        "automatic_accuracy": (
            float((pred[auto] == y_true[auto]).mean())
            if n_auto else np.nan
        ),
        "automatic_errors": int((pred[auto] != y_true[auto]).sum())
    }

baseline_val_metrics = reject_baseline_metrics(
    val_part,
    val_db,
    SELECTED_THRESHOLD
)

baseline_val_metrics


In [ ]:

# ============================================================
# 17. LangGraph state + nodes
# ============================================================

class IntentGraphState(TypedDict, total=False):
    query: str
    distilbert_pred: int
    distilbert_confidence: float
    retrieval_pred: int
    final_pred: Optional[int]
    needs_human_review: bool
    route: str

def predict_one_distilbert(query):
    temp = pd.DataFrame({"text": [query]})
    out = distilbert_predict(temp)
    return int(out["pred"][0]), float(out["confidence"][0])

def retrieve_one_intent(query):
    scores, idx = semantic_retrieve_queries([query], top_k=1)
    train_i = int(idx[0][0])
    return int(train_part.iloc[train_i]["label"])

def classify_node(state: IntentGraphState):
    pred, conf = predict_one_distilbert(state["query"])
    return {
        **state,
        "distilbert_pred": pred,
        "distilbert_confidence": conf
    }

def route_after_classify(state: IntentGraphState):
    if state["distilbert_confidence"] >= SELECTED_THRESHOLD:
        return "direct"
    return "retrieve"

def direct_node(state: IntentGraphState):
    return {
        **state,
        "final_pred": state["distilbert_pred"],
        "needs_human_review": False,
        "route": "direct"
    }

def retrieve_node(state: IntentGraphState):
    rpred = retrieve_one_intent(state["query"])
    return {
        **state,
        "retrieval_pred": rpred
    }

def consensus_node(state: IntentGraphState):
    if state["retrieval_pred"] == state["distilbert_pred"]:
        return {
            **state,
            "final_pred": state["distilbert_pred"],
            "needs_human_review": False,
            "route": "retrieval_consensus"
        }

    return {
        **state,
        "final_pred": None,
        "needs_human_review": True,
        "route": "human_review"
    }

intent_graph = StateGraph(IntentGraphState)
intent_graph.add_node("classify", classify_node)
intent_graph.add_node("direct", direct_node)
intent_graph.add_node("retrieve", retrieve_node)
intent_graph.add_node("consensus", consensus_node)

intent_graph.add_edge(START, "classify")
intent_graph.add_conditional_edges(
    "classify",
    route_after_classify,
    {
        "direct": "direct",
        "retrieve": "retrieve"
    }
)
intent_graph.add_edge("direct", END)
intent_graph.add_edge("retrieve", "consensus")
intent_graph.add_edge("consensus", END)

intent_graph_app = intent_graph.compile()

print("LangGraph compiled.")


In [ ]:

# ============================================================
# 18. Vectorized equivalent evaluation of LangGraph on validation
# ============================================================

def langgraph_selective_metrics(df, db_output, retrieved_top1, threshold):
    y_true = df["label"].to_numpy()
    db_pred = db_output["pred"]
    conf = db_output["confidence"]

    high = conf >= threshold
    low = ~high

    consensus = low & (db_pred == retrieved_top1)
    auto = high | consensus

    final_pred = db_pred.copy()

    return {
        "system": "LangGraph selective consensus",
        "threshold": threshold,
        "n": len(df),
        "auto_closed": int(auto.sum()),
        "coverage": float(auto.mean()),
        "human_review": int((~auto).sum()),
        "review_rate": float((~auto).mean()),
        "automatic_accuracy": (
            float((final_pred[auto] == y_true[auto]).mean())
            if auto.any() else np.nan
        ),
        "automatic_errors": int((final_pred[auto] != y_true[auto]).sum()),
        "extra_closed_vs_baseline": int(consensus.sum())
    }

langgraph_val_metrics = langgraph_selective_metrics(
    val_part,
    val_db,
    val_retrieved_labels[:, 0],
    SELECTED_THRESHOLD
)

comparison_val_df = pd.DataFrame([
    baseline_val_metrics,
    langgraph_val_metrics
])

display(comparison_val_df)

comparison_val_df.to_csv(
    VALIDATION_DIR / "langgraph_vs_reject_validation.csv",
    index=False
)


## 7. Document RAG

Сначала фиксируем RAG pipeline, затем через API создаётся новый QA benchmark по случайным chunks.


In [ ]:
# ============================================================
# БЛОК 142. Синтетическая база внутренних документов
# для Document RAG
# ============================================================

from pathlib import Path
import pandas as pd
import json

DOC_RAG_DIR = PROJECT_DIR / "document_rag"
DOC_RAG_DATA_DIR = DOC_RAG_DIR / "data"
DOC_RAG_RESULTS_DIR = DOC_RAG_DIR / "results"

DOC_RAG_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOC_RAG_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Набор внутренних документов
# ------------------------------------------------------------

internal_documents = [

    {
        "doc_id": "DOC_KYC_001",
        "title": "Регламент проверки личности клиента",
        "category": "KYC",
        "text": """
Проверка личности клиента проводится в рамках процедур KYC.

Если клиент спрашивает, зачем необходимо подтверждение личности,
сотруднику следует объяснить, что проверка используется для соблюдения
требований безопасности, предотвращения мошенничества и выполнения
регуляторных требований.

Для подтверждения личности клиенту могут потребоваться документ,
удостоверяющий личность, фотография документа и дополнительная
проверка личности через приложение.

Если клиент не может пройти проверку личности, необходимо проверить
качество фотографии, корректность данных документа и срок его действия.
При повторной неудаче обращение передаётся в службу поддержки.
""".strip()
    },

    {
        "doc_id": "DOC_CARD_001",
        "title": "Инструкция по выпуску и доставке банковских карт",
        "category": "Cards",
        "text": """
После заказа банковской карты клиент получает информацию
о предполагаемом сроке доставки.

Срок доставки зависит от страны и способа доставки.
До истечения стандартного срока клиенту следует сообщать
ориентировочную дату доставки.

Если стандартный срок уже прошёл, а карта не была получена,
необходимо проверить статус доставки и возможность отслеживания.

Запрос о том, сколько обычно занимает доставка новой карты,
относится к оценке срока доставки.
Запрос о карте, которая должна была уже прийти, но не пришла,
рассматривается как проблема с прибытием карты.
""".strip()
    },

    {
        "doc_id": "DOC_TRANSFER_001",
        "title": "Регламент обработки банковских переводов",
        "category": "Transfers",
        "text": """
Банковский перевод может находиться в статусе обработки
до его окончательного завершения.

Если операция имеет статус pending, сотруднику следует сообщить,
что перевод ещё обрабатывается и необходимо дождаться завершения.

Если перевод отмечен как завершённый, но получатель не получил деньги,
необходимо проверить реквизиты и срок зачисления в банке получателя.

Если перевод был отклонён, сотрудник проверяет ограничения,
доступность получателя и возможные технические причины отказа.

Комиссия за перевод должна проверяться отдельно от комиссии
за пополнение счёта банковским переводом.
""".strip()
    },

    {
        "doc_id": "DOC_TOPUP_001",
        "title": "Инструкция по пополнению счёта",
        "category": "Top Up",
        "text": """
Пополнение банковской карты может иметь несколько состояний.

Failed означает, что попытка пополнения не была успешно завершена.

Pending означает, что операция ещё находится в обработке.

Reverted означает, что операция сначала была инициирована,
но затем средства были возвращены на исходный способ оплаты.

Если клиент сообщает, что пополнение не прошло,
следует уточнить текущий статус операции.

Если средства вернулись обратно, обращение относится к reverted.
Если операция всё ещё ожидает завершения, она относится к pending.
Если операция завершилась ошибкой, она относится к failed.
""".strip()
    },

    {
        "doc_id": "DOC_VCARD_001",
        "title": "Регламент виртуальных и одноразовых карт",
        "category": "Virtual Cards",
        "text": """
Виртуальная карта доступна клиенту в приложении после её выпуска.

Если виртуальная карта уже создана, но не работает при оплате,
необходимо проверить её статус, доступный баланс и ограничения.

Одноразовая виртуальная карта используется для повышения
безопасности интернет-платежей и может иметь ограничения
по количеству операций.

Запрос о создании виртуальной карты отличается от запроса
о неисправности уже существующей виртуальной карты.
""".strip()
    },

    {
        "doc_id": "DOC_PAYMENT_001",
        "title": "Инструкция по проблемам с карточными платежами",
        "category": "Card Payments",
        "text": """
Если клиент не узнаёт карточный платёж, необходимо проверить
название торговой точки, дату и сумму операции.

Если клиент подтверждает, что не совершал операцию,
следует рассматривать её как нераспознанный карточный платёж
и при необходимости рекомендовать блокировку карты.

Отклонённый карточный платёж отличается от банковского перевода:
он относится к покупке с использованием карты.

Комиссия за карточный платёж и неверный обменный курс
рассматриваются как отдельные категории проблем.
""".strip()
    },

    {
        "doc_id": "DOC_CASH_001",
        "title": "Регламент операций с банкоматами",
        "category": "Cash Withdrawal",
        "text": """
При операциях снятия наличных возможны несколько типов проблем.

Если банкомат не выдал наличные и операция была отклонена,
необходимо проверить статус операции и ограничения карты.

Если операция снятия наличных отображается в истории,
но клиент её не совершал, это нераспознанное снятие наличных.

Если клиент получил неправильную сумму наличных,
необходимо зафиксировать сумму операции и фактически полученную сумму.

Ошибочный обменный курс при снятии наличных рассматривается
отдельно от обменного курса карточной покупки.
""".strip()
    },

    {
        "doc_id": "DOC_SECURITY_001",
        "title": "Инструкция по безопасности карт и PIN-кода",
        "category": "Security",
        "text": """
При подозрении на компрометацию карты сотруднику необходимо
рекомендовать клиенту временно заблокировать карту.

Если карта потеряна или украдена, она должна быть заблокирована,
а клиенту может потребоваться выпуск новой карты.

Если PIN-код заблокирован после нескольких неправильных попыток,
необходимо использовать процедуру разблокировки PIN.

Изменение PIN-кода и разблокировка заблокированного PIN-кода
являются разными процедурами.
""".strip()
    }
]

# ------------------------------------------------------------
# 2. DataFrame
# ------------------------------------------------------------

documents_df = pd.DataFrame(
    internal_documents
)

print(
    "Количество документов:",
    len(documents_df)
)

display(
    documents_df[
        [
            "doc_id",
            "title",
            "category"
        ]
    ]
)

# ------------------------------------------------------------
# 3. Сохраняем
# ------------------------------------------------------------

DOCUMENTS_CSV_PATH = (
    DOC_RAG_DATA_DIR / "internal_documents.csv"
)

DOCUMENTS_JSON_PATH = (
    DOC_RAG_DATA_DIR / "internal_documents.json"
)

documents_df.to_csv(
    DOCUMENTS_CSV_PATH,
    index=False
)

with open(
    DOCUMENTS_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        internal_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nСохранено:")
print(DOCUMENTS_CSV_PATH)
print(DOCUMENTS_JSON_PATH)


In [ ]:

# ============================================================
# 20. Chunking
# ============================================================

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", text.strip()))
    return [p.strip() for p in parts if p.strip()]

chunks = []

for _, row in documents_df.iterrows():
    sentences = split_sentences(row["text"])

    for i in range(len(sentences)):
        chunk_sentences = sentences[i:i+2]

        if not chunk_sentences:
            continue

        chunks.append({
            "chunk_id": f"{row['doc_id']}_CHUNK_{i:02d}",
            "doc_id": row["doc_id"],
            "title": row["title"],
            "category": row["category"],
            "text": " ".join(chunk_sentences)
        })

document_chunks_df = pd.DataFrame(chunks).drop_duplicates(
    subset=["doc_id", "text"]
).reset_index(drop=True)

document_chunks_df.to_csv(
    DOC_RAG_DATA_DIR / "document_chunks.csv",
    index=False
)

print("Chunks:", len(document_chunks_df))
display(document_chunks_df.head())


In [ ]:

# ============================================================
# 21. Document embeddings + FAISS
# ============================================================

DOC_EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
doc_embedding_model = SentenceTransformer(DOC_EMBED_MODEL, device="cpu")

doc_embeddings = doc_embedding_model.encode(
    document_chunks_df["text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

doc_index = faiss.IndexFlatIP(doc_embeddings.shape[1])
doc_index.add(doc_embeddings)

print("Document vectors:", doc_index.ntotal)


In [ ]:

# ============================================================
# 22. Query type classifier — fixed before QA generation
# ============================================================

QUERY_TYPES = {
    "kyc": "проверка и подтверждение личности клиента, KYC, идентификация",
    "cards": "выпуск, получение и доставка физической банковской карты",
    "transfers": "банковские переводы, получатель, статусы переводов",
    "top_up": "пополнение счёта или карты, pending failed reverted",
    "virtual_cards": "виртуальные и одноразовые банковские карты",
    "card_payments": "оплата банковской картой и проблемы карточных платежей",
    "cash": "банкомат, снятие наличных, проблемы при выдаче наличных",
    "security": "безопасность карты, потеря, кража, PIN-код"
}

QUERY_TYPE_TO_DOC_ID = {
    "kyc": "DOC_KYC_001",
    "cards": "DOC_CARD_001",
    "transfers": "DOC_TRANSFER_001",
    "top_up": "DOC_TOPUP_001",
    "virtual_cards": "DOC_VCARD_001",
    "card_payments": "DOC_PAYMENT_001",
    "cash": "DOC_CASH_001",
    "security": "DOC_SECURITY_001"
}

query_type_names = list(QUERY_TYPES)
query_type_embeddings = doc_embedding_model.encode(
    [QUERY_TYPES[x] for x in query_type_names],
    convert_to_numpy=True,
    normalize_embeddings=True
)

def classify_query_type_semantic(query):
    q = doc_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = query_type_embeddings @ q
    order = np.argsort(scores)[::-1]

    top1 = int(order[0])
    top2 = int(order[1])

    return {
        "query_type": query_type_names[top1],
        "score": float(scores[top1]),
        "margin": float(scores[top1] - scores[top2])
    }

print("Query type classifier frozen.")


In [ ]:

# ============================================================
# 23. Final Document RAG retrieval + API answer
# Pipeline is frozen BEFORE QA benchmark generation.
# ============================================================

def retrieve_document_chunks(query, top_k=3):
    qtype = classify_query_type_semantic(query)["query_type"]
    expected_doc = QUERY_TYPE_TO_DOC_ID[qtype]

    subset = document_chunks_df[
        document_chunks_df["doc_id"] == expected_doc
    ].copy()

    subset_embeddings = doc_embedding_model.encode(
        subset["text"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    q = doc_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = subset_embeddings @ q
    order = np.argsort(scores)[::-1][:top_k]

    out = subset.iloc[order].copy()
    out["similarity"] = scores[order]

    return qtype, out.reset_index(drop=True)

def answer_document_question(query, top_k=3):
    qtype, retrieved = retrieve_document_chunks(query, top_k=top_k)

    context = "\n\n".join(
        f"[{r.doc_id} | {r.chunk_id}] {r.text}"
        for r in retrieved.itertuples()
    )

    prompt = f"""
Ты внутренний ассистент сотрудников банка.

Ответь на вопрос ТОЛЬКО по предоставленному контексту.
Не добавляй внешние знания и не придумывай процедуры.
Если в контексте недостаточно информации, прямо скажи об этом.
Ответ должен быть кратким и конкретным.

Контекст:
{context}

Вопрос:
{query}
""".strip()

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt
    )

    return {
        "query_type": qtype,
        "answer": response.output_text.strip(),
        "sources": retrieved["doc_id"].tolist(),
        "chunk_ids": retrieved["chunk_id"].tolist(),
        "retrieved": retrieved
    }

print("Document RAG pipeline frozen.")


## 8. Новый QA benchmark через API

Вопросы создаются **после фиксации pipeline**.
Каждый вопрос генерируется из случайно выбранного chunk и должен быть ответим только по нему.
Старые 27 вопросов здесь не используются.


In [ ]:
# ============================================================
# ЯЧЕЙКА 24. ГЕНЕРАЦИЯ QA ЧЕРЕЗ GROQ API
# Автоматически выбираем доступную модель
# ============================================================

!pip -q install groq

import os
import json
import pandas as pd

from groq import Groq


# ------------------------------------------------------------
# 1. Проверяем API key
# ------------------------------------------------------------

if "GROQ_API_KEY" not in os.environ:
    raise RuntimeError(
        "GROQ_API_KEY не найден. "
        "Сначала запусти ячейку 13."
    )


# ------------------------------------------------------------
# 2. Создаём Groq client
# ------------------------------------------------------------

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)


# ------------------------------------------------------------
# 3. Получаем список моделей,
# доступных конкретно этому API key
# ------------------------------------------------------------

models_response = client.models.list()

available_models = sorted(
    [
        model.id
        for model in models_response.data
    ]
)


print("=" * 70)
print("AVAILABLE GROQ MODELS")
print("=" * 70)

for model_id in available_models:
    print(model_id)


# ------------------------------------------------------------
# 4. Выбираем модель автоматически
#
# Приоритет:
# 1. GPT-OSS 120B
# 2. Qwen 27B
# 3. GPT-OSS 20B
# 4. любая другая текстовая модель
# ------------------------------------------------------------

preferred_models = [
    "openai/gpt-oss-120b",
    "qwen/qwen3.8-27b",
    "openai/gpt-oss-20b"
]


GROQ_MODEL = None


for candidate in preferred_models:

    if candidate in available_models:

        GROQ_MODEL = candidate

        break


if GROQ_MODEL is None:

    excluded_fragments = [
        "whisper",
        "guard",
        "tts",
        "speech"
    ]

    fallback_models = [
        model_id
        for model_id in available_models
        if not any(
            fragment in model_id.lower()
            for fragment in excluded_fragments
        )
    ]

    if fallback_models:

        GROQ_MODEL = fallback_models[0]


if GROQ_MODEL is None:

    raise RuntimeError(
        "Не удалось найти подходящую текстовую "
        "модель в Groq."
    )


print()
print("=" * 70)
print("SELECTED MODEL")
print("=" * 70)

print(
    GROQ_MODEL
)


# ------------------------------------------------------------
# 5. Проверяем модель одним коротким запросом
#
# Важно: если здесь ошибка, не запускаем 30 запросов.
# ------------------------------------------------------------

print()
print(
    "Проверяем доступ к выбранной модели..."
)


test_completion = (
    client
    .chat
    .completions
    .create(

        model=GROQ_MODEL,

        messages=[
            {
                "role": "user",
                "content": (
                    'Ответь только словом "OK".'
                )
            }
        ],

        temperature=0
    )
)


test_answer = (
    test_completion
    .choices[0]
    .message
    .content
    .strip()
)


print(
    "API test response:",
    test_answer
)


# ------------------------------------------------------------
# 6. Количество QA-примеров
# ------------------------------------------------------------

N_QA = min(
    30,
    len(document_chunks_df)
)


# ------------------------------------------------------------
# 7. Случайно выбираем chunks
#
# Этот набор НЕ используется для настройки RAG.
# ------------------------------------------------------------

qa_source_df = (
    document_chunks_df
    .sample(
        n=N_QA,
        random_state=SEED
    )
    .reset_index(drop=True)
)


print()
print("=" * 70)
print("GROQ API-GENERATED QA DATASET")
print("=" * 70)

print(
    "Selected chunks:",
    len(qa_source_df)
)

print(
    "Model:",
    GROQ_MODEL
)


# ------------------------------------------------------------
# 8. Генерация одного QA-примера
# ------------------------------------------------------------

def generate_qa_from_chunk(row):

    chunk_text = str(
        row["text"]
    )

    doc_id = str(
        row["doc_id"]
    )

    chunk_id = str(
        row["chunk_id"]
    )


    prompt = f"""
Ты создаёшь независимый тестовый вопрос
для оценки RAG-системы.

Ниже дан фрагмент внутреннего документа.

DOCUMENT ID:
{doc_id}

CHUNK ID:
{chunk_id}

TEXT:
{chunk_text}

Создай ОДИН естественный вопрос сотрудника
на русском языке, на который можно ответить
по этому фрагменту.

Затем создай краткий эталонный ответ.

Требования:

- вопрос не должен дословно повторять фрагмент;
- вопрос должен быть реалистичным;
- ответ должен полностью подтверждаться фрагментом;
- не используй внешние знания;
- не добавляй новых правил;
- не добавляй сведений, которых нет во фрагменте;
- ответ должен быть коротким и конкретным.

Верни только JSON без Markdown:

{{
  "question": "...",
  "reference_answer": "..."
}}
""".strip()


    completion = (
        client
        .chat
        .completions
        .create(

            model=GROQ_MODEL,

            messages=[
                {
                    "role":
                        "system",

                    "content":
                        (
                            "Ты создаёшь независимые QA-примеры "
                            "для оценки RAG. "
                            "Возвращай только валидный JSON."
                        )
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }
            ],

            temperature=0.2
        )
    )


    raw_text = (
        completion
        .choices[0]
        .message
        .content
        .strip()
    )


    cleaned = (
        raw_text
        .replace(
            "```json",
            ""
        )
        .replace(
            "```JSON",
            ""
        )
        .replace(
            "```",
            ""
        )
        .strip()
    )


    parsed = json.loads(
        cleaned
    )


    question = (
        str(
            parsed["question"]
        )
        .strip()
    )

    reference_answer = (
        str(
            parsed["reference_answer"]
        )
        .strip()
    )


    if not question:

        raise ValueError(
            "Empty generated question."
        )


    if not reference_answer:

        raise ValueError(
            "Empty reference answer."
        )


    return {

        "question":
            question,

        "reference_answer":
            reference_answer,

        "expected_doc_id":
            doc_id,

        "expected_chunk_id":
            chunk_id,

        "source_text":
            chunk_text
    }


# ------------------------------------------------------------
# 9. Генерируем QA
#
# При системной ошибке модели прерываем цикл,
# а не отправляем 30 заведомо неуспешных запросов.
# ------------------------------------------------------------

qa_rows = []

generation_errors = []


for idx, row in qa_source_df.iterrows():

    print(
        f"Generating QA "
        f"{idx + 1}/{len(qa_source_df)}..."
    )

    try:

        qa_item = generate_qa_from_chunk(
            row
        )

        qa_rows.append(
            qa_item
        )


    except Exception as e:

        error_text = str(e)

        generation_errors.append({

            "index":
                int(idx),

            "doc_id":
                str(
                    row["doc_id"]
                ),

            "chunk_id":
                str(
                    row["chunk_id"]
                ),

            "error":
                error_text
        })


        print(
            "ERROR:",
            error_text
        )


        # Если проблема системная,
        # нет смысла повторять запрос 30 раз.

        if (
            "model_not_found" in error_text
            or "401" in error_text
            or "403" in error_text
        ):

            print()
            print(
                "System/API error detected."
            )

            print(
                "Generation stopped early."
            )

            break


# ------------------------------------------------------------
# 10. Создаём DataFrame
# ------------------------------------------------------------

generated_qa_df = pd.DataFrame(
    qa_rows
)


print()
print("=" * 70)
print("QA GENERATION COMPLETE")
print("=" * 70)

print(
    "Requested:",
    len(qa_source_df)
)

print(
    "Generated:",
    len(generated_qa_df)
)

print(
    "Errors:",
    len(generation_errors)
)


# ------------------------------------------------------------
# 11. Проверяем результат
# ------------------------------------------------------------

if len(generated_qa_df) == 0:

    raise RuntimeError(
        "Не удалось сгенерировать "
        "ни одного QA-примера."
    )


# ------------------------------------------------------------
# 12. Проверяем дубликаты вопросов
# ------------------------------------------------------------

duplicate_questions = (
    generated_qa_df[
        "question"
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicate questions:",
    int(
        duplicate_questions
    )
)


# ------------------------------------------------------------
# 13. Показываем примеры
# ------------------------------------------------------------

display(
    generated_qa_df[
        [
            "question",
            "reference_answer",
            "expected_doc_id",
            "expected_chunk_id"
        ]
    ]
    .head(10)
)


# ------------------------------------------------------------
# 14. Сохраняем CSV
# ------------------------------------------------------------

GENERATED_QA_PATH = (
    DOC_RAG_DATA_DIR
    / "api_generated_qa.csv"
)


generated_qa_df.to_csv(
    GENERATED_QA_PATH,
    index=False
)


# ------------------------------------------------------------
# 15. Сохраняем JSON
# ------------------------------------------------------------

GENERATED_QA_JSON_PATH = (
    DOC_RAG_DATA_DIR
    / "api_generated_qa.json"
)


with open(
    GENERATED_QA_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        generated_qa_df.to_dict(
            orient="records"
        ),
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 16. Сохраняем ошибки
# ------------------------------------------------------------

QA_ERRORS_PATH = (
    DOC_RAG_RESULTS_DIR
    / "qa_generation_errors.json"
)


with open(
    QA_ERRORS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        generation_errors,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 17. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 24 COMPLETE")
print("=" * 70)

print(
    "Model:",
    GROQ_MODEL
)

print(
    "Generated QA:",
    len(generated_qa_df)
)

print(
    "Saved:"
)

print(
    GENERATED_QA_PATH
)

print()

print(
    "Важно: эти вопросы не использовались "
    "для настройки RAG."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 25. ОЦЕНКА DOCUMENT RAG НА API-GENERATED QA
#
# Используем:
# generated_qa_df -> вопросы, сгенерированные по случайным chunks
# answer_document_question(...) -> текущий RAG
#
# ============================================================

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Проверяем наличие QA dataset
# ------------------------------------------------------------

if "generated_qa_df" not in globals():

    GENERATED_QA_PATH = (
        DOC_RAG_DATA_DIR
        / "api_generated_qa.csv"
    )

    if not GENERATED_QA_PATH.exists():

        raise RuntimeError(
            "generated_qa_df не найден и "
            "api_generated_qa.csv отсутствует. "
            "Сначала запусти ячейку 24."
        )

    generated_qa_df = pd.read_csv(
        GENERATED_QA_PATH
    )


print("=" * 70)
print("DOCUMENT RAG EVALUATION")
print("=" * 70)

print(
    "QA examples:",
    len(generated_qa_df)
)


# ------------------------------------------------------------
# 2. Проверяем функцию RAG
# ------------------------------------------------------------

if "answer_document_question" not in globals():

    raise RuntimeError(
        "Функция answer_document_question "
        "не определена. "
        "Сначала запусти предыдущие ячейки Document RAG."
    )


# ------------------------------------------------------------
# 3. Вспомогательная функция для нормализации sources
# ------------------------------------------------------------

def normalize_source_ids(value):

    if value is None:
        return []

    if isinstance(value, str):
        return [value]

    if isinstance(value, dict):

        if "doc_id" in value:
            return [
                str(
                    value["doc_id"]
                )
            ]

        return []

    if isinstance(value, list):

        result = []

        for item in value:

            if isinstance(item, str):

                result.append(
                    item
                )

            elif isinstance(item, dict):

                if "doc_id" in item:

                    result.append(
                        str(
                            item["doc_id"]
                        )
                    )

        return result

    return []


# ------------------------------------------------------------
# 4. Вспомогательная функция для chunk IDs
# ------------------------------------------------------------

def extract_chunk_ids(result):

    chunk_ids = []

    candidates = []

    if isinstance(result, dict):

        for key in [
            "retrieved_documents",
            "documents",
            "retrieved_chunks",
            "chunks"
        ]:

            if key in result:
                candidates = result[key]
                break

    if not isinstance(
        candidates,
        list
    ):

        return []


    for item in candidates:

        if isinstance(
            item,
            dict
        ):

            if "chunk_id" in item:

                chunk_ids.append(
                    str(
                        item[
                            "chunk_id"
                        ]
                    )
                )

    return chunk_ids


# ------------------------------------------------------------
# 5. Запускаем RAG на всех API-generated вопросах
# ------------------------------------------------------------

rag_rows = []


for i, row in enumerate(
    generated_qa_df.itertuples(
        index=False
    ),
    start=1
):

    print(
        f"Evaluating "
        f"{i}/{len(generated_qa_df)}..."
    )


    try:

        result = answer_document_question(
            row.question,
            top_k=3
        )


        # ----------------------------------------------------
        # Финальный ответ
        # ----------------------------------------------------

        if isinstance(
            result,
            dict
        ):

            answer = (
                result.get(
                    "answer"
                )
                or result.get(
                    "final_answer"
                )
                or result.get(
                    "response"
                )
                or ""
            )

            sources = (
                result.get(
                    "sources"
                )
                or []
            )

        else:

            answer = str(
                result
            )

            sources = []


        source_doc_ids = normalize_source_ids(
            sources
        )


        retrieved_chunk_ids = extract_chunk_ids(
            result
        )


        # ----------------------------------------------------
        # Если answer_document_question не возвращает chunks,
        # дополнительно делаем retrieval напрямую
        # ----------------------------------------------------

        if not retrieved_chunk_ids:

            if "search_internal_documents" in globals():

                retrieved_df = (
                    search_internal_documents(
                        row.question,
                        top_k=3
                    )
                )


                if (
                    retrieved_df is not None
                    and len(
                        retrieved_df
                    ) > 0
                ):

                    retrieved_chunk_ids = (
                        retrieved_df[
                            "chunk_id"
                        ]
                        .astype(str)
                        .tolist()
                    )


                    if not source_doc_ids:

                        source_doc_ids = (
                            retrieved_df[
                                "doc_id"
                            ]
                            .astype(str)
                            .tolist()
                        )


        # ----------------------------------------------------
        # Метрики retrieval
        # ----------------------------------------------------

        expected_doc_id = str(
            row.expected_doc_id
        )

        expected_chunk_id = str(
            row.expected_chunk_id
        )


        source_correct = (
            expected_doc_id
            in source_doc_ids
        )


        chunk_hit_at_1 = (
            len(
                retrieved_chunk_ids
            ) >= 1
            and
            retrieved_chunk_ids[0]
            == expected_chunk_id
        )


        chunk_hit_at_3 = (
            expected_chunk_id
            in retrieved_chunk_ids[
                :3
            ]
        )


        rag_rows.append({

            "question":
                row.question,

            "reference_answer":
                row.reference_answer,

            "expected_doc_id":
                expected_doc_id,

            "expected_chunk_id":
                expected_chunk_id,

            "rag_answer":
                str(
                    answer
                ),

            "retrieved_doc_ids":
                source_doc_ids,

            "retrieved_chunk_ids":
                retrieved_chunk_ids,

            "source_correct":
                bool(
                    source_correct
                ),

            "chunk_hit_at_1":
                bool(
                    chunk_hit_at_1
                ),

            "chunk_hit_at_3":
                bool(
                    chunk_hit_at_3
                ),

            "error":
                None
        })


    except Exception as e:

        rag_rows.append({

            "question":
                row.question,

            "reference_answer":
                row.reference_answer,

            "expected_doc_id":
                str(
                    row.expected_doc_id
                ),

            "expected_chunk_id":
                str(
                    row.expected_chunk_id
                ),

            "rag_answer":
                "",

            "retrieved_doc_ids":
                [],

            "retrieved_chunk_ids":
                [],

            "source_correct":
                False,

            "chunk_hit_at_1":
                False,

            "chunk_hit_at_3":
                False,

            "error":
                str(e)
        })


        print(
            "ERROR:",
            str(e)
        )


# ------------------------------------------------------------
# 6. DataFrame
# ------------------------------------------------------------

rag_eval_df = pd.DataFrame(
    rag_rows
)


# ------------------------------------------------------------
# 7. Основные retrieval/source metrics
# ------------------------------------------------------------

n_examples = len(
    rag_eval_df
)


source_accuracy = float(
    rag_eval_df[
        "source_correct"
    ]
    .mean()
)


chunk_hit_at_1 = float(
    rag_eval_df[
        "chunk_hit_at_1"
    ]
    .mean()
)


chunk_hit_at_3 = float(
    rag_eval_df[
        "chunk_hit_at_3"
    ]
    .mean()
)


n_errors = int(
    rag_eval_df[
        "error"
    ]
    .notna()
    .sum()
)


# ------------------------------------------------------------
# 8. Итоговые метрики
# ------------------------------------------------------------

rag_metrics = {

    "n_examples":
        int(
            n_examples
        ),

    "source_accuracy":
        source_accuracy,

    "chunk_hit_at_1":
        chunk_hit_at_1,

    "chunk_hit_at_3":
        chunk_hit_at_3,

    "runtime_errors":
        n_errors
}


print()
print("=" * 70)
print("DOCUMENT RAG RETRIEVAL METRICS")
print("=" * 70)


for key, value in rag_metrics.items():

    if isinstance(
        value,
        float
    ):

        print(
            f"{key}: "
            f"{value:.4f}"
        )

    else:

        print(
            f"{key}: "
            f"{value}"
        )


# ------------------------------------------------------------
# 9. Показываем несколько результатов
# ------------------------------------------------------------

display_columns = [

    "question",
    "expected_doc_id",
    "expected_chunk_id",
    "retrieved_doc_ids",
    "retrieved_chunk_ids",
    "source_correct",
    "chunk_hit_at_1",
    "chunk_hit_at_3",
    "rag_answer"
]


display(
    rag_eval_df[
        display_columns
    ]
    .head(10)
)


# ------------------------------------------------------------
# 10. Сохраняем результаты
# ------------------------------------------------------------

RAG_EVAL_PATH = (
    DOC_RAG_RESULTS_DIR
    / "api_generated_qa_rag_evaluation.csv"
)


RAG_METRICS_PATH = (
    DOC_RAG_RESULTS_DIR
    / "api_generated_qa_rag_metrics.json"
)


rag_eval_df.to_csv(
    RAG_EVAL_PATH,
    index=False
)


with open(
    RAG_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 11. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 25 COMPLETE")
print("=" * 70)

print(
    "Evaluation saved:"
)

print(
    RAG_EVAL_PATH
)

print()

print(
    "Metrics saved:"
)

print(
    RAG_METRICS_PATH
)

print()

print(
    "ВАЖНО: результаты этого QA benchmark "
    "не используем для дополнительной "
    "настройки retrieval/RAG."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 26. LLM-JUDGE ДЛЯ ANSWER CORRECTNESS
# Оценка RAG-ответов через GROQ API
# - judge НЕ используется для настройки RAG
# - этот benchmark остаётся отложенной оценкой
# ============================================================

import json
import os
import pandas as pd

from groq import Groq


# ------------------------------------------------------------
# 1. Проверяем необходимые объекты
# ------------------------------------------------------------

if "GROQ_API_KEY" not in os.environ:
    raise RuntimeError(
        "GROQ_API_KEY не найден. "
        "Сначала запусти ячейку 13."
    )

if "GROQ_MODEL" not in globals():
    raise RuntimeError(
        "GROQ_MODEL не найден. "
        "Сначала запусти ячейку 24."
    )

if "rag_eval_df" not in globals():
    raise RuntimeError(
        "rag_eval_df не найден. "
        "Сначала запусти ячейку 25."
    )

if "generated_qa_df" not in globals():
    raise RuntimeError(
        "generated_qa_df не найден. "
        "Сначала запусти ячейку 24."
    )


# ------------------------------------------------------------
# 2. Создаём Groq client
# ------------------------------------------------------------

judge_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)


# ------------------------------------------------------------
# 3. Добавляем source_text к evaluation,
# если его там ещё нет
# ------------------------------------------------------------

if "source_text" not in rag_eval_df.columns:

    source_lookup = (
        generated_qa_df[
            [
                "question",
                "source_text"
            ]
        ]
        .drop_duplicates(
            subset=["question"]
        )
    )

    rag_eval_df = (
        rag_eval_df
        .merge(
            source_lookup,
            on="question",
            how="left"
        )
    )


# ------------------------------------------------------------
# 4. Функция judge
# ------------------------------------------------------------

def judge_rag_answer(
    question,
    reference_answer,
    rag_answer,
    source_text
):

    prompt = f"""
Ты оцениваешь качество ответа RAG-системы.

Тебе даны:

QUESTION:
{question}

SOURCE CHUNK:
{source_text}

REFERENCE ANSWER:
{reference_answer}

RAG ANSWER:
{rag_answer}

Оцени ответ RAG по двум критериям:

1. correctness
   Насколько ответ правильно отвечает на вопрос
   относительно reference answer.

2. groundedness
   Подтверждается ли ответ содержимым SOURCE CHUNK.

Верни только JSON:

{{
  "correct": true,
  "grounded": true,
  "score": 0,
  "reason": "..."
}}

Правила score:
0 = ответ неверный или отсутствует
1 = в основном неверный
2 = частично правильный
3 = в основном правильный
4 = полностью правильный и подтверждён источником

Не используй внешние знания.
Оценивай только по SOURCE CHUNK и REFERENCE ANSWER.
""".strip()


    completion = (
        judge_client
        .chat
        .completions
        .create(

            model=GROQ_MODEL,

            messages=[
                {
                    "role":
                        "system",

                    "content":
                        (
                            "Ты строгий независимый evaluator "
                            "для RAG-системы. "
                            "Возвращай только валидный JSON."
                        )
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }
            ],

            temperature=0
        )
    )


    raw_text = (
        completion
        .choices[0]
        .message
        .content
        .strip()
    )


    cleaned = (
        raw_text
        .replace(
            "```json",
            ""
        )
        .replace(
            "```JSON",
            ""
        )
        .replace(
            "```",
            ""
        )
        .strip()
    )


    parsed = json.loads(
        cleaned
    )


    return {
        "judge_correct":
            bool(
                parsed.get(
                    "correct",
                    False
                )
            ),

        "judge_grounded":
            bool(
                parsed.get(
                    "grounded",
                    False
                )
            ),

        "judge_score":
            int(
                parsed.get(
                    "score",
                    0
                )
            ),

        "judge_reason":
            str(
                parsed.get(
                    "reason",
                    ""
                )
            )
    }


# ------------------------------------------------------------
# 5. Оцениваем все RAG answers
# ------------------------------------------------------------

judge_rows = []


print("=" * 70)
print("LLM-JUDGE EVALUATION")
print("=" * 70)

print(
    "Examples:",
    len(rag_eval_df)
)

print(
    "Judge model:",
    GROQ_MODEL
)


for i, row in enumerate(
    rag_eval_df.itertuples(
        index=False
    ),
    start=1
):

    print(
        f"Judging "
        f"{i}/{len(rag_eval_df)}..."
    )


    rag_answer = str(
        getattr(
            row,
            "rag_answer",
            ""
        )
        or ""
    )


    source_text = str(
        getattr(
            row,
            "source_text",
            ""
        )
        or ""
    )


    # --------------------------------------------------------
    # Если RAG вообще не дал ответ,
    # API можно не вызывать
    # --------------------------------------------------------

    if not rag_answer.strip():

        judge_rows.append({

            "judge_correct":
                False,

            "judge_grounded":
                False,

            "judge_score":
                0,

            "judge_reason":
                "RAG returned an empty answer.",

            "judge_error":
                None
        })

        continue


    try:

        result = judge_rag_answer(

            question=str(
                row.question
            ),

            reference_answer=str(
                row.reference_answer
            ),

            rag_answer=rag_answer,

            source_text=source_text
        )


        judge_rows.append({

            **result,

            "judge_error":
                None
        })


    except Exception as e:

        print(
            "ERROR:",
            str(e)
        )

        judge_rows.append({

            "judge_correct":
                False,

            "judge_grounded":
                False,

            "judge_score":
                0,

            "judge_reason":
                "",

            "judge_error":
                str(e)
        })


# ------------------------------------------------------------
# 6. Добавляем результаты judge
# ------------------------------------------------------------

judge_df = pd.DataFrame(
    judge_rows
)


assert len(judge_df) == len(
    rag_eval_df
)


for column in judge_df.columns:

    rag_eval_df[
        column
    ] = judge_df[
        column
    ].values


# ------------------------------------------------------------
# 7. Считаем answer metrics
# ------------------------------------------------------------

answer_accuracy = float(
    rag_eval_df[
        "judge_correct"
    ]
    .mean()
)


groundedness_rate = float(
    rag_eval_df[
        "judge_grounded"
    ]
    .mean()
)


mean_judge_score = float(
    rag_eval_df[
        "judge_score"
    ]
    .mean()
)


judge_success_rate = float(
    rag_eval_df[
        "judge_error"
    ]
    .isna()
    .mean()
)


# ------------------------------------------------------------
# 8. Combined pipeline metric
#
# Считаем успехом:
# - правильный expected document
# - judge считает ответ correct
# - judge считает ответ grounded
# ------------------------------------------------------------

rag_eval_df[
    "overall_rag_success"
] = (

    rag_eval_df[
        "source_correct"
    ]

    &

    rag_eval_df[
        "judge_correct"
    ]

    &

    rag_eval_df[
        "judge_grounded"
    ]
)


overall_rag_success = float(
    rag_eval_df[
        "overall_rag_success"
    ]
    .mean()
)


# ------------------------------------------------------------
# 9. Итоговые метрики
# ------------------------------------------------------------

final_rag_metrics = {

    "n_examples":
        int(
            len(
                rag_eval_df
            )
        ),

    "source_accuracy":
        float(
            rag_eval_df[
                "source_correct"
            ]
            .mean()
        ),

    "chunk_hit_at_1":
        float(
            rag_eval_df[
                "chunk_hit_at_1"
            ]
            .mean()
        ),

    "chunk_hit_at_3":
        float(
            rag_eval_df[
                "chunk_hit_at_3"
            ]
            .mean()
        ),

    "answer_accuracy_llm_judge":
        answer_accuracy,

    "groundedness_rate":
        groundedness_rate,

    "mean_judge_score":
        mean_judge_score,

    "judge_success_rate":
        judge_success_rate,

    "overall_rag_success":
        overall_rag_success
}


print()
print("=" * 70)
print("FINAL DOCUMENT RAG METRICS")
print("=" * 70)


for key, value in final_rag_metrics.items():

    if isinstance(
        value,
        float
    ):

        print(
            f"{key}: "
            f"{value:.4f}"
        )

    else:

        print(
            f"{key}: "
            f"{value}"
        )


# ------------------------------------------------------------
# 10. Показываем примеры
# ------------------------------------------------------------

display(
    rag_eval_df[
        [
            "question",
            "reference_answer",
            "rag_answer",
            "source_correct",
            "chunk_hit_at_1",
            "chunk_hit_at_3",
            "judge_correct",
            "judge_grounded",
            "judge_score",
            "judge_reason",
            "overall_rag_success"
        ]
    ]
    .head(10)
)


# ------------------------------------------------------------
# 11. Сохраняем подробную evaluation
# ------------------------------------------------------------

FINAL_RAG_EVAL_PATH = (
    DOC_RAG_RESULTS_DIR
    / "api_generated_qa_final_evaluation.csv"
)


rag_eval_df.to_csv(
    FINAL_RAG_EVAL_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. Сохраняем metrics
# ------------------------------------------------------------

FINAL_RAG_METRICS_PATH = (
    DOC_RAG_RESULTS_DIR
    / "api_generated_qa_final_metrics.json"
)


with open(
    FINAL_RAG_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_rag_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 13. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 26 COMPLETE")
print("=" * 70)

print(
    "Evaluation saved:"
)

print(
    FINAL_RAG_EVAL_PATH
)

print()

print(
    "Metrics saved:"
)

print(
    FINAL_RAG_METRICS_PATH
)

print()

print(
    "ВАЖНО: этот benchmark остаётся "
    "отложенной оценкой."
)

print(
    "После просмотра результатов "
    "не меняем RAG под эти 30 вопросов."
)

print("=" * 70)

## 9. FINAL TEST — запускается только теперь

До этой точки все решения уже зафиксированы:
- DistilBERT;
- semantic retrieval;
- `SELECTED_THRESHOLD`;
- baseline reject policy;
- LangGraph routing;
- prompts;
- Document RAG pipeline.

После запуска этой секции **никакие параметры не меняются по результатам test**.


In [ ]:

# ============================================================
# 27. FINAL TEST — DistilBERT + semantic retrieval
# ============================================================

test_db = distilbert_predict(test_df)

test_scores, test_idx = semantic_retrieve_queries(
    test_df["text"],
    top_k=5
)

test_retrieved_labels = train_part["label"].to_numpy()[test_idx]
y_test = test_df["label"].to_numpy()

final_model_metrics = {
    "distilbert_accuracy": float(
        accuracy_score(y_test, test_db["pred"])
    ),
    "distilbert_macro_f1": float(
        f1_score(y_test, test_db["pred"], average="macro")
    ),
    "retrieval_hit@1": hit_at_k(test_retrieved_labels, y_test, 1),
    "retrieval_hit@3": hit_at_k(test_retrieved_labels, y_test, 3),
    "retrieval_hit@5": hit_at_k(test_retrieved_labels, y_test, 5)
}

print(json.dumps(final_model_metrics, indent=2))


In [ ]:

# ============================================================
# 28. FINAL TEST — baseline vs LangGraph
# ============================================================

baseline_test_metrics = reject_baseline_metrics(
    test_df,
    test_db,
    SELECTED_THRESHOLD
)

langgraph_test_metrics = langgraph_selective_metrics(
    test_df,
    test_db,
    test_retrieved_labels[:, 0],
    SELECTED_THRESHOLD
)

final_comparison_df = pd.DataFrame([
    baseline_test_metrics,
    langgraph_test_metrics
])

display(final_comparison_df)

final_comparison_df.to_csv(
    RESULTS_DIR / "final_langgraph_vs_reject_test.csv",
    index=False
)

with open(
    RESULTS_DIR / "final_model_metrics_test.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_model_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Threshold frozen from validation:",
    SELECTED_THRESHOLD
)


## 10. финальные результаты

### Validation
Использовалась для:
- выбора confidence threshold;
- построения diagnostic 100;
- проверки и изменения prompt;
- выбора routing logic.

### Test
Использован **один раз** после фиксации всех решений.

### Document RAG
Оценивается не на вручную подготовленных 27 вопросах, а на новом QA benchmark, сгенерированном через API по случайным chunks уже после фиксации RAG pipeline.

### LangGraph
Сравнивается напрямую с baseline `DistilBERT + reject threshold` при одном и том же threshold.
Главные метрики:
- coverage;
- automatic accuracy;
- review rate;
- automatic errors.

Если LangGraph увеличивает coverage, но резко снижает automatic accuracy, это нужно честно указать — это и есть результат эксперимента.


In [ ]:

# ============================================================
# 29. Финальный summary
# ============================================================

summary = {
    "protocol": protocol,
    "selected_threshold_from_validation": SELECTED_THRESHOLD,
    "validation": {
        "tfidf": val_lr_metrics,
        "distilbert_accuracy": float(
            accuracy_score(val_part["label"], val_db["pred"])
        ),
        "distilbert_macro_f1": float(
            f1_score(val_part["label"], val_db["pred"], average="macro")
        ),
        "retrieval": retrieval_val_metrics,
        "reject_baseline": baseline_val_metrics,
        "langgraph": langgraph_val_metrics
    },
    "final_test": {
        "models": final_model_metrics,
        "reject_baseline": baseline_test_metrics,
        "langgraph": langgraph_test_metrics
    },
    "document_rag_api_qa": rag_metrics
}

with open(
    RESULTS_DIR / "FINAL_CORRECTED_SUMMARY.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved:", RESULTS_DIR / "FINAL_CORRECTED_SUMMARY.json")


In [ ]:
# ============================================================
# ЯЧЕЙКА 30. ДИАГНОСТИКА DOCUMENT RAG RUNTIME ERRORS
# ============================================================

import pandas as pd
from collections import Counter


print("=" * 70)
print("DOCUMENT RAG ERROR DIAGNOSTICS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Проверяем наличие evaluation
# ------------------------------------------------------------

if "rag_eval_df" not in globals():

    RAG_EVAL_PATH = (
        DOC_RAG_RESULTS_DIR
        / "api_generated_qa_rag_evaluation.csv"
    )

    if not RAG_EVAL_PATH.exists():

        raise RuntimeError(
            "rag_eval_df не найден и файл evaluation отсутствует. "
            "Сначала должна быть выполнена ячейка 25."
        )

    rag_eval_df = pd.read_csv(
        RAG_EVAL_PATH
    )


# ------------------------------------------------------------
# 2. Общая статистика
# ------------------------------------------------------------

print(
    "Total QA examples:",
    len(rag_eval_df)
)

error_mask = (
    rag_eval_df["error"]
    .notna()
)

print(
    "Runtime errors:",
    int(
        error_mask.sum()
    )
)


# ------------------------------------------------------------
# 3. Показываем уникальные ошибки
# ------------------------------------------------------------

errors = (
    rag_eval_df.loc[
        error_mask,
        "error"
    ]
    .astype(str)
    .tolist()
)


error_counts = Counter(
    errors
)


print()
print("=" * 70)
print("UNIQUE ERRORS")
print("=" * 70)


for i, (error, count) in enumerate(
    error_counts.most_common(),
    start=1
):

    print()
    print(
        f"[{i}] Count: {count}"
    )

    print(
        error
    )


# ------------------------------------------------------------
# 4. Проверяем основные функции Document RAG
# ------------------------------------------------------------

print()
print("=" * 70)
print("AVAILABLE RAG OBJECTS")
print("=" * 70)


objects_to_check = [

    "answer_document_question",

    "search_internal_documents",

    "document_chunks_df",

    "doc_embedding_model",

    "document_index",

    "generated_qa_df"
]


for name in objects_to_check:

    print(
        f"{name}:",
        "OK"
        if name in globals()
        else "MISSING"
    )


# ------------------------------------------------------------
# 5. Если answer_document_question существует,
# смотрим сигнатуру
# ------------------------------------------------------------

if "answer_document_question" in globals():

    import inspect

    print()
    print("=" * 70)
    print("answer_document_question SIGNATURE")
    print("=" * 70)

    try:

        print(
            inspect.signature(
                answer_document_question
            )
        )

    except Exception as e:

        print(
            "Could not inspect:",
            str(e)
        )


# ------------------------------------------------------------
# 6. Берём только ОДИН вопрос
#
# Это техническая диагностика,
# а не изменение алгоритма.
# ------------------------------------------------------------

if "generated_qa_df" in globals():

    sample = generated_qa_df.iloc[0]

    print()
    print("=" * 70)
    print("SINGLE QA TECHNICAL TEST")
    print("=" * 70)

    print(
        "Question:"
    )

    print(
        sample["question"]
    )

    print()

    print(
        "Expected doc:"
    )

    print(
        sample["expected_doc_id"]
    )

    print()

    print(
        "Expected chunk:"
    )

    print(
        sample["expected_chunk_id"]
    )


    # --------------------------------------------------------
    # Тест retrieval отдельно
    # --------------------------------------------------------

    if "search_internal_documents" in globals():

        print()
        print("-" * 70)
        print("DIRECT RETRIEVAL TEST")
        print("-" * 70)

        try:

            retrieval_test = (
                search_internal_documents(
                    sample["question"],
                    top_k=3
                )
            )

            print(
                "Retrieval type:",
                type(
                    retrieval_test
                )
            )

            print()

            print(
                retrieval_test
            )

        except Exception as e:

            print(
                "RETRIEVAL ERROR:"
            )

            print(
                repr(e)
            )


    # --------------------------------------------------------
    # Тест полного answer function отдельно
    # --------------------------------------------------------

    if "answer_document_question" in globals():

        print()
        print("-" * 70)
        print("FULL RAG TEST")
        print("-" * 70)

        try:

            rag_test = (
                answer_document_question(
                    sample["question"],
                    top_k=3
                )
            )

            print(
                "Result type:",
                type(
                    rag_test
                )
            )

            print()

            print(
                rag_test
            )

        except Exception as e:

            print(
                "FULL RAG ERROR:"
            )

            print(
                repr(e)
            )


print()
print("=" * 70)
print("CELL 30 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 31. ИСПРАВЛЕНИЕ DOCUMENT RAG
#
# Исправляем только техническую инфраструктуру:
# - создаём search_internal_documents()
# - создаём retrieval без отсутствующего document_index
# - переписываем answer_document_question() под Groq
#
# ============================================================

import os
import numpy as np
import pandas as pd

from groq import Groq


# ------------------------------------------------------------
# 1. Проверяем необходимые объекты
# ------------------------------------------------------------

if "document_chunks_df" not in globals():

    raise RuntimeError(
        "document_chunks_df не найден. "
        "Сначала должны быть выполнены ячейки "
        "создания Document RAG knowledge base."
    )


if "doc_embedding_model" not in globals():

    raise RuntimeError(
        "doc_embedding_model не найден. "
        "Сначала должна быть загружена embedding model."
    )


if "GROQ_API_KEY" not in os.environ:

    raise RuntimeError(
        "GROQ_API_KEY не найден. "
        "Сначала запусти ячейку 13."
    )


if "GROQ_MODEL" not in globals():

    raise RuntimeError(
        "GROQ_MODEL не найден. "
        "Сначала запусти ячейку 24."
    )


# ------------------------------------------------------------
# 2. Groq client
# ------------------------------------------------------------

rag_client = Groq(
    api_key=os.environ[
        "GROQ_API_KEY"
    ]
)


# ------------------------------------------------------------
# 3. Проверяем колонки chunks
# ------------------------------------------------------------

print("=" * 70)
print("DOCUMENT RAG TECHNICAL FIX")
print("=" * 70)

print(
    "Chunks:",
    len(document_chunks_df)
)

print(
    "Columns:",
    list(
        document_chunks_df.columns
    )
)


required_columns = {
    "chunk_id",
    "doc_id",
    "text"
}


missing_columns = (
    required_columns
    -
    set(
        document_chunks_df.columns
    )
)


if missing_columns:

    raise RuntimeError(
        f"В document_chunks_df отсутствуют "
        f"колонки: {missing_columns}"
    )


# ------------------------------------------------------------
# 4. Один раз считаем embeddings всех chunks
#
# Это маленькая база (33 chunks), поэтому отдельный FAISS
# здесь не обязателен.
#
# Важно:
# corpus не меняется на основании QA benchmark.
# ------------------------------------------------------------

chunk_texts = (
    document_chunks_df[
        "text"
    ]
    .astype(str)
    .tolist()
)


print()
print(
    "Computing document chunk embeddings..."
)


document_chunk_embeddings = (
    doc_embedding_model.encode(

        chunk_texts,

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=True
    )
)


document_chunk_embeddings = np.asarray(
    document_chunk_embeddings,
    dtype=np.float32
)


print(
    "Embeddings shape:",
    document_chunk_embeddings.shape
)


assert len(
    document_chunk_embeddings
) == len(
    document_chunks_df
)


# ------------------------------------------------------------
# 5. Semantic retrieval
# ------------------------------------------------------------

def search_internal_documents(
    query,
    top_k=3
):

    query = str(
        query
    ).strip()


    if not query:

        raise ValueError(
            "Query is empty."
        )


    query_embedding = (
        doc_embedding_model.encode(

            [query],

            convert_to_numpy=True,

            normalize_embeddings=True
        )[0]
    )


    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )


    # cosine similarity,
    # потому что embeddings нормализованы

    scores = (
        document_chunk_embeddings
        @
        query_embedding
    )


    top_k = min(
        int(top_k),
        len(
            document_chunks_df
        )
    )


    top_indices = (
        np.argsort(
            scores
        )[::-1][:top_k]
    )


    rows = []


    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        source_row = (
            document_chunks_df
            .iloc[
                int(idx)
            ]
        )


        item = {

            "rank":
                rank,

            "chunk_id":
                str(
                    source_row[
                        "chunk_id"
                    ]
                ),

            "doc_id":
                str(
                    source_row[
                        "doc_id"
                    ]
                ),

            "text":
                str(
                    source_row[
                        "text"
                    ]
                ),

            "similarity":
                float(
                    scores[
                        int(idx)
                    ]
                )
        }


        # Добавляем metadata,
        # только если такие колонки существуют.

        for optional_column in [
            "title",
            "category"
        ]:

            if (
                optional_column
                in document_chunks_df.columns
            ):

                item[
                    optional_column
                ] = str(
                    source_row[
                        optional_column
                    ]
                )


        rows.append(
            item
        )


    return pd.DataFrame(
        rows
    )


# ------------------------------------------------------------
# 6. Технический retrieval test
# ------------------------------------------------------------

retrieval_smoke_test = (
    search_internal_documents(
        "Как подтвердить личность клиента?",
        top_k=3
    )
)


print()
print("=" * 70)
print("RETRIEVAL SMOKE TEST")
print("=" * 70)

display(
    retrieval_smoke_test[
        [
            "rank",
            "doc_id",
            "chunk_id",
            "similarity"
        ]
    ]
)


# ------------------------------------------------------------
# 7. Формирование grounded context
# ------------------------------------------------------------

def build_document_context(
    retrieved_df
):

    context_blocks = []


    for row in retrieved_df.itertuples(
        index=False
    ):

        block = (
            f"[DOC_ID: {row.doc_id}]\n"
            f"[CHUNK_ID: {row.chunk_id}]\n"
            f"{row.text}"
        )


        context_blocks.append(
            block
        )


    return "\n\n---\n\n".join(
        context_blocks
    )


# ------------------------------------------------------------
# 8. Финальная функция Document RAG
# ------------------------------------------------------------

def answer_document_question(
    query,
    top_k=3
):

    # --------------------------------------------------------
    # Retrieval
    # --------------------------------------------------------

    retrieved_df = (
        search_internal_documents(
            query=query,
            top_k=top_k
        )
    )


    if retrieved_df.empty:

        return {

            "answer":
                (
                    "В базе документов не найдено "
                    "достаточно информации для ответа."
                ),

            "sources":
                [],

            "retrieved_documents":
                []
        }


    # --------------------------------------------------------
    # Context
    # --------------------------------------------------------

    context = (
        build_document_context(
            retrieved_df
        )
    )


    # --------------------------------------------------------
    # Prompt
    #
    # Он общий и не содержит правил,
    # созданных по benchmark-вопросам.
    # --------------------------------------------------------

    prompt = f"""
Ты внутренний помощник сотрудника компании.

Отвечай на вопрос ТОЛЬКО на основании
предоставленных фрагментов внутренних документов.

ВОПРОС:
{query}

ФРАГМЕНТЫ ДОКУМЕНТОВ:
{context}

Правила:

1. Не используй внешние знания.
2. Не придумывай процедуры или правила.
3. Не добавляй действия, которых нет в документах.
4. Если информации недостаточно, прямо скажи об этом.
5. Ответ должен быть кратким и конкретным.
6. Не упоминай, что ты языковая модель.
7. Не придумывай идентификаторы документов.

Верни только текст ответа.
""".strip()


    # --------------------------------------------------------
    # Groq generation
    # --------------------------------------------------------

    completion = (
        rag_client
        .chat
        .completions
        .create(

            model=GROQ_MODEL,

            messages=[
                {
                    "role":
                        "system",

                    "content":
                        (
                            "Ты корпоративный RAG-ассистент. "
                            "Используй только переданный context."
                        )
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }
            ],

            temperature=0
        )
    )


    answer = (
        completion
        .choices[0]
        .message
        .content
        .strip()
    )


    # --------------------------------------------------------
    # Sources
    # --------------------------------------------------------

    sources = []


    for row in retrieved_df.itertuples(
        index=False
    ):

        sources.append({

            "doc_id":
                str(
                    row.doc_id
                ),

            "chunk_id":
                str(
                    row.chunk_id
                ),

            "similarity":
                float(
                    row.similarity
                )
        })


    # --------------------------------------------------------
    # Полные retrieved documents
    # --------------------------------------------------------

    retrieved_documents = (
        retrieved_df
        .to_dict(
            orient="records"
        )
    )


    return {

        "answer":
            answer,

        "sources":
            sources,

        "retrieved_documents":
            retrieved_documents
    }


# ------------------------------------------------------------
# 9. Проверяем на ОДНОМ техническом запросе
#
# Это не QA benchmark.
# ------------------------------------------------------------

technical_query = (
    "Как подтвердить личность клиента?"
)


print()
print("=" * 70)
print("FULL RAG SMOKE TEST")
print("=" * 70)

print(
    "Query:",
    technical_query
)


technical_result = (
    answer_document_question(
        technical_query,
        top_k=3
    )
)


print()
print(
    "Answer:"
)

print(
    technical_result[
        "answer"
    ]
)


print()
print(
    "Sources:"
)

for source in technical_result[
    "sources"
]:

    print(
        source
    )


# ------------------------------------------------------------
# 10. Контроль интерфейса для ячейки 25
# ------------------------------------------------------------

assert isinstance(
    technical_result,
    dict
)

assert (
    "answer"
    in technical_result
)

assert (
    "sources"
    in technical_result
)

assert (
    "retrieved_documents"
    in technical_result
)


print()
print("=" * 70)
print("CELL 31 COMPLETE")
print("=" * 70)

print(
    "search_internal_documents: READY"
)

print(
    "answer_document_question: READY"
)

print(
    "RAG API backend: GROQ"
)

print(
    "Model:",
    GROQ_MODEL
)

print()
print(
    "Теперь повторно запусти ЯЧЕЙКУ 25."
)

print(
    "Ячейку 24 повторно запускать НЕ нужно."
)

print("=" * 70)

In [ ]:
# ============================================================
# СОХРАНЕНИЕ ВСЕГО ПРОЕКТА В GOOGLE DRIVE
# ============================================================

import os
import shutil
from pathlib import Path

from google.colab import drive


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

LOCAL_PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DRIVE_MOUNT = Path(
    "/content/drive"
)

DRIVE_PROJECT_DIR = (
    DRIVE_MOUNT
    / "MyDrive"
    / "ml_customer_classification_corrected"
)


# ------------------------------------------------------------
# 2. Проверяем локальный проект
# ------------------------------------------------------------

print("=" * 70)
print("PROJECT BACKUP TO GOOGLE DRIVE")
print("=" * 70)

print(
    "Local project:",
    LOCAL_PROJECT_DIR
)


if not LOCAL_PROJECT_DIR.exists():

    raise RuntimeError(
        f"Локальная папка проекта не найдена: "
        f"{LOCAL_PROJECT_DIR}"
    )


# ------------------------------------------------------------
# 3. Подключаем Google Drive
# ------------------------------------------------------------

print()
print(
    "Подключаем Google Drive..."
)


try:

    drive.mount(
        str(
            DRIVE_MOUNT
        ),
        force_remount=True
    )

except Exception as e:

    print()
    print("=" * 70)
    print("DRIVE MOUNT ERROR")
    print("=" * 70)

    print(
        repr(e)
    )

    raise RuntimeError(
        "Google Drive не удалось подключить. "
        "Среду выполнения пока НЕ переключай."
    )


# ------------------------------------------------------------
# 4. Проверяем Drive
# ------------------------------------------------------------

MY_DRIVE = (
    DRIVE_MOUNT
    / "MyDrive"
)


if not MY_DRIVE.exists():

    raise RuntimeError(
        "Google Drive смонтирован, "
        "но папка MyDrive не найдена."
    )


print()
print(
    "Google Drive connected:",
    MY_DRIVE
)


# ------------------------------------------------------------
# 5. Создаём папку назначения
# ------------------------------------------------------------

DRIVE_PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Backup destination:",
    DRIVE_PROJECT_DIR
)


# ------------------------------------------------------------
# 6. Копируем проект целиком
#
# dirs_exist_ok=True:
# если папка уже существует,
# файлы будут обновлены.
# ------------------------------------------------------------

print()
print(
    "Копируем весь проект..."
)


shutil.copytree(
    LOCAL_PROJECT_DIR,
    DRIVE_PROJECT_DIR,
    dirs_exist_ok=True
)


# ------------------------------------------------------------
# 7. Подсчёт файлов
# ------------------------------------------------------------

local_files = [
    p
    for p in LOCAL_PROJECT_DIR.rglob("*")
    if p.is_file()
]


drive_files = [
    p
    for p in DRIVE_PROJECT_DIR.rglob("*")
    if p.is_file()
]


print()
print("=" * 70)
print("BACKUP SUMMARY")
print("=" * 70)

print(
    "Local files:",
    len(local_files)
)

print(
    "Drive files:",
    len(drive_files)
)


# ------------------------------------------------------------
# 8. Проверяем ключевые файлы
# ------------------------------------------------------------

important_relative_paths = [

    "data/train_part.parquet",
    "data/val_part.parquet",
    "data/test.parquet",
    "data/label_mapping.json",

    "document_rag/data/api_generated_qa.csv",

    "document_rag/results/api_generated_qa_rag_evaluation.csv",
    "document_rag/results/api_generated_qa_rag_metrics.json",
    "document_rag/results/api_generated_qa_final_evaluation.csv",
    "document_rag/results/api_generated_qa_final_metrics.json",

    "results/FINAL_CORRECTED_SUMMARY.json"
]


print()
print("=" * 70)
print("IMPORTANT FILE CHECK")
print("=" * 70)


all_important_ok = True


for relative_path in important_relative_paths:

    local_path = (
        LOCAL_PROJECT_DIR
        / relative_path
    )

    drive_path = (
        DRIVE_PROJECT_DIR
        / relative_path
    )


    local_ok = local_path.exists()
    drive_ok = drive_path.exists()


    status = (
        "OK"
        if local_ok and drive_ok
        else "MISSING"
    )


    print(
        f"{status:8} | "
        f"{relative_path}"
    )


    if not (
        local_ok
        and drive_ok
    ):

        all_important_ok = False


# ------------------------------------------------------------
# 9. Размер проекта
# ------------------------------------------------------------

def folder_size_bytes(path):

    total = 0

    for file_path in path.rglob("*"):

        if file_path.is_file():

            total += (
                file_path.stat().st_size
            )

    return total


local_size = folder_size_bytes(
    LOCAL_PROJECT_DIR
)

drive_size = folder_size_bytes(
    DRIVE_PROJECT_DIR
)


print()
print("=" * 70)
print("SIZE CHECK")
print("=" * 70)

print(
    "Local size:",
    f"{local_size / (1024 ** 2):.2f} MB"
)

print(
    "Drive size:",
    f"{drive_size / (1024 ** 2):.2f} MB"
)


# ------------------------------------------------------------
# 10. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)

if all_important_ok:

    print(
        "BACKUP COMPLETE"
    )

    print()
    print(
        "Ключевые файлы успешно сохранены в:"
    )

    print(
        DRIVE_PROJECT_DIR
    )

    print()
    print(
        "После этого можно переключать "
        "runtime на T4 GPU."
    )

else:

    print(
        "BACKUP PARTIALLY COMPLETE"
    )

    print()
    print(
        "Некоторые ключевые файлы не найдены."
    )

    print(
        "Среду выполнения пока НЕ переключай."
    )

print("=" * 70)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU не обнаружен")

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ ПРОЕКТА ИЗ GOOGLE DRIVE
# ============================================================

import os
import shutil
from pathlib import Path

from google.colab import drive


# ------------------------------------------------------------
# 1. Подключаем Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=True
)


# ------------------------------------------------------------
# 2. Пути
# ------------------------------------------------------------

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/ml_customer_classification_corrected"
)

LOCAL_PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)


# ------------------------------------------------------------
# 3. Проверяем backup
# ------------------------------------------------------------

if not DRIVE_PROJECT_DIR.exists():

    raise RuntimeError(
        f"Backup проекта не найден: "
        f"{DRIVE_PROJECT_DIR}"
    )


print("=" * 70)
print("RESTORE PROJECT FROM GOOGLE DRIVE")
print("=" * 70)

print(
    "Source:",
    DRIVE_PROJECT_DIR
)

print(
    "Destination:",
    LOCAL_PROJECT_DIR
)


# ------------------------------------------------------------
# 4. Восстанавливаем проект
# ------------------------------------------------------------

if LOCAL_PROJECT_DIR.exists():

    shutil.rmtree(
        LOCAL_PROJECT_DIR
    )


shutil.copytree(
    DRIVE_PROJECT_DIR,
    LOCAL_PROJECT_DIR
)


# ------------------------------------------------------------
# 5. Проверяем количество файлов
# ------------------------------------------------------------

drive_files = [
    p
    for p in DRIVE_PROJECT_DIR.rglob("*")
    if p.is_file()
]


local_files = [
    p
    for p in LOCAL_PROJECT_DIR.rglob("*")
    if p.is_file()
]


print()
print(
    "Drive files:",
    len(drive_files)
)

print(
    "Local files:",
    len(local_files)
)


# ------------------------------------------------------------
# 6. Проверяем GPU
# ------------------------------------------------------------

import torch


print()
print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    raise RuntimeError(
        "GPU не обнаружен."
    )


# ------------------------------------------------------------
# 7. Восстанавливаем основные пути проекта
# ------------------------------------------------------------

PROJECT_DIR = LOCAL_PROJECT_DIR

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

SEMANTIC_DIR = (
    PROJECT_DIR
    / "semantic"
)

SEMANTIC_RESULTS_DIR = (
    SEMANTIC_DIR
    / "results"
)

RAG_DIR = (
    PROJECT_DIR
    / "rag"
)

RAG_DATA_DIR = (
    RAG_DIR
    / "data"
)

RAG_RESULTS_DIR = (
    RAG_DIR
    / "results"
)

DOC_RAG_DIR = (
    PROJECT_DIR
    / "document_rag"
)

DOC_RAG_DATA_DIR = (
    DOC_RAG_DIR
    / "data"
)

DOC_RAG_RESULTS_DIR = (
    DOC_RAG_DIR
    / "results"
)

LANGGRAPH_DIR = (
    PROJECT_DIR
    / "langgraph"
)

LANGGRAPH_RESULTS_DIR = (
    LANGGRAPH_DIR
    / "results"
)

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
)


# ------------------------------------------------------------
# 8. Проверяем ключевые файлы
# ------------------------------------------------------------

important_files = [

    DATA_DIR
    / "train_part.parquet",

    DATA_DIR
    / "val_part.parquet",

    DATA_DIR
    / "test.parquet",

    DATA_DIR
    / "label_mapping.json",

    DOC_RAG_DATA_DIR
    / "api_generated_qa.csv"
]


print()
print("=" * 70)
print("KEY FILE CHECK")
print("=" * 70)


for path in important_files:

    print(
        "OK"
        if path.exists()
        else "MISSING",
        "|",
        path
    )


# ------------------------------------------------------------
# 9. Загружаем train / validation / test
# ------------------------------------------------------------

import pandas as pd
import json


train_part = pd.read_parquet(
    DATA_DIR
    / "train_part.parquet"
)

val_part = pd.read_parquet(
    DATA_DIR
    / "val_part.parquet"
)

test_df = pd.read_parquet(
    DATA_DIR
    / "test.parquet"
)


with open(
    DATA_DIR
    / "label_mapping.json",
    "r",
    encoding="utf-8"
) as f:

    label_mapping = json.load(f)


print()
print("=" * 70)
print("DATA RESTORED")
print("=" * 70)

print(
    "train_part:",
    train_part.shape
)

print(
    "val_part:",
    val_part.shape
)

print(
    "test_df:",
    test_df.shape
)

print(
    "labels:",
    len(label_mapping)
)


print()
print("=" * 70)
print("RESTORE COMPLETE")
print("=" * 70)

print(
    "Проект восстановлен."
)

print(
    "Следующий шаг: финальное обучение "
    "DistilBERT на 6 эпохах."
)

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path(
    "/content/ml_customer_classification_corrected/data"
)

LABEL_MAPPING_PATH = (
    DATA_DIR
    / "label_mapping.json"
)

with open(
    LABEL_MAPPING_PATH,
    "r",
    encoding="utf-8"
) as f:
    label_mapping = json.load(f)

print("Тип:", type(label_mapping))
print("Ключи верхнего уровня:", list(label_mapping.keys()))
print("Количество ключей верхнего уровня:", len(label_mapping))

for key, value in label_mapping.items():
    print()
    print("KEY:", key)
    print("TYPE:", type(value))

    try:
        print("LENGTH:", len(value))
    except TypeError:
        print("LENGTH: нет")

    if isinstance(value, dict):
        print("Первые 5 элементов:")
        for i, (k, v) in enumerate(value.items()):
            if i >= 5:
                break
            print(f"  {k} -> {v}")

In [ ]:
# ============================================================
# ФИНАЛЬЬНОЕ ОБУЧЕНИЕ DISTILBERT — 6 ЭПОХ НА T4
#
# train_part -> обучение
# val_part   -> выбор лучшего checkpoint
# test_df    -> НЕ ИСПОЛЬЗУЕТСЯ
# ============================================================

!pip -q install -U transformers datasets accelerate scikit-learn

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


# ------------------------------------------------------------
# 1. Seed
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 2. Проверяем GPU
# ------------------------------------------------------------

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA недоступна. "
        "Для финального обучения нужна T4 GPU."
    )


DEVICE = "cuda"

print("=" * 70)
print("FINAL DISTILBERT TRAINING")
print("=" * 70)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)


# ------------------------------------------------------------
# 3. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FINAL_MODEL_DIR = (
    MODELS_DIR
    / "distilbert_final_6epoch"
)

TRAINING_OUTPUT_DIR = (
    MODELS_DIR
    / "distilbert_final_training"
)


# ------------------------------------------------------------
# 4. На случай нового runtime загружаем данные
# ------------------------------------------------------------

if "train_part" not in globals():

    train_part = pd.read_parquet(
        DATA_DIR
        / "train_part.parquet"
    )


if "val_part" not in globals():

    val_part = pd.read_parquet(
        DATA_DIR
        / "val_part.parquet"
    )


# ------------------------------------------------------------
# 5. Проверяем данные
# ------------------------------------------------------------

required_columns = {
    "text",
    "label",
    "label_text"
}


for df_name, df in [
    ("train_part", train_part),
    ("val_part", val_part)
]:

    missing = (
        required_columns
        -
        set(df.columns)
    )

    if missing:

        raise RuntimeError(
            f"{df_name}: отсутствуют колонки {missing}"
        )


NUM_LABELS = int(
    train_part[
        "label"
    ]
    .nunique()
)


print()
print(
    "train_part:",
    train_part.shape
)

print(
    "val_part:",
    val_part.shape
)

print(
    "Number of labels:",
    NUM_LABELS
)


if NUM_LABELS != 77:

    raise RuntimeError(
        f"Ожидалось 77 классов Banking77, "
        f"получено {NUM_LABELS}."
    )


# ------------------------------------------------------------
# 6. Создаём корректные mapping
# ------------------------------------------------------------

label_table = (
    train_part[
        [
            "label",
            "label_text"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "label"
    )
)


id_to_label = {
    int(row.label):
        str(row.label_text)

    for row in label_table.itertuples(
        index=False
    )
}


label_to_id = {
    label_name:
        label_id

    for label_id, label_name
    in id_to_label.items()
}


print(
    "id_to_label:",
    len(id_to_label)
)

print(
    "label_to_id:",
    len(label_to_id)
)


# ------------------------------------------------------------
# 7. Сохраняем исправленный mapping
# ------------------------------------------------------------

CORRECTED_MAPPING_PATH = (
    DATA_DIR
    / "label_mapping_final.json"
)


with open(
    CORRECTED_MAPPING_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "id_to_label": {
                str(k): v
                for k, v
                in id_to_label.items()
            },

            "label_to_id":
                label_to_id
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 8. Hugging Face datasets
# ------------------------------------------------------------

train_hf = Dataset.from_pandas(
    train_part[
        [
            "text",
            "label"
        ]
    ],
    preserve_index=False
)


val_hf = Dataset.from_pandas(
    val_part[
        [
            "text",
            "label"
        ]
    ],
    preserve_index=False
)


# ------------------------------------------------------------
# 9. Tokenizer
# ------------------------------------------------------------

BASE_MODEL = (
    "distilbert-base-uncased"
)

MAX_LENGTH = 64


tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL
    )
)


def tokenize_batch(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_tokenized = (
    train_hf
    .map(
        tokenize_batch,
        batched=True
    )
)


val_tokenized = (
    val_hf
    .map(
        tokenize_batch,
        batched=True
    )
)


train_tokenized = (
    train_tokenized
    .rename_column(
        "label",
        "labels"
    )
)


val_tokenized = (
    val_tokenized
    .rename_column(
        "label",
        "labels"
    )
)


train_tokenized.set_format(
    "torch"
)

val_tokenized.set_format(
    "torch"
)


# ------------------------------------------------------------
# 10. Dynamic padding
# ------------------------------------------------------------

data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)


# ------------------------------------------------------------
# 11. Model
# ------------------------------------------------------------

model = (
    AutoModelForSequenceClassification
    .from_pretrained(

        BASE_MODEL,

        num_labels=NUM_LABELS,

        id2label=id_to_label,

        label2id=label_to_id
    )
)


# ------------------------------------------------------------
# 12. Metrics
# ------------------------------------------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )


    accuracy = accuracy_score(
        labels,
        predictions
    )


    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )


    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )


    return {
        "accuracy":
            float(
                accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "weighted_f1":
            float(
                weighted_f1
            )
    }


# ------------------------------------------------------------
# 13. TrainingArguments
# ------------------------------------------------------------

training_args = TrainingArguments(

    output_dir=str(
        TRAINING_OUTPUT_DIR
    ),

    num_train_epochs=6,

    learning_rate=5e-5,

    per_device_train_batch_size=32,

    per_device_eval_batch_size=64,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    save_total_limit=2,

    fp16=True,

    report_to="none",

    seed=SEED,

    data_seed=SEED
)


# ------------------------------------------------------------
# 14. Trainer
# ------------------------------------------------------------

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


# ------------------------------------------------------------
# 15. Обучение
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAINING START")
print("=" * 70)

print(
    "Epochs: 6"
)

print(
    "Train samples:",
    len(train_part)
)

print(
    "Validation samples:",
    len(val_part)
)

print(
    "TEST DATA IS NOT USED"
)


train_result = trainer.train()


# ------------------------------------------------------------
# 16. Финальная validation evaluation
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL VALIDATION EVALUATION")
print("=" * 70)


final_val_metrics = (
    trainer.evaluate(
        eval_dataset=val_tokenized
    )
)


for key, value in final_val_metrics.items():

    print(
        f"{key}: {value}"
    )


# ------------------------------------------------------------
# 17. Сохраняем лучшую модель
# ------------------------------------------------------------

trainer.save_model(
    str(
        FINAL_MODEL_DIR
    )
)


tokenizer.save_pretrained(
    str(
        FINAL_MODEL_DIR
    )
)


# ------------------------------------------------------------
# 18. Сохраняем validation metrics
# ------------------------------------------------------------

FINAL_VAL_METRICS_PATH = (
    RESULTS_DIR
    / "distilbert_final_validation_metrics.json"
)


serializable_metrics = {}


for key, value in final_val_metrics.items():

    if isinstance(
        value,
        (
            np.floating,
            np.integer
        )
    ):

        serializable_metrics[
            key
        ] = value.item()

    else:

        serializable_metrics[
            key
        ] = value


with open(
    FINAL_VAL_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        serializable_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 19. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL DISTILBERT TRAINING COMPLETE")
print("=" * 70)

print(
    "Model saved:"
)

print(
    FINAL_MODEL_DIR
)

print()

print(
    "Validation metrics saved:"
)

print(
    FINAL_VAL_METRICS_PATH
)

print()

print(
    "Best checkpoint:"
)

print(
    trainer.state.best_model_checkpoint
)

print()

print(
    "Best validation Macro F1:"
)

print(
    trainer.state.best_metric
)

print()

print(
    "IMPORTANT:"
)

print(
    "test_df НЕ использовался при обучении "
    "или выборе модели."
)

print("=" * 70)

In [ ]:
# ============================================================
# FINAL VALIDATION PREDICTIONS + THRESHOLD SELECTION
#
# Используем ТОЛЬКО val_part.
# test_df здесь НЕ используется.
#
# Критерий выбора threshold:
# максимальная coverage при automatic accuracy >= 95%.
# ============================================================

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FINAL_MODEL_DIR = (
    MODELS_DIR
    / "distilbert_final_6epoch"
)


# ------------------------------------------------------------
# 2. Загружаем validation
# ------------------------------------------------------------

if "val_part" not in globals():

    val_part = pd.read_parquet(
        DATA_DIR
        / "val_part.parquet"
    )


print("=" * 70)
print("FINAL VALIDATION THRESHOLD SELECTION")
print("=" * 70)

print(
    "Validation size:",
    len(val_part)
)

print(
    "Model:",
    FINAL_MODEL_DIR
)

print(
    "TEST DATA IS NOT USED"
)


# ------------------------------------------------------------
# 3. Загружаем финальную модель
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


tokenizer_final = (
    AutoTokenizer
    .from_pretrained(
        FINAL_MODEL_DIR
    )
)


model_final = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FINAL_MODEL_DIR
    )
    .to(device)
)


model_final.eval()


print(
    "Device:",
    device
)


# ------------------------------------------------------------
# 4. Batch inference на validation
# ------------------------------------------------------------

BATCH_SIZE = 64
MAX_LENGTH = 64


all_predictions = []
all_confidences = []
all_probabilities = []


texts = (
    val_part[
        "text"
    ]
    .astype(str)
    .tolist()
)


with torch.no_grad():

    for start in range(
        0,
        len(texts),
        BATCH_SIZE
    ):

        batch_texts = texts[
            start:
            start + BATCH_SIZE
        ]


        encoded = tokenizer_final(

            batch_texts,

            padding=True,

            truncation=True,

            max_length=MAX_LENGTH,

            return_tensors="pt"
        )


        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }


        outputs = model_final(
            **encoded
        )


        probabilities = (
            torch.softmax(
                outputs.logits,
                dim=-1
            )
            .cpu()
            .numpy()
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        confidences = np.max(
            probabilities,
            axis=1
        )


        all_predictions.extend(
            predictions.tolist()
        )

        all_confidences.extend(
            confidences.tolist()
        )

        all_probabilities.append(
            probabilities
        )


all_probabilities = np.vstack(
    all_probabilities
)


# ------------------------------------------------------------
# 5. Создаём validation predictions dataframe
# ------------------------------------------------------------

val_predictions_df = (
    val_part
    .reset_index(drop=True)
    .copy()
)


val_predictions_df[
    "distilbert_pred"
] = np.array(
    all_predictions,
    dtype=int
)


val_predictions_df[
    "distilbert_confidence"
] = np.array(
    all_confidences,
    dtype=float
)


val_predictions_df[
    "distilbert_correct"
] = (

    val_predictions_df[
        "distilbert_pred"
    ]

    ==

    val_predictions_df[
        "label"
    ]
)


# ------------------------------------------------------------
# 6. Проверяем общие validation metrics
# ------------------------------------------------------------

val_accuracy = accuracy_score(

    val_predictions_df[
        "label"
    ],

    val_predictions_df[
        "distilbert_pred"
    ]
)


val_macro_f1 = f1_score(

    val_predictions_df[
        "label"
    ],

    val_predictions_df[
        "distilbert_pred"
    ],

    average="macro"
)


print()
print("=" * 70)
print("VALIDATION MODEL METRICS")
print("=" * 70)

print(
    f"Accuracy: {val_accuracy:.6f}"
)

print(
    f"Macro F1: {val_macro_f1:.6f}"
)


# ------------------------------------------------------------
# 7. Threshold grid
#
# Начинаем достаточно низко,
# чтобы увидеть полный trade-off.
# ------------------------------------------------------------

thresholds = np.round(
    np.arange(
        0.00,
        1.00,
        0.01
    ),
    2
)


threshold_rows = []


for threshold in thresholds:

    accepted_mask = (

        val_predictions_df[
            "distilbert_confidence"
        ]

        >= threshold
    )


    auto_closed = int(
        accepted_mask.sum()
    )


    human_review = int(
        len(
            val_predictions_df
        )
        -
        auto_closed
    )


    coverage = (

        auto_closed
        /
        len(
            val_predictions_df
        )

    )


    review_rate = (

        human_review
        /
        len(
            val_predictions_df
        )

    )


    if auto_closed > 0:

        automatic_accuracy = float(

            val_predictions_df.loc[
                accepted_mask,
                "distilbert_correct"
            ]
            .mean()
        )


        automatic_errors = int(

            (
                ~val_predictions_df.loc[
                    accepted_mask,
                    "distilbert_correct"
                ]
            )
            .sum()
        )


    else:

        automatic_accuracy = np.nan
        automatic_errors = 0


    threshold_rows.append({

        "threshold":
            float(
                threshold
            ),

        "auto_closed":
            auto_closed,

        "coverage":
            float(
                coverage
            ),

        "human_review":
            human_review,

        "review_rate":
            float(
                review_rate
            ),

        "automatic_accuracy":
            automatic_accuracy,

        "automatic_errors":
            automatic_errors
    })


threshold_df = pd.DataFrame(
    threshold_rows
)


# ------------------------------------------------------------
# 8. Выбираем threshold
#
# Политика:
# automatic_accuracy >= 95%
# и среди таких thresholds выбираем
# максимальную coverage.
#
# Это решение принимается ТОЛЬКО по validation.
# ------------------------------------------------------------

TARGET_AUTO_ACCURACY = 0.95


eligible = threshold_df[

    (
        threshold_df[
            "automatic_accuracy"
        ]
        >= TARGET_AUTO_ACCURACY
    )

    &

    (
        threshold_df[
            "auto_closed"
        ]
        > 0
    )

].copy()


if len(eligible) == 0:

    raise RuntimeError(
        "Не найден threshold с "
        "automatic_accuracy >= 95%."
    )


selected_row = (

    eligible
    .sort_values(

        by=[
            "coverage",
            "automatic_accuracy",
            "threshold"
        ],

        ascending=[
            False,
            False,
            True
        ]
    )
    .iloc[0]
)


SELECTED_THRESHOLD = float(
    selected_row[
        "threshold"
    ]
)


# ------------------------------------------------------------
# 9. Финальный baseline на validation
# ------------------------------------------------------------

validation_reject_baseline = {

    "system":
        "DistilBERT + reject threshold",

    "threshold":
        SELECTED_THRESHOLD,

    "n":
        int(
            len(
                val_predictions_df
            )
        ),

    "auto_closed":
        int(
            selected_row[
                "auto_closed"
            ]
        ),

    "coverage":
        float(
            selected_row[
                "coverage"
            ]
        ),

    "human_review":
        int(
            selected_row[
                "human_review"
            ]
        ),

    "review_rate":
        float(
            selected_row[
                "review_rate"
            ]
        ),

    "automatic_accuracy":
        float(
            selected_row[
                "automatic_accuracy"
            ]
        ),

    "automatic_errors":
        int(
            selected_row[
                "automatic_errors"
            ]
        )
}


# ------------------------------------------------------------
# 10. Выводим выбранный threshold
# ------------------------------------------------------------

print()
print("=" * 70)
print("SELECTED VALIDATION THRESHOLD")
print("=" * 70)

print(
    "Target automatic accuracy:",
    TARGET_AUTO_ACCURACY
)

print(
    "Selected threshold:",
    SELECTED_THRESHOLD
)

print(
    "Auto closed:",
    validation_reject_baseline[
        "auto_closed"
    ]
)

print(
    "Coverage:",
    f"{validation_reject_baseline['coverage']:.4f}"
)

print(
    "Human review:",
    validation_reject_baseline[
        "human_review"
    ]
)

print(
    "Review rate:",
    f"{validation_reject_baseline['review_rate']:.4f}"
)

print(
    "Automatic accuracy:",
    f"{validation_reject_baseline['automatic_accuracy']:.4f}"
)

print(
    "Automatic errors:",
    validation_reject_baseline[
        "automatic_errors"
    ]
)


# ------------------------------------------------------------
# 11. Показываем thresholds около выбранного
# ------------------------------------------------------------

near_threshold = threshold_df[

    (
        threshold_df[
            "threshold"
        ]
        >= max(
            0,
            SELECTED_THRESHOLD - 0.05
        )
    )

    &

    (
        threshold_df[
            "threshold"
        ]
        <= min(
            0.99,
            SELECTED_THRESHOLD + 0.05
        )
    )

].copy()


print()
print("=" * 70)
print("THRESHOLD NEIGHBORHOOD")
print("=" * 70)

display(
    near_threshold
)


# ------------------------------------------------------------
# 12. Сохраняем validation predictions
# ------------------------------------------------------------

VAL_PREDICTIONS_PATH = (

    VALIDATION_DIR
    / "distilbert_final_val_predictions.csv"
)


val_predictions_df.to_csv(
    VAL_PREDICTIONS_PATH,
    index=False
)


# ------------------------------------------------------------
# 13. Сохраняем полный threshold sweep
# ------------------------------------------------------------

THRESHOLD_SWEEP_PATH = (

    VALIDATION_DIR
    / "distilbert_final_threshold_sweep.csv"
)


threshold_df.to_csv(
    THRESHOLD_SWEEP_PATH,
    index=False
)


# ------------------------------------------------------------
# 14. Сохраняем выбранный threshold
# ------------------------------------------------------------

THRESHOLD_RESULT_PATH = (

    VALIDATION_DIR
    / "selected_threshold_final.json"
)


threshold_result = {

    "selection_dataset":
        "val_part",

    "target_automatic_accuracy":
        TARGET_AUTO_ACCURACY,

    "selected_threshold":
        SELECTED_THRESHOLD,

    "distilbert_validation_accuracy":
        float(
            val_accuracy
        ),

    "distilbert_validation_macro_f1":
        float(
            val_macro_f1
        ),

    "reject_baseline":
        validation_reject_baseline,

    "test_used":
        False
}


with open(
    THRESHOLD_RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_result,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 15. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("VALIDATION THRESHOLD SELECTION COMPLETE")
print("=" * 70)

print(
    "Selected threshold:",
    SELECTED_THRESHOLD
)

print()

print(
    "Predictions saved:"
)

print(
    VAL_PREDICTIONS_PATH
)

print()

print(
    "Threshold sweep saved:"
)

print(
    THRESHOLD_SWEEP_PATH
)

print()

print(
    "Selection result saved:"
)

print(
    THRESHOLD_RESULT_PATH
)

print()

print(
    "IMPORTANT: test_df НЕ использовался."
)

print("=" * 70)

In [ ]:
# ============================================================
# FINAL VALIDATION LANGGRAPH EVALUATION
# + RESTORE SEMANTIC RETRIEVAL AFTER RUNTIME RESET
#
# Retrieval corpus: ONLY train_part
# Threshold: already selected ONLY on val_part
# test_df: NOT USED
# ============================================================

!pip -q install -U sentence-transformers

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

SEMANTIC_DIR = (
    PROJECT_DIR
    / "semantic"
)

SEMANTIC_RESULTS_DIR = (
    SEMANTIC_DIR
    / "results"
)

LANGGRAPH_DIR = (
    PROJECT_DIR
    / "langgraph"
)

LANGGRAPH_RESULTS_DIR = (
    LANGGRAPH_DIR
    / "results"
)


SEMANTIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEMANTIC_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LANGGRAPH_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Загружаем train_part
#
# ВАЖНО:
# semantic retrieval строится ТОЛЬКО по train_part.
# ------------------------------------------------------------

TRAIN_PATH = (
    DATA_DIR
    / "train_part.parquet"
)


if not TRAIN_PATH.exists():

    raise RuntimeError(
        "train_part.parquet не найден."
    )


train_part = pd.read_parquet(
    TRAIN_PATH
)


print("=" * 70)
print("RESTORING SEMANTIC RETRIEVAL")
print("=" * 70)

print(
    "Retrieval corpus:",
    train_part.shape
)


# ------------------------------------------------------------
# 3. Проверяем необходимые колонки
# ------------------------------------------------------------

required_columns = {
    "text",
    "label",
    "label_text"
}


missing_columns = (
    required_columns
    -
    set(train_part.columns)
)


if missing_columns:

    raise RuntimeError(
        f"В train_part отсутствуют колонки: "
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# 4. Восстанавливаем label mapping
# ------------------------------------------------------------

LABEL_MAPPING_PATH = (
    DATA_DIR
    / "label_mapping.json"
)


with open(
    LABEL_MAPPING_PATH,
    "r",
    encoding="utf-8"
) as f:

    label_mapping = json.load(f)


id_to_label = {
    int(k): v
    for k, v
    in label_mapping[
        "id_to_label"
    ].items()
}


label_to_id = {
    k: int(v)
    for k, v
    in label_mapping[
        "label_to_id"
    ].items()
}


print(
    "Labels:",
    len(id_to_label)
)


# ------------------------------------------------------------
# 5. Semantic embedding model
#
# Используем all-MiniLM-L6-v2 для Banking77 retrieval.
# ------------------------------------------------------------

SEMANTIC_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)


semantic_device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print()
print(
    "Semantic model:",
    SEMANTIC_MODEL_NAME
)

print(
    "Device:",
    semantic_device
)


semantic_model = SentenceTransformer(
    SEMANTIC_MODEL_NAME,
    device=semantic_device
)


# ------------------------------------------------------------
# 6. Строим embeddings ТОЛЬКО train_part
# ------------------------------------------------------------

semantic_corpus_df = (
    train_part[
        [
            "text",
            "label",
            "label_text"
        ]
    ]
    .reset_index(drop=True)
    .copy()
)


print()
print(
    "Encoding retrieval corpus..."
)


semantic_corpus_embeddings = (
    semantic_model.encode(

        semantic_corpus_df[
            "text"
        ]
        .astype(str)
        .tolist(),

        batch_size=128,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True
    )
)


semantic_corpus_embeddings = np.asarray(
    semantic_corpus_embeddings,
    dtype=np.float32
)


print(
    "Corpus embeddings shape:",
    semantic_corpus_embeddings.shape
)


assert (
    len(semantic_corpus_embeddings)
    ==
    len(semantic_corpus_df)
)


# ------------------------------------------------------------
# 7. Восстанавливаем semantic_search()
# ------------------------------------------------------------

def semantic_search(
    query,
    top_k=5
):

    query = str(
        query
    ).strip()


    if not query:

        raise ValueError(
            "Query is empty."
        )


    query_embedding = (
        semantic_model.encode(

            [query],

            convert_to_numpy=True,

            normalize_embeddings=True
        )[0]
    )


    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )


    # cosine similarity, поскольку embeddings нормализованы

    scores = (
        semantic_corpus_embeddings
        @
        query_embedding
    )


    top_k = min(
        int(top_k),
        len(semantic_corpus_df)
    )


    top_indices = (
        np.argsort(scores)[::-1][
            :top_k
        ]
    )


    results = (
        semantic_corpus_df
        .iloc[top_indices]
        .copy()
        .reset_index(drop=True)
    )


    results[
        "similarity"
    ] = scores[
        top_indices
    ]


    results.insert(
        0,
        "rank",
        np.arange(
            1,
            len(results) + 1
        )
    )


    return results


# ------------------------------------------------------------
# 8. Smoke test semantic retrieval
#
# НЕ validation benchmark.
# Просто техническая проверка функции.
# ------------------------------------------------------------

print()
print("=" * 70)
print("SEMANTIC SEARCH SMOKE TEST")
print("=" * 70)


semantic_smoke = semantic_search(
    "How do I activate my card?",
    top_k=3
)


display(
    semantic_smoke[
        [
            "rank",
            "label",
            "label_text",
            "similarity",
            "text"
        ]
    ]
)


# ------------------------------------------------------------
# 9. Загружаем FINAL validation predictions
# ------------------------------------------------------------

VAL_PREDICTIONS_PATH = (
    VALIDATION_DIR
    / "distilbert_final_val_predictions.csv"
)


if not VAL_PREDICTIONS_PATH.exists():

    raise RuntimeError(
        "distilbert_final_val_predictions.csv "
        "не найден."
    )


val_predictions_df = pd.read_csv(
    VAL_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# 10. Загружаем уже выбранный threshold
# ------------------------------------------------------------

THRESHOLD_RESULT_PATH = (
    VALIDATION_DIR
    / "selected_threshold_final.json"
)


if not THRESHOLD_RESULT_PATH.exists():

    raise RuntimeError(
        "selected_threshold_final.json "
        "не найден."
    )


with open(
    THRESHOLD_RESULT_PATH,
    "r",
    encoding="utf-8"
) as f:

    threshold_result = json.load(f)


SELECTED_THRESHOLD = float(
    threshold_result[
        "selected_threshold"
    ]
)


baseline = threshold_result[
    "reject_baseline"
]


print()
print("=" * 70)
print("FINAL VALIDATION LANGGRAPH EVALUATION")
print("=" * 70)

print(
    "Validation size:",
    len(val_predictions_df)
)

print(
    "Selected threshold:",
    SELECTED_THRESHOLD
)

print(
    "Retrieval corpus:",
    len(semantic_corpus_df)
)

print(
    "TEST DATA IS NOT USED"
)


# ------------------------------------------------------------
# 11. LangGraph selective consensus
#
# Frozen logic:
#
# confidence >= threshold
#     -> DistilBERT auto close
#
# confidence < threshold
#     -> semantic retrieval
#
# DistilBERT == retrieval prediction
#     -> consensus auto close
#
# disagreement
#     -> human review
# ------------------------------------------------------------

langgraph_rows = []


for i, row in enumerate(
    val_predictions_df.itertuples(
        index=False
    ),
    start=1
):

    if (
        i == 1
        or i % 250 == 0
        or i == len(val_predictions_df)
    ):

        print(
            f"Processing {i}/"
            f"{len(val_predictions_df)}..."
        )


    query = str(
        row.text
    )


    true_label = int(
        row.label
    )


    distilbert_pred = int(
        row.distilbert_pred
    )


    confidence = float(
        row.distilbert_confidence
    )


    retrieval_pred = None


    # --------------------------------------------------------
    # Route 1: high-confidence DistilBERT
    # --------------------------------------------------------

    if confidence >= SELECTED_THRESHOLD:

        final_pred = distilbert_pred

        auto_closed = True

        route = (
            "distilbert_high_confidence"
        )


    # --------------------------------------------------------
    # Route 2: low-confidence -> retrieval
    # --------------------------------------------------------

    else:

        retrieval_result = (
            semantic_search(
                query,
                top_k=1
            )
        )


        if len(retrieval_result) > 0:

            retrieval_pred = int(
                retrieval_result
                .iloc[0][
                    "label"
                ]
            )


        # ----------------------------------------------------
        # Consensus
        # ----------------------------------------------------

        if (
            retrieval_pred is not None
            and
            retrieval_pred
            ==
            distilbert_pred
        ):

            final_pred = (
                distilbert_pred
            )

            auto_closed = True

            route = (
                "low_confidence_consensus"
            )


        # ----------------------------------------------------
        # Human review
        # ----------------------------------------------------

        else:

            final_pred = None

            auto_closed = False

            route = (
                "human_review"
            )


    # --------------------------------------------------------
    # Correctness only for automatically closed requests
    # --------------------------------------------------------

    if auto_closed:

        correct = bool(
            int(final_pred)
            ==
            true_label
        )

    else:

        correct = None


    langgraph_rows.append({

        "text":
            query,

        "true_label":
            true_label,

        "distilbert_pred":
            distilbert_pred,

        "confidence":
            confidence,

        "retrieval_pred":
            retrieval_pred,

        "final_pred":
            final_pred,

        "auto_closed":
            auto_closed,

        "correct":
            correct,

        "route":
            route
    })


# ------------------------------------------------------------
# 12. DataFrame
# ------------------------------------------------------------

langgraph_val_df = pd.DataFrame(
    langgraph_rows
)


# ------------------------------------------------------------
# 13. Метрики
# ------------------------------------------------------------

auto_mask = (
    langgraph_val_df[
        "auto_closed"
    ]
    ==
    True
)


auto_closed = int(
    auto_mask.sum()
)


human_review = int(
    len(langgraph_val_df)
    -
    auto_closed
)


coverage = float(
    auto_closed
    /
    len(langgraph_val_df)
)


review_rate = float(
    human_review
    /
    len(langgraph_val_df)
)


if auto_closed > 0:

    auto_correct_series = (
        langgraph_val_df
        .loc[
            auto_mask,
            "correct"
        ]
        .astype(bool)
    )


    automatic_accuracy = float(
        auto_correct_series.mean()
    )


    automatic_errors = int(
        (
            ~auto_correct_series
        )
        .sum()
    )


else:

    automatic_accuracy = np.nan

    automatic_errors = 0


extra_closed_vs_baseline = (

    auto_closed

    -

    int(
        baseline[
            "auto_closed"
        ]
    )
)


route_counts = (
    langgraph_val_df[
        "route"
    ]
    .value_counts()
    .to_dict()
)


# ------------------------------------------------------------
# 14. Result dict
# ------------------------------------------------------------

validation_langgraph = {

    "system":
        "LangGraph selective consensus",

    "threshold":
        SELECTED_THRESHOLD,

    "n":
        int(
            len(langgraph_val_df)
        ),

    "auto_closed":
        auto_closed,

    "coverage":
        coverage,

    "human_review":
        human_review,

    "review_rate":
        review_rate,

    "automatic_accuracy":
        automatic_accuracy,

    "automatic_errors":
        automatic_errors,

    "extra_closed_vs_baseline":
        int(
            extra_closed_vs_baseline
        ),

    "routes":
        {
            str(k): int(v)
            for k, v
            in route_counts.items()
        }
}


# ------------------------------------------------------------
# 15. Сравнение baseline vs LangGraph
# ------------------------------------------------------------

comparison_df = pd.DataFrame(

    [
        {
            "system":
                "DistilBERT + reject",

            "threshold":
                SELECTED_THRESHOLD,

            "auto_closed":
                int(
                    baseline[
                        "auto_closed"
                    ]
                ),

            "coverage":
                float(
                    baseline[
                        "coverage"
                    ]
                ),

            "human_review":
                int(
                    baseline[
                        "human_review"
                    ]
                ),

            "review_rate":
                float(
                    baseline[
                        "review_rate"
                    ]
                ),

            "automatic_accuracy":
                float(
                    baseline[
                        "automatic_accuracy"
                    ]
                ),

            "automatic_errors":
                int(
                    baseline[
                        "automatic_errors"
                    ]
                )
        },

        {
            "system":
                "LangGraph consensus",

            "threshold":
                SELECTED_THRESHOLD,

            "auto_closed":
                auto_closed,

            "coverage":
                coverage,

            "human_review":
                human_review,

            "review_rate":
                review_rate,

            "automatic_accuracy":
                automatic_accuracy,

            "automatic_errors":
                automatic_errors
        }
    ]
)


print()
print("=" * 70)
print("BASELINE VS LANGGRAPH — VALIDATION")
print("=" * 70)

display(
    comparison_df
)


print()
print(
    "Extra closed vs baseline:",
    extra_closed_vs_baseline
)

print(
    "Routes:",
    route_counts
)


# ------------------------------------------------------------
# 16. Дополнительная проверка consensus route
# ------------------------------------------------------------

consensus_mask = (

    langgraph_val_df[
        "route"
    ]

    ==

    "low_confidence_consensus"
)


consensus_count = int(
    consensus_mask.sum()
)


if consensus_count > 0:

    consensus_accuracy = float(

        langgraph_val_df
        .loc[
            consensus_mask,
            "correct"
        ]
        .astype(bool)
        .mean()
    )

else:

    consensus_accuracy = np.nan


print()
print(
    "Low-confidence consensus count:",
    consensus_count
)

print(
    "Low-confidence consensus accuracy:",
    (
        f"{consensus_accuracy:.4f}"
        if not np.isnan(
            consensus_accuracy
        )
        else "N/A"
    )
)


# ------------------------------------------------------------
# 17. Сохраняем predictions
# ------------------------------------------------------------

LANGGRAPH_VAL_PATH = (

    LANGGRAPH_RESULTS_DIR
    / "langgraph_final_validation_predictions.csv"
)


langgraph_val_df.to_csv(
    LANGGRAPH_VAL_PATH,
    index=False
)


# ------------------------------------------------------------
# 18. Сохраняем metrics
# ------------------------------------------------------------

LANGGRAPH_METRICS_PATH = (

    LANGGRAPH_RESULTS_DIR
    / "langgraph_final_validation_metrics.json"
)


with open(
    LANGGRAPH_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        validation_langgraph,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 19. Сохраняем информацию о retrieval
# ------------------------------------------------------------

SEMANTIC_CONFIG_PATH = (

    SEMANTIC_RESULTS_DIR
    / "semantic_retrieval_final_config.json"
)


semantic_config = {

    "model":
        SEMANTIC_MODEL_NAME,

    "corpus":
        "train_part",

    "corpus_size":
        int(
            len(
                semantic_corpus_df
            )
        ),

    "embedding_dimension":
        int(
            semantic_corpus_embeddings.shape[
                1
            ]
        ),

    "normalized_embeddings":
        True,

    "test_used":
        False
}


with open(
    SEMANTIC_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        semantic_config,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 20. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL VALIDATION LANGGRAPH COMPLETE")
print("=" * 70)

print(
    "Threshold frozen:",
    SELECTED_THRESHOLD
)

print(
    "Semantic retrieval: READY"
)

print(
    "Retrieval corpus: train_part ONLY"
)

print(
    "TEST DATA IS STILL NOT USED"
)

print()
print(
    "Saved:"
)

print(
    LANGGRAPH_VAL_PATH
)

print(
    LANGGRAPH_METRICS_PATH
)

print("=" * 70)

In [ ]:
# ============================================================
# FINAL LANGGRAPH ARCHITECTURE SELECTION ON VALIDATION
#
# DistilBERT threshold уже зафиксирован: 0.65
#
# Теперь на val_part выбираем дополнительный порог:
# MIN_RETRIEVAL_SIMILARITY
#
# Low-confidence запрос закрываем автоматически ТОЛЬКО если:
# 1) retrieval prediction == DistilBERT prediction
# 2) retrieval similarity >= MIN_RETRIEVAL_SIMILARITY
#
# Цель:
# максимальная coverage при automatic_accuracy >= 95%
#
# TEST DATA НЕ ИСПОЛЬЗУЕТСЯ.
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

LANGGRAPH_RESULTS_DIR = (
    PROJECT_DIR
    / "langgraph"
    / "results"
)

LANGGRAPH_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Загружаем validation predictions
# ------------------------------------------------------------

VAL_PREDICTIONS_PATH = (
    VALIDATION_DIR
    / "distilbert_final_val_predictions.csv"
)

THRESHOLD_RESULT_PATH = (
    VALIDATION_DIR
    / "selected_threshold_final.json"
)


val_predictions_df = pd.read_csv(
    VAL_PREDICTIONS_PATH
)


with open(
    THRESHOLD_RESULT_PATH,
    "r",
    encoding="utf-8"
) as f:

    threshold_result = json.load(f)


SELECTED_THRESHOLD = float(
    threshold_result[
        "selected_threshold"
    ]
)

baseline = threshold_result[
    "reject_baseline"
]


print("=" * 70)
print("LANGGRAPH ARCHITECTURE SELECTION — VALIDATION")
print("=" * 70)

print(
    "DistilBERT threshold:",
    SELECTED_THRESHOLD
)

print(
    "Validation size:",
    len(val_predictions_df)
)

print(
    "Baseline coverage:",
    f"{baseline['coverage']:.4f}"
)

print(
    "Baseline automatic accuracy:",
    f"{baseline['automatic_accuracy']:.4f}"
)

print(
    "TEST DATA IS NOT USED"
)


# ------------------------------------------------------------
# 3. Проверяем semantic_search
# ------------------------------------------------------------

if "semantic_search" not in globals():

    raise RuntimeError(
        "semantic_search не найден. "
        "Сначала выполни предыдущую ячейку "
        "восстановления semantic retrieval."
    )


# ------------------------------------------------------------
# 4. Собираем retrieval данные только
# для low-confidence validation запросов
# ------------------------------------------------------------

analysis_rows = []


low_conf_df = (
    val_predictions_df[
        val_predictions_df[
            "distilbert_confidence"
        ]
        <
        SELECTED_THRESHOLD
    ]
    .copy()
    .reset_index(drop=True)
)


print()
print(
    "Low-confidence validation examples:",
    len(low_conf_df)
)


for i, row in enumerate(
    low_conf_df.itertuples(
        index=False
    ),
    start=1
):

    if (
        i == 1
        or i % 25 == 0
        or i == len(low_conf_df)
    ):

        print(
            f"Retrieval {i}/{len(low_conf_df)}..."
        )


    result = semantic_search(
        str(row.text),
        top_k=1
    )


    if len(result) == 0:

        retrieval_pred = None
        retrieval_similarity = np.nan

    else:

        retrieval_pred = int(
            result.iloc[0][
                "label"
            ]
        )

        retrieval_similarity = float(
            result.iloc[0][
                "similarity"
            ]
        )


    distilbert_pred = int(
        row.distilbert_pred
    )

    true_label = int(
        row.label
    )


    consensus = (
        retrieval_pred is not None
        and
        retrieval_pred
        ==
        distilbert_pred
    )


    consensus_correct = (

        consensus
        and
        distilbert_pred
        ==
        true_label
    )


    analysis_rows.append({

        "text":
            str(row.text),

        "true_label":
            true_label,

        "distilbert_pred":
            distilbert_pred,

        "distilbert_confidence":
            float(
                row.distilbert_confidence
            ),

        "retrieval_pred":
            retrieval_pred,

        "retrieval_similarity":
            retrieval_similarity,

        "consensus":
            bool(
                consensus
            ),

        "consensus_correct":
            bool(
                consensus_correct
            )
    })


low_conf_analysis_df = pd.DataFrame(
    analysis_rows
)


# ------------------------------------------------------------
# 5. Базовая статистика consensus
# ------------------------------------------------------------

consensus_df = (
    low_conf_analysis_df[
        low_conf_analysis_df[
            "consensus"
        ]
        ==
        True
    ]
    .copy()
)


print()
print("=" * 70)
print("LOW-CONFIDENCE CONSENSUS ANALYSIS")
print("=" * 70)

print(
    "Consensus examples:",
    len(consensus_df)
)


if len(consensus_df) > 0:

    print(
        "Raw consensus accuracy:",
        f"{consensus_df['consensus_correct'].mean():.4f}"
    )

    print(
        "Similarity min:",
        f"{consensus_df['retrieval_similarity'].min():.4f}"
    )

    print(
        "Similarity max:",
        f"{consensus_df['retrieval_similarity'].max():.4f}"
    )

    print(
        "Similarity median:",
        f"{consensus_df['retrieval_similarity'].median():.4f}"
    )


# ------------------------------------------------------------
# 6. Sweep retrieval similarity threshold
#
# DistilBERT high-confidence часть остаётся неизменной.
# Добавляем только надёжные consensus cases.
# ------------------------------------------------------------

TARGET_AUTO_ACCURACY = 0.95


similarity_thresholds = np.round(
    np.arange(
        0.00,
        1.001,
        0.01
    ),
    2
)


sweep_rows = []


baseline_auto_closed = int(
    baseline[
        "auto_closed"
    ]
)

baseline_correct = (

    baseline_auto_closed
    -
    int(
        baseline[
            "automatic_errors"
        ]
    )
)


for similarity_threshold in similarity_thresholds:

    accepted_consensus = (
        consensus_df[
            consensus_df[
                "retrieval_similarity"
            ]
            >=
            similarity_threshold
        ]
    )


    extra_closed = int(
        len(
            accepted_consensus
        )
    )


    extra_correct = int(
        accepted_consensus[
            "consensus_correct"
        ]
        .sum()
    )


    total_auto_closed = (
        baseline_auto_closed
        +
        extra_closed
    )


    total_correct = (
        baseline_correct
        +
        extra_correct
    )


    total_errors = (
        total_auto_closed
        -
        total_correct
    )


    coverage = (
        total_auto_closed
        /
        len(
            val_predictions_df
        )
    )


    automatic_accuracy = (
        total_correct
        /
        total_auto_closed
        if total_auto_closed > 0
        else np.nan
    )


    human_review = (
        len(
            val_predictions_df
        )
        -
        total_auto_closed
    )


    review_rate = (
        human_review
        /
        len(
            val_predictions_df
        )
    )


    consensus_accuracy = (

        extra_correct
        /
        extra_closed

        if extra_closed > 0

        else np.nan
    )


    sweep_rows.append({

        "retrieval_similarity_threshold":
            float(
                similarity_threshold
            ),

        "extra_closed":
            extra_closed,

        "extra_correct":
            extra_correct,

        "consensus_accuracy":
            consensus_accuracy,

        "total_auto_closed":
            total_auto_closed,

        "coverage":
            float(
                coverage
            ),

        "human_review":
            int(
                human_review
            ),

        "review_rate":
            float(
                review_rate
            ),

        "automatic_accuracy":
            float(
                automatic_accuracy
            ),

        "automatic_errors":
            int(
                total_errors
            )
    })


similarity_sweep_df = pd.DataFrame(
    sweep_rows
)


# ------------------------------------------------------------
# 7. Выбираем лучшую архитектуру
#
# Максимальная coverage при общей
# automatic accuracy >= 95%.
# ------------------------------------------------------------

eligible = (
    similarity_sweep_df[
        similarity_sweep_df[
            "automatic_accuracy"
        ]
        >=
        TARGET_AUTO_ACCURACY
    ]
    .copy()
)


if len(eligible) == 0:

    raise RuntimeError(
        "Не найден retrieval similarity threshold "
        "с общей automatic accuracy >= 95%."
    )


selected_architecture = (

    eligible
    .sort_values(

        by=[
            "coverage",
            "automatic_accuracy",
            "retrieval_similarity_threshold"
        ],

        ascending=[
            False,
            False,
            False
        ]
    )
    .iloc[0]
)


MIN_RETRIEVAL_SIMILARITY = float(
    selected_architecture[
        "retrieval_similarity_threshold"
    ]
)


# ------------------------------------------------------------
# 8. Финальные validation metrics
# ------------------------------------------------------------

final_validation_langgraph = {

    "system":
        "LangGraph selective consensus",

    "distilbert_threshold":
        SELECTED_THRESHOLD,

    "retrieval_similarity_threshold":
        MIN_RETRIEVAL_SIMILARITY,

    "selection_dataset":
        "val_part",

    "n":
        int(
            len(
                val_predictions_df
            )
        ),

    "auto_closed":
        int(
            selected_architecture[
                "total_auto_closed"
            ]
        ),

    "coverage":
        float(
            selected_architecture[
                "coverage"
            ]
        ),

    "human_review":
        int(
            selected_architecture[
                "human_review"
            ]
        ),

    "review_rate":
        float(
            selected_architecture[
                "review_rate"
            ]
        ),

    "automatic_accuracy":
        float(
            selected_architecture[
                "automatic_accuracy"
            ]
        ),

    "automatic_errors":
        int(
            selected_architecture[
                "automatic_errors"
            ]
        ),

    "extra_closed_vs_baseline":
        int(
            selected_architecture[
                "extra_closed"
            ]
        ),

    "accepted_consensus_accuracy":
        (
            None

            if pd.isna(
                selected_architecture[
                    "consensus_accuracy"
                ]
            )

            else float(
                selected_architecture[
                    "consensus_accuracy"
                ]
            )
        ),

    "target_automatic_accuracy":
        TARGET_AUTO_ACCURACY,

    "test_used":
        False
}


# ------------------------------------------------------------
# 9. Вывод результата
# ------------------------------------------------------------

print()
print("=" * 70)
print("SELECTED LANGGRAPH ARCHITECTURE")
print("=" * 70)

print(
    "DistilBERT threshold:",
    SELECTED_THRESHOLD
)

print(
    "Min retrieval similarity:",
    MIN_RETRIEVAL_SIMILARITY
)

print(
    "Auto closed:",
    final_validation_langgraph[
        "auto_closed"
    ]
)

print(
    "Coverage:",
    f"{final_validation_langgraph['coverage']:.4f}"
)

print(
    "Human review:",
    final_validation_langgraph[
        "human_review"
    ]
)

print(
    "Review rate:",
    f"{final_validation_langgraph['review_rate']:.4f}"
)

print(
    "Automatic accuracy:",
    f"{final_validation_langgraph['automatic_accuracy']:.4f}"
)

print(
    "Automatic errors:",
    final_validation_langgraph[
        "automatic_errors"
    ]
)

print(
    "Extra closed vs baseline:",
    final_validation_langgraph[
        "extra_closed_vs_baseline"
    ]
)

print(
    "Accepted consensus accuracy:",
    final_validation_langgraph[
        "accepted_consensus_accuracy"
    ]
)


# ------------------------------------------------------------
# 10. Сравнение с baseline
# ------------------------------------------------------------

comparison_df = pd.DataFrame(
    [
        {
            "system":
                "DistilBERT + reject",

            "coverage":
                baseline[
                    "coverage"
                ],

            "automatic_accuracy":
                baseline[
                    "automatic_accuracy"
                ],

            "review_rate":
                baseline[
                    "review_rate"
                ],

            "automatic_errors":
                baseline[
                    "automatic_errors"
                ],

            "auto_closed":
                baseline[
                    "auto_closed"
                ]
        },

        {
            "system":
                "LangGraph + similarity gate",

            "coverage":
                final_validation_langgraph[
                    "coverage"
                ],

            "automatic_accuracy":
                final_validation_langgraph[
                    "automatic_accuracy"
                ],

            "review_rate":
                final_validation_langgraph[
                    "review_rate"
                ],

            "automatic_errors":
                final_validation_langgraph[
                    "automatic_errors"
                ],

            "auto_closed":
                final_validation_langgraph[
                    "auto_closed"
                ]
        }
    ]
)


print()
print("=" * 70)
print("FINAL BASELINE VS LANGGRAPH — VALIDATION")
print("=" * 70)

display(
    comparison_df
)


# ------------------------------------------------------------
# 11. Показываем область около выбранного threshold
# ------------------------------------------------------------

near_selected = (
    similarity_sweep_df[
        (
            similarity_sweep_df[
                "retrieval_similarity_threshold"
            ]
            >=
            max(
                0,
                MIN_RETRIEVAL_SIMILARITY
                -
                0.05
            )
        )
        &
        (
            similarity_sweep_df[
                "retrieval_similarity_threshold"
            ]
            <=
            min(
                1,
                MIN_RETRIEVAL_SIMILARITY
                +
                0.05
            )
        )
    ]
)


print()
print("=" * 70)
print("SIMILARITY THRESHOLD NEIGHBORHOOD")
print("=" * 70)

display(
    near_selected
)


# ------------------------------------------------------------
# 12. Сохраняем analysis
# ------------------------------------------------------------

LOW_CONF_ANALYSIS_PATH = (
    LANGGRAPH_RESULTS_DIR
    / "low_confidence_consensus_validation.csv"
)


low_conf_analysis_df.to_csv(
    LOW_CONF_ANALYSIS_PATH,
    index=False
)


# ------------------------------------------------------------
# 13. Сохраняем sweep
# ------------------------------------------------------------

SIMILARITY_SWEEP_PATH = (
    LANGGRAPH_RESULTS_DIR
    / "retrieval_similarity_threshold_sweep.csv"
)


similarity_sweep_df.to_csv(
    SIMILARITY_SWEEP_PATH,
    index=False
)


# ------------------------------------------------------------
# 14. Сохраняем финальную архитектуру
# ------------------------------------------------------------

FINAL_ARCHITECTURE_PATH = (
    LANGGRAPH_RESULTS_DIR
    / "final_langgraph_architecture.json"
)


with open(
    FINAL_ARCHITECTURE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_validation_langgraph,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 15. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("LANGGRAPH ARCHITECTURE SELECTION COMPLETE")
print("=" * 70)

print(
    "Configuration selected ONLY on val_part."
)

print(
    "test_df has NOT been used."
)

print()

print(
    "Saved architecture:"
)

print(
    FINAL_ARCHITECTURE_PATH
)

print()

print(
    "После этой ячейки параметры "
    "DistilBERT threshold и retrieval similarity "
    "больше не изменяем."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 51.
# SINGLE FINAL TEST EVALUATION
#
# Все решения уже заморожены на val_part:
#
# DistilBERT threshold = 0.65
# Retrieval similarity gate = 1.0
#
# Здесь:
# 1. Финальная оценка DistilBERT на test
# 2. Финальная оценка semantic retrieval на test
# 3. DistilBERT + reject threshold на test
# 4. Frozen LangGraph на test
#
# После этой ячейки НЕЛЬЗЯ менять threshold,
# retrieval gate или архитектуру по результатам test.
# ============================================================

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

LANGGRAPH_RESULTS_DIR = (
    PROJECT_DIR
    / "langgraph"
    / "results"
)

SEMANTIC_RESULTS_DIR = (
    PROJECT_DIR
    / "semantic"
    / "results"
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FINAL_MODEL_DIR = (
    MODELS_DIR
    / "distilbert_final_6epoch"
)


# ------------------------------------------------------------
# 2. Загружаем test
# ------------------------------------------------------------

TEST_PATH = (
    DATA_DIR
    / "test.parquet"
)


if not TEST_PATH.exists():

    raise RuntimeError(
        "test.parquet не найден."
    )


test_df = pd.read_parquet(
    TEST_PATH
)


print("=" * 70)
print("SINGLE FINAL TEST EVALUATION")
print("=" * 70)

print(
    "Test size:",
    len(test_df)
)


# ------------------------------------------------------------
# 3. Загружаем ЗАМОРОЖЕННЫЕ validation decisions
# ------------------------------------------------------------

THRESHOLD_RESULT_PATH = (
    VALIDATION_DIR
    / "selected_threshold_final.json"
)


FINAL_ARCHITECTURE_PATH = (
    LANGGRAPH_RESULTS_DIR
    / "final_langgraph_architecture.json"
)


with open(
    THRESHOLD_RESULT_PATH,
    "r",
    encoding="utf-8"
) as f:

    threshold_result = json.load(f)


with open(
    FINAL_ARCHITECTURE_PATH,
    "r",
    encoding="utf-8"
) as f:

    architecture_result = json.load(f)


SELECTED_THRESHOLD = float(
    threshold_result[
        "selected_threshold"
    ]
)


MIN_RETRIEVAL_SIMILARITY = float(
    architecture_result[
        "retrieval_similarity_threshold"
    ]
)


print()
print("FROZEN PARAMETERS")

print(
    "DistilBERT threshold:",
    SELECTED_THRESHOLD
)

print(
    "Retrieval similarity gate:",
    MIN_RETRIEVAL_SIMILARITY
)


# ------------------------------------------------------------
# 4. Проверяем, что semantic retrieval остался в памяти
# ------------------------------------------------------------

required_objects = [
    "semantic_model",
    "semantic_corpus_df",
    "semantic_corpus_embeddings"
]


missing = [
    name
    for name in required_objects
    if name not in globals()
]


if missing:

    raise RuntimeError(
        f"Не найдены semantic retrieval объекты: {missing}. "
        "Повтори ячейку 50 перед этой ячейкой."
    )


# ------------------------------------------------------------
# 5. Загружаем финальный DistilBERT
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


tokenizer_final = (
    AutoTokenizer
    .from_pretrained(
        FINAL_MODEL_DIR
    )
)


model_final = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FINAL_MODEL_DIR
    )
    .to(device)
)


model_final.eval()


print()
print(
    "DistilBERT device:",
    device
)


# ------------------------------------------------------------
# 6. DistilBERT inference на test
# ------------------------------------------------------------

BATCH_SIZE = 64
MAX_LENGTH = 64


test_texts = (
    test_df[
        "text"
    ]
    .astype(str)
    .tolist()
)


all_test_predictions = []
all_test_confidences = []


print()
print(
    "Running final DistilBERT inference..."
)


with torch.no_grad():

    for start in range(
        0,
        len(test_texts),
        BATCH_SIZE
    ):

        batch_texts = test_texts[
            start:
            start + BATCH_SIZE
        ]


        encoded = tokenizer_final(

            batch_texts,

            padding=True,

            truncation=True,

            max_length=MAX_LENGTH,

            return_tensors="pt"
        )


        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }


        outputs = model_final(
            **encoded
        )


        probabilities = (
            torch.softmax(
                outputs.logits,
                dim=-1
            )
            .cpu()
            .numpy()
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        confidences = np.max(
            probabilities,
            axis=1
        )


        all_test_predictions.extend(
            predictions.tolist()
        )

        all_test_confidences.extend(
            confidences.tolist()
        )


# ------------------------------------------------------------
# 7. Test predictions dataframe
# ------------------------------------------------------------

final_test_df = (
    test_df
    .reset_index(drop=True)
    .copy()
)


final_test_df[
    "distilbert_pred"
] = np.asarray(
    all_test_predictions,
    dtype=int
)


final_test_df[
    "distilbert_confidence"
] = np.asarray(
    all_test_confidences,
    dtype=float
)


final_test_df[
    "distilbert_correct"
] = (

    final_test_df[
        "distilbert_pred"
    ]

    ==

    final_test_df[
        "label"
    ]
)


# ------------------------------------------------------------
# 8. Финальные DistilBERT metrics
# ------------------------------------------------------------

distilbert_test_accuracy = float(
    accuracy_score(

        final_test_df[
            "label"
        ],

        final_test_df[
            "distilbert_pred"
        ]
    )
)


distilbert_test_macro_f1 = float(
    f1_score(

        final_test_df[
            "label"
        ],

        final_test_df[
            "distilbert_pred"
        ],

        average="macro"
    )
)


distilbert_test_weighted_f1 = float(
    f1_score(

        final_test_df[
            "label"
        ],

        final_test_df[
            "distilbert_pred"
        ],

        average="weighted"
    )
)


print()
print("=" * 70)
print("FINAL DISTILBERT TEST METRICS")
print("=" * 70)

print(
    f"Accuracy: {distilbert_test_accuracy:.6f}"
)

print(
    f"Macro F1: {distilbert_test_macro_f1:.6f}"
)

print(
    f"Weighted F1: {distilbert_test_weighted_f1:.6f}"
)


# ------------------------------------------------------------
# 9. Финальная semantic retrieval evaluation
#
# Encode test in batch.
# Corpus = train_part ONLY.
# ------------------------------------------------------------

print()
print(
    "Encoding test queries for semantic retrieval..."
)


test_embeddings = (
    semantic_model.encode(

        test_texts,

        batch_size=128,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True
    )
)


test_embeddings = np.asarray(
    test_embeddings,
    dtype=np.float32
)


# ------------------------------------------------------------
# 10. Retrieval top-5
#
# Работаем batch-wise, чтобы не держать
# лишнюю большую similarity matrix в памяти.
# ------------------------------------------------------------

retrieval_top_labels = []
retrieval_top_scores = []


RETRIEVAL_BATCH_SIZE = 256


print()
print(
    "Running final semantic retrieval..."
)


for start in range(
    0,
    len(test_embeddings),
    RETRIEVAL_BATCH_SIZE
):

    batch_embeddings = test_embeddings[
        start:
        start + RETRIEVAL_BATCH_SIZE
    ]


    similarities = (
        batch_embeddings
        @
        semantic_corpus_embeddings.T
    )


    # Получаем top-5 indices
    top_indices_unsorted = np.argpartition(
        -similarities,
        kth=4,
        axis=1
    )[:, :5]


    # Сортируем эти top-5 по similarity
    top_scores_unsorted = np.take_along_axis(
        similarities,
        top_indices_unsorted,
        axis=1
    )


    ordering = np.argsort(
        -top_scores_unsorted,
        axis=1
    )


    top_indices = np.take_along_axis(
        top_indices_unsorted,
        ordering,
        axis=1
    )


    top_scores = np.take_along_axis(
        similarities,
        top_indices,
        axis=1
    )


    corpus_labels = (
        semantic_corpus_df[
            "label"
        ]
        .to_numpy()
    )


    batch_labels = corpus_labels[
        top_indices
    ]


    retrieval_top_labels.append(
        batch_labels
    )

    retrieval_top_scores.append(
        top_scores
    )


retrieval_top_labels = np.vstack(
    retrieval_top_labels
)


retrieval_top_scores = np.vstack(
    retrieval_top_scores
)


# ------------------------------------------------------------
# 11. Retrieval metrics
# ------------------------------------------------------------

true_labels = (
    final_test_df[
        "label"
    ]
    .astype(int)
    .to_numpy()
)


retrieval_hit_at_1 = float(
    np.mean(
        retrieval_top_labels[
            :, 0
        ]
        ==
        true_labels
    )
)


retrieval_hit_at_3 = float(
    np.mean(
        [
            true_label
            in predicted_labels[:3]

            for true_label, predicted_labels
            in zip(
                true_labels,
                retrieval_top_labels
            )
        ]
    )
)


retrieval_hit_at_5 = float(
    np.mean(
        [
            true_label
            in predicted_labels[:5]

            for true_label, predicted_labels
            in zip(
                true_labels,
                retrieval_top_labels
            )
        ]
    )
)


print()
print("=" * 70)
print("FINAL SEMANTIC RETRIEVAL TEST METRICS")
print("=" * 70)

print(
    f"Hit@1: {retrieval_hit_at_1:.6f}"
)

print(
    f"Hit@3: {retrieval_hit_at_3:.6f}"
)

print(
    f"Hit@5: {retrieval_hit_at_5:.6f}"
)


# ------------------------------------------------------------
# 12. Добавляем top-1 retrieval prediction
# ------------------------------------------------------------

final_test_df[
    "retrieval_pred"
] = (
    retrieval_top_labels[
        :, 0
    ]
    .astype(int)
)


final_test_df[
    "retrieval_similarity"
] = (
    retrieval_top_scores[
        :, 0
    ]
    .astype(float)
)


# ------------------------------------------------------------
# 13. FINAL BASELINE:
# DistilBERT + frozen reject threshold
# ------------------------------------------------------------

baseline_auto_mask = (

    final_test_df[
        "distilbert_confidence"
    ]

    >=

    SELECTED_THRESHOLD
)


baseline_auto_closed = int(
    baseline_auto_mask.sum()
)


baseline_human_review = int(
    len(final_test_df)
    -
    baseline_auto_closed
)


baseline_coverage = float(
    baseline_auto_closed
    /
    len(final_test_df)
)


baseline_review_rate = float(
    baseline_human_review
    /
    len(final_test_df)
)


baseline_auto_correct = (
    final_test_df.loc[
        baseline_auto_mask,
        "distilbert_correct"
    ]
)


baseline_automatic_accuracy = float(
    baseline_auto_correct.mean()
)


baseline_automatic_errors = int(
    (
        ~baseline_auto_correct
    )
    .sum()
)


final_test_baseline = {

    "system":
        "DistilBERT + reject threshold",

    "threshold":
        SELECTED_THRESHOLD,

    "n":
        int(
            len(final_test_df)
        ),

    "auto_closed":
        baseline_auto_closed,

    "coverage":
        baseline_coverage,

    "human_review":
        baseline_human_review,

    "review_rate":
        baseline_review_rate,

    "automatic_accuracy":
        baseline_automatic_accuracy,

    "automatic_errors":
        baseline_automatic_errors
}


# ------------------------------------------------------------
# 14. FINAL FROZEN LANGGRAPH
#
# High confidence:
#     DistilBERT
#
# Low confidence:
#     close only if:
#     retrieval_pred == distilbert_pred
#     AND similarity >= frozen gate
#
# Otherwise:
#     human review
# ------------------------------------------------------------

langgraph_rows = []


for row in final_test_df.itertuples(
    index=False
):

    true_label = int(
        row.label
    )

    distilbert_pred = int(
        row.distilbert_pred
    )

    confidence = float(
        row.distilbert_confidence
    )

    retrieval_pred = int(
        row.retrieval_pred
    )

    retrieval_similarity = float(
        row.retrieval_similarity
    )


    if confidence >= SELECTED_THRESHOLD:

        final_pred = (
            distilbert_pred
        )

        auto_closed = True

        route = (
            "distilbert_high_confidence"
        )


    elif (
        retrieval_pred
        ==
        distilbert_pred
        and
        retrieval_similarity
        >=
        MIN_RETRIEVAL_SIMILARITY
    ):

        final_pred = (
            distilbert_pred
        )

        auto_closed = True

        route = (
            "low_confidence_consensus"
        )


    else:

        final_pred = None

        auto_closed = False

        route = (
            "human_review"
        )


    if auto_closed:

        correct = bool(
            final_pred
            ==
            true_label
        )

    else:

        correct = None


    langgraph_rows.append({

        "true_label":
            true_label,

        "distilbert_pred":
            distilbert_pred,

        "distilbert_confidence":
            confidence,

        "retrieval_pred":
            retrieval_pred,

        "retrieval_similarity":
            retrieval_similarity,

        "final_pred":
            final_pred,

        "auto_closed":
            auto_closed,

        "correct":
            correct,

        "route":
            route
    })


langgraph_test_df = pd.DataFrame(
    langgraph_rows
)


# ------------------------------------------------------------
# 15. LangGraph test metrics
# ------------------------------------------------------------

langgraph_auto_mask = (
    langgraph_test_df[
        "auto_closed"
    ]
    ==
    True
)


langgraph_auto_closed = int(
    langgraph_auto_mask.sum()
)


langgraph_human_review = int(
    len(langgraph_test_df)
    -
    langgraph_auto_closed
)


langgraph_coverage = float(
    langgraph_auto_closed
    /
    len(langgraph_test_df)
)


langgraph_review_rate = float(
    langgraph_human_review
    /
    len(langgraph_test_df)
)


langgraph_auto_correct = (
    langgraph_test_df.loc[
        langgraph_auto_mask,
        "correct"
    ]
    .astype(bool)
)


langgraph_automatic_accuracy = float(
    langgraph_auto_correct.mean()
)


langgraph_automatic_errors = int(
    (
        ~langgraph_auto_correct
    )
    .sum()
)


extra_closed_vs_baseline = int(
    langgraph_auto_closed
    -
    baseline_auto_closed
)


route_counts = (
    langgraph_test_df[
        "route"
    ]
    .value_counts()
    .to_dict()
)


final_test_langgraph = {

    "system":
        "LangGraph selective consensus",

    "distilbert_threshold":
        SELECTED_THRESHOLD,

    "retrieval_similarity_threshold":
        MIN_RETRIEVAL_SIMILARITY,

    "n":
        int(
            len(langgraph_test_df)
        ),

    "auto_closed":
        langgraph_auto_closed,

    "coverage":
        langgraph_coverage,

    "human_review":
        langgraph_human_review,

    "review_rate":
        langgraph_review_rate,

    "automatic_accuracy":
        langgraph_automatic_accuracy,

    "automatic_errors":
        langgraph_automatic_errors,

    "extra_closed_vs_baseline":
        extra_closed_vs_baseline,

    "routes":
        {
            str(k): int(v)
            for k, v
            in route_counts.items()
        }
}


# ------------------------------------------------------------
# 16. Финальное сравнение
# ------------------------------------------------------------

comparison_df = pd.DataFrame(
    [
        {
            "system":
                "DistilBERT + reject",

            "coverage":
                final_test_baseline[
                    "coverage"
                ],

            "automatic_accuracy":
                final_test_baseline[
                    "automatic_accuracy"
                ],

            "review_rate":
                final_test_baseline[
                    "review_rate"
                ],

            "automatic_errors":
                final_test_baseline[
                    "automatic_errors"
                ],

            "auto_closed":
                final_test_baseline[
                    "auto_closed"
                ]
        },

        {
            "system":
                "Frozen LangGraph",

            "coverage":
                final_test_langgraph[
                    "coverage"
                ],

            "automatic_accuracy":
                final_test_langgraph[
                    "automatic_accuracy"
                ],

            "review_rate":
                final_test_langgraph[
                    "review_rate"
                ],

            "automatic_errors":
                final_test_langgraph[
                    "automatic_errors"
                ],

            "auto_closed":
                final_test_langgraph[
                    "auto_closed"
                ]
        }
    ]
)


print()
print("=" * 70)
print("FINAL BASELINE VS LANGGRAPH — TEST")
print("=" * 70)

display(
    comparison_df
)


print()
print(
    "Extra closed vs baseline:",
    extra_closed_vs_baseline
)

print(
    "LangGraph routes:",
    route_counts
)


# ------------------------------------------------------------
# 17. Итоговый test result
# ------------------------------------------------------------

final_test_metrics = {

    "evaluation_policy":
        (
            "single final test evaluation "
            "after validation decisions were frozen"
        ),

    "models": {

        "distilbert_accuracy":
            distilbert_test_accuracy,

        "distilbert_macro_f1":
            distilbert_test_macro_f1,

        "distilbert_weighted_f1":
            distilbert_test_weighted_f1,

        "retrieval_hit@1":
            retrieval_hit_at_1,

        "retrieval_hit@3":
            retrieval_hit_at_3,

        "retrieval_hit@5":
            retrieval_hit_at_5
    },

    "reject_baseline":
        final_test_baseline,

    "langgraph":
        final_test_langgraph
}


# ------------------------------------------------------------
# 18. Сохраняем test predictions
# ------------------------------------------------------------

FINAL_TEST_PREDICTIONS_PATH = (
    RESULTS_DIR
    / "final_test_predictions.csv"
)


final_test_df.to_csv(
    FINAL_TEST_PREDICTIONS_PATH,
    index=False
)


# ------------------------------------------------------------
# 19. Сохраняем LangGraph predictions
# ------------------------------------------------------------

FINAL_LANGGRAPH_TEST_PATH = (
    RESULTS_DIR
    / "final_langgraph_test_predictions.csv"
)


langgraph_test_df.to_csv(
    FINAL_LANGGRAPH_TEST_PATH,
    index=False
)


# ------------------------------------------------------------
# 20. Сохраняем финальные test metrics
# ------------------------------------------------------------

FINAL_TEST_METRICS_PATH = (
    RESULTS_DIR
    / "final_test_metrics.json"
)


with open(
    FINAL_TEST_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_test_metrics,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 21. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 51 COMPLETE")
print("=" * 70)

print(
    "FINAL TEST HAS BEEN EVALUATED."
)

print()
print(
    "From this point:"
)

print(
    "- thresholds are NOT changed"
)

print(
    "- LangGraph rules are NOT changed"
)

print(
    "- retrieval settings are NOT changed"
)

print()

print(
    "Saved metrics:"
)

print(
    FINAL_TEST_METRICS_PATH
)

print()

print(
    "Saved predictions:"
)

print(
    FINAL_TEST_PREDICTIONS_PATH
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 52.
# FINAL PROJECT SUMMARY
#
# Собираем окончательные результаты проекта:
# - protocol
# - validation
# - frozen architecture
# - single final test
# - Document RAG API-generated QA evaluation
#
# После финального test НИЧЕГО не перенастраиваем.
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

LANGGRAPH_RESULTS_DIR = (
    PROJECT_DIR
    / "langgraph"
    / "results"
)

DOC_RAG_RESULTS_DIR = (
    PROJECT_DIR
    / "document_rag"
    / "results"
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Загружаем datasets
# ------------------------------------------------------------

train_part = pd.read_parquet(
    DATA_DIR
    / "train_part.parquet"
)

val_part = pd.read_parquet(
    DATA_DIR
    / "val_part.parquet"
)

test_df = pd.read_parquet(
    DATA_DIR
    / "test.parquet"
)


# ------------------------------------------------------------
# 3. Загружаем финальный threshold
# ------------------------------------------------------------

THRESHOLD_PATH = (
    VALIDATION_DIR
    / "selected_threshold_final.json"
)


with open(
    THRESHOLD_PATH,
    "r",
    encoding="utf-8"
) as f:

    threshold_result = json.load(f)


SELECTED_THRESHOLD = float(
    threshold_result[
        "selected_threshold"
    ]
)


validation_baseline = (
    threshold_result[
        "reject_baseline"
    ]
)


# ------------------------------------------------------------
# 4. Загружаем финальную LangGraph architecture
# ------------------------------------------------------------

ARCHITECTURE_PATH = (
    LANGGRAPH_RESULTS_DIR
    / "final_langgraph_architecture.json"
)


with open(
    ARCHITECTURE_PATH,
    "r",
    encoding="utf-8"
) as f:

    validation_langgraph = json.load(f)


MIN_RETRIEVAL_SIMILARITY = float(
    validation_langgraph[
        "retrieval_similarity_threshold"
    ]
)


# ------------------------------------------------------------
# 5. Загружаем финальный test
# ------------------------------------------------------------

FINAL_TEST_METRICS_PATH = (
    RESULTS_DIR
    / "final_test_metrics.json"
)


with open(
    FINAL_TEST_METRICS_PATH,
    "r",
    encoding="utf-8"
) as f:

    final_test_metrics = json.load(f)


# ------------------------------------------------------------
# 6. Загружаем финальные validation predictions
# ------------------------------------------------------------

VAL_PREDICTIONS_PATH = (
    VALIDATION_DIR
    / "distilbert_final_val_predictions.csv"
)


val_predictions_df = pd.read_csv(
    VAL_PREDICTIONS_PATH
)


val_distilbert_accuracy = float(
    accuracy_score(
        val_predictions_df["label"],
        val_predictions_df["distilbert_pred"]
    )
)


val_distilbert_macro_f1 = float(
    f1_score(
        val_predictions_df["label"],
        val_predictions_df["distilbert_pred"],
        average="macro"
    )
)


# ------------------------------------------------------------
# 7. Проверяем semantic retrieval objects
# ------------------------------------------------------------

required_semantic_objects = [
    "semantic_model",
    "semantic_corpus_df",
    "semantic_corpus_embeddings"
]


missing_semantic = [
    name
    for name in required_semantic_objects
    if name not in globals()
]


if missing_semantic:

    raise RuntimeError(
        f"Не найдены semantic retrieval объекты: "
        f"{missing_semantic}. "
        f"Выполни ячейку 50."
    )


# ------------------------------------------------------------
# 8. Пересчитываем VALIDATION retrieval metrics
#
# Это НЕ настройка.
# Архитектура уже заморожена.
# Просто сохраняем финальные validation metrics
# для итогового отчёта.
# ------------------------------------------------------------

print("=" * 70)
print("FINAL VALIDATION RETRIEVAL METRICS")
print("=" * 70)


val_texts = (
    val_part[
        "text"
    ]
    .astype(str)
    .tolist()
)


val_embeddings = (
    semantic_model.encode(
        val_texts,
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
)


val_embeddings = np.asarray(
    val_embeddings,
    dtype=np.float32
)


val_retrieval_top_labels = []


RETRIEVAL_BATCH_SIZE = 256


corpus_labels = (
    semantic_corpus_df[
        "label"
    ]
    .astype(int)
    .to_numpy()
)


for start in range(
    0,
    len(val_embeddings),
    RETRIEVAL_BATCH_SIZE
):

    batch_embeddings = val_embeddings[
        start:
        start + RETRIEVAL_BATCH_SIZE
    ]


    similarities = (
        batch_embeddings
        @
        semantic_corpus_embeddings.T
    )


    top_indices_unsorted = np.argpartition(
        -similarities,
        kth=4,
        axis=1
    )[:, :5]


    top_scores_unsorted = np.take_along_axis(
        similarities,
        top_indices_unsorted,
        axis=1
    )


    ordering = np.argsort(
        -top_scores_unsorted,
        axis=1
    )


    top_indices = np.take_along_axis(
        top_indices_unsorted,
        ordering,
        axis=1
    )


    batch_labels = corpus_labels[
        top_indices
    ]


    val_retrieval_top_labels.append(
        batch_labels
    )


val_retrieval_top_labels = np.vstack(
    val_retrieval_top_labels
)


val_true_labels = (
    val_part[
        "label"
    ]
    .astype(int)
    .to_numpy()
)


val_hit_at_1 = float(
    np.mean(
        val_retrieval_top_labels[:, 0]
        ==
        val_true_labels
    )
)


val_hit_at_3 = float(
    np.mean(
        [
            true_label in predictions[:3]

            for true_label, predictions
            in zip(
                val_true_labels,
                val_retrieval_top_labels
            )
        ]
    )
)


val_hit_at_5 = float(
    np.mean(
        [
            true_label in predictions[:5]

            for true_label, predictions
            in zip(
                val_true_labels,
                val_retrieval_top_labels
            )
        ]
    )
)


print(
    f"Hit@1: {val_hit_at_1:.6f}"
)

print(
    f"Hit@3: {val_hit_at_3:.6f}"
)

print(
    f"Hit@5: {val_hit_at_5:.6f}"
)


# ------------------------------------------------------------
# 9. Document RAG retrieval metrics
# ------------------------------------------------------------

DOC_RAG_RETRIEVAL_PATH = (
    DOC_RAG_RESULTS_DIR
    / "api_generated_qa_rag_metrics.json"
)


if not DOC_RAG_RETRIEVAL_PATH.exists():

    raise RuntimeError(
        "Document RAG retrieval metrics не найдены."
    )


with open(
    DOC_RAG_RETRIEVAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    document_rag_retrieval = json.load(f)


# ------------------------------------------------------------
# 10. Document RAG answer/judge metrics
# ------------------------------------------------------------

DOC_RAG_FINAL_PATH = (
    DOC_RAG_RESULTS_DIR
    / "api_generated_qa_final_metrics.json"
)


if not DOC_RAG_FINAL_PATH.exists():

    raise RuntimeError(
        "Document RAG final judge metrics не найдены."
    )


with open(
    DOC_RAG_FINAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    document_rag_final = json.load(f)


# ------------------------------------------------------------
# 11. TF-IDF validation metrics
#
# Берём из предыдущего summary, если он существует.
# Это независимый baseline, который уже был рассчитан
# на validation.
# ------------------------------------------------------------

OLD_SUMMARY_PATH = (
    RESULTS_DIR
    / "FINAL_CORRECTED_SUMMARY.json"
)


tfidf_validation = None


if OLD_SUMMARY_PATH.exists():

    try:

        with open(
            OLD_SUMMARY_PATH,
            "r",
            encoding="utf-8"
        ) as f:

            old_summary = json.load(f)


        tfidf_validation = (
            old_summary
            .get(
                "validation",
                {}
            )
            .get(
                "tfidf"
            )
        )

    except Exception:

        tfidf_validation = None


# ------------------------------------------------------------
# 12. Формируем главный итоговый summary
# ------------------------------------------------------------

FINAL_PROJECT_SUMMARY = {

    "protocol": {

        "seed":
            42,

        "train_part_size":
            int(
                len(train_part)
            ),

        "val_part_size":
            int(
                len(val_part)
            ),

        "test_size":
            int(
                len(test_df)
            ),

        "n_classes":
            int(
                train_part[
                    "label"
                ]
                .nunique()
            ),

        "policy": {

            "train_part":
                (
                    "fit models and build "
                    "retrieval corpus"
                ),

            "val_part":
                (
                    "threshold selection, "
                    "diagnostic analysis and "
                    "architecture selection"
                ),

            "test":
                (
                    "single final evaluation "
                    "after all decisions were frozen"
                )
        }
    },


    "final_configuration": {

        "distilbert_model":
            "distilbert-base-uncased",

        "distilbert_epochs":
            6,

        "best_validation_checkpoint":
            "checkpoint-1330",

        "selected_threshold_from_validation":
            SELECTED_THRESHOLD,

        "semantic_model":
            (
                "sentence-transformers/"
                "all-MiniLM-L6-v2"
            ),

        "retrieval_similarity_gate":
            MIN_RETRIEVAL_SIMILARITY,

        "langgraph_policy":
            (
                "high-confidence DistilBERT auto-close; "
                "low-confidence auto-close only when "
                "DistilBERT and retrieval agree and "
                "retrieval similarity passes frozen gate; "
                "otherwise human review"
            )
    },


    "validation": {

        "tfidf":
            tfidf_validation,

        "distilbert": {

            "accuracy":
                val_distilbert_accuracy,

            "macro_f1":
                val_distilbert_macro_f1
        },

        "semantic_retrieval": {

            "hit@1":
                val_hit_at_1,

            "hit@3":
                val_hit_at_3,

            "hit@5":
                val_hit_at_5
        },

        "reject_baseline":
            validation_baseline,

        "langgraph":
            validation_langgraph
    },


    "final_test": {

        "evaluation_policy":
            (
                "single final evaluation; "
                "no parameters changed afterwards"
            ),

        **final_test_metrics
    },


    "document_rag_api_qa": {

        "evaluation_set": {

            "generation":
                (
                    "30 questions generated through "
                    "Groq API from randomly selected "
                    "document chunks"
                ),

            "generator_model":
                "openai/gpt-oss-120b",

            "n_examples":
                int(
                    document_rag_final[
                        "n_examples"
                    ]
                ),

            "used_for_rag_tuning":
                False
        },

        "retrieval": {

            "source_accuracy":
                float(
                    document_rag_retrieval[
                        "source_accuracy"
                    ]
                ),

            "chunk_hit_at_1":
                float(
                    document_rag_retrieval[
                        "chunk_hit_at_1"
                    ]
                ),

            "chunk_hit_at_3":
                float(
                    document_rag_retrieval[
                        "chunk_hit_at_3"
                    ]
                ),

            "runtime_errors":
                int(
                    document_rag_retrieval[
                        "runtime_errors"
                    ]
                )
        },

        "answer_quality": {

            "answer_accuracy_llm_judge":
                float(
                    document_rag_final[
                        "answer_accuracy_llm_judge"
                    ]
                ),

            "groundedness_rate":
                float(
                    document_rag_final[
                        "groundedness_rate"
                    ]
                ),

            "mean_judge_score":
                float(
                    document_rag_final[
                        "mean_judge_score"
                    ]
                ),

            "judge_success_rate":
                float(
                    document_rag_final[
                        "judge_success_rate"
                    ]
                ),

            "overall_rag_success":
                float(
                    document_rag_final[
                        "overall_rag_success"
                    ]
                )
        }
    },


    "conclusions": {

        "distilbert":
            (
                "Final 6-epoch model substantially "
                "outperformed the 1-epoch debug model."
            ),

        "selective_baseline":
            (
                "Validation-selected confidence threshold "
                "preserved at least 95% automatic accuracy "
                "while maintaining high coverage."
            ),

        "langgraph":
            (
                "The tested low-confidence consensus route "
                "did not provide additional safe coverage "
                "under the >=95% automatic-accuracy "
                "constraint. The validation-optimal frozen "
                "LangGraph therefore reduced to the "
                "DistilBERT reject baseline."
            ),

        "document_rag":
            (
                "Document RAG was evaluated on an "
                "API-generated held-out QA benchmark "
                "rather than on manually tuned questions."
            )
    }
}


# ------------------------------------------------------------
# 13. Убираем None TF-IDF, если старый summary недоступен
# ------------------------------------------------------------

if (
    FINAL_PROJECT_SUMMARY[
        "validation"
    ][
        "tfidf"
    ]
    is None
):

    del (
        FINAL_PROJECT_SUMMARY[
            "validation"
        ][
            "tfidf"
        ]
    )


# ------------------------------------------------------------
# 14. Сохраняем окончательный JSON
# ------------------------------------------------------------

FINAL_SUMMARY_PATH = (
    RESULTS_DIR
    / "FINAL_CORRECTED_SUMMARY.json"
)


with open(
    FINAL_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_PROJECT_SUMMARY,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 15. Краткий итог на экран
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL PROJECT RESULTS")
print("=" * 70)

print()
print("VALIDATION")
print("-" * 70)

print(
    "DistilBERT accuracy:",
    f"{val_distilbert_accuracy:.4f}"
)

print(
    "DistilBERT macro F1:",
    f"{val_distilbert_macro_f1:.4f}"
)

print(
    "Threshold:",
    SELECTED_THRESHOLD
)

print(
    "Baseline coverage:",
    f"{validation_baseline['coverage']:.4f}"
)

print(
    "Baseline auto accuracy:",
    f"{validation_baseline['automatic_accuracy']:.4f}"
)

print(
    "LangGraph coverage:",
    f"{validation_langgraph['coverage']:.4f}"
)

print(
    "LangGraph auto accuracy:",
    f"{validation_langgraph['automatic_accuracy']:.4f}"
)


print()
print("FINAL TEST")
print("-" * 70)

print(
    "DistilBERT accuracy:",
    f"{final_test_metrics['models']['distilbert_accuracy']:.4f}"
)

print(
    "DistilBERT macro F1:",
    f"{final_test_metrics['models']['distilbert_macro_f1']:.4f}"
)

print(
    "Retrieval Hit@1:",
    f"{final_test_metrics['models']['retrieval_hit@1']:.4f}"
)

print(
    "Retrieval Hit@3:",
    f"{final_test_metrics['models']['retrieval_hit@3']:.4f}"
)

print(
    "Retrieval Hit@5:",
    f"{final_test_metrics['models']['retrieval_hit@5']:.4f}"
)

print(
    "Baseline coverage:",
    f"{final_test_metrics['reject_baseline']['coverage']:.4f}"
)

print(
    "Baseline auto accuracy:",
    f"{final_test_metrics['reject_baseline']['automatic_accuracy']:.4f}"
)

print(
    "LangGraph coverage:",
    f"{final_test_metrics['langgraph']['coverage']:.4f}"
)

print(
    "LangGraph auto accuracy:",
    f"{final_test_metrics['langgraph']['automatic_accuracy']:.4f}"
)


print()
print("DOCUMENT RAG")
print("-" * 70)

print(
    "QA examples:",
    document_rag_final[
        "n_examples"
    ]
)

print(
    "Source accuracy:",
    f"{document_rag_retrieval['source_accuracy']:.4f}"
)

print(
    "Chunk Hit@1:",
    f"{document_rag_retrieval['chunk_hit_at_1']:.4f}"
)

print(
    "Chunk Hit@3:",
    f"{document_rag_retrieval['chunk_hit_at_3']:.4f}"
)

print(
    "Answer accuracy:",
    f"{document_rag_final['answer_accuracy_llm_judge']:.4f}"
)

print(
    "Groundedness:",
    f"{document_rag_final['groundedness_rate']:.4f}"
)

print(
    "Mean judge score:",
    f"{document_rag_final['mean_judge_score']:.4f}"
)

print(
    "Overall RAG success:",
    f"{document_rag_final['overall_rag_success']:.4f}"
)


print()
print("=" * 70)
print("CELL 52 COMPLETE")
print("=" * 70)

print(
    "Final summary saved:"
)

print(
    FINAL_SUMMARY_PATH
)

print()
print(
    "Test results are now FINAL and frozen."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 53.
# FINAL PROJECT BACKUP TO GOOGLE DRIVE
#
# Сохраняем всю финальную версию проекта после:
# - 6-epoch DistilBERT
# - validation threshold selection
# - frozen LangGraph architecture
# - single final test
# - Document RAG evaluation
# ============================================================

import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

LOCAL_PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/ml_customer_classification_corrected"
)


# ------------------------------------------------------------
# 2. Проверяем проект
# ------------------------------------------------------------

if not LOCAL_PROJECT_DIR.exists():

    raise RuntimeError(
        f"Локальный проект не найден: "
        f"{LOCAL_PROJECT_DIR}"
    )


if not Path(
    "/content/drive/MyDrive"
).exists():

    raise RuntimeError(
        "Google Drive не подключён."
    )


print("=" * 70)
print("FINAL PROJECT BACKUP")
print("=" * 70)

print(
    "Source:",
    LOCAL_PROJECT_DIR
)

print(
    "Destination:",
    DRIVE_PROJECT_DIR
)


# ------------------------------------------------------------
# 3. Копируем проект
# ------------------------------------------------------------

print()
print(
    "Copying final project to Drive..."
)


DRIVE_PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


shutil.copytree(
    LOCAL_PROJECT_DIR,
    DRIVE_PROJECT_DIR,
    dirs_exist_ok=True
)


# ------------------------------------------------------------
# 4. Список критически важных финальных файлов
# ------------------------------------------------------------

important_relative_paths = [

    # DATA
    "data/train_part.parquet",
    "data/val_part.parquet",
    "data/test.parquet",
    "data/label_mapping.json",
    "data/label_mapping_final.json",

    # FINAL DISTILBERT
    "models/distilbert_final_6epoch/config.json",
    "models/distilbert_final_6epoch/model.safetensors",

    # VALIDATION
    "validation/distilbert_final_val_predictions.csv",
    "validation/distilbert_final_threshold_sweep.csv",
    "validation/selected_threshold_final.json",

    # LANGGRAPH
    "langgraph/results/final_langgraph_architecture.json",
    "langgraph/results/low_confidence_consensus_validation.csv",
    "langgraph/results/retrieval_similarity_threshold_sweep.csv",

    # FINAL TEST
    "results/final_test_metrics.json",
    "results/final_test_predictions.csv",
    "results/final_langgraph_test_predictions.csv",

    # FINAL SUMMARY
    "results/FINAL_CORRECTED_SUMMARY.json",

    # DOCUMENT RAG QA
    "document_rag/data/api_generated_qa.csv",

    # DOCUMENT RAG RETRIEVAL
    "document_rag/results/api_generated_qa_rag_evaluation.csv",
    "document_rag/results/api_generated_qa_rag_metrics.json",

    # DOCUMENT RAG JUDGE
    "document_rag/results/api_generated_qa_final_evaluation.csv",
    "document_rag/results/api_generated_qa_final_metrics.json"
]


# ------------------------------------------------------------
# 5. Проверяем критические файлы на Drive
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL FILE CHECK")
print("=" * 70)


missing_files = []


for relative_path in important_relative_paths:

    local_path = (
        LOCAL_PROJECT_DIR
        / relative_path
    )

    drive_path = (
        DRIVE_PROJECT_DIR
        / relative_path
    )


    local_ok = (
        local_path.exists()
    )

    drive_ok = (
        drive_path.exists()
    )


    if local_ok and drive_ok:

        status = "OK"

    else:

        status = "MISSING"

        missing_files.append(
            relative_path
        )


    print(
        f"{status:8} | "
        f"{relative_path}"
    )


# ------------------------------------------------------------
# 6. Размер проекта
# ------------------------------------------------------------

def get_folder_size(
    folder
):

    total = 0

    for path in folder.rglob("*"):

        if path.is_file():

            try:

                total += (
                    path.stat().st_size
                )

            except OSError:

                pass

    return total


local_size = get_folder_size(
    LOCAL_PROJECT_DIR
)

drive_size = get_folder_size(
    DRIVE_PROJECT_DIR
)


print()
print("=" * 70)
print("SIZE CHECK")
print("=" * 70)

print(
    "Local size:",
    f"{local_size / 1024**2:.2f} MB"
)

print(
    "Drive size:",
    f"{drive_size / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# 7. Проверяем финальный summary
# ------------------------------------------------------------

FINAL_SUMMARY_PATH = (
    DRIVE_PROJECT_DIR
    / "results"
    / "FINAL_CORRECTED_SUMMARY.json"
)


if FINAL_SUMMARY_PATH.exists():

    import json

    with open(
        FINAL_SUMMARY_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        summary = json.load(f)


    print()
    print("=" * 70)
    print("FINAL SUMMARY CHECK")
    print("=" * 70)


    print(
        "DistilBERT test accuracy:",
        summary[
            "final_test"
        ][
            "models"
        ][
            "distilbert_accuracy"
        ]
    )


    print(
        "DistilBERT test macro F1:",
        summary[
            "final_test"
        ][
            "models"
        ][
            "distilbert_macro_f1"
        ]
    )


    print(
        "Final threshold:",
        summary[
            "final_configuration"
        ][
            "selected_threshold_from_validation"
        ]
    )


    print(
        "Document RAG answer accuracy:",
        summary[
            "document_rag_api_qa"
        ][
            "answer_quality"
        ][
            "answer_accuracy_llm_judge"
        ]
    )


    print(
        "Document RAG overall success:",
        summary[
            "document_rag_api_qa"
        ][
            "answer_quality"
        ][
            "overall_rag_success"
        ]
    )


# ------------------------------------------------------------
# 8. Итог
# ------------------------------------------------------------

print()
print("=" * 70)

if len(
    missing_files
) == 0:

    print(
        "FINAL BACKUP COMPLETE"
    )

    print()
    print(
        "Финальная версия проекта полностью "
        "сохранена на Google Drive."
    )

    print()
    print(
        "Location:"
    )

    print(
        DRIVE_PROJECT_DIR
    )

else:

    print(
        "FINAL BACKUP PARTIALLY COMPLETE"
    )

    print()
    print(
        "Не найдены следующие файлы:"
    )

    for path in missing_files:

        print(
            "-",
            path
        )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 54.
# FINAL REPORT TABLES + DEFENSE CONCLUSIONS
#
# ============================================================

import json
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    PROJECT_DIR
    / "validation"
)

LANGGRAPH_RESULTS_DIR = (
    PROJECT_DIR
    / "langgraph"
    / "results"
)

DOC_RAG_RESULTS_DIR = (
    PROJECT_DIR
    / "document_rag"
    / "results"
)


# ------------------------------------------------------------
# 2. Загружаем финальный summary
# ------------------------------------------------------------

FINAL_SUMMARY_PATH = (
    RESULTS_DIR
    / "FINAL_CORRECTED_SUMMARY.json"
)


if not FINAL_SUMMARY_PATH.exists():

    raise RuntimeError(
        "FINAL_CORRECTED_SUMMARY.json не найден. "
        "Сначала выполни ячейку 52."
    )


with open(
    FINAL_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    summary = json.load(f)


# ------------------------------------------------------------
# 3. Основные значения
# ------------------------------------------------------------

validation = summary[
    "validation"
]

final_test = summary[
    "final_test"
]

document_rag = summary[
    "document_rag_api_qa"
]

final_config = summary[
    "final_configuration"
]


# ------------------------------------------------------------
# 4. Таблица 1.
# Classification + retrieval
# ------------------------------------------------------------

classification_table = pd.DataFrame(
    [
        {
            "Metric":
                "DistilBERT Accuracy",

            "Validation":
                validation[
                    "distilbert"
                ][
                    "accuracy"
                ],

            "Final Test":
                final_test[
                    "models"
                ][
                    "distilbert_accuracy"
                ]
        },

        {
            "Metric":
                "DistilBERT Macro F1",

            "Validation":
                validation[
                    "distilbert"
                ][
                    "macro_f1"
                ],

            "Final Test":
                final_test[
                    "models"
                ][
                    "distilbert_macro_f1"
                ]
        },

        {
            "Metric":
                "Semantic Retrieval Hit@1",

            "Validation":
                validation[
                    "semantic_retrieval"
                ][
                    "hit@1"
                ],

            "Final Test":
                final_test[
                    "models"
                ][
                    "retrieval_hit@1"
                ]
        },

        {
            "Metric":
                "Semantic Retrieval Hit@3",

            "Validation":
                validation[
                    "semantic_retrieval"
                ][
                    "hit@3"
                ],

            "Final Test":
                final_test[
                    "models"
                ][
                    "retrieval_hit@3"
                ]
        },

        {
            "Metric":
                "Semantic Retrieval Hit@5",

            "Validation":
                validation[
                    "semantic_retrieval"
                ][
                    "hit@5"
                ],

            "Final Test":
                final_test[
                    "models"
                ][
                    "retrieval_hit@5"
                ]
        }
    ]
)


# Переводим в проценты
for column in [
    "Validation",
    "Final Test"
]:

    classification_table[
        column
    ] = (
        classification_table[
            column
        ]
        * 100
    ).round(2)


print("=" * 70)
print("TABLE 1. CLASSIFICATION AND RETRIEVAL")
print("=" * 70)

display(
    classification_table
)


# ------------------------------------------------------------
# 5. Таблица 2.
# Baseline vs LangGraph
# ------------------------------------------------------------

baseline_test = (
    final_test[
        "reject_baseline"
    ]
)

langgraph_test = (
    final_test[
        "langgraph"
    ]
)


system_comparison_table = pd.DataFrame(
    [
        {
            "System":
                "DistilBERT + reject threshold",

            "Threshold":
                baseline_test[
                    "threshold"
                ],

            "Coverage, %":
                baseline_test[
                    "coverage"
                ]
                * 100,

            "Automatic accuracy, %":
                baseline_test[
                    "automatic_accuracy"
                ]
                * 100,

            "Human review, %":
                baseline_test[
                    "review_rate"
                ]
                * 100,

            "Automatic errors":
                baseline_test[
                    "automatic_errors"
                ],

            "Auto-closed":
                baseline_test[
                    "auto_closed"
                ]
        },

        {
            "System":
                "Frozen LangGraph",

            "Threshold":
                langgraph_test[
                    "distilbert_threshold"
                ],

            "Coverage, %":
                langgraph_test[
                    "coverage"
                ]
                * 100,

            "Automatic accuracy, %":
                langgraph_test[
                    "automatic_accuracy"
                ]
                * 100,

            "Human review, %":
                langgraph_test[
                    "review_rate"
                ]
                * 100,

            "Automatic errors":
                langgraph_test[
                    "automatic_errors"
                ],

            "Auto-closed":
                langgraph_test[
                    "auto_closed"
                ]
        }
    ]
)


for column in [
    "Coverage, %",
    "Automatic accuracy, %",
    "Human review, %"
]:

    system_comparison_table[
        column
    ] = (
        system_comparison_table[
            column
        ]
        .round(2)
    )


print()
print("=" * 70)
print("TABLE 2. BASELINE VS LANGGRAPH")
print("=" * 70)

display(
    system_comparison_table
)


# ------------------------------------------------------------
# 6. Таблица 3.
# Document RAG
# ------------------------------------------------------------

doc_retrieval = (
    document_rag[
        "retrieval"
    ]
)

doc_answer = (
    document_rag[
        "answer_quality"
    ]
)


document_rag_table = pd.DataFrame(
    [
        {
            "Metric":
                "Source accuracy",

            "Value":
                doc_retrieval[
                    "source_accuracy"
                ]
                * 100,

            "Unit":
                "%"
        },

        {
            "Metric":
                "Chunk Hit@1",

            "Value":
                doc_retrieval[
                    "chunk_hit_at_1"
                ]
                * 100,

            "Unit":
                "%"
        },

        {
            "Metric":
                "Chunk Hit@3",

            "Value":
                doc_retrieval[
                    "chunk_hit_at_3"
                ]
                * 100,

            "Unit":
                "%"
        },

        {
            "Metric":
                "Answer accuracy (LLM judge)",

            "Value":
                doc_answer[
                    "answer_accuracy_llm_judge"
                ]
                * 100,

            "Unit":
                "%"
        },

        {
            "Metric":
                "Groundedness",

            "Value":
                doc_answer[
                    "groundedness_rate"
                ]
                * 100,

            "Unit":
                "%"
        },

        {
            "Metric":
                "Mean judge score",

            "Value":
                doc_answer[
                    "mean_judge_score"
                ],

            "Unit":
                "/ 4"
        },

        {
            "Metric":
                "Overall RAG success",

            "Value":
                doc_answer[
                    "overall_rag_success"
                ]
                * 100,

            "Unit":
                "%"
        }
    ]
)


document_rag_table[
    "Value"
] = (
    document_rag_table[
        "Value"
    ]
    .round(2)
)


print()
print("=" * 70)
print("TABLE 3. DOCUMENT RAG")
print("=" * 70)

display(
    document_rag_table
)


# ------------------------------------------------------------
# 7. Методологический протокол
# ------------------------------------------------------------

protocol_table = pd.DataFrame(
    [
        {
            "Stage":
                "Training",

            "Dataset":
                "train_part",

            "Size":
                summary[
                    "protocol"
                ][
                    "train_part_size"
                ],

            "Purpose":
                (
                    "Обучение моделей и построение "
                    "retrieval-корпуса"
                )
        },

        {
            "Stage":
                "Validation",

            "Dataset":
                "val_part",

            "Size":
                summary[
                    "protocol"
                ][
                    "val_part_size"
                ],

            "Purpose":
                (
                    "Выбор threshold и архитектуры"
                )
        },

        {
            "Stage":
                "Final test",

            "Dataset":
                "test",

            "Size":
                summary[
                    "protocol"
                ][
                    "test_size"
                ],

            "Purpose":
                (
                    "Однократная финальная оценка "
                    "после заморозки решений"
                )
        }
    ]
)


print()
print("=" * 70)
print("TABLE 4. EVALUATION PROTOCOL")
print("=" * 70)

display(
    protocol_table
)


# ------------------------------------------------------------
# 8. Ключевые выводы для защиты
# ------------------------------------------------------------

print()
print("=" * 70)
print("DEFENSE CONCLUSIONS")
print("=" * 70)

print()

print(
    "1. DistilBERT:"
)

print(
    "   Финальная модель после 6 эпох показала "
    f"Accuracy = "
    f"{final_test['models']['distilbert_accuracy'] * 100:.2f}% "
    f"и Macro F1 = "
    f"{final_test['models']['distilbert_macro_f1'] * 100:.2f}% "
    "на отложенном test."
)

print()

print(
    "2. Selective classification:"
)

print(
    "   Порог confidence = "
    f"{final_config['selected_threshold_from_validation']:.2f} "
    "был выбран только на validation."
)

print(
    "   На test система автоматически закрывает "
    f"{baseline_test['coverage'] * 100:.2f}% запросов "
    "с точностью "
    f"{baseline_test['automatic_accuracy'] * 100:.2f}%."
)

print(
    "   На ручную проверку направляется "
    f"{baseline_test['review_rate'] * 100:.2f}% запросов."
)

print()

print(
    "3. Semantic retrieval:"
)

print(
    "   На test достигнуты "
    f"Hit@1 = "
    f"{final_test['models']['retrieval_hit@1'] * 100:.2f}%, "
    f"Hit@3 = "
    f"{final_test['models']['retrieval_hit@3'] * 100:.2f}% "
    f"и Hit@5 = "
    f"{final_test['models']['retrieval_hit@5'] * 100:.2f}%."
)

print()

print(
    "4. LangGraph:"
)

print(
    "   На validation была проверена схема "
    "дополнительного закрытия low-confidence запросов "
    "по consensus DistilBERT и semantic retrieval."
)

print(
    "   Однако при требовании automatic accuracy >= 95% "
    "эта ветка не дала безопасного прироста coverage."
)

print(
    "   Поэтому финальная frozen-конфигурация LangGraph "
    "по метрикам совпадает с базовым "
    "DistilBERT + reject threshold."
)

print()

print(
    "5. Document RAG:"
)

print(
    "   Оценка проведена на "
    f"{document_rag['evaluation_set']['n_examples']} "
    "вопросах, сгенерированных через API "
    "по случайно выбранным фрагментам документов."
)

print(
    "   Эти вопросы не использовались для настройки RAG."
)

print(
    "   Source accuracy = "
    f"{doc_retrieval['source_accuracy'] * 100:.2f}%, "
    "Answer accuracy = "
    f"{doc_answer['answer_accuracy_llm_judge'] * 100:.2f}%, "
    "Overall success = "
    f"{doc_answer['overall_rag_success'] * 100:.2f}%."
)

print()

print(
    "6. Методология:"
)

print(
    "   train_part использовался для обучения, "
    "val_part — для выбора решений, "
    "а test — только для однократной финальной оценки."
)

print(
    "   После просмотра test-результатов "
    "threshold и архитектура не менялись."
)


# ------------------------------------------------------------
# 9. Сохраняем таблицы
# ------------------------------------------------------------

classification_table.to_csv(
    RESULTS_DIR
    / "report_classification_metrics.csv",
    index=False
)


system_comparison_table.to_csv(
    RESULTS_DIR
    / "report_baseline_vs_langgraph.csv",
    index=False
)


document_rag_table.to_csv(
    RESULTS_DIR
    / "report_document_rag_metrics.csv",
    index=False
)


protocol_table.to_csv(
    RESULTS_DIR
    / "report_evaluation_protocol.csv",
    index=False
)


# ------------------------------------------------------------
# 10. Сохраняем текст выводов
# ------------------------------------------------------------

DEFENSE_TEXT_PATH = (
    RESULTS_DIR
    / "DEFENSE_CONCLUSIONS.txt"
)


defense_text = f"""
ИТОГОВЫЕ ВЫВОДЫ ПРОЕКТА

1. Классификация намерений

Финальная модель DistilBERT была обучена на train_part
в течение 6 эпох с выбором лучшего checkpoint
по Macro F1 на validation.

На финальном test:
Accuracy = {final_test['models']['distilbert_accuracy'] * 100:.2f}%
Macro F1 = {final_test['models']['distilbert_macro_f1'] * 100:.2f}%


2. Selective classification

Порог confidence {final_config['selected_threshold_from_validation']:.2f}
был выбран исключительно на validation.

На test:
Coverage = {baseline_test['coverage'] * 100:.2f}%
Automatic accuracy = {baseline_test['automatic_accuracy'] * 100:.2f}%
Human review rate = {baseline_test['review_rate'] * 100:.2f}%


3. Semantic retrieval

На финальном test:
Hit@1 = {final_test['models']['retrieval_hit@1'] * 100:.2f}%
Hit@3 = {final_test['models']['retrieval_hit@3'] * 100:.2f}%
Hit@5 = {final_test['models']['retrieval_hit@5'] * 100:.2f}%


4. LangGraph

На validation была проверена схема selective consensus:
для low-confidence запросов semantic retrieval
сравнивался с предсказанием DistilBERT.

При ограничении automatic accuracy >= 95%
дополнительная consensus-ветка не обеспечила
безопасного прироста coverage.

Поэтому финальная frozen-конфигурация LangGraph
совпадает по метрикам с baseline
DistilBERT + reject threshold.

Это является экспериментальным результатом:
усложнение архитектуры не всегда повышает
качество системы при фиксированном SLA по точности.


5. Document RAG

Document RAG оценивался на независимом наборе
из {document_rag['evaluation_set']['n_examples']} QA-примеров,
сгенерированных через API из случайно выбранных chunks.

QA benchmark не использовался для настройки RAG.

Source accuracy = {doc_retrieval['source_accuracy'] * 100:.2f}%
Chunk Hit@1 = {doc_retrieval['chunk_hit_at_1'] * 100:.2f}%
Chunk Hit@3 = {doc_retrieval['chunk_hit_at_3'] * 100:.2f}%
Answer accuracy = {doc_answer['answer_accuracy_llm_judge'] * 100:.2f}%
Groundedness = {doc_answer['groundedness_rate'] * 100:.2f}%
Mean judge score = {doc_answer['mean_judge_score']:.2f}/4
Overall RAG success = {doc_answer['overall_rag_success'] * 100:.2f}%


6. Evaluation protocol

train_part:
обучение моделей и построение retrieval corpus.

val_part:
выбор confidence threshold и архитектуры.

test:
однократная финальная оценка после фиксации
всех параметров.

После просмотра результатов test
параметры системы не изменялись.
""".strip()


with open(
    DEFENSE_TEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        defense_text
    )


# ------------------------------------------------------------
# 11. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 54 COMPLETE")
print("=" * 70)

print(
    "Report tables saved to:"
)

print(
    RESULTS_DIR
)

print()

print(
    "Defense conclusions saved:"
)

print(
    DEFENSE_TEXT_PATH
)

print()

print(
    "Проект готов к оформлению итогового отчёта."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 55.
# SYNC FINAL REPORT FILES TO GOOGLE DRIVE
#
# Сохраняем обновлённые отчётные файлы после ячейки 54.
# ============================================================

import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

LOCAL_PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/ml_customer_classification_corrected"
)

LOCAL_RESULTS_DIR = (
    LOCAL_PROJECT_DIR
    / "results"
)

DRIVE_RESULTS_DIR = (
    DRIVE_PROJECT_DIR
    / "results"
)


# ------------------------------------------------------------
# 2. Проверяем Drive
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():

    raise RuntimeError(
        "Google Drive не подключён."
    )


if not LOCAL_RESULTS_DIR.exists():

    raise RuntimeError(
        "Локальная папка results не найдена."
    )


DRIVE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("FINAL REPORT SYNC TO GOOGLE DRIVE")
print("=" * 70)


# ------------------------------------------------------------
# 3. Файлы отчёта
# ------------------------------------------------------------

report_files = [

    "FINAL_CORRECTED_SUMMARY.json",

    "final_test_metrics.json",

    "final_test_predictions.csv",

    "final_langgraph_test_predictions.csv",

    "report_classification_metrics.csv",

    "report_baseline_vs_langgraph.csv",

    "report_document_rag_metrics.csv",

    "report_evaluation_protocol.csv",

    "DEFENSE_CONCLUSIONS.txt"
]


# ------------------------------------------------------------
# 4. Копируем
# ------------------------------------------------------------

copied = []
missing = []


for filename in report_files:

    source = (
        LOCAL_RESULTS_DIR
        / filename
    )

    destination = (
        DRIVE_RESULTS_DIR
        / filename
    )


    if source.exists():

        shutil.copy2(
            source,
            destination
        )

        copied.append(
            filename
        )

    else:

        missing.append(
            filename
        )


# ------------------------------------------------------------
# 5. Проверка
# ------------------------------------------------------------

print()
print("=" * 70)
print("REPORT FILE CHECK")
print("=" * 70)


for filename in report_files:

    local_path = (
        LOCAL_RESULTS_DIR
        / filename
    )

    drive_path = (
        DRIVE_RESULTS_DIR
        / filename
    )


    status = (
        "OK"
        if (
            local_path.exists()
            and
            drive_path.exists()
        )
        else "MISSING"
    )


    print(
        f"{status:8} | {filename}"
    )


# ------------------------------------------------------------
# 6. Итог
# ------------------------------------------------------------

print()
print("=" * 70)

if len(missing) == 0:

    print(
        "CELL 55 COMPLETE"
    )

    print()
    print(
        "Все финальные отчётные файлы "
        "синхронизированы с Google Drive."
    )

    print()
    print(
        "Drive results:"
    )

    print(
        DRIVE_RESULTS_DIR
    )

else:

    print(
        "CELL 55 PARTIALLY COMPLETE"
    )

    print()
    print(
        "Не найдены локальные файлы:"
    )

    for filename in missing:

        print(
            "-",
            filename
        )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 56.
# CREATE FINAL README.md
#
# Формируем финальный README проекта
# без длинной многострочной f-string конструкции.
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

README_PATH = (
    PROJECT_DIR
    / "README.md"
)

FINAL_SUMMARY_PATH = (
    RESULTS_DIR
    / "FINAL_CORRECTED_SUMMARY.json"
)


# ------------------------------------------------------------
# 2. Загружаем финальный summary
# ------------------------------------------------------------

if not FINAL_SUMMARY_PATH.exists():
    raise RuntimeError(
        "FINAL_CORRECTED_SUMMARY.json не найден. "
        "Сначала выполни ячейку 52."
    )


with open(
    FINAL_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:
    summary = json.load(f)


# ------------------------------------------------------------
# 3. Извлекаем данные
# ------------------------------------------------------------

protocol = summary["protocol"]

config = summary[
    "final_configuration"
]

validation = summary[
    "validation"
]

final_test = summary[
    "final_test"
]

document_rag = summary[
    "document_rag_api_qa"
]

baseline_test = final_test[
    "reject_baseline"
]

langgraph_test = final_test[
    "langgraph"
]

doc_retrieval = document_rag[
    "retrieval"
]

doc_answer = document_rag[
    "answer_quality"
]


# ------------------------------------------------------------
# 4. Форматирование значений
# ------------------------------------------------------------

val_acc = (
    validation["distilbert"]["accuracy"]
    * 100
)

val_f1 = (
    validation["distilbert"]["macro_f1"]
    * 100
)

test_acc = (
    final_test["models"]["distilbert_accuracy"]
    * 100
)

test_f1 = (
    final_test["models"]["distilbert_macro_f1"]
    * 100
)

test_weighted_f1 = (
    final_test["models"]["distilbert_weighted_f1"]
    * 100
)

val_hit1 = (
    validation["semantic_retrieval"]["hit@1"]
    * 100
)

val_hit3 = (
    validation["semantic_retrieval"]["hit@3"]
    * 100
)

val_hit5 = (
    validation["semantic_retrieval"]["hit@5"]
    * 100
)

test_hit1 = (
    final_test["models"]["retrieval_hit@1"]
    * 100
)

test_hit3 = (
    final_test["models"]["retrieval_hit@3"]
    * 100
)

test_hit5 = (
    final_test["models"]["retrieval_hit@5"]
    * 100
)

baseline_coverage = (
    baseline_test["coverage"]
    * 100
)

baseline_accuracy = (
    baseline_test["automatic_accuracy"]
    * 100
)

baseline_review = (
    baseline_test["review_rate"]
    * 100
)

langgraph_coverage = (
    langgraph_test["coverage"]
    * 100
)

langgraph_accuracy = (
    langgraph_test["automatic_accuracy"]
    * 100
)

langgraph_review = (
    langgraph_test["review_rate"]
    * 100
)

source_accuracy = (
    doc_retrieval["source_accuracy"]
    * 100
)

chunk_hit1 = (
    doc_retrieval["chunk_hit_at_1"]
    * 100
)

chunk_hit3 = (
    doc_retrieval["chunk_hit_at_3"]
    * 100
)

answer_accuracy = (
    doc_answer[
        "answer_accuracy_llm_judge"
    ]
    * 100
)

groundedness = (
    doc_answer["groundedness_rate"]
    * 100
)

judge_success = (
    doc_answer["judge_success_rate"]
    * 100
)

overall_rag = (
    doc_answer["overall_rag_success"]
    * 100
)


# ------------------------------------------------------------
# 5. Собираем README построчно
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


add("# ML Customer Support Assistant")
add()

add("## Описание проекта")
add()

add(
    "Прототип интеллектуального ассистента для обработки "
    "запросов сотрудников и клиентов."
)

add()

add("Система объединяет:")
add()
add("- классификацию намерений пользователя;")
add("- semantic retrieval;")
add("- selective classification с reject threshold;")
add("- маршрутизацию через LangGraph;")
add("- Document RAG для ответов по внутренним документам;")
add(
    "- передачу недостаточно уверенных запросов "
    "на ручную проверку."
)

add()
add(
    "Основная задача — не только получить высокую точность, "
    "но и ограничить автоматические ответы в случаях, "
    "когда уверенность системы недостаточна."
)

add()
add("---")
add()

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

add("# 1. Данные")
add()

add("Для классификации использовался датасет Banking77.")
add()

add(
    f"Количество intent-категорий: "
    f"**{protocol['n_classes']}**."
)

add()

add("Разделение данных:")
add()
add(
    f"- train_part: "
    f"{protocol['train_part_size']} примеров;"
)
add(
    f"- val_part: "
    f"{protocol['val_part_size']} примеров;"
)
add(
    f"- test: "
    f"{protocol['test_size']} примеров."
)

add()
add("### Назначение выборок")
add()

add("**train_part**")
add()
add(
    "Используется для обучения моделей "
    "и построения retrieval-корпуса."
)

add()
add("**val_part**")
add()
add(
    "Используется для выбора confidence threshold "
    "и архитектуры системы."
)

add()
add("**test**")
add()
add(
    "Используется только для однократной финальной "
    "оценки после фиксации всех решений."
)

add()
add("---")
add()

# ------------------------------------------------------------
# DISTILBERT
# ------------------------------------------------------------

add("# 2. DistilBERT")
add()

add("Базовая модель:")
add()
add("`distilbert-base-uncased`")

add()
add(
    f"Количество эпох: "
    f"**{config['distilbert_epochs']}**"
)

add()

add(
    f"Лучший checkpoint: "
    f"`{config['best_validation_checkpoint']}`"
)

add()
add("### Validation")
add()

add(
    f"- Accuracy: **{val_acc:.2f}%**"
)

add(
    f"- Macro F1: **{val_f1:.2f}%**"
)

add()
add("### Final test")
add()

add(
    f"- Accuracy: **{test_acc:.2f}%**"
)

add(
    f"- Macro F1: **{test_f1:.2f}%**"
)

add(
    f"- Weighted F1: **{test_weighted_f1:.2f}%**"
)

add()
add("---")
add()

# ------------------------------------------------------------
# SEMANTIC RETRIEVAL
# ------------------------------------------------------------

add("# 3. Semantic Retrieval")
add()

add("Embedding model:")
add()

add(
    f"`{config['semantic_model']}`"
)

add()

add(
    "Retrieval corpus построен только "
    "на `train_part`."
)

add()
add("### Validation")
add()

add(
    f"- Hit@1: **{val_hit1:.2f}%**"
)

add(
    f"- Hit@3: **{val_hit3:.2f}%**"
)

add(
    f"- Hit@5: **{val_hit5:.2f}%**"
)

add()
add("### Final test")
add()

add(
    f"- Hit@1: **{test_hit1:.2f}%**"
)

add(
    f"- Hit@3: **{test_hit3:.2f}%**"
)

add(
    f"- Hit@5: **{test_hit5:.2f}%**"
)

add()
add("---")
add()

# ------------------------------------------------------------
# SELECTIVE CLASSIFICATION
# ------------------------------------------------------------

add("# 4. Selective Classification")
add()

add(
    "Для отказа от недостаточно уверенных "
    "автоматических ответов используется confidence threshold."
)

add()

add(
    f"Финальный threshold: "
    f"**{config['selected_threshold_from_validation']}**"
)

add()

add(
    "Threshold был выбран исключительно "
    "на validation выборке."
)

add()
add("### Final test")
add()

add(
    f"- Coverage: **{baseline_coverage:.2f}%**"
)

add(
    f"- Automatic accuracy: "
    f"**{baseline_accuracy:.2f}%**"
)

add(
    f"- Human review rate: "
    f"**{baseline_review:.2f}%**"
)

add(
    f"- Auto-closed: "
    f"**{baseline_test['auto_closed']}**"
)

add(
    f"- Automatic errors: "
    f"**{baseline_test['automatic_errors']}**"
)

add()
add("---")
add()

# ------------------------------------------------------------
# LANGGRAPH
# ------------------------------------------------------------

add("# 5. LangGraph")
add()

add(
    "В проекте проверена архитектура "
    "selective consensus."
)

add()

add("Логика эксперимента:")
add()

add(
    "1. При confidence DistilBERT выше threshold "
    "запрос закрывается автоматически."
)

add(
    "2. Для low-confidence запросов "
    "запускается semantic retrieval."
)

add(
    "3. При согласии DistilBERT и retrieval "
    "проверяется retrieval similarity."
)

add(
    "4. При отсутствии достаточной уверенности "
    "запрос передаётся человеку."
)

add()

add(
    f"Финальный retrieval similarity gate: "
    f"**{config['retrieval_similarity_gate']}**"
)

add()
add("### Результат эксперимента")
add()

add(
    "При требовании automatic accuracy не ниже 95% "
    "дополнительная consensus-ветка не смогла безопасно "
    "увеличить coverage."
)

add()

add(
    "Поэтому validation-optimal frozen-конфигурация "
    "LangGraph фактически совпала с baseline "
    "`DistilBERT + reject threshold`."
)

add()
add("### Final test")
add()

add(
    f"- Coverage: **{langgraph_coverage:.2f}%**"
)

add(
    f"- Automatic accuracy: "
    f"**{langgraph_accuracy:.2f}%**"
)

add(
    f"- Human review: "
    f"**{langgraph_review:.2f}%**"
)

add(
    f"- Extra closed vs baseline: "
    f"**{langgraph_test['extra_closed_vs_baseline']}**"
)

add()
add("---")
add()

# ------------------------------------------------------------
# DOCUMENT RAG
# ------------------------------------------------------------

add("# 6. Document RAG")
add()

add(
    "Document RAG предназначен для ответа "
    "на вопросы по внутренним документам и инструкциям."
)

add()

add("### Evaluation dataset")
add()

add(
    f"Для финальной оценки использовано "
    f"**{document_rag['evaluation_set']['n_examples']}** "
    f"QA-примеров."
)

add()

add(
    "Вопросы были сгенерированы через внешний API "
    "по случайно выбранным chunks."
)

add()

add(
    f"Generator model: "
    f"`{document_rag['evaluation_set']['generator_model']}`"
)

add()

add(
    "Этот набор не использовался "
    "для настройки retrieval или RAG."
)

add()
add("### Retrieval metrics")
add()

add(
    f"- Source accuracy: "
    f"**{source_accuracy:.2f}%**"
)

add(
    f"- Chunk Hit@1: "
    f"**{chunk_hit1:.2f}%**"
)

add(
    f"- Chunk Hit@3: "
    f"**{chunk_hit3:.2f}%**"
)

add(
    f"- Runtime errors: "
    f"**{doc_retrieval['runtime_errors']}**"
)

add()
add("### Answer quality")
add()

add(
    f"- Answer accuracy: "
    f"**{answer_accuracy:.2f}%**"
)

add(
    f"- Groundedness: "
    f"**{groundedness:.2f}%**"
)

add(
    f"- Mean judge score: "
    f"**{doc_answer['mean_judge_score']:.2f} / 4**"
)

add(
    f"- Judge success rate: "
    f"**{judge_success:.2f}%**"
)

add(
    f"- Overall RAG success: "
    f"**{overall_rag:.2f}%**"
)

add()
add("---")
add()

# ------------------------------------------------------------
# FINAL TABLE
# ------------------------------------------------------------

add("# 7. Финальные результаты")
add()

add(
    "| Компонент | Метрика | Результат |"
)

add(
    "|---|---|---:|"
)

add(
    f"| DistilBERT | Accuracy | "
    f"{test_acc:.2f}% |"
)

add(
    f"| DistilBERT | Macro F1 | "
    f"{test_f1:.2f}% |"
)

add(
    f"| Semantic Retrieval | Hit@1 | "
    f"{test_hit1:.2f}% |"
)

add(
    f"| Semantic Retrieval | Hit@3 | "
    f"{test_hit3:.2f}% |"
)

add(
    f"| Semantic Retrieval | Hit@5 | "
    f"{test_hit5:.2f}% |"
)

add(
    f"| Selective baseline | Coverage | "
    f"{baseline_coverage:.2f}% |"
)

add(
    f"| Selective baseline | Automatic accuracy | "
    f"{baseline_accuracy:.2f}% |"
)

add(
    f"| Selective baseline | Human review | "
    f"{baseline_review:.2f}% |"
)

add(
    f"| Document RAG | Source accuracy | "
    f"{source_accuracy:.2f}% |"
)

add(
    f"| Document RAG | Answer accuracy | "
    f"{answer_accuracy:.2f}% |"
)

add(
    f"| Document RAG | Overall success | "
    f"{overall_rag:.2f}% |"
)

add()
add("---")
add()

# ------------------------------------------------------------
# CONCLUSIONS
# ------------------------------------------------------------

add("# 8. Основные выводы")
add()

add(
    "1. Финальный DistilBERT показал "
    f"Accuracy {test_acc:.2f}% и "
    f"Macro F1 {test_f1:.2f}% "
    "на отложенной test выборке."
)

add()

add(
    "2. Confidence threshold позволяет увеличить "
    "надёжность автоматически закрываемых запросов "
    "до уровня выше 95%."
)

add()

add(
    f"3. При threshold "
    f"{config['selected_threshold_from_validation']} "
    f"автоматически обрабатывается "
    f"{baseline_coverage:.2f}% запросов."
)

add()

add(
    "4. Semantic retrieval показывает высокий Hit@3 "
    "и Hit@5 и может использоваться как "
    "вспомогательный компонент."
)

add()

add(
    "5. Эксперимент с LangGraph показал, "
    "что усложнение архитектуры само по себе "
    "не гарантирует улучшения."
)

add()

add(
    "6. При ограничении automatic accuracy >= 95% "
    "consensus-ветка LangGraph не дала "
    "безопасного прироста coverage."
)

add()

add(
    "7. Document RAG показал высокую точность ответов, "
    "однако groundedness и overall success остаются "
    "направлениями для дальнейшего улучшения."
)

add()

add(
    "8. Evaluation protocol разделяет обучение, "
    "выбор решений и финальную оценку, "
    "что снижает риск data leakage."
)

add()
add("---")
add()

# ------------------------------------------------------------
# PROJECT STRUCTURE
# ------------------------------------------------------------

add("# 9. Структура проекта")
add()

add("```text")
add("ml_customer_classification_corrected/")
add("│")
add("├── data/")
add("│   ├── train_part.parquet")
add("│   ├── val_part.parquet")
add("│   ├── test.parquet")
add("│   ├── label_mapping.json")
add("│   └── label_mapping_final.json")
add("│")
add("├── models/")
add("│   └── distilbert_final_6epoch/")
add("│")
add("├── validation/")
add("│   ├── distilbert_final_val_predictions.csv")
add("│   ├── distilbert_final_threshold_sweep.csv")
add("│   └── selected_threshold_final.json")
add("│")
add("├── semantic/")
add("│   └── results/")
add("│")
add("├── langgraph/")
add("│   └── results/")
add("│       ├── final_langgraph_architecture.json")
add("│       ├── low_confidence_consensus_validation.csv")
add("│       └── retrieval_similarity_threshold_sweep.csv")
add("│")
add("├── document_rag/")
add("│   ├── data/")
add("│   │   └── api_generated_qa.csv")
add("│   └── results/")
add("│       ├── api_generated_qa_rag_evaluation.csv")
add("│       ├── api_generated_qa_rag_metrics.json")
add("│       ├── api_generated_qa_final_evaluation.csv")
add("│       └── api_generated_qa_final_metrics.json")
add("│")
add("├── results/")
add("│   ├── FINAL_CORRECTED_SUMMARY.json")
add("│   ├── final_test_metrics.json")
add("│   ├── final_test_predictions.csv")
add("│   ├── final_langgraph_test_predictions.csv")
add("│   ├── report_classification_metrics.csv")
add("│   ├── report_baseline_vs_langgraph.csv")
add("│   ├── report_document_rag_metrics.csv")
add("│   ├── report_evaluation_protocol.csv")
add("│   └── DEFENSE_CONCLUSIONS.txt")
add("│")
add("└── README.md")
add("```")

add()
add("---")
add()

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

add("# 10. Воспроизводимость")
add()

add(
    f"Seed: `{protocol['seed']}`"
)

add()

add(
    "Все ключевые решения, включая confidence threshold "
    "и LangGraph architecture, были выбраны на validation."
)

add()

add(
    "Test использовался только после "
    "фиксации параметров."
)

add()

add(
    "После получения test-метрик "
    "архитектура и thresholds не изменялись."
)


# ------------------------------------------------------------
# 6. Собираем финальный текст
# ------------------------------------------------------------

readme_text = "\n".join(
    lines
)


# ------------------------------------------------------------
# 7. Сохраняем README
# ------------------------------------------------------------

with open(
    README_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        readme_text
    )


# ------------------------------------------------------------
# 8. Проверяем файл
# ------------------------------------------------------------

if not README_PATH.exists():
    raise RuntimeError(
        "README.md не удалось создать."
    )


readme_size = (
    README_PATH
    .stat()
    .st_size
)


# ------------------------------------------------------------
# 9. Показываем начало README
# ------------------------------------------------------------

print("=" * 70)
print("README PREVIEW")
print("=" * 70)

print()

print(
    readme_text[:4000]
)

print()

print("=" * 70)
print("CELL 56 COMPLETE")
print("=" * 70)

print(
    "README saved:"
)

print(
    README_PATH
)

print(
    "README size:",
    readme_size,
    "bytes"
)

print()

print(
    "README содержит:"
)

print(
    "- описание проекта"
)

print(
    "- evaluation protocol"
)

print(
    "- DistilBERT results"
)

print(
    "- semantic retrieval"
)

print(
    "- selective classification"
)

print(
    "- baseline vs LangGraph"
)

print(
    "- Document RAG"
)

print(
    "- финальные выводы"
)

print(
    "- структуру проекта"
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 57.
# FINAL README SYNC + PROJECT VERIFICATION ON GOOGLE DRIVE
# ============================================================

import json
import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------

LOCAL_PROJECT_DIR = Path(
    "/content/ml_customer_classification_corrected"
)

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/ml_customer_classification_corrected"
)

LOCAL_README = (
    LOCAL_PROJECT_DIR
    / "README.md"
)

DRIVE_README = (
    DRIVE_PROJECT_DIR
    / "README.md"
)


# ------------------------------------------------------------
# 2. Проверяем исходные файлы
# ------------------------------------------------------------

if not LOCAL_PROJECT_DIR.exists():

    raise RuntimeError(
        "Локальная папка проекта не найдена."
    )


if not Path(
    "/content/drive/MyDrive"
).exists():

    raise RuntimeError(
        "Google Drive не подключён."
    )


if not LOCAL_README.exists():

    raise RuntimeError(
        "README.md не найден. "
        "Сначала выполни ячейку 56."
    )


print("=" * 70)
print("FINAL README SYNC")
print("=" * 70)

print(
    "Local README:",
    LOCAL_README
)

print(
    "Drive README:",
    DRIVE_README
)


# ------------------------------------------------------------
# 3. Копируем README
# ------------------------------------------------------------

DRIVE_PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


shutil.copy2(
    LOCAL_README,
    DRIVE_README
)


# ------------------------------------------------------------
# 4. Проверяем README
# ------------------------------------------------------------

local_readme_size = (
    LOCAL_README
    .stat()
    .st_size
)

drive_readme_size = (
    DRIVE_README
    .stat()
    .st_size
)


print()
print("=" * 70)
print("README CHECK")
print("=" * 70)

print(
    "Local README size:",
    local_readme_size,
    "bytes"
)

print(
    "Drive README size:",
    drive_readme_size,
    "bytes"
)


if local_readme_size != drive_readme_size:

    raise RuntimeError(
        "Размер README на Drive отличается "
        "от локальной версии."
    )


print(
    "README: OK"
)


# ------------------------------------------------------------
# 5. Финальный список файлов проекта
# ------------------------------------------------------------

critical_files = [

    # README
    "README.md",

    # DATA
    "data/train_part.parquet",
    "data/val_part.parquet",
    "data/test.parquet",
    "data/label_mapping.json",
    "data/label_mapping_final.json",

    # MODEL
    "models/distilbert_final_6epoch/config.json",
    "models/distilbert_final_6epoch/model.safetensors",

    # VALIDATION
    "validation/distilbert_final_val_predictions.csv",
    "validation/distilbert_final_threshold_sweep.csv",
    "validation/selected_threshold_final.json",

    # LANGGRAPH
    "langgraph/results/final_langgraph_architecture.json",
    "langgraph/results/low_confidence_consensus_validation.csv",
    "langgraph/results/retrieval_similarity_threshold_sweep.csv",

    # FINAL TEST
    "results/final_test_metrics.json",
    "results/final_test_predictions.csv",
    "results/final_langgraph_test_predictions.csv",

    # REPORT
    "results/FINAL_CORRECTED_SUMMARY.json",
    "results/report_classification_metrics.csv",
    "results/report_baseline_vs_langgraph.csv",
    "results/report_document_rag_metrics.csv",
    "results/report_evaluation_protocol.csv",
    "results/DEFENSE_CONCLUSIONS.txt",

    # DOCUMENT RAG
    "document_rag/data/api_generated_qa.csv",
    "document_rag/results/api_generated_qa_rag_evaluation.csv",
    "document_rag/results/api_generated_qa_rag_metrics.json",
    "document_rag/results/api_generated_qa_final_evaluation.csv",
    "document_rag/results/api_generated_qa_final_metrics.json"
]


# ------------------------------------------------------------
# 6. Проверяем Drive
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL DRIVE PROJECT CHECK")
print("=" * 70)


missing_files = []


for relative_path in critical_files:

    path = (
        DRIVE_PROJECT_DIR
        / relative_path
    )


    if path.exists():

        status = "OK"

    else:

        status = "MISSING"

        missing_files.append(
            relative_path
        )


    print(
        f"{status:8} | "
        f"{relative_path}"
    )


# ------------------------------------------------------------
# 7. Проверяем итоговый summary
# ------------------------------------------------------------

SUMMARY_PATH = (
    DRIVE_PROJECT_DIR
    / "results"
    / "FINAL_CORRECTED_SUMMARY.json"
)


if not SUMMARY_PATH.exists():

    raise RuntimeError(
        "FINAL_CORRECTED_SUMMARY.json "
        "не найден на Drive."
    )


with open(
    SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    summary = json.load(f)


# ------------------------------------------------------------
# 8. Контроль ключевых финальных метрик
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL METRICS CHECK")
print("=" * 70)


print(
    "DistilBERT Accuracy:",
    f"{summary['final_test']['models']['distilbert_accuracy'] * 100:.2f}%"
)

print(
    "DistilBERT Macro F1:",
    f"{summary['final_test']['models']['distilbert_macro_f1'] * 100:.2f}%"
)

print(
    "Threshold:",
    summary[
        "final_configuration"
    ][
        "selected_threshold_from_validation"
    ]
)

print(
    "Baseline Coverage:",
    f"{summary['final_test']['reject_baseline']['coverage'] * 100:.2f}%"
)

print(
    "Baseline Automatic Accuracy:",
    f"{summary['final_test']['reject_baseline']['automatic_accuracy'] * 100:.2f}%"
)

print(
    "Document RAG Answer Accuracy:",
    f"{summary['document_rag_api_qa']['answer_quality']['answer_accuracy_llm_judge'] * 100:.2f}%"
)

print(
    "Document RAG Overall Success:",
    f"{summary['document_rag_api_qa']['answer_quality']['overall_rag_success'] * 100:.2f}%"
)


# ------------------------------------------------------------
# 9. Размер проекта на Drive
# ------------------------------------------------------------

def get_folder_size(
    folder
):

    total = 0

    for path in folder.rglob("*"):

        if path.is_file():

            try:

                total += (
                    path.stat().st_size
                )

            except OSError:

                pass

    return total


drive_project_size = get_folder_size(
    DRIVE_PROJECT_DIR
)


print()
print("=" * 70)
print("PROJECT SIZE")
print("=" * 70)

print(
    "Drive project size:",
    f"{drive_project_size / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# 10. Итог
# ------------------------------------------------------------

print()
print("=" * 70)

if len(missing_files) == 0:

    print(
        "CELL 57 COMPLETE"
    )

    print()

    print(
        "FINAL PROJECT STATUS: READY FOR SUBMISSION"
    )

    print()

    print(
        "README.md синхронизирован."
    )

    print(
        "Все критические файлы проекта присутствуют."
    )

    print(
        "Финальные метрики сохранены."
    )

    print(
        "Google Drive содержит актуальную версию проекта."
    )

    print()

    print(
        "Project location:"
    )

    print(
        DRIVE_PROJECT_DIR
    )

else:

    print(
        "CELL 57 PARTIALLY COMPLETE"
    )

    print()

    print(
        "Не найдены файлы:"
    )

    for relative_path in missing_files:

        print(
            "-",
            relative_path
        )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 1.
# RESTORE COLAB ENVIRONMENT + PREPARE GITHUB / MLSD WORKSPACE
#
# Что делает ячейка:
# 1. Подключает Google Drive
# 2. Восстанавливает пути финального проекта
# 3. Проверяет критические файлы
# 4. Загружает FINAL_CORRECTED_SUMMARY.json
# 5. Показывает ключевые финальные метрики
# 6. Создаёт отдельную папку под GitHub-репозиторий
# 7. Создаёт структуру GitHub / MLSD проекта
#
# Ничего не обучаем и не пересчитываем.
# Исходный финальный проект не изменяется.
# ============================================================

import json
from pathlib import Path

from google.colab import drive


# ------------------------------------------------------------
# 1. Подключаем Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=True
)


# ------------------------------------------------------------
# 2. Основные пути
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)


# Финальный рабочий проект
SOURCE_PROJECT_DIR = (
    DRIVE_ROOT
    / "ml_customer_classification_corrected"
)


# Основные директории финального проекта
DATA_DIR = (
    SOURCE_PROJECT_DIR
    / "data"
)

MODELS_DIR = (
    SOURCE_PROJECT_DIR
    / "models"
)

RESULTS_DIR = (
    SOURCE_PROJECT_DIR
    / "results"
)

VALIDATION_DIR = (
    SOURCE_PROJECT_DIR
    / "validation"
)

SEMANTIC_DIR = (
    SOURCE_PROJECT_DIR
    / "semantic"
)

LANGGRAPH_DIR = (
    SOURCE_PROJECT_DIR
    / "langgraph"
)

DOC_RAG_DIR = (
    SOURCE_PROJECT_DIR
    / "document_rag"
)


# Отдельная папка,
# которую будем готовить для GitHub / MLSD
GITHUB_PROJECT_DIR = (
    DRIVE_ROOT
    / "ml_customer_support_assistant_github"
)


# ------------------------------------------------------------
# 3. Проверяем исходный проект
# ------------------------------------------------------------

if not SOURCE_PROJECT_DIR.exists():

    raise RuntimeError(
        f"Финальный проект не найден:\n"
        f"{SOURCE_PROJECT_DIR}"
    )


print("=" * 70)
print("COLAB RESTORE + GITHUB / MLSD PREPARATION")
print("=" * 70)

print()
print(
    "Source project:"
)

print(
    SOURCE_PROJECT_DIR
)


# ------------------------------------------------------------
# 4. Проверяем критические файлы финального проекта
# ------------------------------------------------------------

critical_files = [

    # README
    "README.md",

    # DATA
    "data/train_part.parquet",
    "data/val_part.parquet",
    "data/test.parquet",
    "data/label_mapping.json",
    "data/label_mapping_final.json",

    # MODEL
    "models/distilbert_final_6epoch/config.json",
    "models/distilbert_final_6epoch/model.safetensors",

    # VALIDATION
    "validation/distilbert_final_val_predictions.csv",
    "validation/distilbert_final_threshold_sweep.csv",
    "validation/selected_threshold_final.json",

    # LANGGRAPH
    "langgraph/results/final_langgraph_architecture.json",
    "langgraph/results/low_confidence_consensus_validation.csv",
    "langgraph/results/retrieval_similarity_threshold_sweep.csv",

    # FINAL TEST / REPORTS
    "results/final_test_metrics.json",
    "results/final_test_predictions.csv",
    "results/final_langgraph_test_predictions.csv",
    "results/FINAL_CORRECTED_SUMMARY.json",
    "results/report_classification_metrics.csv",
    "results/report_baseline_vs_langgraph.csv",
    "results/report_document_rag_metrics.csv",
    "results/report_evaluation_protocol.csv",
    "results/DEFENSE_CONCLUSIONS.txt",

    # DOCUMENT RAG
    "document_rag/data/api_generated_qa.csv",
    "document_rag/results/api_generated_qa_rag_evaluation.csv",
    "document_rag/results/api_generated_qa_rag_metrics.json",
    "document_rag/results/api_generated_qa_final_evaluation.csv",
    "document_rag/results/api_generated_qa_final_metrics.json"
]


print()
print("=" * 70)
print("SOURCE PROJECT CHECK")
print("=" * 70)


missing_files = []


for relative_path in critical_files:

    file_path = (
        SOURCE_PROJECT_DIR
        / relative_path
    )


    if file_path.exists():

        status = "OK"

    else:

        status = "MISSING"

        missing_files.append(
            relative_path
        )


    print(
        f"{status:8} | "
        f"{relative_path}"
    )


# ------------------------------------------------------------
# 5. Загружаем финальный summary
# ------------------------------------------------------------

FINAL_SUMMARY_PATH = (
    RESULTS_DIR
    / "FINAL_CORRECTED_SUMMARY.json"
)


if not FINAL_SUMMARY_PATH.exists():

    raise RuntimeError(
        "FINAL_CORRECTED_SUMMARY.json не найден."
    )


with open(
    FINAL_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    FINAL_SUMMARY = json.load(f)


# ------------------------------------------------------------
# 6. Показываем ключевые финальные метрики
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL METRICS LOADED")
print("=" * 70)


print(
    "DistilBERT Accuracy:",
    f"{FINAL_SUMMARY['final_test']['models']['distilbert_accuracy'] * 100:.2f}%"
)

print(
    "DistilBERT Macro F1:",
    f"{FINAL_SUMMARY['final_test']['models']['distilbert_macro_f1'] * 100:.2f}%"
)

print(
    "Threshold:",
    FINAL_SUMMARY[
        "final_configuration"
    ][
        "selected_threshold_from_validation"
    ]
)

print(
    "Baseline Coverage:",
    f"{FINAL_SUMMARY['final_test']['reject_baseline']['coverage'] * 100:.2f}%"
)

print(
    "Baseline Automatic Accuracy:",
    f"{FINAL_SUMMARY['final_test']['reject_baseline']['automatic_accuracy'] * 100:.2f}%"
)

print(
    "Document RAG Answer Accuracy:",
    f"{FINAL_SUMMARY['document_rag_api_qa']['answer_quality']['answer_accuracy_llm_judge'] * 100:.2f}%"
)

print(
    "Document RAG Overall Success:",
    f"{FINAL_SUMMARY['document_rag_api_qa']['answer_quality']['overall_rag_success'] * 100:.2f}%"
)


# ------------------------------------------------------------
# 7. Создаём отдельную GitHub / MLSD папку
# ------------------------------------------------------------

GITHUB_PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 8. Создаём структуру будущего репозитория
# ------------------------------------------------------------

github_folders = [

    "notebooks",

    "src",

    "results",

    "validation",

    "document_rag",

    "langgraph",

    "docs",

    "assets"
]


for folder in github_folders:

    (
        GITHUB_PROJECT_DIR
        / folder
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 9. Показываем структуру GitHub-проекта
# ------------------------------------------------------------

print()
print("=" * 70)
print("GITHUB / MLSD WORKSPACE")
print("=" * 70)

print(
    "GitHub project:"
)

print(
    GITHUB_PROJECT_DIR
)

print()

print(
    """
ml_customer_support_assistant_github/
│
├── notebooks/
├── src/
├── results/
├── validation/
├── document_rag/
├── langgraph/
├── docs/
├── assets/
│
└── README.md
""".strip()
)


# ------------------------------------------------------------
# 10. Итог
# ------------------------------------------------------------

print()
print("=" * 70)

if len(missing_files) == 0:

    print(
        "CELL 1 COMPLETE"
    )

    print()

    print(
        "Среда восстановлена."
    )

    print(
        "Финальный проект найден."
    )

    print(
        "Все критические файлы доступны."
    )

    print(
        "Отдельная GitHub / MLSD папка создана."
    )

    print(
        "Исходный проект не изменён."
    )

    print()

    print(
        "Переобучение моделей НЕ требуется."
    )

    print(
        "GPU и API key для текущего этапа НЕ требуются."
    )

else:

    print(
        "CELL 1 COMPLETE WITH WARNINGS"
    )

    print()

    print(
        "Не найдены файлы:"
    )

    for path in missing_files:

        print(
            "-",
            path
        )

    print()

    print(
        "Перед упаковкой GitHub-проекта "
        "лучше разобраться с отсутствующими файлами."
    )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 2.
# COPY ONLY GITHUB-READY FILES
#
# Копируем в отдельный GitHub workspace:
# - README
# - итоговые результаты
# - validation artifacts
# - LangGraph artifacts
# - Document RAG evaluation
#
# НЕ копируем:
# - model.safetensors
# - train/val/test parquet
# - checkpoints
# - временные и тяжёлые файлы
# ============================================================

import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем пути из ячейки 1
# ------------------------------------------------------------

if "SOURCE_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "SOURCE_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


print("=" * 70)
print("COPYING GITHUB-READY PROJECT FILES")
print("=" * 70)


# ------------------------------------------------------------
# 2. Вспомогательная функция
# ------------------------------------------------------------

def copy_file(
    source_relative,
    destination_relative=None
):

    source = (
        SOURCE_PROJECT_DIR
        / source_relative
    )

    if destination_relative is None:
        destination_relative = source_relative

    destination = (
        GITHUB_PROJECT_DIR
        / destination_relative
    )

    if not source.exists():

        print(
            f"MISSING  | {source_relative}"
        )

        return False

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        source,
        destination
    )

    print(
        f"COPIED   | {destination_relative}"
    )

    return True


# ------------------------------------------------------------
# 3. README
# ------------------------------------------------------------

copy_file(
    "README.md",
    "README.md"
)


# ------------------------------------------------------------
# 4. Results
# ------------------------------------------------------------

result_files = [

    "FINAL_CORRECTED_SUMMARY.json",

    "final_test_metrics.json",

    "report_classification_metrics.csv",

    "report_baseline_vs_langgraph.csv",

    "report_document_rag_metrics.csv",

    "report_evaluation_protocol.csv",

    "DEFENSE_CONCLUSIONS.txt"
]


for filename in result_files:

    copy_file(
        f"results/{filename}",
        f"results/{filename}"
    )


# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

validation_files = [

    "selected_threshold_final.json",

    "distilbert_final_threshold_sweep.csv"
]


for filename in validation_files:

    copy_file(
        f"validation/{filename}",
        f"validation/{filename}"
    )


# ------------------------------------------------------------
# 6. LangGraph
# ------------------------------------------------------------

langgraph_files = [

    "final_langgraph_architecture.json",

    "low_confidence_consensus_validation.csv",

    "retrieval_similarity_threshold_sweep.csv"
]


for filename in langgraph_files:

    copy_file(
        f"langgraph/results/{filename}",
        f"langgraph/{filename}"
    )


# ------------------------------------------------------------
# 7. Document RAG
# ------------------------------------------------------------

copy_file(
    "document_rag/data/api_generated_qa.csv",
    "document_rag/api_generated_qa.csv"
)


document_rag_result_files = [

    "api_generated_qa_rag_metrics.json",

    "api_generated_qa_final_metrics.json",

    "api_generated_qa_final_evaluation.csv"
]


for filename in document_rag_result_files:

    copy_file(
        f"document_rag/results/{filename}",
        f"document_rag/{filename}"
    )


# ------------------------------------------------------------
# 8. Label mapping
#
# Небольшой файл, полезен для воспроизводимости.
# ------------------------------------------------------------

copy_file(
    "data/label_mapping.json",
    "src/label_mapping.json"
)


# ------------------------------------------------------------
# 9. Создаём .gitignore
# ------------------------------------------------------------

gitignore_text = """
# Python
__pycache__/
*.py[cod]
*.so
.ipynb_checkpoints/

# Environments
.env
.venv/
venv/

# Secrets
*.key
.env.*

# Models / checkpoints
*.safetensors
*.bin
*.pt
*.pth
checkpoints/
models/

# Large datasets
*.parquet
*.arrow

# Colab / local
/content/
drive/

# OS
.DS_Store
Thumbs.db
""".strip()


GITIGNORE_PATH = (
    GITHUB_PROJECT_DIR
    / ".gitignore"
)


with open(
    GITIGNORE_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        gitignore_text
    )


print(
    "CREATED  | .gitignore"
)


# ------------------------------------------------------------
# 10. Создаём placeholder для MLSD
# ------------------------------------------------------------

MLSD_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "mlsd.md"
)


if not MLSD_PATH.exists():

    with open(
        MLSD_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "# ML System Design\n\n"
            "Документ будет сформирован на следующих этапах.\n"
        )


    print(
        "CREATED  | docs/mlsd.md"
    )


# ------------------------------------------------------------
# 11. Размер GitHub workspace
# ------------------------------------------------------------

def get_folder_size_mb(path):

    total = 0

    for file_path in path.rglob("*"):

        if file_path.is_file():

            total += (
                file_path.stat().st_size
            )

    return (
        total
        /
        1024**2
    )


github_size_mb = get_folder_size_mb(
    GITHUB_PROJECT_DIR
)


# ------------------------------------------------------------
# 12. Показываем файлы
# ------------------------------------------------------------

print()
print("=" * 70)
print("GITHUB WORKSPACE FILES")
print("=" * 70)


for path in sorted(
    GITHUB_PROJECT_DIR.rglob("*")
):

    if path.is_file():

        relative_path = (
            path.relative_to(
                GITHUB_PROJECT_DIR
            )
        )

        size_kb = (
            path.stat().st_size
            /
            1024
        )

        print(
            f"{size_kb:10.1f} KB | "
            f"{relative_path}"
        )


# ------------------------------------------------------------
# 13. Финальный контроль
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 2 COMPLETE")
print("=" * 70)

print(
    "GitHub workspace:"
)

print(
    GITHUB_PROJECT_DIR
)

print()

print(
    "Workspace size:",
    f"{github_size_mb:.2f} MB"
)

print()

print(
    "Тяжёлая модель и parquet-датасеты "
    "в GitHub workspace НЕ копировались."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 3.
# CREATE SOURCE CODE STRUCTURE
#
# Создаём минимальную src-структуру для GitHub:
# - config.py
# - classifier.py
# - retrieval.py
# - routing.py
# - rag.py
# - __init__.py
#
# Это не переобучение и не новый эксперимент.
# Мы оформляем уже реализованную архитектуру проекта.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем GitHub workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


SRC_DIR = (
    GITHUB_PROJECT_DIR
    / "src"
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. __init__.py
# ------------------------------------------------------------

init_text = """
\"\"\"ML Customer Support Assistant package.\"\"\"

__version__ = "1.0.0"
""".strip()


with open(
    SRC_DIR / "__init__.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(init_text)


# ------------------------------------------------------------
# 3. config.py
# ------------------------------------------------------------

config_text = """
\"\"\"Final frozen configuration of the project.\"\"\"

SEED = 42

# Intent classification
DISTILBERT_MODEL = "distilbert-base-uncased"
NUM_LABELS = 77
NUM_EPOCHS = 6
MAX_LENGTH = 64

# Selected on validation
DISTILBERT_CONFIDENCE_THRESHOLD = 0.65

# Semantic retrieval
SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Validation-optimal LangGraph gate.
# The value 1.0 means the low-confidence consensus branch
# did not provide safe additional coverage under the
# >=95% automatic-accuracy constraint.
RETRIEVAL_SIMILARITY_THRESHOLD = 1.0

# Document RAG
RAG_TOP_K = 3
RAG_GENERATOR_MODEL = "openai/gpt-oss-120b"

# Quality target
TARGET_AUTOMATIC_ACCURACY = 0.95
""".strip()


with open(
    SRC_DIR / "config.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(config_text)


# ------------------------------------------------------------
# 4. classifier.py
# ------------------------------------------------------------

classifier_text = """
\"\"\"Intent classification interface.

The final experimental model is DistilBERT trained for
Banking77 intent classification.

Heavy model weights are intentionally not stored in GitHub.
\"\"\"

from dataclasses import dataclass
from typing import Optional


@dataclass
class ClassificationResult:
    intent_id: int
    intent_name: str
    confidence: float
    accepted: bool


def apply_reject_policy(
    intent_id: int,
    intent_name: str,
    confidence: float,
    threshold: float,
) -> ClassificationResult:
    \"\"\"Apply the validation-selected reject threshold.\"\"\"

    return ClassificationResult(
        intent_id=int(intent_id),
        intent_name=str(intent_name),
        confidence=float(confidence),
        accepted=bool(confidence >= threshold),
    )
""".strip()


with open(
    SRC_DIR / "classifier.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(classifier_text)


# ------------------------------------------------------------
# 5. retrieval.py
# ------------------------------------------------------------

retrieval_text = """
\"\"\"Semantic retrieval utilities.

The project uses normalized sentence embeddings and cosine
similarity over a retrieval corpus built only from train_part.
\"\"\"

from dataclasses import dataclass
from typing import List


@dataclass
class RetrievalResult:
    label: int
    label_name: str
    similarity: float
    text: str


def consensus_is_safe(
    classifier_label: int,
    retrieval_label: int,
    retrieval_similarity: float,
    similarity_threshold: float,
) -> bool:
    \"\"\"Check whether low-confidence consensus may auto-close.

    In the final experiment the selected similarity threshold
    was 1.0, so no additional low-confidence requests were
    safely auto-closed under the >=95% accuracy constraint.
    \"\"\"

    return (
        int(classifier_label) == int(retrieval_label)
        and float(retrieval_similarity) >= float(similarity_threshold)
    )
""".strip()


with open(
    SRC_DIR / "retrieval.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(retrieval_text)


# ------------------------------------------------------------
# 6. routing.py
# ------------------------------------------------------------

routing_text = """
\"\"\"Frozen routing policy corresponding to the final experiment.\"\"\"

from dataclasses import dataclass

from .config import (
    DISTILBERT_CONFIDENCE_THRESHOLD,
    RETRIEVAL_SIMILARITY_THRESHOLD,
)


@dataclass
class RoutingDecision:
    route: str
    auto_closed: bool
    final_label: int | None


def route_request(
    classifier_label: int,
    classifier_confidence: float,
    retrieval_label: int | None = None,
    retrieval_similarity: float | None = None,
) -> RoutingDecision:
    \"\"\"Route one request according to the frozen policy.

    Routes:
    - high-confidence DistilBERT -> automatic close
    - low-confidence consensus + similarity gate -> automatic close
    - otherwise -> human review
    \"\"\"

    if classifier_confidence >= DISTILBERT_CONFIDENCE_THRESHOLD:
        return RoutingDecision(
            route="distilbert_high_confidence",
            auto_closed=True,
            final_label=int(classifier_label),
        )

    if (
        retrieval_label is not None
        and retrieval_similarity is not None
        and int(retrieval_label) == int(classifier_label)
        and float(retrieval_similarity) >= RETRIEVAL_SIMILARITY_THRESHOLD
    ):
        return RoutingDecision(
            route="low_confidence_consensus",
            auto_closed=True,
            final_label=int(classifier_label),
        )

    return RoutingDecision(
        route="human_review",
        auto_closed=False,
        final_label=None,
    )
""".strip()


with open(
    SRC_DIR / "routing.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(routing_text)


# ------------------------------------------------------------
# 7. rag.py
# ------------------------------------------------------------

rag_text = """
\"\"\"Document RAG interfaces.

The evaluation pipeline used:
1. semantic retrieval over document chunks;
2. top-k grounded context;
3. LLM answer generation;
4. independent offline QA evaluation.

The API key and provider configuration are not stored here.
\"\"\"

from dataclasses import dataclass
from typing import List


@dataclass
class DocumentChunk:
    doc_id: str
    chunk_id: str
    text: str
    similarity: float


@dataclass
class RAGResult:
    answer: str
    sources: List[DocumentChunk]


def build_context(chunks: List[DocumentChunk]) -> str:
    \"\"\"Build grounded context from retrieved document chunks.\"\"\"

    blocks = []

    for chunk in chunks:
        blocks.append(
            f"[DOC_ID: {chunk.doc_id}]\\n"
            f"[CHUNK_ID: {chunk.chunk_id}]\\n"
            f"{chunk.text}"
        )

    return "\\n\\n---\\n\\n".join(blocks)
""".strip()


with open(
    SRC_DIR / "rag.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(rag_text)


# ------------------------------------------------------------
# 8. Проверяем созданные файлы
# ------------------------------------------------------------

print("=" * 70)
print("SOURCE CODE STRUCTURE")
print("=" * 70)


source_files = [
    "__init__.py",
    "config.py",
    "classifier.py",
    "retrieval.py",
    "routing.py",
    "rag.py",
    "label_mapping.json",
]


for filename in source_files:

    path = (
        SRC_DIR
        / filename
    )

    status = (
        "OK"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{status:8} | src/{filename}"
    )


# ------------------------------------------------------------
# 9. Показываем итоговую структуру src
# ------------------------------------------------------------

print()
print("=" * 70)
print("SRC TREE")
print("=" * 70)


for path in sorted(
    SRC_DIR.iterdir()
):

    if path.is_file():

        print(
            f"- {path.name}"
        )


print()
print("=" * 70)
print("CELL 3 COMPLETE")
print("=" * 70)

print(
    "Минимальная кодовая структура проекта создана."
)

print(
    "Следующий шаг — оформить ML System Design "
    "в docs/mlsd.md."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 4.
# CREATE ML SYSTEM DESIGN DOCUMENT
#
# Формируем docs/mlsd.md:
# - постановка задачи
# - бизнес-цель
# - функциональные требования
# - нефункциональные требования
# - архитектура
# - data flow
# - ML-компоненты
# - offline / online metrics
# - deployment
# - monitoring
# - retraining
# - риски
#
# Используем только финальные результаты проекта.
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


DOCS_DIR = (
    GITHUB_PROJECT_DIR
    / "docs"
)

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MLSD_PATH = (
    DOCS_DIR
    / "mlsd.md"
)


# ------------------------------------------------------------
# 2. Загружаем финальный summary
# ------------------------------------------------------------

SUMMARY_PATH = (
    GITHUB_PROJECT_DIR
    / "results"
    / "FINAL_CORRECTED_SUMMARY.json"
)


if not SUMMARY_PATH.exists():
    raise RuntimeError(
        "FINAL_CORRECTED_SUMMARY.json не найден. "
        "Сначала выполни ячейку 2."
    )


with open(
    SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    summary = json.load(f)


protocol = summary[
    "protocol"
]

config = summary[
    "final_configuration"
]

validation = summary[
    "validation"
]

final_test = summary[
    "final_test"
]

document_rag = summary[
    "document_rag_api_qa"
]


baseline_test = (
    final_test[
        "reject_baseline"
    ]
)

langgraph_test = (
    final_test[
        "langgraph"
    ]
)

doc_retrieval = (
    document_rag[
        "retrieval"
    ]
)

doc_answer = (
    document_rag[
        "answer_quality"
    ]
)


# ------------------------------------------------------------
# 3. Форматируем основные метрики
# ------------------------------------------------------------

test_accuracy = (
    final_test[
        "models"
    ][
        "distilbert_accuracy"
    ]
    * 100
)

test_macro_f1 = (
    final_test[
        "models"
    ][
        "distilbert_macro_f1"
    ]
    * 100
)

retrieval_hit1 = (
    final_test[
        "models"
    ][
        "retrieval_hit@1"
    ]
    * 100
)

retrieval_hit3 = (
    final_test[
        "models"
    ][
        "retrieval_hit@3"
    ]
    * 100
)

retrieval_hit5 = (
    final_test[
        "models"
    ][
        "retrieval_hit@5"
    ]
    * 100
)

coverage = (
    baseline_test[
        "coverage"
    ]
    * 100
)

auto_accuracy = (
    baseline_test[
        "automatic_accuracy"
    ]
    * 100
)

review_rate = (
    baseline_test[
        "review_rate"
    ]
    * 100
)

rag_source_accuracy = (
    doc_retrieval[
        "source_accuracy"
    ]
    * 100
)

rag_answer_accuracy = (
    doc_answer[
        "answer_accuracy_llm_judge"
    ]
    * 100
)

rag_groundedness = (
    doc_answer[
        "groundedness_rate"
    ]
    * 100
)

rag_overall = (
    doc_answer[
        "overall_rag_success"
    ]
    * 100
)


# ------------------------------------------------------------
# 4. Формируем MLSD построчно
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


# ------------------------------------------------------------
# TITLE
# ------------------------------------------------------------

add("# ML System Design")
add()
add("## Интеллектуальный ассистент для поддержки сотрудников")
add()

add(
    "Система предназначена для автоматической обработки "
    "типовых запросов сотрудников, классификации намерений, "
    "поиска информации во внутренних документах и формирования "
    "ответов с возможностью передачи сложных случаев человеку."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 1. PROBLEM
# ------------------------------------------------------------

add("# 1. Постановка задачи")
add()

add(
    "Необходимо разработать интеллектуального ассистента, "
    "который отвечает на вопросы сотрудников по внутренним "
    "документам, регламентам, продуктам и инструкциям."
)

add()

add(
    "Система должна не только выдавать ответ, "
    "но и определять тип запроса, находить релевантную информацию, "
    "проверять достаточность уверенности и при необходимости "
    "передавать запрос на ручную обработку."
)

add()

add("Целевая логика обработки:")
add()

add(
    "`определение типа запроса → поиск информации → "
    "дополнительный поиск при необходимости → "
    "проверка ответа → финальный результат`"
)

add()
add("---")
add()


# ------------------------------------------------------------
# 2. BUSINESS GOAL
# ------------------------------------------------------------

add("# 2. Бизнес-цель")
add()

add(
    "Сократить количество типовых обращений, "
    "которые требуют ручной обработки специалистом, "
    "при сохранении контролируемого качества автоматических ответов."
)

add()

add("Основные бизнес-эффекты:")
add()

add("- сокращение нагрузки на сотрудников поддержки;")
add("- уменьшение времени ответа;")
add("- единообразие ответов по внутренним документам;")
add("- автоматическая эскалация сложных запросов;")
add("- накопление данных для дальнейшего улучшения системы.")

add()
add("---")
add()


# ------------------------------------------------------------
# 3. USERS
# ------------------------------------------------------------

add("# 3. Пользователи системы")
add()

add("Основные пользователи:")
add()

add("- сотрудники компании;")
add("- специалисты внутренней поддержки;")
add("- операторы, которые получают запросы на human review;")
add("- ML / Data-команда, сопровождающая систему.")

add()
add("---")
add()


# ------------------------------------------------------------
# 4. FUNCTIONAL REQUIREMENTS
# ------------------------------------------------------------

add("# 4. Функциональные требования")
add()

add("Система должна:")
add()

add("1. Принимать текстовый запрос пользователя.")
add("2. Определять intent запроса.")
add("3. Возвращать confidence классификатора.")
add("4. Использовать reject threshold для неуверенных запросов.")
add("5. Выполнять semantic retrieval.")
add("6. Искать релевантные фрагменты внутренних документов.")
add("7. Формировать grounded RAG-ответ.")
add("8. Маршрутизировать запросы через orchestration layer.")
add("9. Передавать недостаточно уверенные случаи человеку.")
add("10. Логировать предсказания, confidence и маршрут.")
add("11. Поддерживать обновление документной базы.")
add()

add("---")
add()


# ------------------------------------------------------------
# 5. NON-FUNCTIONAL REQUIREMENTS
# ------------------------------------------------------------

add("# 5. Нефункциональные требования")
add()

add("### Качество")
add()

add(
    "Для автоматически закрываемых intent-запросов "
    "целевой уровень automatic accuracy: **не ниже 95%**."
)

add()

add("### Надёжность")
add()

add(
    "При недостаточной уверенности запрос не должен "
    "автоматически закрываться и направляется на human review."
)

add()

add("### Latency")
add()

add(
    "Классификационный маршрут должен работать быстрее RAG-маршрута. "
    "LLM вызывается только там, где действительно требуется генерация."
)

add()

add("### Безопасность")
add()

add(
    "API-ключи и служебные credentials не должны храниться "
    "в GitHub-репозитории."
)

add()

add(
    "Document RAG должен формировать ответы только "
    "на основе найденных внутренних документов."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 6. ARCHITECTURE
# ------------------------------------------------------------

add("# 6. Архитектура системы")
add()

add("```mermaid")
add("flowchart TD")
add("    U[Employee / User] --> API[Application API]")
add("    API --> LG[LangGraph Orchestrator]")
add("    LG --> C[DistilBERT Intent Classifier]")
add("    C --> G{Confidence >= 0.65?}")
add("    G -->|Yes| A[Automatic intent response / action]")
add("    G -->|No| SR[Semantic Retrieval]")
add("    SR --> CG{Safe consensus?}")
add("    CG -->|Yes| A")
add("    CG -->|No| HR[Human Review]")
add("    LG --> DR[Document RAG Route]")
add("    DR --> E[Document Embeddings]")
add("    E --> VS[Vector Search]")
add("    VS --> CTX[Top-k Document Chunks]")
add("    CTX --> LLM[LLM Grounded Generation]")
add("    LLM --> RESP[Final Answer]")
add("    A --> RESP")
add("    HR --> RESP")
add("```")

add()

add(
    "В экспериментальной конфигурации дополнительная "
    "low-confidence consensus-ветка не дала безопасного "
    "прироста coverage при SLA >=95% automatic accuracy."
)

add()

add(
    "Поэтому финальная frozen-policy фактически использует "
    "DistilBERT + reject threshold как основной безопасный маршрут."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 7. COMPONENTS
# ------------------------------------------------------------

add("# 7. Основные компоненты")
add()

add("## 7.1 Intent Classifier")
add()

add(
    f"Модель: `{config['distilbert_model']}`"
)

add()

add(
    f"Количество классов: **{protocol['n_classes']}**"
)

add()

add(
    f"Количество эпох обучения: **{config['distilbert_epochs']}**"
)

add()

add(
    f"Confidence threshold: "
    f"**{config['selected_threshold_from_validation']}**"
)

add()

add("Назначение:")
add()

add("- определение intent;")
add("- выдача confidence;")
add("- принятие решения об auto-close или escalation.")

add()

add("## 7.2 Semantic Retrieval")
add()

add(
    f"Embedding model: `{config['semantic_model']}`"
)

add()

add(
    "Retrieval corpus строится только по `train_part`."
)

add()

add("Используется для:")
add()

add("- поиска похожих пользовательских запросов;")
add("- дополнительной проверки low-confidence classification;")
add("- retrieval-метрик Hit@K.")

add()

add("## 7.3 Document RAG")
add()

add("Основные этапы:")
add()

add("1. Разбиение документов на chunks.")
add("2. Построение embeddings.")
add("3. Vector similarity search.")
add("4. Получение top-k chunks.")
add("5. Формирование grounded prompt.")
add("6. Генерация ответа LLM.")
add("7. Возврат ответа вместе с источниками.")

add()

add(
    f"Evaluation generator model: "
    f"`{document_rag['evaluation_set']['generator_model']}`"
)

add()

add("## 7.4 LangGraph")
add()

add(
    "LangGraph используется как orchestration layer "
    "для маршрутизации запросов между ML-компонентами."
)

add()

add(
    "Его задача — управлять workflow, "
    "а не обязательно улучшать accuracy сам по себе."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 8. DATA
# ------------------------------------------------------------

add("# 8. Данные")
add()

add("### Banking77")
add()

add(
    f"- train_part: **{protocol['train_part_size']}**"
)

add(
    f"- val_part: **{protocol['val_part_size']}**"
)

add(
    f"- test: **{protocol['test_size']}**"
)

add()

add(
    f"Количество классов: **{protocol['n_classes']}**"
)

add()

add("Разделение:")
add()

add("- train_part — обучение моделей и retrieval corpus;")
add("- val_part — threshold и architecture selection;")
add("- test — только финальная оценка.")

add()

add("### Document RAG data")
add()

add(
    "Отдельная коллекция внутренних документов "
    "разбивается на semantic chunks."
)

add()

add(
    f"Для независимой RAG-оценки использовано "
    f"**{document_rag['evaluation_set']['n_examples']}** "
    "API-generated QA-примеров."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 9. TRAINING PIPELINE
# ------------------------------------------------------------

add("# 9. Training pipeline")
add()

add("1. Загрузка Banking77.")
add("2. Формирование train / validation / test.")
add("3. Обучение DistilBERT на train_part.")
add("4. Evaluation после каждой эпохи на val_part.")
add("5. Выбор лучшего checkpoint по Macro F1.")
add("6. Построение semantic retrieval corpus.")
add("7. Получение validation predictions.")
add("8. Sweep confidence threshold.")
add("9. Выбор threshold при automatic accuracy >=95%.")
add("10. Проверка LangGraph architecture на validation.")
add("11. Freeze всех параметров.")
add("12. Однократная final test evaluation.")

add()
add("---")
add()


# ------------------------------------------------------------
# 10. INFERENCE FLOW
# ------------------------------------------------------------

add("# 10. Online inference flow")
add()

add("### Intent-запрос")
add()

add(
    "`request → preprocessing → DistilBERT → confidence gate`"
)

add()

add(
    "Если confidence >= 0.65, "
    "intent принимается автоматически."
)

add()

add(
    "Если confidence ниже, "
    "запрос может пройти дополнительную retrieval-проверку "
    "или быть передан человеку."
)

add()

add("### Document-запрос")
add()

add(
    "`request → embedding → vector search → top-k chunks → "
    "LLM generation → answer`"
)

add()

add(
    "Если подходящего контекста недостаточно, "
    "система должна отказаться от уверенного ответа "
    "или передать запрос на ручную обработку."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 11. OFFLINE METRICS
# ------------------------------------------------------------

add("# 11. Offline metrics")
add()

add("## Intent classification")
add()

add(
    f"- Test Accuracy: **{test_accuracy:.2f}%**"
)

add(
    f"- Test Macro F1: **{test_macro_f1:.2f}%**"
)

add()

add("## Semantic retrieval")
add()

add(
    f"- Hit@1: **{retrieval_hit1:.2f}%**"
)

add(
    f"- Hit@3: **{retrieval_hit3:.2f}%**"
)

add(
    f"- Hit@5: **{retrieval_hit5:.2f}%**"
)

add()

add("## Selective classification")
add()

add(
    f"- Coverage: **{coverage:.2f}%**"
)

add(
    f"- Automatic accuracy: **{auto_accuracy:.2f}%**"
)

add(
    f"- Human review rate: **{review_rate:.2f}%**"
)

add()

add("## Document RAG")
add()

add(
    f"- Source accuracy: **{rag_source_accuracy:.2f}%**"
)

add(
    f"- Answer accuracy: **{rag_answer_accuracy:.2f}%**"
)

add(
    f"- Groundedness: **{rag_groundedness:.2f}%**"
)

add(
    f"- Overall RAG success: **{rag_overall:.2f}%**"
)

add()
add("---")
add()


# ------------------------------------------------------------
# 12. ONLINE METRICS
# ------------------------------------------------------------

add("# 12. Online metrics")
add()

add("Для production-системы необходимо мониторить:")
add()

add("- request volume;")
add("- p50 / p95 latency;")
add("- auto-resolution rate;")
add("- human review rate;")
add("- human override rate;")
add("- low-confidence rate;")
add("- no-answer rate;")
add("- RAG retrieval failure rate;")
add("- LLM/API error rate;")
add("- intent distribution drift;")
add("- confidence distribution drift;")
add("- document freshness.")

add()
add("---")
add()


# ------------------------------------------------------------
# 13. DEPLOYMENT
# ------------------------------------------------------------

add("# 13. Production deployment")
add()

add("Предлагаемая production-архитектура:")
add()

add("- FastAPI — внешний API;")
add("- Docker — упаковка сервисов;")
add("- DistilBERT inference service;")
add("- embedding service;")
add("- vector database: FAISS для prototype или Qdrant для production;")
add("- LangGraph orchestration service;")
add("- LLM API provider;")
add("- PostgreSQL / log storage;")
add("- monitoring stack;")
add("- human-review interface.")

add()

add("```mermaid")
add("flowchart LR")
add("    USER[User] --> API[FastAPI]")
add("    API --> GRAPH[LangGraph]")
add("    GRAPH --> CLS[Classifier Service]")
add("    GRAPH --> RET[Embedding / Retrieval Service]")
add("    RET --> VDB[(Vector DB)]")
add("    GRAPH --> LLM[LLM API]")
add("    GRAPH --> HUMAN[Human Review]")
add("    GRAPH --> LOG[(Logs / Metrics)]")
add("    GRAPH --> USER")
add("```")

add()
add("---")
add()


# ------------------------------------------------------------
# 14. MONITORING
# ------------------------------------------------------------

add("# 14. Monitoring")
add()

add("Необходимо контролировать:")
add()

add("- Accuracy по размеченным production-примерам;")
add("- изменение confidence distribution;")
add("- рост human review rate;")
add("- изменение частоты intents;")
add("- retrieval similarity distribution;")
add("- процент RAG-ответов без релевантного источника;")
add("- hallucination / groundedness signals;")
add("- latency и API failures.")

add()
add("---")
add()


# ------------------------------------------------------------
# 15. RETRAINING
# ------------------------------------------------------------

add("# 15. Retraining strategy")
add()

add("Источники новых данных:")
add()

add("- human review;")
add("- исправленные ответы операторов;")
add("- новые типы запросов;")
add("- новые документы;")
add("- production logs.")

add()

add("Предлагаемый цикл:")
add()

add("1. Сбор новых размеченных запросов.")
add("2. Data quality check.")
add("3. Обновление train corpus.")
add("4. Retraining.")
add("5. Validation threshold selection.")
add("6. Offline comparison с текущей production-моделью.")
add("7. Shadow / canary deployment.")
add("8. Production rollout при прохождении quality gate.")

add()
add("---")
add()


# ------------------------------------------------------------
# 16. DOCUMENT UPDATE
# ------------------------------------------------------------

add("# 16. Обновление документной базы")
add()

add(
    "Document RAG не требует переобучения intent-классификатора "
    "при каждом обновлении документа."
)

add()

add("При изменении внутренних документов:")
add()

add("1. Загружается новая версия документа.")
add("2. Документ повторно разбивается на chunks.")
add("3. Пересчитываются embeddings изменённых chunks.")
add("4. Vector index обновляется.")
add("5. Старая версия документа архивируется.")
add("6. Выполняется retrieval smoke-test.")

add()
add("---")
add()


# ------------------------------------------------------------
# 17. RISKS
# ------------------------------------------------------------

add("# 17. Риски")
add()

add("| Риск | Механизм снижения |")
add("|---|---|")
add(
    "| Неверная intent-классификация | "
    "Confidence threshold + human review |"
)
add(
    "| Overconfidence | "
    "Selective classification и мониторинг confidence |"
)
add(
    "| Hallucination RAG | "
    "Grounded prompt + retrieval context + source tracking |"
)
add(
    "| Устаревшие документы | "
    "Versioning и регулярное обновление vector index |"
)
add(
    "| API outage | "
    "Fallback / human review |"
)
add(
    "| Data drift | "
    "Monitoring + periodic retraining |"
)
add(
    "| Утечка API keys | "
    "Secrets storage, .env и .gitignore |"
)
add(
    "| Рост latency | "
    "Кэширование и разделение быстрых и LLM-маршрутов |"
)

add()
add("---")
add()


# ------------------------------------------------------------
# 18. LANGGRAPH CONCLUSION
# ------------------------------------------------------------

add("# 18. Результат LangGraph-эксперимента")
add()

add(
    "На validation была проверена гипотеза, "
    "что согласие DistilBERT и semantic retrieval "
    "позволит безопасно закрывать дополнительные "
    "low-confidence запросы."
)

add()

add(
    "Raw accuracy consensus-группы оказалась недостаточной "
    "для требуемого SLA."
)

add()

add(
    "После добавления similarity gate ни один дополнительный "
    "low-confidence запрос не мог быть безопасно закрыт "
    "при сохранении общей automatic accuracy >=95%."
)

add()

add(
    "Поэтому более сложная схема не была искусственно выбрана "
    "как лучшая. Финальная policy совпала с простым "
    "`DistilBERT + reject threshold` baseline."
)

add()

add(
    "LangGraph при этом остаётся полезным orchestration layer "
    "для объединения классификации, RAG, fallback и human review."
)

add()
add("---")
add()


# ------------------------------------------------------------
# 19. FUTURE WORK
# ------------------------------------------------------------

add("# 19. Возможные улучшения")
add()

add("- добавить cross-encoder reranker для Document RAG;")
add("- перейти с in-memory/FAISS retrieval на Qdrant;")
add("- использовать hybrid search: dense + BM25;")
add("- внедрить structured output;")
add("- добавить tool calling для внутренних сервисов;")
add("- реализовать query rewriting;")
add("- реализовать document versioning;")
add("- добавить online feedback loop;")
add("- добавить observability для LangGraph nodes;")
add("- протестировать более сильный embedding model;")
add("- расширить независимый RAG benchmark.")

add()
add("---")
add()


# ------------------------------------------------------------
# 20. FINAL CONCLUSION
# ------------------------------------------------------------

add("# 20. Итог")
add()

add(
    "Разработанный prototype показывает, что система "
    "может автоматически обрабатывать большую долю запросов "
    "при контролируемом уровне качества."
)

add()

add(
    f"На финальном test selective baseline автоматически "
    f"закрывает **{coverage:.2f}%** запросов "
    f"с automatic accuracy **{auto_accuracy:.2f}%**."
)

add()

add(
    "Document RAG показывает высокую точность ответов, "
    "но groundedness и overall success остаются "
    "основными направлениями улучшения."
)

add()

add(
    "Разделение train / validation / test и отдельный "
    "API-generated RAG benchmark позволяют избежать "
    "подгонки финальных метрик под evaluation dataset."
)


# ------------------------------------------------------------
# 5. Сохраняем MLSD
# ------------------------------------------------------------

mlsd_text = "\n".join(
    lines
)


with open(
    MLSD_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        mlsd_text
    )


# ------------------------------------------------------------
# 6. Проверяем
# ------------------------------------------------------------

if not MLSD_PATH.exists():
    raise RuntimeError(
        "docs/mlsd.md не удалось создать."
    )


print("=" * 70)
print("ML SYSTEM DESIGN CREATED")
print("=" * 70)

print(
    "Path:"
)

print(
    MLSD_PATH
)

print()

print(
    "Size:",
    MLSD_PATH.stat().st_size,
    "bytes"
)

print()

print(
    "Sections:",
    20
)

print()

print("=" * 70)
print("CELL 4 COMPLETE")
print("=" * 70)

print(
    "MLSD документ создан."
)

print(
    "Следующий шаг — проверить структуру "
    "и подготовить архитектурные схемы/assets."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 5.
# CREATE ARCHITECTURE DIAGRAMS
#
# Создаём отдельные Mermaid-схемы:
# 1. High-level architecture
# 2. Intent inference flow
# 3. Document RAG flow
# 4. Training / evaluation flow
#
# Сохраняем в assets/
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


ASSETS_DIR = (
    GITHUB_PROJECT_DIR
    / "assets"
)

ASSETS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Вспомогательная функция
# ------------------------------------------------------------

def build_text(lines):
    return "\n".join(lines)


# ------------------------------------------------------------
# 3. High-level architecture
# ------------------------------------------------------------

high_level = build_text([
    "# High-Level Architecture",
    "",
    "```mermaid",
    "flowchart LR",
    "    U[Employee / User] --> API[Application API]",
    "    API --> LG[LangGraph Orchestrator]",
    "    LG --> CLS[DistilBERT Intent Classifier]",
    "    CLS --> CONF{Confidence >= 0.65?}",
    "    CONF -->|Yes| AUTO[Automatic Intent Resolution]",
    "    CONF -->|No| SR[Semantic Retrieval]",
    "    SR --> CONS{Safe Consensus?}",
    "    CONS -->|Yes| AUTO",
    "    CONS -->|No| HUMAN[Human Review]",
    "    LG --> DOC{Document Question?}",
    "    DOC -->|Yes| RAG[Document RAG]",
    "    RAG --> EMB[Document Embeddings]",
    "    EMB --> VDB[Vector Search]",
    "    VDB --> CHUNKS[Top-k Chunks]",
    "    CHUNKS --> LLM[Grounded LLM Generation]",
    "    AUTO --> RESP[Final Response]",
    "    HUMAN --> RESP",
    "    LLM --> RESP",
    "```",
])


# ------------------------------------------------------------
# 4. Intent inference flow
# ------------------------------------------------------------

intent_flow = build_text([
    "# Intent Classification Flow",
    "",
    "```mermaid",
    "flowchart TD",
    "    Q[Incoming Query] --> PRE[Preprocessing]",
    "    PRE --> DB[DistilBERT]",
    "    DB --> P[Intent Prediction + Confidence]",
    "    P --> G{Confidence >= 0.65?}",
    "    G -->|Yes| CLOSE[Auto-close]",
    "    G -->|No| RET[Semantic Retrieval]",
    "    RET --> AGREE{DistilBERT label = Retrieval label?}",
    "    AGREE -->|No| REVIEW[Human Review]",
    "    AGREE -->|Yes| SIM{Similarity passes gate?}",
    "    SIM -->|Yes| CLOSE",
    "    SIM -->|No| REVIEW",
    "```",
    "",
    "В финальном эксперименте similarity gate был выбран как 1.0,",
    "поэтому low-confidence consensus не дал дополнительной безопасной coverage.",
])


# ------------------------------------------------------------
# 5. Document RAG flow
# ------------------------------------------------------------

rag_flow = build_text([
    "# Document RAG Flow",
    "",
    "```mermaid",
    "flowchart TD",
    "    Q[Employee Question] --> QE[Query Embedding]",
    "    QE --> SEARCH[Vector Similarity Search]",
    "    SEARCH --> TOPK[Top-k Document Chunks]",
    "    TOPK --> CTX[Grounded Context Builder]",
    "    CTX --> PROMPT[Prompt]",
    "    PROMPT --> LLM[LLM Generator]",
    "    LLM --> ANSWER[Answer]",
    "    TOPK --> SOURCES[Source Metadata]",
    "    SOURCES --> ANSWER",
    "    ANSWER --> CHECK{Sufficient grounded answer?}",
    "    CHECK -->|Yes| FINAL[Return Answer]",
    "    CHECK -->|No| HUMAN[Human Review / Escalation]",
    "```",
])


# ------------------------------------------------------------
# 6. Training / evaluation flow
# ------------------------------------------------------------

training_flow = build_text([
    "# Training and Evaluation Flow",
    "",
    "```mermaid",
    "flowchart TD",
    "    DATA[Banking77] --> SPLIT[Train / Validation / Test Split]",
    "    SPLIT --> TRAIN[train_part]",
    "    SPLIT --> VAL[val_part]",
    "    SPLIT --> TEST[test]",
    "    TRAIN --> DBTRAIN[Train DistilBERT]",
    "    TRAIN --> RETCORPUS[Build Retrieval Corpus]",
    "    DBTRAIN --> VALEVAL[Validation Evaluation]",
    "    RETCORPUS --> VALEVAL",
    "    VAL --> VALEVAL",
    "    VALEVAL --> THRESH[Select Confidence Threshold]",
    "    VALEVAL --> ARCH[Select LangGraph Architecture]",
    "    THRESH --> FREEZE[Freeze Decisions]",
    "    ARCH --> FREEZE",
    "    FREEZE --> FINALTEST[Single Final Test]",
    "    TEST --> FINALTEST",
    "    FINALTEST --> METRICS[Final Metrics]",
    "```",
    "",
    "Методологическое правило:",
    "test используется только после выбора всех параметров на validation.",
])


# ------------------------------------------------------------
# 7. Сохраняем файлы
# ------------------------------------------------------------

diagrams = {
    "architecture_high_level.md": high_level,
    "intent_inference_flow.md": intent_flow,
    "document_rag_flow.md": rag_flow,
    "training_evaluation_flow.md": training_flow,
}


for filename, content in diagrams.items():

    path = (
        ASSETS_DIR
        / filename
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            content
        )


# ------------------------------------------------------------
# 8. Проверяем
# ------------------------------------------------------------

print("=" * 70)
print("ARCHITECTURE DIAGRAMS")
print("=" * 70)


for filename in diagrams:

    path = (
        ASSETS_DIR
        / filename
    )

    status = (
        "OK"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{status:8} | assets/{filename}"
    )


print()
print("=" * 70)
print("CELL 5 COMPLETE")
print("=" * 70)

print(
    "Mermaid-схемы созданы."
)

print(
    "Следующий шаг — добавить ссылки на MLSD "
    "и схемы в README."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 6.
# UPDATE README WITH MLSD + ARCHITECTURE LINKS
#
# Добавляем в README:
# - ссылку на MLSD
# - ссылки на Mermaid-схемы
# - структуру репозитория
# - краткие итоговые метрики
#
# Существующий README сохраняется.
# Новый блок добавляется только один раз.
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


README_PATH = (
    GITHUB_PROJECT_DIR
    / "README.md"
)

SUMMARY_PATH = (
    GITHUB_PROJECT_DIR
    / "results"
    / "FINAL_CORRECTED_SUMMARY.json"
)


if not README_PATH.exists():
    raise RuntimeError(
        "README.md не найден. "
        "Сначала выполни ячейку 2."
    )

if not SUMMARY_PATH.exists():
    raise RuntimeError(
        "FINAL_CORRECTED_SUMMARY.json не найден."
    )


# ------------------------------------------------------------
# 2. Загружаем summary
# ------------------------------------------------------------

with open(
    SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:
    summary = json.load(f)


final_test = summary["final_test"]
document_rag = summary["document_rag_api_qa"]


accuracy = (
    final_test["models"]["distilbert_accuracy"]
    * 100
)

macro_f1 = (
    final_test["models"]["distilbert_macro_f1"]
    * 100
)

coverage = (
    final_test["reject_baseline"]["coverage"]
    * 100
)

auto_accuracy = (
    final_test["reject_baseline"]["automatic_accuracy"]
    * 100
)

rag_answer_accuracy = (
    document_rag["answer_quality"]["answer_accuracy_llm_judge"]
    * 100
)

rag_overall = (
    document_rag["answer_quality"]["overall_rag_success"]
    * 100
)


# ------------------------------------------------------------
# 3. Читаем текущий README
# ------------------------------------------------------------

with open(
    README_PATH,
    "r",
    encoding="utf-8"
) as f:
    current_readme = f.read()


# ------------------------------------------------------------
# 4. Уникальный маркер блока
# ------------------------------------------------------------

BLOCK_MARKER = (
    "<!-- GITHUB_MLSD_SECTION -->"
)


# ------------------------------------------------------------
# 5. Формируем новый блок
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


add()
add("---")
add()
add(BLOCK_MARKER)
add()
add("# ML System Design и архитектура")
add()

add(
    "Для проекта подготовлен отдельный ML System Design документ, "
    "описывающий архитектуру системы, data flow, ML-компоненты, "
    "deployment, monitoring, retraining strategy и основные риски."
)

add()

add("### Основные документы")
add()

add("- [ML System Design](docs/mlsd.md)")
add("- [High-Level Architecture](assets/architecture_high_level.md)")
add("- [Intent Inference Flow](assets/intent_inference_flow.md)")
add("- [Document RAG Flow](assets/document_rag_flow.md)")
add("- [Training & Evaluation Flow](assets/training_evaluation_flow.md)")

add()
add("## Финальные результаты")
add()

add("| Метрика | Значение |")
add("|---|---:|")
add(
    f"| DistilBERT Test Accuracy | {accuracy:.2f}% |"
)
add(
    f"| DistilBERT Test Macro F1 | {macro_f1:.2f}% |"
)
add(
    f"| Selective Coverage | {coverage:.2f}% |"
)
add(
    f"| Automatic Accuracy | {auto_accuracy:.2f}% |"
)
add(
    f"| Document RAG Answer Accuracy | {rag_answer_accuracy:.2f}% |"
)
add(
    f"| Document RAG Overall Success | {rag_overall:.2f}% |"
)

add()
add("## Структура репозитория")
add()

add("```text")
add("ml_customer_support_assistant/")
add("│")
add("├── README.md")
add("├── .gitignore")
add("│")
add("├── src/")
add("│   ├── __init__.py")
add("│   ├── classifier.py")
add("│   ├── config.py")
add("│   ├── rag.py")
add("│   ├── retrieval.py")
add("│   ├── routing.py")
add("│   └── label_mapping.json")
add("│")
add("├── validation/")
add("│   ├── selected_threshold_final.json")
add("│   └── distilbert_final_threshold_sweep.csv")
add("│")
add("├── langgraph/")
add("│   ├── final_langgraph_architecture.json")
add("│   ├── low_confidence_consensus_validation.csv")
add("│   └── retrieval_similarity_threshold_sweep.csv")
add("│")
add("├── document_rag/")
add("│   ├── api_generated_qa.csv")
add("│   ├── api_generated_qa_rag_metrics.json")
add("│   ├── api_generated_qa_final_metrics.json")
add("│   └── api_generated_qa_final_evaluation.csv")
add("│")
add("├── results/")
add("│   ├── FINAL_CORRECTED_SUMMARY.json")
add("│   ├── final_test_metrics.json")
add("│   ├── report_classification_metrics.csv")
add("│   ├── report_baseline_vs_langgraph.csv")
add("│   ├── report_document_rag_metrics.csv")
add("│   ├── report_evaluation_protocol.csv")
add("│   └── DEFENSE_CONCLUSIONS.txt")
add("│")
add("├── docs/")
add("│   └── mlsd.md")
add("│")
add("└── assets/")
add("    ├── architecture_high_level.md")
add("    ├── intent_inference_flow.md")
add("    ├── document_rag_flow.md")
add("    └── training_evaluation_flow.md")
add("```")

add()
add("## Ограничения")
add()

add(
    "Тяжёлые model checkpoints и parquet-датасеты "
    "не хранятся в GitHub-репозитории."
)

add()

add(
    "Финальная модель и исходные train / validation / test artifacts "
    "сохраняются отдельно, а репозиторий содержит код, "
    "конфигурацию и воспроизводимые результаты."
)

add()

add(
    "API keys и другие secrets в репозиторий не включаются."
)


new_block = "\n".join(lines)


# ------------------------------------------------------------
# 6. Добавляем блок только если его ещё нет
# ------------------------------------------------------------

if BLOCK_MARKER in current_readme:

    print(
        "MLSD section already exists in README."
    )

    updated_readme = current_readme

else:

    updated_readme = (
        current_readme.rstrip()
        + "\n"
        + new_block
        + "\n"
    )

    with open(
        README_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            updated_readme
        )

    print(
        "MLSD section added to README."
    )


# ------------------------------------------------------------
# 7. Проверяем ссылки
# ------------------------------------------------------------

required_paths = [

    "docs/mlsd.md",

    "assets/architecture_high_level.md",

    "assets/intent_inference_flow.md",

    "assets/document_rag_flow.md",

    "assets/training_evaluation_flow.md"
]


print()
print("=" * 70)
print("README LINK CHECK")
print("=" * 70)


missing = []


for relative_path in required_paths:

    full_path = (
        GITHUB_PROJECT_DIR
        / relative_path
    )

    if full_path.exists():

        status = "OK"

    else:

        status = "MISSING"
        missing.append(
            relative_path
        )

    print(
        f"{status:8} | {relative_path}"
    )


# ------------------------------------------------------------
# 8. Итоговая информация
# ------------------------------------------------------------

readme_size_kb = (
    README_PATH.stat().st_size
    / 1024
)


print()
print("=" * 70)
print("CELL 6 COMPLETE")
print("=" * 70)

print(
    "README:"
)

print(
    README_PATH
)

print()

print(
    "README size:",
    f"{readme_size_kb:.1f} KB"
)

print()

if len(missing) == 0:

    print(
        "Все ссылки на MLSD и архитектурные схемы валидны."
    )

else:

    print(
        "Есть отсутствующие файлы:"
    )

    for item in missing:

        print(
            "-",
            item
        )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 7.
# AUDIT PROJECT AGAINST ORIGINAL REQUIREMENTS
#
# Проверяем исходные требования проекта:
# - LangGraph
# - LLM
# - RAG
# - Embeddings
# - Vector Search
# - Reranker
# - Structured Output
# - Tool Calling
# - Python
#
# ВАЖНО:
# Разделяем:
# 1. implemented in prototype
# 2. partially implemented
# 3. proposed for production
#
# Это нужно для корректного MLSD и GitHub README.
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


DOCS_DIR = (
    GITHUB_PROJECT_DIR
    / "docs"
)

AUDIT_PATH = (
    DOCS_DIR
    / "requirements_audit.md"
)


# ------------------------------------------------------------
# 2. Определяем требования и статус
# ------------------------------------------------------------

requirements = [

    {
        "component": "Python",
        "status": "IMPLEMENTED",
        "evidence": (
            "Основной prototype, обучение, evaluation, "
            "retrieval и RAG реализованы на Python."
        ),
        "production_note": (
            "Python остаётся основным языком backend/ML service."
        )
    },

    {
        "component": "DistilBERT / Intent Classification",
        "status": "IMPLEMENTED",
        "evidence": (
            "Обучен DistilBERT для 77 intents Banking77. "
            "Финальная test accuracy 92.56%, Macro F1 92.55%."
        ),
        "production_note": (
            "Используется как быстрый intent-classification service."
        )
    },

    {
        "component": "Embeddings",
        "status": "IMPLEMENTED",
        "evidence": (
            "Использован sentence-transformers/all-MiniLM-L6-v2 "
            "для semantic retrieval и Document RAG."
        ),
        "production_note": (
            "Embedding service может быть выделен "
            "в отдельный production-компонент."
        )
    },

    {
        "component": "Semantic Retrieval",
        "status": "IMPLEMENTED",
        "evidence": (
            "Построен retrieval corpus только по train_part. "
            "На final test Hit@1=91.42%, Hit@3=95.48%, "
            "Hit@5=96.72%."
        ),
        "production_note": (
            "Может использоваться для fallback и поиска похожих запросов."
        )
    },

    {
        "component": "RAG",
        "status": "IMPLEMENTED",
        "evidence": (
            "Document RAG реализован и оценён на независимом "
            "API-generated QA benchmark."
        ),
        "production_note": (
            "Production-версия должна включать обновляемый "
            "vector index и document versioning."
        )
    },

    {
        "component": "LLM",
        "status": "IMPLEMENTED",
        "evidence": (
            "Для Document RAG и offline evaluation использовался "
            "LLM через Groq API."
        ),
        "production_note": (
            "В production LLM должен использоваться только "
            "для генеративных маршрутов."
        )
    },

    {
        "component": "LangGraph",
        "status": "PARTIAL",
        "evidence": (
            "Проверена orchestration / routing логика для "
            "selective classification и consensus route. "
            "Финальная безопасная policy не дала прироста "
            "к baseline при SLA >=95% automatic accuracy."
        ),
        "production_note": (
            "В MLSD LangGraph рассматривается как единый "
            "orchestration layer для classifier, RAG, fallback "
            "и human review."
        )
    },

    {
        "component": "Vector Database",
        "status": "PARTIAL",
        "evidence": (
            "В prototype использован semantic vector search "
            "по embeddings."
        ),
        "production_note": (
            "Для production предлагается Qdrant. "
            "FAISS подходит для локального prototype."
        )
    },

    {
        "component": "Reranker",
        "status": "PROPOSED",
        "evidence": (
            "Отдельный cross-encoder reranker в финальном "
            "prototype не реализован."
        ),
        "production_note": (
            "Предлагается добавить после initial retrieval "
            "для повышения качества Document RAG."
        )
    },

    {
        "component": "Structured Output",
        "status": "PROPOSED",
        "evidence": (
            "Формальный schema-enforced structured output "
            "в prototype не является отдельным production-компонентом."
        ),
        "production_note": (
            "Предлагается использовать Pydantic/JSON schema "
            "для router и LLM responses."
        )
    },

    {
        "component": "Tool Calling",
        "status": "PROPOSED",
        "evidence": (
            "Внешние business tools в prototype не подключались."
        ),
        "production_note": (
            "Можно добавить вызов внутренних сервисов: "
            "поиск статуса заявки, справочники, CRM и т.д."
        )
    },

    {
        "component": "Human Review",
        "status": "IMPLEMENTED_AS_POLICY",
        "evidence": (
            "Reject-policy рассчитывает долю запросов, "
            "которые должны быть переданы человеку."
        ),
        "production_note": (
            "Для production нужен отдельный интерфейс "
            "оператора и feedback loop."
        )
    },

    {
        "component": "Monitoring",
        "status": "DESIGNED",
        "evidence": (
            "Offline metrics рассчитаны; production monitoring "
            "описан в MLSD."
        ),
        "production_note": (
            "Нужно реализовать сбор latency, drift, review rate, "
            "errors и RAG quality."
        )
    }
]


# ------------------------------------------------------------
# 3. Формируем audit document
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


add("# Audit of Original Project Requirements")
add()

add(
    "Этот документ разделяет реально реализованные "
    "компоненты prototype и элементы, которые описаны "
    "только как production design."
)

add()

add("| Компонент | Статус | Что есть сейчас | Production design |")
add("|---|---|---|---|")


for item in requirements:

    add(
        f"| {item['component']} "
        f"| {item['status']} "
        f"| {item['evidence']} "
        f"| {item['production_note']} |"
    )


add()
add("## Обозначения")
add()

add(
    "- **IMPLEMENTED** — компонент реально реализован "
    "и использовался в эксперименте."
)

add(
    "- **IMPLEMENTED_AS_POLICY** — логика реализована "
    "на уровне experimental routing / evaluation policy."
)

add(
    "- **PARTIAL** — ключевая идея реализована, "
    "но не в полном production-виде."
)

add(
    "- **DESIGNED** — компонент описан в MLSD, "
    "но production infrastructure ещё не развёрнута."
)

add(
    "- **PROPOSED** — компонент является рекомендуемым "
    "расширением архитектуры и не должен описываться "
    "как уже реализованный."
)

add()
add("## Ключевой вывод")
add()

add(
    "Prototype закрывает основную ML-часть задачи: "
    "intent classification, embeddings, semantic retrieval, "
    "Document RAG, LLM evaluation и selective routing."
)

add()

add(
    "Reranker, полноценная vector database, structured output, "
    "tool calling, production monitoring и полноценный "
    "end-to-end LangGraph workflow относятся к production design."
)

add()

add(
    "В README и MLSD эти компоненты необходимо явно отделять "
    "от того, что было реально реализовано и измерено."
)


audit_text = "\n".join(lines)


# ------------------------------------------------------------
# 4. Сохраняем
# ------------------------------------------------------------

with open(
    AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        audit_text
    )


# ------------------------------------------------------------
# 5. Выводим таблицу статусов
# ------------------------------------------------------------

print("=" * 70)
print("ORIGINAL REQUIREMENTS AUDIT")
print("=" * 70)


for item in requirements:

    print(
        f"{item['status']:22} | "
        f"{item['component']}"
    )


print()
print("=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

implemented = sum(
    item["status"] == "IMPLEMENTED"
    for item in requirements
)

partial = sum(
    item["status"] == "PARTIAL"
    for item in requirements
)

proposed = sum(
    item["status"] == "PROPOSED"
    for item in requirements
)

designed = sum(
    item["status"] == "DESIGNED"
    for item in requirements
)

policy = sum(
    item["status"] == "IMPLEMENTED_AS_POLICY"
    for item in requirements
)


print(
    "Implemented:",
    implemented
)

print(
    "Implemented as policy:",
    policy
)

print(
    "Partial:",
    partial
)

print(
    "Designed:",
    designed
)

print(
    "Proposed:",
    proposed
)

print()

print(
    "Saved:"
)

print(
    AUDIT_PATH
)

print()
print("=" * 70)
print("CELL 7 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 8.
# CREATE END-TO-END LANGGRAPH ORCHESTRATOR
#
# Закрываем один из главных пунктов исходного ТЗ:
#
# determine query type
#        ↓
# classify / retrieve
#        ↓
# additional search if needed
#        ↓
# check result
#        ↓
# final answer / human review
#
# ВАЖНО:
# - граф реальный;
# - orchestration отделён от ML-моделей;
# - модели подключаются через dependency injection;
# - никаких fake production results;
# - test metrics не пересчитываются.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


SRC_DIR = (
    GITHUB_PROJECT_DIR
    / "src"
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True
)


LANGGRAPH_APP_PATH = (
    SRC_DIR
    / "langgraph_app.py"
)


# ------------------------------------------------------------
# 2. Формируем LangGraph module
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

add('"""End-to-end LangGraph orchestration for the assistant.')
add()
add("The graph coordinates:")
add("- query type detection")
add("- intent classification")
add("- semantic/document retrieval")
add("- optional additional search")
add("- answer validation")
add("- human-review fallback")
add("- final response")
add()
add("ML models and external services are injected as adapters.")
add('"""')
add()

add("from __future__ import annotations")
add()

add("from dataclasses import dataclass")
add("from typing import Any, Callable, Dict, List, Literal, Optional, TypedDict")
add()

add("from langgraph.graph import END, START, StateGraph")
add()


# ------------------------------------------------------------
# Types
# ------------------------------------------------------------

add()
add("# ------------------------------------------------------------")
add("# State")
add("# ------------------------------------------------------------")
add()

add("QueryType = Literal['intent', 'document']")
add()

add("class AssistantState(TypedDict, total=False):")
add("    query: str")
add("    query_type: QueryType")
add()
add("    intent_id: Optional[int]")
add("    intent_name: Optional[str]")
add("    classifier_confidence: Optional[float]")
add()
add("    retrieval_label: Optional[int]")
add("    retrieval_similarity: Optional[float]")
add()
add("    retrieved_chunks: List[Dict[str, Any]]")
add("    additional_chunks: List[Dict[str, Any]]")
add()
add("    generated_answer: Optional[str]")
add()
add("    answer_is_valid: bool")
add("    needs_additional_search: bool")
add("    needs_human_review: bool")
add()
add("    route: Optional[str]")
add("    final_answer: Optional[str]")
add()


# ------------------------------------------------------------
# Adapter dataclass
# ------------------------------------------------------------

add()
add("# ------------------------------------------------------------")
add("# External ML/service adapters")
add("# ------------------------------------------------------------")
add()

add("@dataclass")
add("class AssistantAdapters:")
add('    """Functions supplied by the application layer."""')
add()
add("    determine_query_type: Callable[[str], QueryType]")
add()
add(
    "    classify_intent: "
    "Callable[[str], Dict[str, Any]]"
)
add()
add(
    "    semantic_retrieve: "
    "Callable[[str], Dict[str, Any]]"
)
add()
add(
    "    document_retrieve: "
    "Callable[[str, int], List[Dict[str, Any]]]"
)
add()
add(
    "    generate_answer: "
    "Callable[[str, List[Dict[str, Any]]], str]"
)
add()
add(
    "    validate_answer: "
    "Callable[[str, str, List[Dict[str, Any]]], bool]"
)
add()


# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

add()
add("# ------------------------------------------------------------")
add("# Frozen experimental policy")
add("# ------------------------------------------------------------")
add()

add("DISTILBERT_THRESHOLD = 0.65")
add("RETRIEVAL_SIMILARITY_GATE = 1.0")
add("RAG_TOP_K = 3")
add("RAG_ADDITIONAL_TOP_K = 5")
add()


# ------------------------------------------------------------
# Graph builder
# ------------------------------------------------------------

add()
add("# ------------------------------------------------------------")
add("# Graph builder")
add("# ------------------------------------------------------------")
add()

add("def build_assistant_graph(adapters: AssistantAdapters):")
add('    """Build and compile the assistant LangGraph."""')
add()


# determine_query_type
add("    def determine_query_type_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        query_type = adapters.determine_query_type(")
add("            state['query']")
add("        )")
add()
add("        return {")
add("            'query_type': query_type,")
add("        }")
add()


# classify
add("    def classify_intent_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        result = adapters.classify_intent(")
add("            state['query']")
add("        )")
add()
add("        return {")
add("            'intent_id': result.get('intent_id'),")
add("            'intent_name': result.get('intent_name'),")
add("            'classifier_confidence': float(")
add("                result.get('confidence', 0.0)")
add("            ),")
add("        }")
add()


# semantic retrieval
add("    def semantic_retrieval_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        result = adapters.semantic_retrieve(")
add("            state['query']")
add("        )")
add()
add("        return {")
add("            'retrieval_label': result.get('label'),")
add("            'retrieval_similarity': float(")
add("                result.get('similarity', 0.0)")
add("            ),")
add("        }")
add()


# intent check
add("    def check_intent_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        confidence = float(")
add("            state.get('classifier_confidence') or 0.0")
add("        )")
add()
add("        classifier_label = state.get('intent_id')")
add("        retrieval_label = state.get('retrieval_label')")
add()
add("        retrieval_similarity = float(")
add("            state.get('retrieval_similarity') or 0.0")
add("        )")
add()
add("        if confidence >= DISTILBERT_THRESHOLD:")
add("            return {")
add("                'needs_human_review': False,")
add("                'route': 'distilbert_high_confidence',")
add("            }")
add()
add("        safe_consensus = (")
add("            classifier_label is not None")
add("            and retrieval_label is not None")
add("            and classifier_label == retrieval_label")
add("            and retrieval_similarity")
add("            >= RETRIEVAL_SIMILARITY_GATE")
add("        )")
add()
add("        if safe_consensus:")
add("            return {")
add("                'needs_human_review': False,")
add("                'route': 'low_confidence_consensus',")
add("            }")
add()
add("        return {")
add("            'needs_human_review': True,")
add("            'route': 'human_review',")
add("        }")
add()


# intent response
add("    def form_intent_answer_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        intent_name = state.get('intent_name')")
add()
add("        return {")
add("            'final_answer': (")
add("                f'Intent: {intent_name}'")
add("                if intent_name")
add("                else 'Intent identified.'")
add("            )")
add("        }")
add()


# document retrieve
add("    def document_retrieval_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        chunks = adapters.document_retrieve(")
add("            state['query'],")
add("            RAG_TOP_K,")
add("        )")
add()
add("        return {")
add("            'retrieved_chunks': chunks,")
add("        }")
add()


# generation
add("    def generate_answer_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        chunks = state.get(")
add("            'retrieved_chunks',")
add("            [],")
add("        )")
add()
add("        answer = adapters.generate_answer(")
add("            state['query'],")
add("            chunks,")
add("        )")
add()
add("        return {")
add("            'generated_answer': answer,")
add("        }")
add()


# validate
add("    def validate_answer_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        chunks = state.get(")
add("            'retrieved_chunks',")
add("            [],")
add("        )")
add()
add("        answer = state.get(")
add("            'generated_answer'")
add("        ) or ''")
add()
add("        valid = adapters.validate_answer(")
add("            state['query'],")
add("            answer,")
add("            chunks,")
add("        )")
add()
add("        return {")
add("            'answer_is_valid': bool(valid),")
add("            'needs_additional_search': not bool(valid),")
add("        }")
add()


# additional retrieval
add("    def additional_search_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        chunks = adapters.document_retrieve(")
add("            state['query'],")
add("            RAG_ADDITIONAL_TOP_K,")
add("        )")
add()
add("        return {")
add("            'retrieved_chunks': chunks,")
add("            'additional_chunks': chunks,")
add("            'needs_additional_search': False,")
add("        }")
add()


# validate after additional search
add("    def validate_after_additional_search_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        chunks = state.get(")
add("            'retrieved_chunks',")
add("            [],")
add("        )")
add()
add("        answer = adapters.generate_answer(")
add("            state['query'],")
add("            chunks,")
add("        )")
add()
add("        valid = adapters.validate_answer(")
add("            state['query'],")
add("            answer,")
add("            chunks,")
add("        )")
add()
add("        return {")
add("            'generated_answer': answer,")
add("            'answer_is_valid': bool(valid),")
add("            'needs_human_review': not bool(valid),")
add("            'route': (")
add("                'document_rag'")
add("                if valid")
add("                else 'human_review'")
add("            ),")
add("        }")
add()


# final doc answer
add("    def form_document_answer_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        return {")
add("            'final_answer': state.get(")
add("                'generated_answer'")
add("            )")
add("        }")
add()


# human review
add("    def human_review_node(")
add("        state: AssistantState,")
add("    ) -> AssistantState:")
add()
add("        return {")
add("            'needs_human_review': True,")
add("            'route': 'human_review',")
add("            'final_answer': (")
add("                'Request requires human review.'")
add("            ),")
add("        }")
add()


# routers
add()
add("    def route_by_query_type(")
add("        state: AssistantState,")
add("    ) -> str:")
add()
add("        if state.get('query_type') == 'document':")
add("            return 'document'")
add("        return 'intent'")
add()

add()
add("    def route_after_intent_check(")
add("        state: AssistantState,")
add("    ) -> str:")
add()
add("        if state.get('needs_human_review'):")
add("            return 'human_review'")
add("        return 'automatic'")
add()

add()
add("    def route_after_rag_validation(")
add("        state: AssistantState,")
add("    ) -> str:")
add()
add("        if state.get('answer_is_valid'):")
add("            return 'valid'")
add("        return 'additional_search'")
add()

add()
add("    def route_after_second_validation(")
add("        state: AssistantState,")
add("    ) -> str:")
add()
add("        if state.get('answer_is_valid'):")
add("            return 'valid'")
add("        return 'human_review'")
add()


# ------------------------------------------------------------
# Build graph
# ------------------------------------------------------------

add()
add("    graph = StateGraph(AssistantState)")
add()

add("    graph.add_node(")
add("        'determine_query_type',")
add("        determine_query_type_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'classify_intent',")
add("        classify_intent_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'semantic_retrieval',")
add("        semantic_retrieval_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'check_intent',")
add("        check_intent_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'form_intent_answer',")
add("        form_intent_answer_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'document_retrieval',")
add("        document_retrieval_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'generate_answer',")
add("        generate_answer_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'validate_answer',")
add("        validate_answer_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'additional_search',")
add("        additional_search_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'validate_after_additional_search',")
add("        validate_after_additional_search_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'form_document_answer',")
add("        form_document_answer_node,")
add("    )")
add()

add("    graph.add_node(")
add("        'human_review',")
add("        human_review_node,")
add("    )")
add()


# edges
add()
add("    graph.add_edge(")
add("        START,")
add("        'determine_query_type',")
add("    )")
add()

add("    graph.add_conditional_edges(")
add("        'determine_query_type',")
add("        route_by_query_type,")
add("        {")
add("            'intent': 'classify_intent',")
add("            'document': 'document_retrieval',")
add("        },")
add("    )")
add()

add("    graph.add_edge(")
add("        'classify_intent',")
add("        'semantic_retrieval',")
add("    )")
add()

add("    graph.add_edge(")
add("        'semantic_retrieval',")
add("        'check_intent',")
add("    )")
add()

add("    graph.add_conditional_edges(")
add("        'check_intent',")
add("        route_after_intent_check,")
add("        {")
add("            'automatic': 'form_intent_answer',")
add("            'human_review': 'human_review',")
add("        },")
add("    )")
add()

add("    graph.add_edge(")
add("        'form_intent_answer',")
add("        END,")
add("    )")
add()

add("    graph.add_edge(")
add("        'document_retrieval',")
add("        'generate_answer',")
add("    )")
add()

add("    graph.add_edge(")
add("        'generate_answer',")
add("        'validate_answer',")
add("    )")
add()

add("    graph.add_conditional_edges(")
add("        'validate_answer',")
add("        route_after_rag_validation,")
add("        {")
add("            'valid': 'form_document_answer',")
add("            'additional_search': 'additional_search',")
add("        },")
add("    )")
add()

add("    graph.add_edge(")
add("        'additional_search',")
add("        'validate_after_additional_search',")
add("    )")
add()

add("    graph.add_conditional_edges(")
add("        'validate_after_additional_search',")
add("        route_after_second_validation,")
add("        {")
add("            'valid': 'form_document_answer',")
add("            'human_review': 'human_review',")
add("        },")
add("    )")
add()

add("    graph.add_edge(")
add("        'form_document_answer',")
add("        END,")
add("    )")
add()

add("    graph.add_edge(")
add("        'human_review',")
add("        END,")
add("    )")
add()

add("    return graph.compile()")


# ------------------------------------------------------------
# Public runner
# ------------------------------------------------------------

add()
add()
add("# ------------------------------------------------------------")
add("# Public helper")
add("# ------------------------------------------------------------")
add()

add("def run_assistant(")
add("    graph,")
add("    query: str,")
add(") -> AssistantState:")
add('    """Run one query through the compiled graph."""')
add()

add("    return graph.invoke(")
add("        {")
add("            'query': query,")
add("            'needs_human_review': False,")
add("            'needs_additional_search': False,")
add("            'answer_is_valid': False,")
add("            'retrieved_chunks': [],")
add("            'additional_chunks': [],")
add("        }")
add("    )")


# ------------------------------------------------------------
# 3. Сохраняем файл
# ------------------------------------------------------------

langgraph_code = "\n".join(lines)


with open(
    LANGGRAPH_APP_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(langgraph_code)


# ------------------------------------------------------------
# 4. Создаём requirements.txt
# ------------------------------------------------------------

REQUIREMENTS_PATH = (
    GITHUB_PROJECT_DIR
    / "requirements.txt"
)


requirements = [
    "torch",
    "transformers",
    "datasets",
    "scikit-learn",
    "pandas",
    "numpy",
    "sentence-transformers",
    "langgraph",
    "langchain-core",
    "groq",
]


with open(
    REQUIREMENTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(
        "\n".join(requirements)
        + "\n"
    )


# ------------------------------------------------------------
# 5. Проверяем содержимое
# ------------------------------------------------------------

required_terms = [
    "StateGraph",
    "determine_query_type",
    "classify_intent",
    "semantic_retrieval",
    "document_retrieval",
    "additional_search",
    "validate_answer",
    "human_review",
    "form_document_answer",
]


content = LANGGRAPH_APP_PATH.read_text(
    encoding="utf-8"
)


print("=" * 70)
print("END-TO-END LANGGRAPH ORCHESTRATOR")
print("=" * 70)


all_ok = True


for term in required_terms:

    if term in content:
        status = "OK"
    else:
        status = "MISSING"
        all_ok = False

    print(
        f"{status:8} | {term}"
    )


# ------------------------------------------------------------
# 6. Итог
# ------------------------------------------------------------

print()
print("=" * 70)
print("CREATED FILES")
print("=" * 70)

print(
    "OK       | src/langgraph_app.py"
)

print(
    "OK       | requirements.txt"
)

print()

print(
    "LangGraph module size:",
    f"{LANGGRAPH_APP_PATH.stat().st_size / 1024:.1f} KB"
)

print()
print("=" * 70)

if all_ok:

    print("CELL 8 COMPLETE")
    print()
    print(
        "End-to-end LangGraph orchestration layer создан."
    )
    print()
    print(
        "Следующий шаг — установить LangGraph и провести "
        "smoke-test графа на mock adapters, чтобы проверить "
        "все маршруты без повторного запуска ML-моделей."
    )

else:

    print(
        "CELL 8 COMPLETE WITH WARNINGS"
    )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 9.
# FIXED: INSTALL LANGGRAPH + SMOKE TEST ALL ROUTES
#
# Исправление:
# - модуль langgraph_app регистрируется в sys.modules
#   ДО exec_module(), чтобы dataclass корректно работал
#   в Python 3.13 / Colab.
#
# Проверяем:
# 1. high-confidence intent -> auto answer
# 2. low-confidence intent -> human review
# 3. document RAG -> valid answer
# 4. document RAG -> additional search -> valid answer
#
# ML-модели заново НЕ загружаем.
# ============================================================

import sys
import subprocess
import importlib.util
import json
from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


SRC_DIR = (
    GITHUB_PROJECT_DIR
    / "src"
)

LANGGRAPH_APP_PATH = (
    SRC_DIR
    / "langgraph_app.py"
)


if not LANGGRAPH_APP_PATH.exists():
    raise RuntimeError(
        "src/langgraph_app.py не найден. "
        "Сначала выполни ячейку 8."
    )


# ------------------------------------------------------------
# 2. Устанавливаем LangGraph при необходимости
# ------------------------------------------------------------

try:
    import langgraph

    print(
        "LangGraph already installed."
    )

except ImportError:

    print(
        "Installing LangGraph..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langgraph",
            "langchain-core",
        ]
    )

    print(
        "LangGraph installed."
    )


# ------------------------------------------------------------
# 3. Корректный импорт langgraph_app.py
#
# ВАЖНО:
# Регистрируем модуль в sys.modules ДО exec_module().
# Это исправляет ошибку dataclass в Python 3.13.
# ------------------------------------------------------------

module_name = (
    "langgraph_app"
)


# Если модуль уже был частично загружен
# после неудачной попытки — удаляем
if module_name in sys.modules:
    del sys.modules[
        module_name
    ]


spec = (
    importlib.util.spec_from_file_location(
        module_name,
        LANGGRAPH_APP_PATH
    )
)


if spec is None or spec.loader is None:

    raise RuntimeError(
        "Не удалось создать import spec "
        "для src/langgraph_app.py"
    )


langgraph_app = (
    importlib.util.module_from_spec(
        spec
    )
)


# КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ
sys.modules[
    module_name
] = langgraph_app


spec.loader.exec_module(
    langgraph_app
)


print(
    "langgraph_app imported successfully."
)


# ------------------------------------------------------------
# 4. Получаем объекты из модуля
# ------------------------------------------------------------

AssistantAdapters = (
    langgraph_app.AssistantAdapters
)

build_assistant_graph = (
    langgraph_app.build_assistant_graph
)

run_assistant = (
    langgraph_app.run_assistant
)


# ------------------------------------------------------------
# 5. Mock state для проверки additional search
# ------------------------------------------------------------

mock_state = {
    "document_retrieval_calls": {}
}


# ------------------------------------------------------------
# 6. Mock adapters
# ------------------------------------------------------------

def mock_determine_query_type(
    query
):

    query_lower = (
        query.lower()
    )

    if (
        "policy" in query_lower
        or "document" in query_lower
    ):

        return "document"

    return "intent"


def mock_classify_intent(
    query
):

    query_lower = (
        query.lower()
    )

    if "activate card" in query_lower:

        return {
            "intent_id": 10,
            "intent_name": "activate_my_card",
            "confidence": 0.97,
        }

    return {
        "intent_id": 20,
        "intent_name": "cash_withdrawal",
        "confidence": 0.40,
    }


def mock_semantic_retrieve(
    query
):

    query_lower = (
        query.lower()
    )

    if "activate card" in query_lower:

        return {
            "label": 10,
            "similarity": 0.95,
        }

    return {
        "label": 21,
        "similarity": 0.80,
    }


def mock_document_retrieve(
    query,
    top_k
):

    key = (
        query.lower()
    )

    calls = (
        mock_state[
            "document_retrieval_calls"
        ].get(
            key,
            0
        )
    )

    mock_state[
        "document_retrieval_calls"
    ][
        key
    ] = calls + 1


    # --------------------------------------------
    # Сценарий additional search
    # --------------------------------------------

    if "additional" in key:

        if calls == 0:

            return [
                {
                    "doc_id": "DOC_001",
                    "chunk_id": "CHUNK_001",
                    "text": (
                        "Partial information only."
                    ),
                    "similarity": 0.70,
                }
            ]

        return [
            {
                "doc_id": "DOC_001",
                "chunk_id": "CHUNK_001",
                "text": (
                    "Full policy information."
                ),
                "similarity": 0.90,
            },
            {
                "doc_id": "DOC_001",
                "chunk_id": "CHUNK_002",
                "text": (
                    "Additional supporting "
                    "policy details."
                ),
                "similarity": 0.88,
            },
        ]


    # --------------------------------------------
    # Обычный успешный Document RAG
    # --------------------------------------------

    return [
        {
            "doc_id": "DOC_002",
            "chunk_id": "CHUNK_010",
            "text": (
                "Employees must follow "
                "the internal policy."
            ),
            "similarity": 0.91,
        }
    ]


def mock_generate_answer(
    query,
    chunks
):

    texts = [
        chunk.get(
            "text",
            ""
        )
        for chunk in chunks
    ]

    if any(
        "Full policy information"
        in text
        for text in texts
    ):

        return (
            "The full internal policy "
            "allows the requested action."
        )

    if any(
        "Partial information only"
        in text
        for text in texts
    ):

        return (
            "The available context "
            "is incomplete."
        )

    return (
        "According to the internal policy, "
        "employees must follow "
        "the documented procedure."
    )


def mock_validate_answer(
    query,
    answer,
    chunks
):

    if "additional" in query.lower():

        return any(
            "Full policy information"
            in chunk.get(
                "text",
                ""
            )
            for chunk in chunks
        )

    return bool(
        answer
        and len(chunks) > 0
    )


# ------------------------------------------------------------
# 7. Собираем adapters и graph
# ------------------------------------------------------------

adapters = (
    AssistantAdapters(

        determine_query_type=(
            mock_determine_query_type
        ),

        classify_intent=(
            mock_classify_intent
        ),

        semantic_retrieve=(
            mock_semantic_retrieve
        ),

        document_retrieve=(
            mock_document_retrieve
        ),

        generate_answer=(
            mock_generate_answer
        ),

        validate_answer=(
            mock_validate_answer
        ),
    )
)


graph = (
    build_assistant_graph(
        adapters
    )
)


print(
    "LangGraph compiled successfully."
)


# ------------------------------------------------------------
# 8. Тестовые сценарии
# ------------------------------------------------------------

scenarios = [

    {
        "name": (
            "High-confidence intent"
        ),
        "query": (
            "How do I activate card?"
        ),
        "expected_route": (
            "distilbert_high_confidence"
        ),
        "expected_human_review": False,
    },

    {
        "name": (
            "Low-confidence intent"
        ),
        "query": (
            "Why was my cash withdrawal rejected?"
        ),
        "expected_route": (
            "human_review"
        ),
        "expected_human_review": True,
    },

    {
        "name": (
            "Document RAG direct success"
        ),
        "query": (
            "What does the internal policy say?"
        ),
        "expected_route": None,
        "expected_human_review": False,
    },

    {
        "name": (
            "Document RAG + additional search"
        ),
        "query": (
            "What does the additional "
            "policy document say?"
        ),
        "expected_route": (
            "document_rag"
        ),
        "expected_human_review": False,
    },
]


# ------------------------------------------------------------
# 9. Запускаем smoke tests
# ------------------------------------------------------------

print()
print("=" * 70)
print("LANGGRAPH SMOKE TEST")
print("=" * 70)


all_passed = True

results = []


for scenario in scenarios:

    result = (
        run_assistant(
            graph,
            scenario["query"]
        )
    )


    route = (
        result.get(
            "route"
        )
    )

    human_review = bool(
        result.get(
            "needs_human_review",
            False
        )
    )

    final_answer = (
        result.get(
            "final_answer"
        )
    )

    query_type = (
        result.get(
            "query_type"
        )
    )


    route_ok = True


    if (
        scenario[
            "expected_route"
        ]
        is not None
    ):

        route_ok = (
            route
            == scenario[
                "expected_route"
            ]
        )


    human_ok = (
        human_review
        == scenario[
            "expected_human_review"
        ]
    )


    answer_ok = bool(
        final_answer
    )


    passed = (
        route_ok
        and human_ok
        and answer_ok
    )


    if not passed:

        all_passed = False


    results.append(
        {
            "scenario": (
                scenario[
                    "name"
                ]
            ),
            "query_type": (
                query_type
            ),
            "route": (
                route
            ),
            "human_review": (
                human_review
            ),
            "final_answer": (
                final_answer
            ),
            "passed": (
                passed
            ),
        }
    )


    status = (
        "PASS"
        if passed
        else "FAIL"
    )


    print()

    print(
        f"{status:6} | "
        f"{scenario['name']}"
    )

    print(
        "         query_type:",
        query_type
    )

    print(
        "         route:",
        route
    )

    print(
        "         human_review:",
        human_review
    )

    print(
        "         final_answer:",
        final_answer
    )


# ------------------------------------------------------------
# 10. Проверка additional search
# ------------------------------------------------------------

additional_query = (
    "what does the additional "
    "policy document say?"
)


additional_calls = (
    mock_state[
        "document_retrieval_calls"
    ].get(
        additional_query,
        0
    )
)


print()
print("=" * 70)
print("ADDITIONAL SEARCH CHECK")
print("=" * 70)


print(
    "Document retrieval calls:",
    additional_calls
)


if additional_calls >= 2:

    print(
        "PASS     | additional search executed"
    )

else:

    print(
        "FAIL     | additional search was not executed"
    )

    all_passed = False


# ------------------------------------------------------------
# 11. Сохраняем smoke-test report
# ------------------------------------------------------------

SMOKE_TEST_PATH = (
    GITHUB_PROJECT_DIR
    / "results"
    / "langgraph_smoke_test.json"
)


with open(
    SMOKE_TEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "all_passed": (
                all_passed
            ),
            "additional_search_calls": (
                additional_calls
            ),
            "scenarios": (
                results
            ),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)


if all_passed:

    print(
        "CELL 9 COMPLETE"
    )

    print()

    print(
        "ALL LANGGRAPH SMOKE TESTS PASSED"
    )

    print()

    print(
        "Проверены маршруты:"
    )

    print(
        "- high-confidence auto resolution"
    )

    print(
        "- low-confidence human review"
    )

    print(
        "- direct Document RAG"
    )

    print(
        "- additional document search"
    )

    print()

    print(
        "Saved:"
    )

    print(
        SMOKE_TEST_PATH
    )

else:

    print(
        "CELL 9 COMPLETE WITH FAILURES"
    )

    print()

    print(
        "Есть ошибки маршрутизации."
    )


print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 10.
# UPDATE AUDIT + MLSD AFTER LANGGRAPH IMPLEMENTATION
#
# Что делаем:
# 1. Обновляем requirements_audit.md
# 2. Меняем LangGraph: PARTIAL -> IMPLEMENTED
# 3. Добавляем ссылку на smoke-test
# 4. Добавляем в MLSD факт реального выполнения графа
#
# Метрики не пересчитываем.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


AUDIT_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "requirements_audit.md"
)

MLSD_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "mlsd.md"
)

SMOKE_TEST_PATH = (
    GITHUB_PROJECT_DIR
    / "results"
    / "langgraph_smoke_test.json"
)


for path in [
    AUDIT_PATH,
    MLSD_PATH,
    SMOKE_TEST_PATH,
]:

    if not path.exists():
        raise RuntimeError(
            f"Не найден файл: {path}"
        )


# ------------------------------------------------------------
# 2. Обновляем requirements_audit.md
# ------------------------------------------------------------

audit_text = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)


old_langgraph_row = (
    "| LangGraph | PARTIAL | "
    "Проверена orchestration / routing логика для "
    "selective classification и consensus route. "
    "Финальная безопасная policy не дала прироста "
    "к baseline при SLA >=95% automatic accuracy. "
    "| В MLSD LangGraph рассматривается как единый "
    "orchestration layer для classifier, RAG, fallback "
    "и human review. |"
)


new_langgraph_row = (
    "| LangGraph | IMPLEMENTED | "
    "Реализован end-to-end LangGraph workflow: "
    "query type detection, intent classification, "
    "semantic retrieval, Document RAG, additional search, "
    "answer validation и human review. "
    "Smoke-test успешно прошёл все ключевые маршруты. "
    "| Production-версия может подключать реальные "
    "model services, vector database и monitoring через adapters. |"
)


if old_langgraph_row in audit_text:

    audit_text = (
        audit_text.replace(
            old_langgraph_row,
            new_langgraph_row
        )
    )

else:

    print(
        "WARNING: исходная строка LangGraph "
        "в audit-файле не найдена автоматически."
    )


# Исправляем итоговый текст
audit_text = audit_text.replace(
    "Reranker, полноценная vector database, structured output, "
    "tool calling, production monitoring и полноценный "
    "end-to-end LangGraph workflow относятся к production design.",
    "Reranker, полноценная vector database, structured output, "
    "tool calling и production monitoring относятся "
    "к production design. End-to-end LangGraph workflow "
    "уже реализован и проверен smoke-test."
)


# Добавляем секцию smoke-test
smoke_marker = (
    "## LangGraph smoke-test"
)


if smoke_marker not in audit_text:

    audit_text += (
        "\n\n"
        "## LangGraph smoke-test\n\n"
        "Исполняемый граф был проверен на четырёх сценариях:\n\n"
        "- high-confidence intent → automatic resolution;\n"
        "- low-confidence intent → human review;\n"
        "- Document RAG → direct successful answer;\n"
        "- Document RAG → additional search → successful answer.\n\n"
        "Все smoke-tests завершились успешно.\n\n"
        "Результаты: "
        "`results/langgraph_smoke_test.json`\n"
    )


AUDIT_PATH.write_text(
    audit_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. Обновляем MLSD
# ------------------------------------------------------------

mlsd_text = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
)


implementation_marker = (
    "## 7.5 Реализованный LangGraph workflow"
)


if implementation_marker not in mlsd_text:

    langgraph_section = "\n".join([
        "",
        "## 7.5 Реализованный LangGraph workflow",
        "",
        "После подготовки MLSD orchestration layer был реализован",
        "как исполняемый LangGraph workflow.",
        "",
        "Граф включает следующие узлы:",
        "",
        "- `determine_query_type`;",
        "- `classify_intent`;",
        "- `semantic_retrieval`;",
        "- `check_intent`;",
        "- `document_retrieval`;",
        "- `generate_answer`;",
        "- `validate_answer`;",
        "- `additional_search`;",
        "- `validate_after_additional_search`;",
        "- `human_review`;",
        "- формирование финального ответа.",
        "",
        "Smoke-test проверил четыре ключевых сценария:",
        "",
        "1. high-confidence intent → auto resolution;",
        "2. low-confidence intent → human review;",
        "3. Document RAG → direct success;",
        "4. Document RAG → additional search → success.",
        "",
        "Все сценарии успешно прошли.",
        "",
        "Результат сохранён в:",
        "",
        "`results/langgraph_smoke_test.json`",
        "",
        "Таким образом, LangGraph в текущем репозитории является",
        "не только design-компонентом, но и реально реализованным",
        "исполняемым orchestration layer.",
        ""
    ])


    insert_before = (
        "# 8. Данные"
    )


    if insert_before in mlsd_text:

        mlsd_text = (
            mlsd_text.replace(
                insert_before,
                langgraph_section
                + "\n---\n\n"
                + insert_before
            )
        )

    else:

        mlsd_text += (
            "\n"
            + langgraph_section
        )


MLSD_PATH.write_text(
    mlsd_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. Проверяем
# ------------------------------------------------------------

updated_audit = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)

updated_mlsd = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
)


checks = {
    "LangGraph IMPLEMENTED in audit": (
        "| LangGraph | IMPLEMENTED |"
        in updated_audit
    ),

    "Smoke-test in audit": (
        "results/langgraph_smoke_test.json"
        in updated_audit
    ),

    "LangGraph implementation in MLSD": (
        "## 7.5 Реализованный LangGraph workflow"
        in updated_mlsd
    ),

    "Smoke-test reference in MLSD": (
        "results/langgraph_smoke_test.json"
        in updated_mlsd
    ),
}


print("=" * 70)
print("DOCUMENTATION UPDATE CHECK")
print("=" * 70)


all_ok = True


for name, status in checks.items():

    if status:
        label = "OK"
    else:
        label = "MISSING"
        all_ok = False

    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 5. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)

if all_ok:

    print(
        "CELL 10 COMPLETE"
    )

    print()

    print(
        "LangGraph status updated to IMPLEMENTED."
    )

    print(
        "Smoke-test зафиксирован в audit и MLSD."
    )

else:

    print(
        "CELL 10 COMPLETE WITH WARNINGS"
    )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 11.
# ADD STRUCTURED OUTPUT WITH PYDANTIC
#
# Что делаем:
# 1. Создаём src/schemas.py
# 2. Описываем строгие схемы ответа:
#    - IntentResult
#    - RAGSource
#    - AssistantResponse
# 3. Проверяем Pydantic validation
# 4. Сохраняем smoke-test результата
#
# Это закрывает пункт Structured Output как IMPLEMENTED.
# ============================================================

from pathlib import Path
import json
import sys
import subprocess


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


SRC_DIR = (
    GITHUB_PROJECT_DIR
    / "src"
)

RESULTS_DIR = (
    GITHUB_PROJECT_DIR
    / "results"
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SCHEMAS_PATH = (
    SRC_DIR
    / "schemas.py"
)

SMOKE_TEST_PATH = (
    RESULTS_DIR
    / "structured_output_smoke_test.json"
)


# ------------------------------------------------------------
# 2. Проверяем Pydantic
# ------------------------------------------------------------

try:
    import pydantic

    print(
        "Pydantic already installed:",
        pydantic.__version__
    )

except ImportError:

    print(
        "Installing Pydantic..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pydantic>=2",
        ]
    )

    import pydantic

    print(
        "Pydantic installed:",
        pydantic.__version__
    )


# ------------------------------------------------------------
# 3. Создаём schemas.py
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


add('"""Structured output schemas for the assistant."""')
add()
add("from __future__ import annotations")
add()
add("from typing import List, Literal, Optional")
add()
add("from pydantic import BaseModel, Field")
add()


# ------------------------------------------------------------
# Intent schema
# ------------------------------------------------------------

add()
add("class IntentResult(BaseModel):")
add('    """Structured intent-classification result."""')
add()
add(
    "    intent_id: Optional[int] = "
    "Field(default=None, ge=0)"
)
add()
add(
    "    intent_name: Optional[str] = None"
)
add()
add(
    "    confidence: float = "
    "Field(ge=0.0, le=1.0)"
)
add()
add(
    "    accepted: bool"
)
add()


# ------------------------------------------------------------
# Source schema
# ------------------------------------------------------------

add()
add("class RAGSource(BaseModel):")
add('    """One document source used by RAG."""')
add()
add(
    "    doc_id: str"
)
add()
add(
    "    chunk_id: str"
)
add()
add(
    "    similarity: float = "
    "Field(ge=-1.0, le=1.0)"
)
add()


# ------------------------------------------------------------
# Final assistant response
# ------------------------------------------------------------

add()
add("class AssistantResponse(BaseModel):")
add('    """Final structured response returned by the assistant."""')
add()
add(
    "    query_type: Literal['intent', 'document']"
)
add()
add(
    "    route: Literal["
    "'distilbert_high_confidence', "
    "'low_confidence_consensus', "
    "'document_rag', "
    "'human_review'"
    "]"
)
add()
add(
    "    answer: str = "
    "Field(min_length=1)"
)
add()
add(
    "    needs_human_review: bool"
)
add()
add(
    "    intent: Optional[IntentResult] = None"
)
add()
add(
    "    sources: List[RAGSource] = "
    "Field(default_factory=list)"
)
add()


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

add()
add("def validate_response(data: dict) -> AssistantResponse:")
add('    """Validate and normalize assistant output."""')
add()
add(
    "    return AssistantResponse.model_validate(data)"
)


schemas_code = "\n".join(
    lines
)


with open(
    SCHEMAS_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        schemas_code
    )


print(
    "Created:",
    SCHEMAS_PATH
)


# ------------------------------------------------------------
# 4. Импортируем schemas.py
# ------------------------------------------------------------

import importlib.util


module_name = (
    "assistant_schemas"
)


if module_name in sys.modules:
    del sys.modules[module_name]


spec = (
    importlib.util.spec_from_file_location(
        module_name,
        SCHEMAS_PATH
    )
)


if spec is None or spec.loader is None:
    raise RuntimeError(
        "Не удалось импортировать schemas.py"
    )


schemas_module = (
    importlib.util.module_from_spec(
        spec
    )
)


sys.modules[
    module_name
] = schemas_module


spec.loader.exec_module(
    schemas_module
)


AssistantResponse = (
    schemas_module.AssistantResponse
)

IntentResult = (
    schemas_module.IntentResult
)

RAGSource = (
    schemas_module.RAGSource
)


# ------------------------------------------------------------
# 5. Валидный intent response
# ------------------------------------------------------------

intent_example = {
    "query_type": "intent",
    "route": "distilbert_high_confidence",
    "answer": "Intent: activate_my_card",
    "needs_human_review": False,
    "intent": {
        "intent_id": 10,
        "intent_name": "activate_my_card",
        "confidence": 0.97,
        "accepted": True,
    },
    "sources": [],
}


intent_validated = (
    AssistantResponse.model_validate(
        intent_example
    )
)


# ------------------------------------------------------------
# 6. Валидный Document RAG response
# ------------------------------------------------------------

rag_example = {
    "query_type": "document",
    "route": "document_rag",
    "answer": (
        "According to the internal policy, "
        "employees must follow the documented procedure."
    ),
    "needs_human_review": False,
    "intent": None,
    "sources": [
        {
            "doc_id": "DOC_002",
            "chunk_id": "CHUNK_010",
            "similarity": 0.91,
        }
    ],
}


rag_validated = (
    AssistantResponse.model_validate(
        rag_example
    )
)


# ------------------------------------------------------------
# 7. Human review response
# ------------------------------------------------------------

review_example = {
    "query_type": "intent",
    "route": "human_review",
    "answer": "Request requires human review.",
    "needs_human_review": True,
    "intent": {
        "intent_id": 20,
        "intent_name": "cash_withdrawal",
        "confidence": 0.40,
        "accepted": False,
    },
    "sources": [],
}


review_validated = (
    AssistantResponse.model_validate(
        review_example
    )
)


# ------------------------------------------------------------
# 8. Проверяем, что некорректный output отклоняется
# ------------------------------------------------------------

invalid_rejected = False
invalid_error = None


invalid_example = {
    "query_type": "intent",

    # Такой route запрещён schema
    "route": "random_route",

    # Пустой answer тоже запрещён
    "answer": "",

    "needs_human_review": False,

    "intent": {
        "intent_id": 10,
        "intent_name": "activate_my_card",

        # Некорректная confidence > 1
        "confidence": 1.50,

        "accepted": True,
    },

    "sources": [],
}


try:

    AssistantResponse.model_validate(
        invalid_example
    )

except Exception as e:

    invalid_rejected = True

    invalid_error = (
        type(e).__name__
    )


# ------------------------------------------------------------
# 9. Проверяем JSON serialization
# ------------------------------------------------------------

intent_json = (
    intent_validated.model_dump(
        mode="json"
    )
)

rag_json = (
    rag_validated.model_dump(
        mode="json"
    )
)

review_json = (
    review_validated.model_dump(
        mode="json"
    )
)


# ------------------------------------------------------------
# 10. Итоги smoke-test
# ------------------------------------------------------------

checks = {
    "intent_response_valid": (
        intent_validated.route
        == "distilbert_high_confidence"
    ),

    "rag_response_valid": (
        rag_validated.route
        == "document_rag"
    ),

    "rag_source_preserved": (
        len(
            rag_validated.sources
        )
        == 1
    ),

    "human_review_valid": (
        review_validated.needs_human_review
        is True
    ),

    "invalid_output_rejected": (
        invalid_rejected
    ),

    "json_serialization": (
        isinstance(
            intent_json,
            dict
        )
        and isinstance(
            rag_json,
            dict
        )
        and isinstance(
            review_json,
            dict
        )
    ),
}


all_passed = all(
    checks.values()
)


print()
print("=" * 70)
print("STRUCTURED OUTPUT SMOKE TEST")
print("=" * 70)


for name, status in checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"{label:6} | {name}"
    )


# ------------------------------------------------------------
# 11. Сохраняем smoke-test
# ------------------------------------------------------------

report = {
    "all_passed": (
        all_passed
    ),

    "schema": (
        "AssistantResponse"
    ),

    "valid_routes": [
        "distilbert_high_confidence",
        "low_confidence_consensus",
        "document_rag",
        "human_review",
    ],

    "tests": checks,

    "invalid_example_rejected": (
        invalid_rejected
    ),

    "invalid_error_type": (
        invalid_error
    ),

    "examples": {
        "intent": (
            intent_json
        ),
        "document_rag": (
            rag_json
        ),
        "human_review": (
            review_json
        ),
    },
}


with open(
    SMOKE_TEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Проверяем requirements.txt
# ------------------------------------------------------------

REQUIREMENTS_PATH = (
    GITHUB_PROJECT_DIR
    / "requirements.txt"
)


if REQUIREMENTS_PATH.exists():

    requirements_text = (
        REQUIREMENTS_PATH.read_text(
            encoding="utf-8"
        )
    )

else:

    requirements_text = ""


if "pydantic" not in requirements_text.lower():

    with open(
        REQUIREMENTS_PATH,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            "pydantic>=2\n"
        )


# ------------------------------------------------------------
# 13. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)

if all_passed:

    print(
        "CELL 11 COMPLETE"
    )

    print()

    print(
        "ALL STRUCTURED OUTPUT TESTS PASSED"
    )

    print()

    print(
        "Structured Output теперь реально реализован "
        "через Pydantic schema."
    )

    print()

    print(
        "Created:"
    )

    print(
        SCHEMAS_PATH
    )

    print()

    print(
        "Saved:"
    )

    print(
        SMOKE_TEST_PATH
    )

else:

    print(
        "CELL 11 COMPLETE WITH FAILURES"
    )

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 12.
# UPDATE DOCUMENTATION AFTER STRUCTURED OUTPUT IMPLEMENTATION
#
# Что делаем:
# 1. Structured Output: PROPOSED -> IMPLEMENTED
# 2. Добавляем описание Pydantic schemas
# 3. Добавляем ссылку на smoke-test
# 4. Обновляем MLSD
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


AUDIT_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "requirements_audit.md"
)

MLSD_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "mlsd.md"
)

STRUCTURED_TEST_PATH = (
    GITHUB_PROJECT_DIR
    / "results"
    / "structured_output_smoke_test.json"
)


for path in [
    AUDIT_PATH,
    MLSD_PATH,
    STRUCTURED_TEST_PATH,
]:

    if not path.exists():
        raise RuntimeError(
            f"Не найден файл: {path}"
        )


# ------------------------------------------------------------
# 2. Обновляем requirements_audit.md
# ------------------------------------------------------------

audit_text = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)


old_row = (
    "| Structured Output | PROPOSED | "
    "Формальный schema-enforced structured output "
    "в prototype не является отдельным production-компонентом. "
    "| Предлагается использовать Pydantic/JSON schema "
    "для router и LLM responses. |"
)


new_row = (
    "| Structured Output | IMPLEMENTED | "
    "Реализованы строгие Pydantic-схемы для intent results, "
    "RAG sources и финального AssistantResponse. "
    "Проверены validation, invalid-output rejection "
    "и JSON serialization. "
    "| В production эти схемы могут использоваться "
    "как контракт API и LangGraph nodes. |"
)


if old_row in audit_text:

    audit_text = (
        audit_text.replace(
            old_row,
            new_row
        )
    )

else:

    print(
        "WARNING: строка Structured Output "
        "не найдена автоматически."
    )


# Исправляем итоговый вывод audit
audit_text = audit_text.replace(
    "Reranker, полноценная vector database, structured output, "
    "tool calling и production monitoring относятся "
    "к production design.",
    "Reranker, полноценная vector database, "
    "tool calling и production monitoring относятся "
    "к production design. Structured Output уже реализован "
    "через Pydantic."
)


structured_marker = (
    "## Structured Output implementation"
)


if structured_marker not in audit_text:

    audit_text += (
        "\n\n"
        "## Structured Output implementation\n\n"
        "Structured Output реализован через Pydantic v2.\n\n"
        "Добавлены схемы:\n\n"
        "- `IntentResult`;\n"
        "- `RAGSource`;\n"
        "- `AssistantResponse`.\n\n"
        "Smoke-test проверяет:\n\n"
        "- валидный intent-response;\n"
        "- валидный Document RAG response;\n"
        "- сохранение source metadata;\n"
        "- human-review response;\n"
        "- отклонение некорректного output;\n"
        "- JSON serialization.\n\n"
        "Все проверки завершились успешно.\n\n"
        "Результаты: "
        "`results/structured_output_smoke_test.json`\n"
    )


AUDIT_PATH.write_text(
    audit_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. Обновляем MLSD
# ------------------------------------------------------------

mlsd_text = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
)


section_marker = (
    "## 7.6 Structured Output"
)


if section_marker not in mlsd_text:

    section = "\n".join([
        "",
        "## 7.6 Structured Output",
        "",
        "Для стандартизации ответа системы реализован",
        "schema-enforced Structured Output на базе Pydantic v2.",
        "",
        "Основные схемы:",
        "",
        "- `IntentResult` — результат intent classification;",
        "- `RAGSource` — metadata источника Document RAG;",
        "- `AssistantResponse` — единый финальный ответ системы.",
        "",
        "В `AssistantResponse` фиксируются:",
        "",
        "- тип запроса;",
        "- выбранный route;",
        "- финальный answer;",
        "- необходимость human review;",
        "- intent result;",
        "- список RAG sources.",
        "",
        "Pydantic validation не позволяет вернуть:",
        "",
        "- неизвестный route;",
        "- confidence вне диапазона 0..1;",
        "- пустой answer;",
        "- некорректную структуру результата.",
        "",
        "Structured Output был проверен smoke-test:",
        "",
        "`results/structured_output_smoke_test.json`",
        "",
        "Все проверки завершились успешно.",
        ""
    ])


    insert_before = (
        "# 8. Данные"
    )


    if insert_before in mlsd_text:

        mlsd_text = (
            mlsd_text.replace(
                insert_before,
                section
                + "\n---\n\n"
                + insert_before
            )
        )

    else:

        mlsd_text += (
            "\n"
            + section
        )


MLSD_PATH.write_text(
    mlsd_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. Проверяем результат
# ------------------------------------------------------------

updated_audit = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)

updated_mlsd = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
)


checks = {

    "Structured Output IMPLEMENTED in audit": (
        "| Structured Output | IMPLEMENTED |"
        in updated_audit
    ),

    "Smoke-test in audit": (
        "structured_output_smoke_test.json"
        in updated_audit
    ),

    "Structured Output section in MLSD": (
        "## 7.6 Structured Output"
        in updated_mlsd
    ),

    "Pydantic mentioned in MLSD": (
        "Pydantic"
        in updated_mlsd
    ),

    "Smoke-test reference in MLSD": (
        "structured_output_smoke_test.json"
        in updated_mlsd
    ),
}


print("=" * 70)
print("STRUCTURED OUTPUT DOCUMENTATION CHECK")
print("=" * 70)


all_ok = True


for name, status in checks.items():

    label = (
        "OK"
        if status
        else "MISSING"
    )

    if not status:
        all_ok = False

    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 5. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)


if all_ok:

    print(
        "CELL 12 COMPLETE"
    )

    print()

    print(
        "Structured Output status updated to IMPLEMENTED."
    )

    print(
        "Audit и MLSD синхронизированы."
    )

else:

    print(
        "CELL 12 COMPLETE WITH WARNINGS"
    )


print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 13.
# IMPLEMENT TOOL CALLING LAYER
#
# Что делаем:
# 1. Создаём src/tools.py
# 2. Реализуем registry внутренних tools
# 3. Добавляем Pydantic-валидацию аргументов
# 4. Проверяем:
#    - корректный tool call
#    - неизвестный tool
#    - некорректные аргументы
#    - JSON-совместимый результат
#
# После этого Tool Calling можно считать IMPLEMENTED.
# ============================================================

from pathlib import Path
import json
import sys
import importlib.util


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


SRC_DIR = (
    GITHUB_PROJECT_DIR
    / "src"
)

RESULTS_DIR = (
    GITHUB_PROJECT_DIR
    / "results"
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TOOLS_PATH = (
    SRC_DIR
    / "tools.py"
)

SMOKE_TEST_PATH = (
    RESULTS_DIR
    / "tool_calling_smoke_test.json"
)


# ------------------------------------------------------------
# 2. Формируем tools.py
# ------------------------------------------------------------

lines = []


def add(text=""):
    lines.append(str(text))


add('"""Minimal internal tool-calling layer for the assistant."""')
add()
add("from __future__ import annotations")
add()
add("from typing import Any, Callable, Dict, Type")
add()
add("from pydantic import BaseModel, Field, ValidationError")
add()


# ------------------------------------------------------------
# Argument schemas
# ------------------------------------------------------------

add()
add("class AccountStatusArgs(BaseModel):")
add('    """Arguments for account-status lookup."""')
add()
add("    account_id: str = Field(min_length=3)")
add()


add()
add("class ProductInfoArgs(BaseModel):")
add('    """Arguments for product-information lookup."""')
add()
add("    product_name: str = Field(min_length=2)")
add()


# ------------------------------------------------------------
# Tool functions
# ------------------------------------------------------------

add()
add("def get_account_status(args: AccountStatusArgs) -> Dict[str, Any]:")
add('    """Mock internal account-status tool."""')
add()
add("    return {")
add("        'tool': 'get_account_status',")
add("        'account_id': args.account_id,")
add("        'status': 'active',")
add("        'source': 'mock_internal_service',")
add("    }")
add()


add()
add("def get_product_info(args: ProductInfoArgs) -> Dict[str, Any]:")
add('    """Mock internal product-information tool."""')
add()
add("    return {")
add("        'tool': 'get_product_info',")
add("        'product_name': args.product_name,")
add("        'available': True,")
add("        'source': 'mock_internal_catalog',")
add("    }")
add()


# ------------------------------------------------------------
# Tool registry
# ------------------------------------------------------------

add()
add("TOOL_REGISTRY = {")
add("    'get_account_status': {")
add("        'schema': AccountStatusArgs,")
add("        'function': get_account_status,")
add("    },")
add("    'get_product_info': {")
add("        'schema': ProductInfoArgs,")
add("        'function': get_product_info,")
add("    },")
add("}")
add()


# ------------------------------------------------------------
# Public executor
# ------------------------------------------------------------

add()
add("def call_tool(")
add("    tool_name: str,")
add("    arguments: Dict[str, Any],")
add(") -> Dict[str, Any]:")
add('    """Validate arguments and execute a registered tool."""')
add()

add("    if tool_name not in TOOL_REGISTRY:")
add("        return {")
add("            'ok': False,")
add("            'error': 'unknown_tool',")
add("            'tool_name': tool_name,")
add("        }")
add()

add("    tool = TOOL_REGISTRY[tool_name]")
add("    schema: Type[BaseModel] = tool['schema']")
add("    function: Callable = tool['function']")
add()

add("    try:")
add("        validated_args = schema.model_validate(arguments)")
add("    except ValidationError as exc:")
add("        return {")
add("            'ok': False,")
add("            'error': 'invalid_arguments',")
add("            'tool_name': tool_name,")
add("            'details': exc.errors(),")
add("        }")
add()

add("    result = function(validated_args)")
add()

add("    return {")
add("        'ok': True,")
add("        'tool_name': tool_name,")
add("        'result': result,")
add("    }")
add()


# ------------------------------------------------------------
# Tool descriptions for an LLM/router
# ------------------------------------------------------------

add()
add("def get_tool_descriptions() -> Dict[str, Dict[str, Any]]:")
add('    """Return lightweight metadata for available tools."""')
add()
add("    return {")
add("        'get_account_status': {")
add("            'description': 'Get account status by account_id.',")
add("            'arguments': {")
add("                'account_id': 'string',")
add("            },")
add("        },")
add("        'get_product_info': {")
add("            'description': 'Get product information by product name.',")
add("            'arguments': {")
add("                'product_name': 'string',")
add("            },")
add("        },")
add("    }")
add()


tools_code = "\n".join(lines)


with open(
    TOOLS_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(tools_code)


print(
    "Created:",
    TOOLS_PATH
)


# ------------------------------------------------------------
# 3. Импортируем tools.py
# ------------------------------------------------------------

module_name = (
    "assistant_tools"
)


if module_name in sys.modules:
    del sys.modules[module_name]


spec = (
    importlib.util.spec_from_file_location(
        module_name,
        TOOLS_PATH
    )
)


if spec is None or spec.loader is None:
    raise RuntimeError(
        "Не удалось импортировать src/tools.py"
    )


tools_module = (
    importlib.util.module_from_spec(
        spec
    )
)


sys.modules[
    module_name
] = tools_module


spec.loader.exec_module(
    tools_module
)


call_tool = (
    tools_module.call_tool
)

get_tool_descriptions = (
    tools_module.get_tool_descriptions
)


# ------------------------------------------------------------
# 4. Smoke-test: успешный account tool
# ------------------------------------------------------------

account_result = (
    call_tool(
        "get_account_status",
        {
            "account_id": "ACC-1001"
        }
    )
)


# ------------------------------------------------------------
# 5. Smoke-test: успешный product tool
# ------------------------------------------------------------

product_result = (
    call_tool(
        "get_product_info",
        {
            "product_name": "Premium Card"
        }
    )
)


# ------------------------------------------------------------
# 6. Smoke-test: неизвестный tool
# ------------------------------------------------------------

unknown_result = (
    call_tool(
        "delete_everything",
        {}
    )
)


# ------------------------------------------------------------
# 7. Smoke-test: некорректные аргументы
# ------------------------------------------------------------

invalid_args_result = (
    call_tool(
        "get_account_status",
        {
            "account_id": ""
        }
    )
)


# ------------------------------------------------------------
# 8. Smoke-test: tool descriptions
# ------------------------------------------------------------

tool_descriptions = (
    get_tool_descriptions()
)


# ------------------------------------------------------------
# 9. Проверки
# ------------------------------------------------------------

checks = {

    "account_tool_success": (
        account_result.get("ok")
        is True
        and account_result.get(
            "result",
            {}
        ).get(
            "status"
        )
        == "active"
    ),

    "product_tool_success": (
        product_result.get("ok")
        is True
        and product_result.get(
            "result",
            {}
        ).get(
            "available"
        )
        is True
    ),

    "unknown_tool_rejected": (
        unknown_result.get(
            "error"
        )
        == "unknown_tool"
    ),

    "invalid_arguments_rejected": (
        invalid_args_result.get(
            "error"
        )
        == "invalid_arguments"
    ),

    "tool_registry_exposed": (
        "get_account_status"
        in tool_descriptions
        and "get_product_info"
        in tool_descriptions
    ),

    "json_serializable": True,
}


# Проверяем JSON serialization отдельно
try:

    json.dumps(
        {
            "account_result": account_result,
            "product_result": product_result,
            "unknown_result": unknown_result,
            "invalid_args_result": invalid_args_result,
            "tool_descriptions": tool_descriptions,
        },
        ensure_ascii=False,
    )

except Exception:

    checks[
        "json_serializable"
    ] = False


all_passed = all(
    checks.values()
)


# ------------------------------------------------------------
# 10. Вывод
# ------------------------------------------------------------

print()
print("=" * 70)
print("TOOL CALLING SMOKE TEST")
print("=" * 70)


for name, status in checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"{label:6} | {name}"
    )


print()
print("Account tool result:")
print(
    json.dumps(
        account_result,
        ensure_ascii=False,
        indent=2,
    )
)


print()
print("Product tool result:")
print(
    json.dumps(
        product_result,
        ensure_ascii=False,
        indent=2,
    )
)


# ------------------------------------------------------------
# 11. Сохраняем smoke-test
# ------------------------------------------------------------

report = {
    "all_passed": (
        all_passed
    ),

    "tests": (
        checks
    ),

    "available_tools": (
        list(
            tool_descriptions.keys()
        )
    ),

    "examples": {
        "account_tool": (
            account_result
        ),
        "product_tool": (
            product_result
        ),
        "unknown_tool": (
            unknown_result
        ),
        "invalid_arguments": (
            invalid_args_result
        ),
    },
}


with open(
    SMOKE_TEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)


if all_passed:

    print(
        "CELL 13 COMPLETE"
    )

    print()

    print(
        "ALL TOOL CALLING TESTS PASSED"
    )

    print()

    print(
        "Tool Calling теперь реально реализован "
        "через внутренний tool registry."
    )

    print()

    print(
        "Created:"
    )

    print(
        TOOLS_PATH
    )

    print()

    print(
        "Saved:"
    )

    print(
        SMOKE_TEST_PATH
    )

else:

    print(
        "CELL 13 COMPLETE WITH FAILURES"
    )


print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 14.
# UPDATE DOCUMENTATION AFTER TOOL CALLING IMPLEMENTATION
#
# Что делаем:
# 1. Tool Calling: PROPOSED -> IMPLEMENTED
# 2. Добавляем описание tool registry
# 3. Добавляем ссылку на smoke-test
# 4. Обновляем MLSD
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


AUDIT_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "requirements_audit.md"
)

MLSD_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "mlsd.md"
)

TOOL_TEST_PATH = (
    GITHUB_PROJECT_DIR
    / "results"
    / "tool_calling_smoke_test.json"
)


for path in [
    AUDIT_PATH,
    MLSD_PATH,
    TOOL_TEST_PATH,
]:

    if not path.exists():
        raise RuntimeError(
            f"Не найден файл: {path}"
        )


# ------------------------------------------------------------
# 2. Обновляем audit
# ------------------------------------------------------------

audit_text = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)


old_row = (
    "| Tool Calling | PROPOSED | "
    "Внешние business tools в prototype не подключались. "
    "| Можно добавить вызов внутренних сервисов: "
    "поиск статуса заявки, справочники, CRM и т.д. |"
)


new_row = (
    "| Tool Calling | IMPLEMENTED | "
    "Реализован внутренний tool registry с Pydantic-валидацией "
    "аргументов и безопасным executor. "
    "Проверены корректный вызов, неизвестный tool, "
    "ошибочные аргументы и JSON serialization. "
    "| В production mock-tools заменяются реальными "
    "внутренними API/CRM/справочниками. |"
)


if old_row in audit_text:

    audit_text = (
        audit_text.replace(
            old_row,
            new_row
        )
    )

else:

    print(
        "WARNING: строка Tool Calling "
        "не найдена автоматически."
    )


audit_text = audit_text.replace(
    "Reranker, полноценная vector database, "
    "tool calling и production monitoring относятся "
    "к production design.",
    "Reranker, полноценная vector database "
    "и production monitoring относятся "
    "к production design. Tool Calling уже реализован "
    "через внутренний tool registry."
)


tool_marker = (
    "## Tool Calling implementation"
)


if tool_marker not in audit_text:

    audit_text += (
        "\n\n"
        "## Tool Calling implementation\n\n"
        "Tool Calling реализован через внутренний registry.\n\n"
        "Добавлены tools:\n\n"
        "- `get_account_status`;\n"
        "- `get_product_info`.\n\n"
        "Executor выполняет:\n\n"
        "- проверку имени tool;\n"
        "- Pydantic-валидацию аргументов;\n"
        "- вызов зарегистрированной функции;\n"
        "- возврат JSON-совместимого результата.\n\n"
        "Smoke-test проверяет:\n\n"
        "- успешный account tool call;\n"
        "- успешный product tool call;\n"
        "- отклонение неизвестного tool;\n"
        "- отклонение некорректных аргументов;\n"
        "- выдачу metadata registry;\n"
        "- JSON serialization.\n\n"
        "Все проверки завершились успешно.\n\n"
        "Результаты: "
        "`results/tool_calling_smoke_test.json`\n"
    )


AUDIT_PATH.write_text(
    audit_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. Обновляем MLSD
# ------------------------------------------------------------

mlsd_text = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
)


section_marker = (
    "## 7.7 Tool Calling"
)


if section_marker not in mlsd_text:

    section = "\n".join([
        "",
        "## 7.7 Tool Calling",
        "",
        "В проект добавлен минимальный исполняемый Tool Calling layer.",
        "",
        "Он включает:",
        "",
        "- внутренний tool registry;",
        "- Pydantic-схемы аргументов;",
        "- проверку существования tool;",
        "- безопасный executor;",
        "- JSON-совместимый ответ;",
        "- metadata доступных tools для router/LLM.",
        "",
        "Реализованы два демонстрационных инструмента:",
        "",
        "- `get_account_status`;",
        "- `get_product_info`.",
        "",
        "Оба инструмента используют mock internal services,",
        "поэтому они демонстрируют механизм Tool Calling,",
        "но не подключены к реальной production CRM.",
        "",
        "В production registry может быть расширен tools для:",
        "",
        "- CRM;",
        "- внутренних справочников;",
        "- статуса заявок;",
        "- продуктового каталога;",
        "- внутренних workflow/API.",
        "",
        "Smoke-test:",
        "",
        "`results/tool_calling_smoke_test.json`",
        "",
        "Все проверки завершились успешно.",
        ""
    ])


    insert_before = (
        "# 8. Данные"
    )


    if insert_before in mlsd_text:

        mlsd_text = (
            mlsd_text.replace(
                insert_before,
                section
                + "\n---\n\n"
                + insert_before
            )
        )

    else:

        mlsd_text += (
            "\n"
            + section
        )


MLSD_PATH.write_text(
    mlsd_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. Проверяем
# ------------------------------------------------------------

updated_audit = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)

updated_mlsd = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
)


checks = {

    "Tool Calling IMPLEMENTED in audit": (
        "| Tool Calling | IMPLEMENTED |"
        in updated_audit
    ),

    "Tool smoke-test in audit": (
        "tool_calling_smoke_test.json"
        in updated_audit
    ),

    "Tool Calling section in MLSD": (
        "## 7.7 Tool Calling"
        in updated_mlsd
    ),

    "Tool registry mentioned in MLSD": (
        "tool registry"
        in updated_mlsd.lower()
    ),

    "Tool smoke-test reference in MLSD": (
        "tool_calling_smoke_test.json"
        in updated_mlsd
    ),
}


print("=" * 70)
print("TOOL CALLING DOCUMENTATION CHECK")
print("=" * 70)


all_ok = True


for name, status in checks.items():

    label = (
        "OK"
        if status
        else "MISSING"
    )

    if not status:
        all_ok = False

    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 5. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)


if all_ok:

    print(
        "CELL 14 COMPLETE"
    )

    print()

    print(
        "Tool Calling status updated to IMPLEMENTED."
    )

    print(
        "Audit и MLSD синхронизированы."
    )

else:

    print(
        "CELL 14 COMPLETE WITH WARNINGS"
    )


print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 15.
# FINAL REPOSITORY ASSEMBLY + PRE-GITHUB CHECK
#
# Что делаем:
# 1. Проверяем структуру репозитория
# 2. Проверяем новые src-файлы
# 3. Проверяем smoke-tests
# 4. Проверяем MLSD и audit
# 5. Обновляем README
# 6. Проверяем requirements.txt
# 7. Создаём FINAL_REPOSITORY_CHECKLIST.md
# 8. Даём итоговый статус готовности к GitHub
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


README_PATH = (
    GITHUB_PROJECT_DIR
    / "README.md"
)

REQUIREMENTS_PATH = (
    GITHUB_PROJECT_DIR
    / "requirements.txt"
)

MLSD_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "mlsd.md"
)

AUDIT_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "requirements_audit.md"
)

CHECKLIST_PATH = (
    GITHUB_PROJECT_DIR
    / "docs"
    / "FINAL_REPOSITORY_CHECKLIST.md"
)


# ------------------------------------------------------------
# 2. Критические файлы репозитория
# ------------------------------------------------------------

critical_files = [

    # Root
    "README.md",
    ".gitignore",
    "requirements.txt",

    # SRC
    "src/__init__.py",
    "src/config.py",
    "src/classifier.py",
    "src/retrieval.py",
    "src/routing.py",
    "src/rag.py",
    "src/langgraph_app.py",
    "src/schemas.py",
    "src/tools.py",
    "src/label_mapping.json",

    # Docs
    "docs/mlsd.md",
    "docs/requirements_audit.md",

    # Validation
    "validation/selected_threshold_final.json",
    "validation/distilbert_final_threshold_sweep.csv",

    # LangGraph
    "langgraph/final_langgraph_architecture.json",
    "langgraph/low_confidence_consensus_validation.csv",
    "langgraph/retrieval_similarity_threshold_sweep.csv",

    # Document RAG
    "document_rag/api_generated_qa.csv",
    "document_rag/api_generated_qa_rag_metrics.json",
    "document_rag/api_generated_qa_final_metrics.json",
    "document_rag/api_generated_qa_final_evaluation.csv",

    # Results
    "results/FINAL_CORRECTED_SUMMARY.json",
    "results/final_test_metrics.json",
    "results/report_classification_metrics.csv",
    "results/report_baseline_vs_langgraph.csv",
    "results/report_document_rag_metrics.csv",
    "results/report_evaluation_protocol.csv",
    "results/DEFENSE_CONCLUSIONS.txt",

    # Smoke tests
    "results/langgraph_smoke_test.json",
    "results/structured_output_smoke_test.json",
    "results/tool_calling_smoke_test.json",

    # Architecture
    "assets/architecture_high_level.md",
    "assets/intent_inference_flow.md",
    "assets/document_rag_flow.md",
    "assets/training_evaluation_flow.md",
]


# ------------------------------------------------------------
# 3. Проверка файлов
# ------------------------------------------------------------

print("=" * 70)
print("FINAL REPOSITORY FILE CHECK")
print("=" * 70)


missing_files = []


for relative_path in critical_files:

    path = (
        GITHUB_PROJECT_DIR
        / relative_path
    )

    if path.exists():

        status = "OK"

    else:

        status = "MISSING"

        missing_files.append(
            relative_path
        )


    print(
        f"{status:8} | {relative_path}"
    )


# ------------------------------------------------------------
# 4. Проверяем smoke-tests
# ------------------------------------------------------------

smoke_tests = {
    "LangGraph": (
        GITHUB_PROJECT_DIR
        / "results"
        / "langgraph_smoke_test.json"
    ),

    "Structured Output": (
        GITHUB_PROJECT_DIR
        / "results"
        / "structured_output_smoke_test.json"
    ),

    "Tool Calling": (
        GITHUB_PROJECT_DIR
        / "results"
        / "tool_calling_smoke_test.json"
    ),
}


print()
print("=" * 70)
print("SMOKE TEST CHECK")
print("=" * 70)


smoke_results = {}


for name, path in smoke_tests.items():

    passed = False

    if path.exists():

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        passed = bool(
            data.get(
                "all_passed",
                False
            )
        )


    smoke_results[
        name
    ] = passed


    label = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 5. Проверяем requirements.txt
# ------------------------------------------------------------

required_packages = [

    "torch",
    "transformers",
    "datasets",
    "scikit-learn",
    "pandas",
    "numpy",
    "sentence-transformers",
    "langgraph",
    "langchain-core",
    "groq",
    "pydantic",
]


requirements_text = ""


if REQUIREMENTS_PATH.exists():

    requirements_text = (
        REQUIREMENTS_PATH.read_text(
            encoding="utf-8"
        )
        .lower()
    )


print()
print("=" * 70)
print("REQUIREMENTS CHECK")
print("=" * 70)


missing_packages = []


for package in required_packages:

    found = (
        package.lower()
        in requirements_text
    )

    if found:

        status = "OK"

    else:

        status = "MISSING"

        missing_packages.append(
            package
        )


    print(
        f"{status:8} | {package}"
    )


# ------------------------------------------------------------
# 6. Обновляем README новыми компонентами
# ------------------------------------------------------------

if not README_PATH.exists():
    raise RuntimeError(
        "README.md не найден."
    )


readme_text = (
    README_PATH.read_text(
        encoding="utf-8"
    )
)


FINAL_MARKER = (
    "<!-- FINAL_IMPLEMENTATION_SECTION -->"
)


if FINAL_MARKER not in readme_text:

    lines = []

    def add(text=""):
        lines.append(str(text))


    add()
    add("---")
    add()
    add(FINAL_MARKER)
    add()
    add("# Реализованные production-oriented компоненты")
    add()

    add(
        "Помимо экспериментальной ML-части, "
        "в репозитории реализованы дополнительные "
        "компоненты system design."
    )

    add()

    add("### LangGraph orchestration")
    add()

    add(
        "Реализован исполняемый end-to-end граф "
        "с маршрутизацией между intent classification, "
        "semantic retrieval, Document RAG, "
        "additional search и human review."
    )

    add()

    add(
        "Код: `src/langgraph_app.py`"
    )

    add()

    add(
        "Smoke-test: `results/langgraph_smoke_test.json`"
    )

    add()

    add("### Structured Output")
    add()

    add(
        "Реализованы строгие Pydantic-схемы "
        "для intent results, RAG sources "
        "и финального ответа ассистента."
    )

    add()

    add(
        "Код: `src/schemas.py`"
    )

    add()

    add(
        "Smoke-test: "
        "`results/structured_output_smoke_test.json`"
    )

    add()

    add("### Tool Calling")
    add()

    add(
        "Реализован внутренний tool registry "
        "с Pydantic-валидацией аргументов "
        "и безопасным executor."
    )

    add()

    add(
        "Код: `src/tools.py`"
    )

    add()

    add(
        "Smoke-test: "
        "`results/tool_calling_smoke_test.json`"
    )

    add()

    add("### Что остаётся production design")
    add()

    add(
        "- отдельная production vector database "
        "(например, Qdrant);"
    )

    add(
        "- cross-encoder reranker;"
    )

    add(
        "- production monitoring / observability stack;"
    )

    add(
        "- интеграция tools с реальными CRM/API;"
    )

    add(
        "- production deployment infrastructure."
    )


    readme_text = (
        readme_text.rstrip()
        + "\n"
        + "\n".join(lines)
        + "\n"
    )


    README_PATH.write_text(
        readme_text,
        encoding="utf-8"
    )


# ------------------------------------------------------------
# 7. Проверяем audit-статусы
# ------------------------------------------------------------

audit_text = (
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
    if AUDIT_PATH.exists()
    else ""
)


audit_checks = {

    "LangGraph IMPLEMENTED": (
        "| LangGraph | IMPLEMENTED |"
        in audit_text
    ),

    "Structured Output IMPLEMENTED": (
        "| Structured Output | IMPLEMENTED |"
        in audit_text
    ),

    "Tool Calling IMPLEMENTED": (
        "| Tool Calling | IMPLEMENTED |"
        in audit_text
    ),

    "Vector Database PARTIAL": (
        "| Vector Database | PARTIAL |"
        in audit_text
    ),

    "Reranker PROPOSED": (
        "| Reranker | PROPOSED |"
        in audit_text
    ),

    "Monitoring DESIGNED": (
        "| Monitoring | DESIGNED |"
        in audit_text
    ),
}


print()
print("=" * 70)
print("REQUIREMENTS AUDIT STATUS")
print("=" * 70)


for name, status in audit_checks.items():

    label = (
        "OK"
        if status
        else "MISSING"
    )

    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 8. Проверяем MLSD
# ------------------------------------------------------------

mlsd_text = (
    MLSD_PATH.read_text(
        encoding="utf-8"
    )
    if MLSD_PATH.exists()
    else ""
)


mlsd_checks = {

    "Problem statement": (
        "# 1. Постановка задачи"
        in mlsd_text
    ),

    "Architecture": (
        "# 6. Архитектура системы"
        in mlsd_text
    ),

    "Data": (
        "# 8. Данные"
        in mlsd_text
    ),

    "Offline metrics": (
        "# 11. Offline metrics"
        in mlsd_text
    ),

    "Online metrics": (
        "# 12. Online metrics"
        in mlsd_text
    ),

    "Deployment": (
        "# 13. Production deployment"
        in mlsd_text
    ),

    "Monitoring": (
        "# 14. Monitoring"
        in mlsd_text
    ),

    "Retraining": (
        "# 15. Retraining strategy"
        in mlsd_text
    ),

    "Risks": (
        "# 17. Риски"
        in mlsd_text
    ),

    "LangGraph implementation": (
        "## 7.5 Реализованный LangGraph workflow"
        in mlsd_text
    ),

    "Structured Output": (
        "## 7.6 Structured Output"
        in mlsd_text
    ),

    "Tool Calling": (
        "## 7.7 Tool Calling"
        in mlsd_text
    ),
}


print()
print("=" * 70)
print("MLSD CHECK")
print("=" * 70)


for name, status in mlsd_checks.items():

    label = (
        "OK"
        if status
        else "MISSING"
    )

    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 9. Создаём финальный checklist
# ------------------------------------------------------------

checklist_lines = []


def cadd(text=""):
    checklist_lines.append(str(text))


cadd("# Final Repository Checklist")
cadd()

cadd("## Core ML")
cadd()
cadd("- [x] Banking77 data protocol")
cadd("- [x] DistilBERT intent classification")
cadd("- [x] Validation-selected confidence threshold")
cadd("- [x] Frozen final test")
cadd("- [x] Semantic retrieval")
cadd("- [x] Document RAG")
cadd("- [x] Independent API-generated QA evaluation")
cadd()

cadd("## System Design")
cadd()
cadd("- [x] End-to-end LangGraph orchestration")
cadd("- [x] Additional search branch")
cadd("- [x] Human-review fallback")
cadd("- [x] Structured Output")
cadd("- [x] Tool Calling layer")
cadd("- [x] Architecture diagrams")
cadd("- [x] MLSD document")
cadd()

cadd("## Production Design")
cadd()
cadd("- [x] Deployment architecture described")
cadd("- [x] Monitoring strategy described")
cadd("- [x] Retraining strategy described")
cadd("- [x] Risks described")
cadd("- [x] Document update strategy described")
cadd()
cadd("- [ ] Production Qdrant deployment")
cadd("- [ ] Cross-encoder reranker")
cadd("- [ ] Real CRM/API tool integrations")
cadd("- [ ] Production monitoring infrastructure")
cadd()
cadd(
    "Последние четыре пункта являются "
    "production extensions и не выдаются "
    "за уже реализованные компоненты prototype."
)
cadd()

cadd("## Reproducibility")
cadd()
cadd("- [x] requirements.txt")
cadd("- [x] .gitignore")
cadd("- [x] Frozen configuration")
cadd("- [x] Final metrics")
cadd("- [x] Smoke-tests")
cadd("- [x] Requirements audit")
cadd()

cadd("## GitHub")
cadd()
cadd("- [x] README")
cadd("- [x] src/")
cadd("- [x] results/")
cadd("- [x] validation/")
cadd("- [x] document_rag/")
cadd("- [x] langgraph/")
cadd("- [x] docs/")
cadd("- [x] assets/")
cadd("- [x] No model weights committed")
cadd("- [x] No API keys committed")


CHECKLIST_PATH.write_text(
    "\n".join(
        checklist_lines
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 10. Проверяем отсутствие тяжёлых файлов
# ------------------------------------------------------------

forbidden_extensions = {
    ".safetensors",
    ".bin",
    ".pt",
    ".pth",
    ".parquet",
}


heavy_or_forbidden = []


for path in GITHUB_PROJECT_DIR.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() in forbidden_extensions:

        heavy_or_forbidden.append(
            str(
                path.relative_to(
                    GITHUB_PROJECT_DIR
                )
            )
        )


# ------------------------------------------------------------
# 11. Считаем размер репозитория
# ------------------------------------------------------------

total_bytes = 0


for path in GITHUB_PROJECT_DIR.rglob("*"):

    if path.is_file():

        total_bytes += (
            path.stat().st_size
        )


repo_size_mb = (
    total_bytes
    / 1024**2
)


# ------------------------------------------------------------
# 12. Итоговый readiness
# ------------------------------------------------------------

all_files_ok = (
    len(missing_files) == 0
)

all_smoke_ok = all(
    smoke_results.values()
)

all_requirements_ok = (
    len(missing_packages) == 0
)

all_audit_ok = all(
    audit_checks.values()
)

all_mlsd_ok = all(
    mlsd_checks.values()
)

no_forbidden = (
    len(heavy_or_forbidden) == 0
)


ready = all([
    all_files_ok,
    all_smoke_ok,
    all_requirements_ok,
    all_audit_ok,
    all_mlsd_ok,
    no_forbidden,
])


# ------------------------------------------------------------
# 13. Финальный вывод
# ------------------------------------------------------------

print()
print("=" * 70)
print("PRE-GITHUB FINAL STATUS")
print("=" * 70)

print(
    "Critical files:",
    "PASS" if all_files_ok else "FAIL"
)

print(
    "Smoke tests:",
    "PASS" if all_smoke_ok else "FAIL"
)

print(
    "requirements.txt:",
    "PASS" if all_requirements_ok else "FAIL"
)

print(
    "Requirements audit:",
    "PASS" if all_audit_ok else "FAIL"
)

print(
    "MLSD:",
    "PASS" if all_mlsd_ok else "FAIL"
)

print(
    "No forbidden heavy files:",
    "PASS" if no_forbidden else "FAIL"
)

print(
    "Repository size:",
    f"{repo_size_mb:.2f} MB"
)

print()


if heavy_or_forbidden:

    print(
        "Forbidden/heavy files found:"
    )

    for item in heavy_or_forbidden:
        print(
            "-",
            item
        )


print()
print(
    "Checklist:"
)

print(
    CHECKLIST_PATH
)

print()
print("=" * 70)


if ready:

    print(
        "CELL 15 COMPLETE"
    )

    print()

    print(
        "FINAL REPOSITORY STATUS: READY FOR GITHUB"
    )

else:

    print(
        "CELL 15 COMPLETE WITH WARNINGS"
    )

    print()

    print(
        "FINAL REPOSITORY STATUS: NOT READY"
    )


print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 16.
# INITIALIZE LOCAL GIT REPOSITORY + FIRST COMMIT
#
# Что делаем:
# 1. Проверяем Git
# 2. Инициализируем репозиторий
# 3. Устанавливаем ветку main
# 4. Проверяем .gitignore
# 5. Добавляем файлы
# 6. Проверяем, что тяжёлые файлы не попали в commit
# 7. Создаём первый commit
#
# Пока НИЧЕГО не отправляем на GitHub.
# ============================================================

import subprocess
from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


REPO_DIR = Path(
    GITHUB_PROJECT_DIR
)


if not REPO_DIR.exists():
    raise RuntimeError(
        f"Repository directory not found: {REPO_DIR}"
    )


# ------------------------------------------------------------
# 2. Helper для git-команд
# ------------------------------------------------------------

def run_git(
    *args,
    check=True
):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )

    if (
        check
        and result.returncode != 0
    ):
        raise RuntimeError(
            "Git command failed:\n"
            f"git {' '.join(args)}\n\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


# ------------------------------------------------------------
# 3. Проверяем Git
# ------------------------------------------------------------

git_version = subprocess.run(
    [
        "git",
        "--version",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()


print("=" * 70)
print("GIT INITIALIZATION")
print("=" * 70)

print(
    git_version
)


# ------------------------------------------------------------
# 4. Инициализируем repository
# ------------------------------------------------------------

git_dir = (
    REPO_DIR
    / ".git"
)


if git_dir.exists():

    print(
        "Git repository already initialized."
    )

else:

    run_git(
        "init"
    )

    print(
        "Git repository initialized."
    )


# ------------------------------------------------------------
# 5. Устанавливаем main
# ------------------------------------------------------------

run_git(
    "branch",
    "-M",
    "main"
)


# ------------------------------------------------------------
# 6. Настраиваем локального Git author
#
# Эти данные относятся только к commit metadata.
# При необходимости позже можно заменить.
# ------------------------------------------------------------

current_name = run_git(
    "config",
    "user.name",
    check=False
).stdout.strip()


current_email = run_git(
    "config",
    "user.email",
    check=False
).stdout.strip()


if not current_name:

    run_git(
        "config",
        "user.name",
        "ML Project"
    )


if not current_email:

    run_git(
        "config",
        "user.email",
        "ml-project@example.com"
    )


print(
    "Git user.name:",
    run_git(
        "config",
        "user.name"
    ).stdout.strip()
)

print(
    "Git user.email:",
    run_git(
        "config",
        "user.email"
    ).stdout.strip()
)


# ------------------------------------------------------------
# 7. Проверяем .gitignore
# ------------------------------------------------------------

GITIGNORE_PATH = (
    REPO_DIR
    / ".gitignore"
)


if not GITIGNORE_PATH.exists():
    raise RuntimeError(
        ".gitignore отсутствует."
    )


print()
print("=" * 70)
print(".GITIGNORE CHECK")
print("=" * 70)


forbidden_patterns = [
    "*.safetensors",
    "*.bin",
    "*.pt",
    "*.pth",
    "*.parquet",
    ".env",
]


gitignore_text = (
    GITIGNORE_PATH.read_text(
        encoding="utf-8"
    )
)


for pattern in forbidden_patterns:

    found = (
        pattern
        in gitignore_text
    )

    print(
        f"{'OK' if found else 'MISSING':8} | "
        f"{pattern}"
    )


# ------------------------------------------------------------
# 8. Добавляем файлы
# ------------------------------------------------------------

run_git(
    "add",
    "."
)


# ------------------------------------------------------------
# 9. Получаем staged files
# ------------------------------------------------------------

staged_result = run_git(
    "diff",
    "--cached",
    "--name-only"
)


staged_files = [
    line.strip()
    for line
    in staged_result.stdout.splitlines()
    if line.strip()
]


print()
print("=" * 70)
print("STAGED FILES")
print("=" * 70)


for file in staged_files:

    print(
        file
    )


# ------------------------------------------------------------
# 10. Проверяем запрещённые файлы
# ------------------------------------------------------------

forbidden_extensions = {
    ".safetensors",
    ".bin",
    ".pt",
    ".pth",
    ".parquet",
}


bad_files = []


for relative_path in staged_files:

    path = Path(
        relative_path
    )

    if path.suffix.lower() in forbidden_extensions:

        bad_files.append(
            relative_path
        )


if bad_files:

    print()
    print(
        "ERROR: forbidden files detected:"
    )

    for file in bad_files:

        print(
            "-",
            file
        )

    raise RuntimeError(
        "Запрещённые тяжёлые файлы "
        "попали в Git staging area."
    )


# ------------------------------------------------------------
# 11. Проверяем возможные secrets
# ------------------------------------------------------------

secret_patterns = [
    "gsk_",
    "sk-",
    "github_pat_",
    "ghp_",
]


possible_secret_files = []


for relative_path in staged_files:

    path = (
        REPO_DIR
        / relative_path
    )

    if not path.is_file():
        continue

    try:

        text = path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    if any(
        pattern in text
        for pattern in secret_patterns
    ):

        possible_secret_files.append(
            relative_path
        )


print()
print("=" * 70)
print("SECRET SCAN")
print("=" * 70)


if possible_secret_files:

    for file in possible_secret_files:

        print(
            "WARNING  |",
            file
        )

    raise RuntimeError(
        "Обнаружена строка, похожая на secret/API key. "
        "Commit остановлен."
    )

else:

    print(
        "PASS     | No obvious API keys detected"
    )


# ------------------------------------------------------------
# 12. Создаём первый commit
# ------------------------------------------------------------

existing_commit = run_git(
    "rev-parse",
    "--verify",
    "HEAD",
    check=False
)


if existing_commit.returncode == 0:

    # Репозиторий уже имеет commit.
    status = run_git(
        "status",
        "--porcelain"
    ).stdout.strip()

    if status:

        commit_result = run_git(
            "commit",
            "-m",
            "Prepare ML assistant project for GitHub"
        )

        print()
        print(
            commit_result.stdout
        )

    else:

        print()
        print(
            "No new changes to commit."
        )

else:

    commit_result = run_git(
        "commit",
        "-m",
        "Initial ML assistant project"
    )

    print()
    print(
        commit_result.stdout
    )


# ------------------------------------------------------------
# 13. Проверяем историю
# ------------------------------------------------------------

log_result = run_git(
    "log",
    "--oneline",
    "-5"
)


status_result = run_git(
    "status",
    "--short"
)


print()
print("=" * 70)
print("GIT LOG")
print("=" * 70)

print(
    log_result.stdout.strip()
)


print()
print("=" * 70)
print("WORKING TREE STATUS")
print("=" * 70)


if status_result.stdout.strip():

    print(
        status_result.stdout
    )

else:

    print(
        "CLEAN"
    )


# ------------------------------------------------------------
# 14. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 16 COMPLETE")
print("=" * 70)

print(
    "Local Git repository is ready."
)

print(
    "Branch: main"
)

print(
    "Heavy model/data files are not staged."
)

print(
    "No obvious API secrets detected."
)

print()

print(
    "Next step: connect this repository "
    "to GitHub and push main."
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 17.
# GITHUB CLI: INSTALL + AUTH + CREATE REPO + PUSH
#
# Что делает:
# 1. Проверяет/устанавливает gh
# 2. Запрашивает GitHub token скрытым вводом
# 3. Логинится через gh auth login
# 4. Создаёт GitHub repository
# 5. Подключает origin
# 6. Пушит ветку main
# 7. Проверяет результат
#
# Token не печатается.
# ============================================================

import os
import getpass
import subprocess
from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем локальный Git repository
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


REPO_DIR = Path(
    GITHUB_PROJECT_DIR
)


if not (
    REPO_DIR
    / ".git"
).exists():

    raise RuntimeError(
        "Git repository не найден. "
        "Сначала выполни ячейку 16."
    )


# ------------------------------------------------------------
# 2. Настройки GitHub repository
# ------------------------------------------------------------

REPO_NAME = (
    "ml-customer-support-assistant"
)

REPO_DESCRIPTION = (
    "ML customer support assistant with DistilBERT, "
    "semantic retrieval, Document RAG, LangGraph, "
    "structured output and tool calling."
)

VISIBILITY = "public"


# ------------------------------------------------------------
# 3. Helper
# ------------------------------------------------------------

def run_cmd(
    command,
    cwd=None,
    input_text=None,
    env=None,
    check=True,
):

    result = subprocess.run(
        command,
        cwd=cwd,
        input=input_text,
        capture_output=True,
        text=True,
        env=env,
    )

    if (
        check
        and result.returncode != 0
    ):

        raise RuntimeError(
            "Command failed:\n"
            + " ".join(command)
            + "\n\nSTDOUT:\n"
            + result.stdout
            + "\nSTDERR:\n"
            + result.stderr
        )

    return result


# ------------------------------------------------------------
# 4. Проверяем gh
# ------------------------------------------------------------

gh_check = run_cmd(
    ["gh", "--version"],
    check=False,
)


if gh_check.returncode != 0:

    print(
        "GitHub CLI not found. Installing..."
    )

    # --------------------------------------------
    # Устанавливаем GitHub CLI
    # --------------------------------------------

    run_cmd(
        [
            "bash",
            "-c",
            (
                "type -p curl >/dev/null || "
                "(sudo apt update && sudo apt install curl -y)"
            ),
        ]
    )

    run_cmd(
        [
            "bash",
            "-c",
            (
                "curl -fsSL "
                "https://cli.github.com/packages/"
                "githubcli-archive-keyring.gpg "
                "| sudo dd of=/usr/share/keyrings/"
                "githubcli-archive-keyring.gpg"
            ),
        ]
    )

    run_cmd(
        [
            "bash",
            "-c",
            (
                "sudo chmod go+r "
                "/usr/share/keyrings/"
                "githubcli-archive-keyring.gpg"
            ),
        ]
    )

    run_cmd(
        [
            "bash",
            "-c",
            (
                'echo "deb '
                '[arch=$(dpkg --print-architecture) '
                'signed-by=/usr/share/keyrings/'
                'githubcli-archive-keyring.gpg] '
                'https://cli.github.com/packages stable main" '
                '| sudo tee '
                '/etc/apt/sources.list.d/'
                'github-cli.list > /dev/null'
            ),
        ]
    )

    run_cmd(
        [
            "bash",
            "-c",
            "sudo apt update -qq",
        ]
    )

    run_cmd(
        [
            "bash",
            "-c",
            "sudo apt install gh -y -qq",
        ]
    )


# ------------------------------------------------------------
# 5. Показываем установленную версию
# ------------------------------------------------------------

gh_version = run_cmd(
    ["gh", "--version"]
)


print("=" * 70)
print("GITHUB CLI")
print("=" * 70)

print(
    gh_version.stdout.splitlines()[0]
)


# ------------------------------------------------------------
# 6. Проверяем текущую авторизацию
# ------------------------------------------------------------

auth_status = run_cmd(
    [
        "gh",
        "auth",
        "status",
    ],
    check=False,
)


if auth_status.returncode == 0:

    print()
    print(
        "GitHub CLI already authenticated."
    )

else:

    # --------------------------------------------------------
    # 7. Запрашиваем token
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("GITHUB AUTHENTICATION")
    print("=" * 70)

    print()
    print(
        "Вставь GitHub Personal Access Token."
    )

    print(
        "Ввод скрыт."
    )

    token = getpass.getpass(
        "GitHub token: "
    ).strip()


    if not token:

        raise RuntimeError(
            "Token не введён."
        )


    # --------------------------------------------------------
    # 8. Логинимся через stdin
    # --------------------------------------------------------

    login_result = run_cmd(
        [
            "gh",
            "auth",
            "login",
            "--hostname",
            "github.com",
            "--git-protocol",
            "https",
            "--with-token",
        ],
        input_text=token + "\n",
    )


    # Убираем token из переменной
    token = None


    print()
    print(
        "GitHub authentication successful."
    )


# ------------------------------------------------------------
# 9. Проверяем аккаунт
# ------------------------------------------------------------

username_result = run_cmd(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".login",
    ]
)


GITHUB_USERNAME = (
    username_result.stdout.strip()
)


print()
print(
    "Authenticated as:",
    GITHUB_USERNAME
)


# ------------------------------------------------------------
# 10. Исправляем Git author
# ------------------------------------------------------------

run_cmd(
    [
        "git",
        "config",
        "user.name",
        GITHUB_USERNAME,
    ],
    cwd=REPO_DIR,
)


noreply_email = (
    f"{GITHUB_USERNAME}"
    "@users.noreply.github.com"
)


run_cmd(
    [
        "git",
        "config",
        "user.email",
        noreply_email,
    ],
    cwd=REPO_DIR,
)


# Перезаписываем author последнего commit
run_cmd(
    [
        "git",
        "commit",
        "--amend",
        "--reset-author",
        "--no-edit",
    ],
    cwd=REPO_DIR,
)


# ------------------------------------------------------------
# 11. Проверяем, существует ли repo
# ------------------------------------------------------------

repo_full_name = (
    f"{GITHUB_USERNAME}/{REPO_NAME}"
)


repo_check = run_cmd(
    [
        "gh",
        "repo",
        "view",
        repo_full_name,
    ],
    check=False,
)


print()
print("=" * 70)
print("GITHUB REPOSITORY")
print("=" * 70)


if repo_check.returncode == 0:

    print(
        "Repository already exists:",
        repo_full_name
    )


    # Настраиваем origin вручную
    remote_url = (
        f"https://github.com/"
        f"{repo_full_name}.git"
    )


    origin_check = run_cmd(
        [
            "git",
            "remote",
            "get-url",
            "origin",
        ],
        cwd=REPO_DIR,
        check=False,
    )


    if origin_check.returncode == 0:

        run_cmd(
            [
                "git",
                "remote",
                "set-url",
                "origin",
                remote_url,
            ],
            cwd=REPO_DIR,
        )

    else:

        run_cmd(
            [
                "git",
                "remote",
                "add",
                "origin",
                remote_url,
            ],
            cwd=REPO_DIR,
        )


    # Push
    run_cmd(
        [
            "git",
            "push",
            "-u",
            "origin",
            "main",
        ],
        cwd=REPO_DIR,
    )


else:

    # --------------------------------------------------------
    # 12. Создаём repo + push одной командой gh
    # --------------------------------------------------------

    create_command = [
        "gh",
        "repo",
        "create",
        REPO_NAME,
        f"--{VISIBILITY}",
        "--description",
        REPO_DESCRIPTION,
        "--source",
        str(REPO_DIR),
        "--remote",
        "origin",
        "--push",
    ]


    create_result = run_cmd(
        create_command,
        cwd=REPO_DIR,
    )


    print(
        create_result.stdout.strip()
    )


# ------------------------------------------------------------
# 13. Проверяем remote
# ------------------------------------------------------------

remote_result = run_cmd(
    [
        "git",
        "remote",
        "-v",
    ],
    cwd=REPO_DIR,
)


status_result = run_cmd(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=REPO_DIR,
)


branch_result = run_cmd(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=REPO_DIR,
)


# ------------------------------------------------------------
# 14. Проверяем remote через gh
# ------------------------------------------------------------

verify_result = run_cmd(
    [
        "gh",
        "repo",
        "view",
        repo_full_name,
        "--json",
        "nameWithOwner,url,visibility,defaultBranchRef",
    ]
)


print()
print("=" * 70)
print("REMOTE CHECK")
print("=" * 70)

print(
    verify_result.stdout.strip()
)

print()
print("Git remote:")
print(
    remote_result.stdout.strip()
)

print()
print(
    "Branch:",
    branch_result.stdout.strip()
)

print(
    "Working tree:",
    "CLEAN"
    if not status_result.stdout.strip()
    else status_result.stdout.strip()
)


# ------------------------------------------------------------
# 15. Итог
# ------------------------------------------------------------

REPO_URL = (
    f"https://github.com/"
    f"{repo_full_name}"
)


print()
print("=" * 70)
print("CELL 17 COMPLETE")
print("=" * 70)

print(
    "GITHUB REPOSITORY CREATED / UPDATED"
)

print()

print(
    "Repository:"
)

print(
    REPO_URL
)

print()

print(
    "Branch: main"
)

print(
    "Push: COMPLETE"
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 18.
# FIXED REMOTE GITHUB VERIFICATION + FINAL CHECKLIST
#
# Исправление:
# - добавлен gh auth setup-git
# - git push теперь использует авторизацию GitHub CLI
#
# Проверяем:
# 1. gh authentication
# 2. git credential helper
# 3. remote metadata
# 4. local HEAD == remote main
# 5. remote files
# 6. README / MLSD / smoke tests
# 7. сохраняем verification report
# 8. commit + push verification report
# ============================================================

import json
import subprocess
from pathlib import Path


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


REPO_DIR = Path(
    GITHUB_PROJECT_DIR
)


if not (
    REPO_DIR
    / ".git"
).exists():

    raise RuntimeError(
        "Локальный Git repository не найден."
    )


# ------------------------------------------------------------
# 2. Helper
# ------------------------------------------------------------

def run_cmd(
    command,
    cwd=None,
    check=True,
):

    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
    )

    if (
        check
        and result.returncode != 0
    ):

        raise RuntimeError(
            "Command failed:\n"
            + " ".join(command)
            + "\n\nSTDOUT:\n"
            + result.stdout
            + "\nSTDERR:\n"
            + result.stderr
        )

    return result


# ------------------------------------------------------------
# 3. Проверяем gh auth
# ------------------------------------------------------------

print("=" * 70)
print("GITHUB AUTH CHECK")
print("=" * 70)


auth_result = run_cmd(
    [
        "gh",
        "auth",
        "status",
        "--hostname",
        "github.com",
    ],
    check=False,
)


auth_ok = (
    auth_result.returncode == 0
)


print(
    "PASS"
    if auth_ok
    else "FAIL",
    "| gh authentication"
)


if not auth_ok:

    print(
        auth_result.stderr
    )

    raise RuntimeError(
        "GitHub CLI authentication отсутствует."
    )


# ------------------------------------------------------------
# 4. КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ:
# Настраиваем git использовать gh credentials
# ------------------------------------------------------------

setup_git_result = run_cmd(
    [
        "gh",
        "auth",
        "setup-git",
        "--hostname",
        "github.com",
    ],
    check=False,
)


setup_git_ok = (
    setup_git_result.returncode == 0
)


print(
    "PASS"
    if setup_git_ok
    else "FAIL",
    "| gh auth setup-git"
)


if not setup_git_ok:

    print(
        setup_git_result.stderr
    )

    raise RuntimeError(
        "Не удалось настроить Git credential helper."
    )


# ------------------------------------------------------------
# 5. Проверяем credential helper
# ------------------------------------------------------------

credential_helper = run_cmd(
    [
        "git",
        "config",
        "--get-all",
        "credential.https://github.com.helper",
    ],
    cwd=REPO_DIR,
    check=False,
)


print()
print("=" * 70)
print("GIT CREDENTIAL CHECK")
print("=" * 70)


if credential_helper.stdout.strip():

    print(
        "PASS     | Git credential helper configured"
    )

    print(
        credential_helper.stdout.strip()
    )

else:

    print(
        "INFO     | gh configured credentials globally"
    )


# ------------------------------------------------------------
# 6. Получаем username
# ------------------------------------------------------------

username = run_cmd(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".login",
    ]
).stdout.strip()


REPO_NAME = (
    "ml-customer-support-assistant"
)

REPO_FULL_NAME = (
    f"{username}/{REPO_NAME}"
)


# ------------------------------------------------------------
# 7. Получаем remote metadata
# ------------------------------------------------------------

repo_json_raw = run_cmd(
    [
        "gh",
        "repo",
        "view",
        REPO_FULL_NAME,
        "--json",
        (
            "nameWithOwner,"
            "url,"
            "visibility,"
            "defaultBranchRef,"
            "description"
        ),
    ]
).stdout


repo_info = json.loads(
    repo_json_raw
)


print()
print("=" * 70)
print("REMOTE REPOSITORY METADATA")
print("=" * 70)

print(
    "Repository:",
    repo_info.get(
        "nameWithOwner"
    )
)

print(
    "URL:",
    repo_info.get(
        "url"
    )
)

print(
    "Visibility:",
    repo_info.get(
        "visibility"
    )
)


default_branch = (
    repo_info.get(
        "defaultBranchRef"
    )
    or {}
).get(
    "name"
)


print(
    "Default branch:",
    default_branch
)


# ------------------------------------------------------------
# 8. Сначала пушим возможный локальный commit,
# оставшийся после прошлой неудачной попытки
# ------------------------------------------------------------

ahead_result = run_cmd(
    [
        "git",
        "rev-list",
        "--count",
        "origin/main..HEAD",
    ],
    cwd=REPO_DIR,
    check=False,
)


ahead_count = 0


if ahead_result.returncode == 0:

    try:

        ahead_count = int(
            ahead_result.stdout.strip()
            or "0"
        )

    except Exception:

        ahead_count = 0


if ahead_count > 0:

    print()
    print("=" * 70)
    print("PENDING LOCAL COMMITS")
    print("=" * 70)

    print(
        "Commits ahead of origin/main:",
        ahead_count
    )

    push_pending = run_cmd(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        cwd=REPO_DIR,
    )

    print(
        "PASS     | pending commits pushed"
    )


# ------------------------------------------------------------
# 9. Fetch и сравнение HEAD
# ------------------------------------------------------------

run_cmd(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ],
    cwd=REPO_DIR,
)


local_head = run_cmd(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO_DIR,
).stdout.strip()


remote_head = run_cmd(
    [
        "git",
        "rev-parse",
        "origin/main",
    ],
    cwd=REPO_DIR,
).stdout.strip()


commit_match = (
    local_head
    == remote_head
)


print()
print("=" * 70)
print("COMMIT SYNC CHECK")
print("=" * 70)

print(
    "Local HEAD :",
    local_head
)

print(
    "Remote main:",
    remote_head
)

print(
    "PASS"
    if commit_match
    else "FAIL",
    "| local HEAD == origin/main"
)


# ------------------------------------------------------------
# 10. Проверяем remote files
# ------------------------------------------------------------

required_remote_files = [

    "README.md",

    "requirements.txt",

    "src/langgraph_app.py",
    "src/schemas.py",
    "src/tools.py",

    "docs/mlsd.md",
    "docs/requirements_audit.md",
    "docs/FINAL_REPOSITORY_CHECKLIST.md",

    "results/FINAL_CORRECTED_SUMMARY.json",
    "results/langgraph_smoke_test.json",
    "results/structured_output_smoke_test.json",
    "results/tool_calling_smoke_test.json",

    "assets/architecture_high_level.md",
    "assets/document_rag_flow.md",
    "assets/intent_inference_flow.md",
    "assets/training_evaluation_flow.md",
]


print()
print("=" * 70)
print("REMOTE FILE CHECK")
print("=" * 70)


remote_file_results = {}


for relative_path in required_remote_files:

    result = run_cmd(
        [
            "gh",
            "api",
            (
                f"repos/{REPO_FULL_NAME}/contents/"
                f"{relative_path}"
            ),
            "--jq",
            ".path",
        ],
        check=False,
    )


    exists = (
        result.returncode == 0
    )


    remote_file_results[
        relative_path
    ] = exists


    print(
        f"{'OK' if exists else 'MISSING':8} | "
        f"{relative_path}"
    )


# ------------------------------------------------------------
# 11. Проверяем README
# ------------------------------------------------------------

readme_result = run_cmd(
    [
        "gh",
        "api",
        f"repos/{REPO_FULL_NAME}/readme",
        "--jq",
        ".name",
    ],
    check=False,
)


readme_ok = (
    readme_result.returncode == 0
    and readme_result.stdout.strip()
    == "README.md"
)


# ------------------------------------------------------------
# 12. Проверяем branch main
# ------------------------------------------------------------

branch_result = run_cmd(
    [
        "gh",
        "api",
        (
            f"repos/{REPO_FULL_NAME}/branches/main"
        ),
        "--jq",
        ".name",
    ],
    check=False,
)


branch_ok = (
    branch_result.returncode == 0
    and branch_result.stdout.strip()
    == "main"
)


# ------------------------------------------------------------
# 13. Проверяем working tree ДО нового report
# ------------------------------------------------------------

local_status_before = run_cmd(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=REPO_DIR,
).stdout.strip()


# ------------------------------------------------------------
# 14. Финальная сводка remote
# ------------------------------------------------------------

all_remote_files_ok = all(
    remote_file_results.values()
)


checks = {

    "GitHub auth": (
        auth_ok
    ),

    "Git credential helper": (
        setup_git_ok
    ),

    "Repository exists": (
        repo_info.get(
            "nameWithOwner"
        )
        == REPO_FULL_NAME
    ),

    "Default branch main": (
        default_branch
        == "main"
    ),

    "Main branch available": (
        branch_ok
    ),

    "Local/remote commit match": (
        commit_match
    ),

    "README available": (
        readme_ok
    ),

    "All required remote files": (
        all_remote_files_ok
    ),
}


print()
print("=" * 70)
print("FINAL REMOTE VERIFICATION")
print("=" * 70)


all_passed = True


for name, status in checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    if not status:
        all_passed = False


    print(
        f"{label:8} | {name}"
    )


# ------------------------------------------------------------
# 15. Сохраняем verification report
# ------------------------------------------------------------

report = {

    "repository": (
        REPO_FULL_NAME
    ),

    "url": (
        repo_info.get(
            "url"
        )
    ),

    "visibility": (
        repo_info.get(
            "visibility"
        )
    ),

    "default_branch": (
        default_branch
    ),

    "local_head_before_report_commit": (
        local_head
    ),

    "remote_head_before_report_commit": (
        remote_head
    ),

    "checks": (
        checks
    ),

    "remote_files": (
        remote_file_results
    ),

    "all_passed": (
        all_passed
    ),
}


REMOTE_REPORT_PATH = (
    REPO_DIR
    / "results"
    / "github_remote_verification.json"
)


with open(
    REMOTE_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 16. Commit verification report при необходимости
# ------------------------------------------------------------

run_cmd(
    [
        "git",
        "add",
        "results/github_remote_verification.json",
    ],
    cwd=REPO_DIR,
)


staged = run_cmd(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ],
    cwd=REPO_DIR,
).stdout.strip()


if staged:

    run_cmd(
        [
            "git",
            "commit",
            "-m",
            "Add GitHub remote verification report",
        ],
        cwd=REPO_DIR,
    )

    print()
    print(
        "PASS     | verification report committed"
    )

else:

    print()
    print(
        "INFO     | verification report unchanged"
    )


# ------------------------------------------------------------
# 17. Push через настроенный gh credential helper
# ------------------------------------------------------------

push_result = run_cmd(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    cwd=REPO_DIR,
)


print(
    "PASS     | verification report pushed"
)


# ------------------------------------------------------------
# 18. Финальная синхронизация
# ------------------------------------------------------------

run_cmd(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ],
    cwd=REPO_DIR,
)


final_local_head = run_cmd(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO_DIR,
).stdout.strip()


final_remote_head = run_cmd(
    [
        "git",
        "rev-parse",
        "origin/main",
    ],
    cwd=REPO_DIR,
).stdout.strip()


final_commit_match = (
    final_local_head
    == final_remote_head
)


final_status = run_cmd(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=REPO_DIR,
).stdout.strip()


working_tree_clean = (
    not final_status
)


# ------------------------------------------------------------
# 19. Финальный статус
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL PUSH CHECK")
print("=" * 70)

print(
    "Local HEAD :",
    final_local_head
)

print(
    "Remote main:",
    final_remote_head
)

print(
    "PASS"
    if final_commit_match
    else "FAIL",
    "| final commit sync"
)

print(
    "PASS"
    if working_tree_clean
    else "FAIL",
    "| working tree clean"
)


print()
print("=" * 70)


if (
    all_passed
    and final_commit_match
    and working_tree_clean
):

    print(
        "CELL 18 COMPLETE"
    )

    print()

    print(
        "REMOTE GITHUB STATUS: VERIFIED"
    )

    print()

    print(
        "Repository:"
    )

    print(
        repo_info.get(
            "url"
        )
    )

    print()

    print(
        "Все ключевые файлы доступны."
    )

    print(
        "Verification report загружен."
    )

    print(
        "main полностью синхронизирован."
    )

else:

    print(
        "CELL 18 COMPLETE WITH WARNINGS"
    )

    print()

    print(
        "REMOTE GITHUB STATUS: CHECK REQUIRED"
    )


print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 19.
# FINAL CONTENT AUDIT:
# README + MLSD + MENTOR FEEDBACK + ORIGINAL REQUIREMENTS
#
# Проверяем:
# 1. Исходную постановку задачи
# 2. Основные ML-компоненты
# 3. Все замечания ментора
# 4. Финальные метрики
# 5. Корректность train/val/test protocol
# 6. LangGraph / RAG / Human Review
# 7. MLSD production sections
# 8. Наличие воспроизводимости
# 9. Наличие/отсутствие финального notebook
#
# НИЧЕГО НЕ МЕНЯЕМ.
# Только аудит.
# ============================================================

from pathlib import Path
import json
import re


# ------------------------------------------------------------
# 1. Проверяем workspace
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


REPO_DIR = Path(
    GITHUB_PROJECT_DIR
)

README_PATH = (
    REPO_DIR
    / "README.md"
)

MLSD_PATH = (
    REPO_DIR
    / "docs"
    / "mlsd.md"
)

AUDIT_PATH = (
    REPO_DIR
    / "docs"
    / "requirements_audit.md"
)

SUMMARY_PATH = (
    REPO_DIR
    / "results"
    / "FINAL_CORRECTED_SUMMARY.json"
)


for path in [
    README_PATH,
    MLSD_PATH,
    AUDIT_PATH,
    SUMMARY_PATH,
]:

    if not path.exists():

        raise RuntimeError(
            f"Не найден файл: {path}"
        )


# ------------------------------------------------------------
# 2. Загружаем документы
# ------------------------------------------------------------

readme = (
    README_PATH
    .read_text(
        encoding="utf-8"
    )
)

mlsd = (
    MLSD_PATH
    .read_text(
        encoding="utf-8"
    )
)

requirements_audit = (
    AUDIT_PATH
    .read_text(
        encoding="utf-8"
    )
)


with open(
    SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    summary = json.load(f)


combined = (
    readme
    + "\n"
    + mlsd
    + "\n"
    + requirements_audit
)


combined_lower = (
    combined.lower()
)


# ------------------------------------------------------------
# 3. Helper
# ------------------------------------------------------------

audit_rows = []


def check(
    category,
    requirement,
    condition,
    importance="REQUIRED",
    note=""
):

    audit_rows.append(
        {
            "category": category,
            "requirement": requirement,
            "status": (
                "PASS"
                if condition
                else (
                    "WARN"
                    if importance == "RECOMMENDED"
                    else "FAIL"
                )
            ),
            "importance": importance,
            "note": note,
        }
    )


# ============================================================
# 4. ИСХОДНАЯ ЗАДАЧА
# ============================================================

check(
    "Original task",
    "Employee/internal support assistant",
    (
        "ассистент" in combined_lower
        or "assistant" in combined_lower
    ),
)

check(
    "Original task",
    "Internal documents / regulations / instructions",
    (
        "внутренн" in combined_lower
        and "документ" in combined_lower
    ),
)

check(
    "Original task",
    "Query type detection",
    (
        "determine_query_type"
        in combined
        or "определ" in combined_lower
    ),
)

check(
    "Original task",
    "Information retrieval",
    (
        "retrieval"
        in combined_lower
    ),
)

check(
    "Original task",
    "Additional search",
    (
        "additional_search"
        in combined
        or "дополнительн" in combined_lower
    ),
)

check(
    "Original task",
    "Answer validation",
    (
        "validate_answer"
        in combined
        or "провер" in combined_lower
    ),
)

check(
    "Original task",
    "Final result / response",
    (
        "final_answer"
        in combined
        or "финальн" in combined_lower
    ),
)


# ============================================================
# 5. ТЕХНОЛОГИЧЕСКИЕ КОМПОНЕНТЫ
# ============================================================

tech_checks = {
    "Python": "python",
    "DistilBERT": "distilbert",
    "Embeddings": "embedding",
    "Semantic Retrieval": "semantic retrieval",
    "RAG": "rag",
    "LLM": "llm",
    "LangGraph": "langgraph",
    "Structured Output": "structured output",
    "Tool Calling": "tool calling",
    "Human Review": "human review",
}


for name, term in tech_checks.items():

    check(
        "Technology",
        name,
        term in combined_lower,
    )


check(
    "Technology",
    "Vector DB design",
    (
        "qdrant" in combined_lower
        or "faiss" in combined_lower
        or "vector database" in combined_lower
    ),
    importance="RECOMMENDED",
    note=(
        "Допускается как production design, "
        "не обязательно как deployed component."
    ),
)


check(
    "Technology",
    "Reranker",
    (
        "reranker" in combined_lower
    ),
    importance="RECOMMENDED",
    note=(
        "Может оставаться proposed/future work."
    ),
)


# ============================================================
# 6. ЗАМЕЧАНИЯ МЕНТОРА
# ============================================================

# 6.1 Validation, а не test, для выбора threshold/architecture

mentor_1 = (
    (
        "validation" in combined_lower
        or "val_part" in combined_lower
    )
    and (
        "threshold" in combined_lower
    )
    and (
        "test" in combined_lower
    )
)

check(
    "Mentor feedback",
    "Threshold/architecture selected on validation, not test",
    mentor_1,
    note=(
        "Главное методологическое замечание ментора."
    ),
)


# 6.2 API-generated unseen QA для RAG

mentor_2 = (
    (
        "api-generated" in combined_lower
        or "api generated" in combined_lower
        or "api_generated" in combined_lower
    )
    and (
        "qa" in combined_lower
    )
)

check(
    "Mentor feedback",
    "Document RAG evaluated on API-generated QA",
    mentor_2,
)


# 6.3 Baseline vs LangGraph

mentor_3 = (
    (
        "baseline" in combined_lower
    )
    and (
        "langgraph" in combined_lower
    )
    and (
        "coverage" in combined_lower
    )
    and (
        "automatic accuracy" in combined_lower
        or "automatic_accuracy" in combined_lower
    )
)

check(
    "Mentor feedback",
    "Direct baseline vs LangGraph comparison",
    mentor_3,
)


# ============================================================
# 7. DATA PROTOCOL
# ============================================================

check(
    "Data protocol",
    "train_part documented",
    "8494" in combined,
)

check(
    "Data protocol",
    "val_part documented",
    "1499" in combined,
)

check(
    "Data protocol",
    "test documented",
    "3076" in combined,
)

check(
    "Data protocol",
    "77 classes documented",
    "77" in combined,
)

check(
    "Data protocol",
    "Test used only after decisions frozen",
    (
        "freeze" in combined_lower
        or "заморож" in combined_lower
        or "однократ" in combined_lower
    ),
)


# ============================================================
# 8. ФИНАЛЬНЫЕ МЕТРИКИ
# ============================================================

required_metrics = {
    "DistilBERT Accuracy 92.56%": "92.56",
    "Macro F1 92.55%": "92.55",
    "Threshold 0.65": "0.65",
    "Coverage 93.95%": "93.95",
    "Automatic Accuracy 95.22%": "95.22",
    "RAG Answer Accuracy 93.33%": "93.33",
    "RAG Overall Success 76.67%": "76.67",
}


for name, value in required_metrics.items():

    check(
        "Final metrics",
        name,
        value in combined,
    )


# ============================================================
# 9. LANGGRAPH
# ============================================================

langgraph_required = [
    "determine_query_type",
    "classify_intent",
    "semantic_retrieval",
    "document_retrieval",
    "additional_search",
    "validate_answer",
    "human_review",
]


for node in langgraph_required:

    check(
        "LangGraph",
        node,
        node in combined,
    )


check(
    "LangGraph",
    "Smoke-test documented",
    (
        "langgraph_smoke_test.json"
        in combined
    ),
)


# ============================================================
# 10. MLSD
# ============================================================

mlsd_sections = [
    "# 1. Постановка задачи",
    "# 2. Бизнес-цель",
    "# 4. Функциональные требования",
    "# 5. Нефункциональные требования",
    "# 6. Архитектура системы",
    "# 8. Данные",
    "# 9. Training pipeline",
    "# 10. Online inference flow",
    "# 11. Offline metrics",
    "# 12. Online metrics",
    "# 13. Production deployment",
    "# 14. Monitoring",
    "# 15. Retraining strategy",
    "# 17. Риски",
]


for section in mlsd_sections:

    check(
        "MLSD",
        section,
        section in mlsd,
    )


# ============================================================
# 11. REPRODUCIBILITY
# ============================================================

reproducibility_files = [
    "requirements.txt",
    ".gitignore",
    "src/config.py",
    "validation/selected_threshold_final.json",
    "results/FINAL_CORRECTED_SUMMARY.json",
]


for relative_path in reproducibility_files:

    check(
        "Reproducibility",
        relative_path,
        (
            REPO_DIR
            / relative_path
        ).exists(),
    )


# ============================================================
# 12. NOTEBOOK CHECK
# ============================================================

notebooks = list(
    REPO_DIR.rglob(
        "*.ipynb"
    )
)


check(
    "Repository completeness",
    "Final notebook included",
    len(notebooks) > 0,
    importance="RECOMMENDED",
    note=(
        "Сейчас notebook не является обязательным для MLSD, "
        "но для учебной/портфолио сдачи его наличие желательно."
    ),
)


# ============================================================
# 13. ПРОВЕРКА СТАРЫХ / DEBUG МЕТРИК
# ============================================================

# Ищем потенциально опасные значения.
# Они не должны выглядеть как финальные результаты.

legacy_patterns = {
    "Old threshold 0.90": r"\b0\.90\b",
    "Debug threshold 0.50": r"\b0\.50\b",
    "Old 100% RAG claim": r"\b100(?:\.0+)?%\b",
}


print("=" * 70)
print("LEGACY / DEBUG VALUE CHECK")
print("=" * 70)


legacy_hits = {}


for name, pattern in legacy_patterns.items():

    matches = re.findall(
        pattern,
        combined,
        flags=re.IGNORECASE,
    )

    legacy_hits[
        name
    ] = len(matches)


    if len(matches) == 0:

        print(
            f"PASS     | {name}: not found"
        )

    else:

        print(
            f"REVIEW   | {name}: "
            f"{len(matches)} occurrence(s)"
        )


# ============================================================
# 14. ВЫВОДИМ АУДИТ
# ============================================================

print()
print("=" * 70)
print("FINAL CONTENT AUDIT")
print("=" * 70)


categories = []


for row in audit_rows:

    if row["category"] not in categories:

        categories.append(
            row["category"]
        )


for category in categories:

    print()
    print(
        f"[{category}]"
    )


    for row in audit_rows:

        if row["category"] != category:

            continue


        print(
            f"{row['status']:6} | "
            f"{row['requirement']}"
        )


        if (
            row["status"] != "PASS"
            and row["note"]
        ):

            print(
                "       ",
                row["note"]
            )


# ============================================================
# 15. SUMMARY
# ============================================================

required_rows = [
    row
    for row in audit_rows
    if row[
        "importance"
    ] == "REQUIRED"
]


recommended_rows = [
    row
    for row in audit_rows
    if row[
        "importance"
    ] == "RECOMMENDED"
]


required_failed = [
    row
    for row in required_rows
    if row[
        "status"
    ] != "PASS"
]


recommended_warn = [
    row
    for row in recommended_rows
    if row[
        "status"
    ] != "PASS"
]


required_passed = (
    len(required_rows)
    - len(required_failed)
)


print()
print("=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print(
    "Required checks:",
    len(required_rows)
)

print(
    "Required passed:",
    required_passed
)

print(
    "Required failed:",
    len(required_failed)
)

print(
    "Recommended warnings:",
    len(recommended_warn)
)


# ------------------------------------------------------------
# 16. Сохраняем отчёт
# ------------------------------------------------------------

audit_report = {
    "required_checks": (
        len(required_rows)
    ),

    "required_passed": (
        required_passed
    ),

    "required_failed": [
        row[
            "requirement"
        ]
        for row in required_failed
    ],

    "recommended_warnings": [
        {
            "requirement": row[
                "requirement"
            ],
            "note": row[
                "note"
            ],
        }
        for row in recommended_warn
    ],

    "legacy_value_occurrences": (
        legacy_hits
    ),

    "notebooks_found": [
        str(
            path.relative_to(
                REPO_DIR
            )
        )
        for path in notebooks
    ],

    "checks": (
        audit_rows
    ),
}


CONTENT_AUDIT_PATH = (
    REPO_DIR
    / "results"
    / "final_content_audit.json"
)


with open(
    CONTENT_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_report,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 17. ИТОГ
# ============================================================

print()
print("=" * 70)


if len(
    required_failed
) == 0:

    print(
        "CELL 19 COMPLETE"
    )

    print()

    print(
        "CONTENT STATUS: CORE REQUIREMENTS PASSED"
    )

else:

    print(
        "CELL 19 COMPLETE WITH ISSUES"
    )

    print()

    print(
        "CONTENT STATUS: FIXES REQUIRED"
    )


print()

print(
    "Audit report:"
)

print(
    CONTENT_AUDIT_PATH
)

print("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА 20.
# FIND FINAL COLAB NOTEBOOK
#
# Цель:
# 1. Найти .ipynb на Google Drive
# 2. Отдельно проверить исходный проект
# 3. Показать наиболее вероятные финальные notebooks
# 4. НИЧЕГО пока не копировать автоматически
#
# После этого выберем правильный notebook и добавим его
# в notebooks/ GitHub-репозитория.
# ============================================================

from pathlib import Path
from datetime import datetime


# ------------------------------------------------------------
# 1. Проверяем основные пути
# ------------------------------------------------------------

if "GITHUB_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "GITHUB_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


if "SOURCE_PROJECT_DIR" not in globals():
    raise RuntimeError(
        "SOURCE_PROJECT_DIR не найден. "
        "Сначала выполни ячейку 1."
    )


DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)


NOTEBOOKS_DIR = (
    GITHUB_PROJECT_DIR
    / "notebooks"
)

NOTEBOOKS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Сначала ищем notebook внутри исходного проекта
# ------------------------------------------------------------

source_notebooks = list(
    SOURCE_PROJECT_DIR.rglob(
        "*.ipynb"
    )
)


print("=" * 70)
print("NOTEBOOKS INSIDE SOURCE PROJECT")
print("=" * 70)


if source_notebooks:

    for path in source_notebooks:

        stat = path.stat()

        modified = datetime.fromtimestamp(
            stat.st_mtime
        ).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        size_mb = (
            stat.st_size
            / 1024**2
        )

        print()
        print(
            "FILE:",
            path
        )

        print(
            "Modified:",
            modified
        )

        print(
            "Size:",
            f"{size_mb:.2f} MB"
        )

else:

    print(
        "No .ipynb files found "
        "inside SOURCE_PROJECT_DIR."
    )


# ------------------------------------------------------------
# 3. Ищем notebooks по всему MyDrive
# ------------------------------------------------------------

print()
print("=" * 70)
print("SEARCHING GOOGLE DRIVE FOR .IPYNB")
print("=" * 70)


all_notebooks = []


for path in DRIVE_ROOT.rglob(
    "*.ipynb"
):

    try:

        stat = path.stat()

        all_notebooks.append(
            {
                "path": path,
                "mtime": stat.st_mtime,
                "size": stat.st_size,
            }
        )

    except Exception:

        pass


# ------------------------------------------------------------
# 4. Сортируем по дате изменения
# ------------------------------------------------------------

all_notebooks = sorted(
    all_notebooks,
    key=lambda x: x["mtime"],
    reverse=True,
)


print(
    "Total notebooks found:",
    len(all_notebooks)
)


# ------------------------------------------------------------
# 5. Ищем наиболее вероятные candidates
# ------------------------------------------------------------

keywords = [
    "ml",
    "customer",
    "classification",
    "banking",
    "rag",
    "langgraph",
    "assistant",
    "corrected",
    "final",
]


candidates = []


for item in all_notebooks:

    search_text = str(
        item["path"]
    ).lower()

    score = sum(
        keyword in search_text
        for keyword in keywords
    )

    if score > 0:

        item = dict(
            item
        )

        item[
            "score"
        ] = score

        candidates.append(
            item
        )


candidates = sorted(
    candidates,
    key=lambda x: (
        x["score"],
        x["mtime"],
    ),
    reverse=True,
)


# ------------------------------------------------------------
# 6. Показываем top candidates
# ------------------------------------------------------------

print()
print("=" * 70)
print("LIKELY PROJECT NOTEBOOKS")
print("=" * 70)


if candidates:

    for index, item in enumerate(
        candidates[:20],
        start=1,
    ):

        modified = datetime.fromtimestamp(
            item["mtime"]
        ).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        size_mb = (
            item["size"]
            / 1024**2
        )

        print()
        print(
            f"[{index}]"
        )

        print(
            "Path:",
            item["path"]
        )

        print(
            "Keyword score:",
            item["score"]
        )

        print(
            "Modified:",
            modified
        )

        print(
            "Size:",
            f"{size_mb:.2f} MB"
        )

else:

    print(
        "No likely project notebooks found."
    )


# ------------------------------------------------------------
# 7. Показываем последние notebooks вообще
# ------------------------------------------------------------

print()
print("=" * 70)
print("MOST RECENT NOTEBOOKS ON DRIVE")
print("=" * 70)


for index, item in enumerate(
    all_notebooks[:15],
    start=1,
):

    modified = datetime.fromtimestamp(
        item["mtime"]
    ).strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    size_mb = (
        item["size"]
        / 1024**2
    )

    print()
    print(
        f"[{index}]"
    )

    print(
        "Path:",
        item["path"]
    )

    print(
        "Modified:",
        modified
    )

    print(
        "Size:",
        f"{size_mb:.2f} MB"
    )


# ------------------------------------------------------------
# 8. Проверяем, что уже лежит в GitHub notebooks/
# ------------------------------------------------------------

existing_repo_notebooks = list(
    NOTEBOOKS_DIR.glob(
        "*.ipynb"
    )
)


print()
print("=" * 70)
print("CURRENT GITHUB NOTEBOOKS DIRECTORY")
print("=" * 70)


if existing_repo_notebooks:

    for path in existing_repo_notebooks:

        print(
            "FOUND |",
            path.name
        )

else:

    print(
        "EMPTY | no notebook added yet"
    )


# ------------------------------------------------------------
# 9. Итог
# ------------------------------------------------------------

print()
print("=" * 70)
print("CELL 20 COMPLETE")
print("=" * 70)

print(
    "Notebook search completed."
)

print(
    "Ничего автоматически не копировалось."
)

print(
    "Следующий шаг — выбрать найденный финальный notebook, "
    "очистить его от secrets/output при необходимости "
    "и добавить в notebooks/."
)

print("=" * 70)